# 4\.4\.1 Train Tree\-Based Panel Forecasting Models

The forecasting pipeline now moves from local time-series models to a fundamentally different modeling strategy: **global panel trees**.

The classical models in 4.3.1 learn primarily from the ordered history of individual mobility series. Panel-tree models instead pool forecasting examples across Taxi Zones and learn whether combinations of recent mobility, calendar state, weather, multimodal context, transportation connectivity, geography, and place identity are associated with the future value of a target.

The modeling unit is therefore one **Modeling Track × Target × Horizon** problem. Within each problem, observations from all eligible Forecast Sequences are pooled into one supervised-learning dataset. The model still knows which place it is forecasting: the 84 target-eligible predictors inherited from the completed forecasting feature catalog are supplemented with **`taxi_zone_id` as categorical series identity**, producing 85 logical model inputs.

This notebook does **not** reopen the forecasting decisions already established in 4.1.* and 4.2.*. Forecast targets, horizons, track boundaries, rolling-validation dates, sequence eligibility, feature availability, leakage rules, and the frozen forecasting benchmark are treated as completed upstream contracts. The 2026 final holdout remains untouched throughout model development.

Only the two ordinary forecasting tracks participate:

- **Post-CP Primary** — learns from post-congestion-pricing history only.
- **Full-History Regime-Aware** — forecasts the same post-CP period while allowing older pre-policy history into the training window.

The Pre-CP Reference surface belongs to the later counterfactual analysis and is intentionally absent from 4.4.1.

The modeling progression is deliberately staged. We first establish a computationally bounded but temporally honest screen of meaningfully different tree architectures. An independent PyCaret comparison then checks whether that hand-selected roster misses an obviously competitive alternative. Only the families that survive this first decision point will advance to the more expensive questions of refit cadence, hyperparameter tuning, full rolling validation, and model interpretation.

> **Primary question:** Can a global nonlinear model learn reusable structure across Taxi Zones well enough to outperform the frozen forecasting benchmark—and which tree architectures are promising enough to justify deeper validation?

## Notebook Setup

Define all notebook-wide imports, paths, execution controls, modeling constants, branding resources, and shared utility functions before beginning the numbered analysis.

The large 4.2.3 forecasting matrices remain the authoritative supervised-learning surfaces. They are read selectively by Target × Horizon rather than loaded in full. The compact registries from 4.1.2, 4.2.3, and 4.2.4 provide the timing, feature, modeling-scope, and benchmark contracts needed to interpret those rows.

Long-running screens are restartable. Each completed model fit is checkpointed independently, so rerunning a partially completed experiment skips work that has already finished.

In [1]:
# ---------------------------------------------------------------------
# 4.4.1.0A — Install the frozen scikit-learn runtime
# ---------------------------------------------------------------------
# The completed panel-tree experiments use scikit-learn 1.4.2. Pin that
# modeling runtime explicitly without reinstalling the optional family-screen
# packages or allowing them to renegotiate the notebook-wide dependency stack.

%pip install -q --no-cache-dir "scikit-learn==1.4.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---------------------------------------------------------------------
# 4.4.1.3B.0 — Install optional family-screen dependencies
# ---------------------------------------------------------------------
# LightGBM and CatBoost participate in the completed tree-family comparison.
# Keep these optional dependencies close to the experiment that needs them
# rather than allowing them to renegotiate the frozen scikit-learn runtime.

%pip install -q --no-cache-dir lightgbm catboost

%pip install -q --no-cache-dir lightgbm catboost


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sklearn

if sklearn.__version__ != "1.4.2":
    raise RuntimeError(
        f"4.4.1 requires scikit-learn 1.4.2; found {sklearn.__version__}."
    )

In [4]:
# ---------------------------------------------------------------------
# 4.4.1.0B — Imports, paths, toggles, and fixed modeling constants
# ---------------------------------------------------------------------

from pathlib import Path
from time import perf_counter
import gc
import re
import shutil

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import plotly.express as px
import plotly.graph_objects as go

from tqdm.auto import tqdm

import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_SEQUENTIAL_HEATMAP_SCALE,
    apply_branding,
)


# ---------------------------------------------------------------------
# Reproducibility and ordinary-forecasting scope
# ---------------------------------------------------------------------
# 4.4.1 is deliberately restricted to the two ordinary forecasting tracks.
# Pre-CP Reference belongs to Chapter 5 and must never enter this workload.

RANDOM_SEED = 696
PANEL_TREE_MODEL_FAMILY = "panel_tree"

ORDINARY_TRACKS = ["post_cp_primary", "full_history_regime_aware"]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
}

TRACK_SHORT_LABELS = {
    "post_cp_primary": "Post-CP",
    "full_history_regime_aware": "Full-History",
}

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trip count",
    "taxi_avg_trip_speed": "Taxi average trip speed",
    "fhvhv_trip_count": "FHVHV trip count",
    "fhvhv_avg_trip_speed": "FHVHV average trip speed",
    "subway_ridership": "Subway ridership",
}

TARGET_SHORT_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway",
}

FORECAST_HORIZONS = [1, 2, 5]


# ---------------------------------------------------------------------
# Authoritative upstream paths
# ---------------------------------------------------------------------
# Compact registries define the inherited experiment. The large 4.2.3
# forecasting matrices are read selectively later rather than loaded here.

PIPELINE_DIR = Path("pipeline_data")

INPUT_412_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_423_DIR = PIPELINE_DIR / "4.2.3.final_tables"
INPUT_424_FINAL_DIR = PIPELINE_DIR / "4.2.4.final_tables"
INPUT_424_INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.4.intermediate_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.4.1.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.4.1.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

TRACK_CONTRACT_PATH = INPUT_412_DIR / "forecasting_track_contract_registry.parquet"
FEATURE_CATALOG_PATH = INPUT_423_DIR / "forecast_feature_catalog_complete.parquet"
SUPPORTED_SCOPE_PATH = INPUT_424_FINAL_DIR / "supported_modeling_scope.parquet"
BENCHMARK_REGISTRY_PATH = INPUT_424_FINAL_DIR / "advanced_model_benchmark_registry.parquet"

FORECASTING_MATRIX_PATHS = {
    "post_cp_primary": INPUT_423_DIR / "forecasting_matrix__post_cp_primary.parquet",
    "full_history_regime_aware": INPUT_423_DIR / "forecasting_matrix__full_history_regime_aware.parquet",
}

BASELINE_PREDICTION_PATHS = {
    "post_cp_primary": INPUT_424_INTERMEDIATE_DIR / "baseline_predictions__post_cp_primary.parquet",
    "full_history_regime_aware": INPUT_424_INTERMEDIATE_DIR / "baseline_predictions__full_history_regime_aware.parquet",
}


# ---------------------------------------------------------------------
# First screening contract
# ---------------------------------------------------------------------
# The first experiment compares broad model architectures, not final tuned
# definitions. It intentionally uses a bounded training surface and one fixed
# validation period so family selection does not become a full production run.

SCREEN_VALIDATION_DAYS = 28
SCREEN_TRAIN_ROW_CAP = 250_000
PYCARET_TRAIN_ROW_CAP = 100_000

# Post-CP is represented across all Target × Horizon combinations. Full-History
# contributes one long-horizon transfer/stress check for each target.
FULL_HISTORY_SCREEN_HORIZONS = [5]

SCREEN_CONTRACT_ID = "family_screen_v1_28d_250k"
PYCARET_SCREEN_CONTRACT_ID = "pycaret_screen_v1_28d_100k"

FAMILY_SCREEN_PARTS_DIR = INTERMEDIATE_DIR / f"family_screen_parts__{SCREEN_CONTRACT_ID}"
PYCARET_SCREEN_PARTS_DIR = INTERMEDIATE_DIR / f"pycaret_screen_parts__{PYCARET_SCREEN_CONTRACT_ID}"

FAMILY_SCREEN_RESULTS_PATH = (
    INTERMEDIATE_DIR / f"panel_tree_family_screen_results__{SCREEN_CONTRACT_ID}.parquet"
)
PYCARET_SCREEN_RESULTS_PATH = (
    INTERMEDIATE_DIR / f"panel_tree_pycaret_screen_results__{PYCARET_SCREEN_CONTRACT_ID}.parquet"
)


# ---------------------------------------------------------------------
# Execution controls
# ---------------------------------------------------------------------
# Long runs write one checkpoint per completed unit. RESET_* deletes only the
# versioned experiment it names; it never touches upstream forecasting assets.

RUN_FAMILY_SCREEN = False
RESET_FAMILY_SCREEN_CHECKPOINTS = False

RUN_PYCARET_SCREEN = False
RESET_PYCARET_SCREEN_CHECKPOINTS = False

RUN_CADENCE_SCREEN = False
RESET_CADENCE_SCREEN_CHECKPOINTS = False

RUN_TUNING_SCREEN = False
RESET_TUNING_SCREEN_CHECKPOINTS = False

RUN_VISUALS = True
WRITE_CHECKPOINTS = True
WRITE_CONSOLIDATED_INTERMEDIATES = True

# ---------------------------------------------------------------------
# This chapter advances one architecture only. The cadence experiment therefore
# isolates whether refreshing that same Extra Trees definition during rolling
# validation materially improves forecasts enough to justify repeated fitting.

ADVANCING_TREE_MODEL_ID = "extra_trees"
ADVANCING_TREE_MODEL_LABEL = "Extra Trees"

CADENCE_POLICIES = ["validation_start_only", "monthly"]
CADENCE_POLICY_LABELS = {
    "validation_start_only": "Validation-start only",
    "monthly": "Monthly refresh",
}

# Reuse the bounded, origin-balanced training surface from architecture
# screening. This experiment decides refresh cadence, not final training size;
# the cap does not automatically become the production-validation contract.
CADENCE_TRAIN_ROW_CAP = 250_000

# Three contrasting sentinels cover both modeling tracks, short and long
# horizons, demand and speed behavior, and a different transit modality.
CADENCE_SENTINEL_KEYS = [
    ("post_cp_primary", "taxi_trip_count", 1),
    ("post_cp_primary", "taxi_avg_trip_speed", 5),
    ("full_history_regime_aware", "subway_ridership", 5),
]

CADENCE_CONTRACT_ID = "extra_trees_cadence_v1_3sentinel_250k"

CADENCE_SCREEN_PARTS_DIR = (
    INTERMEDIATE_DIR / f"cadence_screen_parts__{CADENCE_CONTRACT_ID}"
)
CADENCE_SCREEN_RESULTS_PATH = (
    INTERMEDIATE_DIR
    / f"panel_tree_cadence_screen_results__{CADENCE_CONTRACT_ID}.parquet"
)

/root/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The shared helpers below are limited to operations needed across multiple later chapters: schema validation, forecasting metrics, checkpoint handling, and reconstruction of the target-specific predictor lists.

Model-family construction, categorical encoders, screen-data loading, and other logic used only by candidate screening are deferred until Section 4.4.1.3.

In [5]:
# ---------------------------------------------------------------------
# 4.4.1.0C — Shared notebook helpers
# ---------------------------------------------------------------------

def parquet_columns(path):
    """Return physical Parquet column names without materializing the file."""
    return pq.ParquetFile(path).schema_arrow.names


def require_columns(df, required_columns, object_name):
    """Fail at the notebook boundary when an inherited contract is incomplete."""
    missing = sorted(set(required_columns) - set(df.columns))

    if missing:
        raise ValueError(
            f"{object_name} is missing required columns: {missing}"
        )


def regression_metrics(y_true, y_pred):
    """Calculate the forecasting metrics inherited from the baseline layer.

    MAE remains the model-selection metric. RMSE, NMAE, and bias are retained
    to explain the scale and character of forecast error.
    """
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)

    # Model evaluation should use only rows where both the observed target and
    # prediction are finite. Invalid rows are excluded explicitly rather than
    # being silently coerced into an error value.
    valid = np.isfinite(actual) & np.isfinite(predicted)

    if not valid.any():
        return {
            "evaluated_rows": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "nmae": np.nan,
            "bias": np.nan,
        }

    actual = actual[valid]
    predicted = predicted[valid]

    error = predicted - actual
    absolute_error = np.abs(error)
    absolute_actual_sum = np.abs(actual).sum()

    return {
        "evaluated_rows": int(len(actual)),
        "mae": float(absolute_error.mean()),
        "rmse": float(np.sqrt(np.mean(np.square(error)))),
        "nmae": (
            float(absolute_error.sum() / absolute_actual_sum)
            if absolute_actual_sum > 0
            else np.nan
        ),
        "bias": float(error.mean()),
    }


def slugify(value):
    """Create stable filesystem-safe identifiers for checkpoint artifacts."""
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-")


def reset_checkpoint_dir(path):
    """Reset only the requested versioned checkpoint surface.

    Experiment resets are intentionally scoped to one versioned directory so
    resetting a downstream screen cannot delete unrelated notebook evidence.
    """
    path = Path(path)

    if path.exists():
        shutil.rmtree(path)

    path.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Durable Parquet persistence
# ---------------------------------------------------------------------

def atomic_parquet_write(df, path):
    """Write one Parquet artifact atomically.

    Long-running experiments must never mistake a partially written file for
    durable completed work. The frame is therefore written to a temporary path
    beside the destination and promoted into place only after serialization
    succeeds.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # The temporary suffix deliberately does not end in ".parquet", so normal
    # checkpoint discovery will never treat an interrupted temporary write as a
    # completed checkpoint part.
    temporary_path = path.with_name(f".{path.name}.tmp")

    df.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    temporary_path.replace(path)


def write_checkpoint_frame(df, path):
    """Persist one completed logical experiment unit atomically."""
    atomic_parquet_write(df, path)


def collect_checkpoint_parts(path):
    """Read every durable checkpoint part in deterministic filename order.

    Physical filenames are storage locations only. This helper intentionally
    does not decide which rows are complete, current, duplicated, or reusable.
    Those experiment semantics are validated from fields stored inside the
    checkpoint rows themselves.
    """
    path = Path(path)
    parts = sorted(path.glob("*.parquet"))

    if not parts:
        return pd.DataFrame()

    frames = []

    for part in parts:
        try:
            frames.append(pd.read_parquet(part))

        except Exception as exc:
            raise RuntimeError(
                f"Could not read checkpoint part {part.name}: {exc}"
            ) from exc

    return pd.concat(frames, ignore_index=True)


# ---------------------------------------------------------------------
# Logical experiment identity and restart-state helpers
# ---------------------------------------------------------------------

def logical_key_set(df, key_columns):
    """Return immutable logical experiment keys represented by a result frame.

    Restartability is defined by experiment identity such as
    Job × Model Family or Track × Target × Horizon × Capacity × Representation,
    not by whichever physical filename happened to store the result.
    """
    if df.empty:
        return set()

    require_columns(
        df,
        key_columns,
        "Logical-key frame",
    )

    return set(
        df[list(key_columns)].itertuples(
            index=False,
            name=None,
        )
    )


def filter_experiment_rows(
    df,
    *,
    contract_column,
    contract_id,
    row_filter=None,
):
    """Keep only rows belonging to one explicit versioned experiment contract.

    The contract identifier prevents evidence from another experiment version
    from being reused accidentally. Some experiments also require an additional
    identity rule, such as retaining only the advancing Extra Trees definition;
    callers supply that rule explicitly through row_filter.
    """
    if df.empty:
        return df.copy()

    require_columns(
        df,
        [contract_column, "status"],
        "Experiment result frame",
    )

    filtered = df.loc[
        df[contract_column].eq(contract_id)
    ].copy()

    if row_filter is not None and not filtered.empty:
        filtered = row_filter(filtered)

    return filtered.reset_index(drop=True)


def canonical_checkpoint_rows(
    parts_dir,
    *,
    contract_column,
    contract_id,
    key_columns,
    row_filter=None,
):
    """Return one canonical checkpoint row per logical experiment key.

    A successful current-contract row is the durable completion signal.
    Failed rows remain visible for audit and retry, but they never block a later
    successful result from becoming canonical.

    Multiple successful rows for the same logical key are rejected because the
    experiment would otherwise contain ambiguous duplicated evidence.
    """
    raw_df = collect_checkpoint_parts(parts_dir)

    if raw_df.empty:
        return raw_df

    filtered_df = filter_experiment_rows(
        raw_df,
        contract_column=contract_column,
        contract_id=contract_id,
        row_filter=row_filter,
    )

    if filtered_df.empty:
        return filtered_df

    require_columns(
        filtered_df,
        list(key_columns) + ["status"],
        "Checkpoint result frame",
    )

    success_df = filtered_df.loc[
        filtered_df["status"].eq("success")
    ].copy()

    # A logical experiment unit may have failed in an earlier attempt and later
    # succeeded. That is valid. Two successful rows for the same logical unit
    # are not valid because there is no principled basis for choosing between
    # them silently.
    if success_df.duplicated(list(key_columns)).any():
        duplicate_keys_df = (
            success_df.loc[
                success_df.duplicated(
                    list(key_columns),
                    keep=False,
                ),
                list(key_columns),
            ]
            .drop_duplicates()
            .sort_values(list(key_columns))
            .reset_index(drop=True)
        )

        display(duplicate_keys_df)

        raise ValueError(
            "Multiple successful checkpoint rows share the same logical "
            f"experiment key {list(key_columns)}."
        )

    # Prefer success over failure when both exist for one logical key. If only
    # failures exist, retain one failed row so the experiment remains auditable
    # and the calling cell can correctly identify that key as unfinished.
    ranked_df = filtered_df.assign(
        _status_priority=(
            filtered_df["status"]
            .eq("success")
            .astype(int)
        )
    )

    canonical_df = (
        ranked_df
        .sort_values(
            list(key_columns) + ["_status_priority"]
        )
        .drop_duplicates(
            list(key_columns),
            keep="last",
        )
        .drop(columns="_status_priority")
        .reset_index(drop=True)
    )

    return canonical_df


def load_complete_consolidated_result(
    path,
    *,
    contract_column,
    contract_id,
    key_columns,
    expected_keys,
    row_filter=None,
):
    """Load a consolidated artifact only when it is exactly complete.

    The consolidated Parquet is the normal fast path for notebook reruns, but it
    is trusted only when it is readable and contains exactly one successful row
    for every expected logical key in the current contract.

    Returning None is intentional: it tells the calling long-running cell to
    recover from durable checkpoint parts and compute only unfinished work.
    """
    path = Path(path)

    if not path.exists():
        return None

    try:
        result_df = pd.read_parquet(path)
    except Exception:
        return None

    if result_df.empty:
        return None

    try:
        result_df = filter_experiment_rows(
            result_df,
            contract_column=contract_column,
            contract_id=contract_id,
            row_filter=row_filter,
        )

        require_columns(
            result_df,
            list(key_columns) + ["status"],
            "Consolidated experiment result",
        )

    except (ValueError, KeyError):
        return None

    if result_df.empty:
        return None

    # A complete consolidated artifact must already be canonical. Duplicates,
    # failed rows, missing keys, or unexpected extra keys all force recovery
    # from the checkpoint surface rather than being silently accepted.
    if result_df.duplicated(list(key_columns)).any():
        return None

    if not result_df["status"].eq("success").all():
        return None

    observed_keys = logical_key_set(
        result_df,
        key_columns,
    )

    if observed_keys != set(expected_keys):
        return None

    return result_df.reset_index(drop=True)


def checkpoint_is_success(
    path,
    *,
    contract_column,
    contract_id,
    required_identity=None,
):
    """Validate one known physical checkpoint when direct checking is useful.

    Most long-running cells should recover completion from all checkpoint
    contents by logical key. This helper remains useful for smaller cases where
    one exact checkpoint path is intentionally addressed.

    File existence alone never counts as completion.
    """
    path = Path(path)

    if not path.exists():
        return False

    try:
        checkpoint_df = pd.read_parquet(path)
    except Exception:
        return False

    if checkpoint_df.empty:
        return False

    required_columns = {
        "status",
        contract_column,
    }

    required_identity = required_identity or {}
    required_columns.update(required_identity)

    if not required_columns.issubset(checkpoint_df.columns):
        return False

    if not checkpoint_df[contract_column].eq(
        contract_id
    ).all():
        return False

    if not checkpoint_df["status"].eq(
        "success"
    ).all():
        return False

    # When the caller supplies logical identity fields, every row in the
    # checkpoint must match them. A correctly named file with mismatched
    # contents is therefore never reusable.
    for column, expected_value in required_identity.items():
        if not checkpoint_df[column].eq(
            expected_value
        ).all():
            return False

    return True


# ---------------------------------------------------------------------
# Predictor-contract helpers
# ---------------------------------------------------------------------

def target_predictor_names(feature_catalog, target_metric):
    """Return the frozen target-specific predictors plus Taxi Zone identity.

    4.2.3 defines 86 shared ordinary-forecasting features. Two redundant local
    multimodal fields are suppressed whenever their source metric is the active
    forecast target, leaving 84 inherited predictors. The pooled tree receives
    taxi_zone_id as the 85th logical input.
    """
    ordinary_available = (
        feature_catalog["ordinary_forecasting_available"]
        .fillna(False)
        .astype(bool)
    )

    source_matches_target = (
        feature_catalog["source_metric"]
        .eq(target_metric)
        .fillna(False)
    )

    drop_for_target = (
        feature_catalog[
            "drop_when_source_metric_equals_target_metric"
        ]
        .fillna(False)
        .astype(bool)
        & source_matches_target
    )

    inherited = (
        feature_catalog.loc[
            ordinary_available & ~drop_for_target,
            "feature_name",
        ]
        .astype(str)
        .tolist()
    )

    if len(inherited) != 84:
        raise ValueError(
            f"{target_metric}: expected 84 inherited predictors after the "
            f"frozen target-specific exclusion; found {len(inherited)}."
        )

    if "taxi_zone_id" in inherited:
        raise ValueError(
            "taxi_zone_id unexpectedly appears in the inherited feature catalog."
        )

    return inherited + ["taxi_zone_id"]


def categorical_predictor_names(
    matrix_path,
    predictor_names,
):
    """Recover nominal predictor roles from the persisted matrix schema.

    Taxi Zone ID is categorical by modeling decision even though its stored
    representation is numeric. Persisted string/dictionary predictors retain
    their nominal meaning as well.
    """
    schema = pq.ParquetFile(
        matrix_path
    ).schema_arrow

    schema_fields = {
        field.name: field.type
        for field in schema
    }

    categorical = []

    for column in predictor_names:
        if column == "taxi_zone_id":
            categorical.append(column)
            continue

        if column not in schema_fields:
            raise ValueError(
                f"{column} is not present in the persisted matrix schema "
                f"for {matrix_path}."
            )

        arrow_type = schema_fields[column]

        if (
            pa.types.is_string(arrow_type)
            or pa.types.is_large_string(arrow_type)
            or pa.types.is_dictionary(arrow_type)
        ):
            categorical.append(column)

    return categorical

## 4.4.1.1 Consume the Frozen Ordinary Forecasting Handoff

The tree-modeling notebook starts where the completed forecasting preparation ends.

Four compact upstream contracts define the experiment:

- the 4.1.2 modeling-track registry defines when each ordinary forecasting track can train and validate;
- the 4.2.3 complete feature catalog defines the shared leakage-safe predictor universe;
- the 4.2.4 supported-modeling scope defines the Target × Horizon jobs carried into panel trees;
- the 4.2.4 advanced-model benchmark registry identifies the already-frozen benchmark inherited by each job.

The two large 4.2.3 matrices then supply the actual supervised-learning rows.

This section performs only **handoff-boundary validation**. It does not reopen sequence support, feature availability, missingness, leakage, benchmark selection, or empirical forecastability. Those questions were already resolved upstream. The objective here is simply to fail fast if 4.4.1 is pointed at a different contract than the one those notebooks completed.

For tree modeling, each active target inherits 84 of the 86 shared predictor definitions after the target's two redundant local multimodal fields are excluded. Because one global model pools many Taxi Zones, `taxi_zone_id` is added as explicit categorical series identity. `forecast_sequence_id` remains metadata: once Target is fixed by the model job, it carries no additional identity beyond Taxi Zone.

### Validate the inherited ordinary-forecasting boundary

Load the compact contracts and confirm that 4.4.1 contains exactly the authorized ordinary forecasting workload. These are boundary assertions, not a new support analysis.

In [6]:
# ---------------------------------------------------------------------
# 4.4.1.1A — Load and validate the inherited ordinary forecasting contract
# ---------------------------------------------------------------------

track_contract_registry_df = pd.read_parquet(TRACK_CONTRACT_PATH)
forecast_feature_catalog_df = pd.read_parquet(FEATURE_CATALOG_PATH)
supported_modeling_scope_df = pd.read_parquet(SUPPORTED_SCOPE_PATH)
advanced_model_benchmark_registry_df = pd.read_parquet(BENCHMARK_REGISTRY_PATH)


# Verify only fields that 4.4.1 directly consumes. Upstream notebooks already
# performed the deeper internal QA of these artifacts.
require_columns(
    track_contract_registry_df,
    [
        "modeling_track", "track_role", "training_history_start_date",
        "initial_training_target_start_date", "initial_training_target_end_date",
        "rolling_validation_target_start_date", "rolling_validation_target_end_date",
        "final_holdout_target_start_date", "final_holdout_target_end_date",
        "native_sequence_count",
    ],
    "forecasting_track_contract_registry",
)

require_columns(
    forecast_feature_catalog_df,
    [
        "feature_name", "source_metric", "ordinary_forecasting_available",
        "drop_when_source_metric_equals_target_metric",
    ],
    "forecast_feature_catalog_complete",
)

require_columns(
    supported_modeling_scope_df,
    [
        "model_family", "modeling_track", "metric", "horizon", "scope_status",
        "baseline_id", "baseline_label", "benchmark_selection_metric",
        "benchmark_selection_stage",
    ],
    "supported_modeling_scope",
)

require_columns(
    advanced_model_benchmark_registry_df,
    [
        "modeling_track", "metric", "horizon", "baseline_id", "baseline_label",
        "selection_metric", "selection_stage",
    ],
    "advanced_model_benchmark_registry",
)


# Restrict immediately to the two ordinary forecasting tracks. Pre-CP remains
# physically available upstream but is intentionally outside this notebook.
ordinary_track_contract_df = (
    track_contract_registry_df.loc[
        track_contract_registry_df["modeling_track"].isin(ORDINARY_TRACKS)
    ]
    .copy()
    .sort_values("modeling_track")
    .reset_index(drop=True)
)

panel_tree_scope_df = (
    supported_modeling_scope_df.loc[
        supported_modeling_scope_df["model_family"].eq(PANEL_TREE_MODEL_FAMILY)
        & supported_modeling_scope_df["modeling_track"].isin(ORDINARY_TRACKS)
    ]
    .copy()
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)

ordinary_benchmark_df = (
    advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df["modeling_track"].isin(ORDINARY_TRACKS)
    ]
    .copy()
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)


# Normalize inherited date fields once so later temporal slicing uses one
# unambiguous representation.
track_date_columns = [
    "training_history_start_date",
    "initial_training_target_start_date",
    "initial_training_target_end_date",
    "rolling_validation_target_start_date",
    "rolling_validation_target_end_date",
    "final_holdout_target_start_date",
    "final_holdout_target_end_date",
]

for column in track_date_columns:
    ordinary_track_contract_df[column] = pd.to_datetime(
        ordinary_track_contract_df[column]
    ).dt.normalize()


# The downstream scope is already a frozen contract. These checks detect an
# unexpected upstream replacement rather than recomputing eligibility.
assert set(ordinary_track_contract_df["modeling_track"]) == set(ORDINARY_TRACKS)

assert len(panel_tree_scope_df) == 30
assert not panel_tree_scope_df.duplicated(["modeling_track", "metric", "horizon"]).any()
assert set(panel_tree_scope_df["modeling_track"]) == set(ORDINARY_TRACKS)
assert set(panel_tree_scope_df["metric"]) == set(FORECAST_TARGETS)
assert set(panel_tree_scope_df["horizon"].astype(int)) == set(FORECAST_HORIZONS)
assert panel_tree_scope_df["scope_status"].eq("continue").all()

assert panel_tree_scope_df["benchmark_selection_metric"].astype(str).str.lower().eq("mae").all()
assert panel_tree_scope_df["benchmark_selection_stage"].astype(str).eq("rolling_validation").all()

assert len(forecast_feature_catalog_df) == 86
assert forecast_feature_catalog_df["feature_name"].is_unique
assert forecast_feature_catalog_df["ordinary_forecasting_available"].fillna(False).astype(bool).all()

# Both ordinary tracks forecast the same rolling-validation calendar. This is
# what makes later Post-CP versus Full-History comparisons meaningful.
assert ordinary_track_contract_df["rolling_validation_target_start_date"].nunique() == 1
assert ordinary_track_contract_df["rolling_validation_target_end_date"].nunique() == 1

# The benchmark carried in supported_modeling_scope must still match the
# authoritative downstream benchmark registry exactly.
benchmark_alignment_df = panel_tree_scope_df[
    ["modeling_track", "metric", "horizon", "baseline_id"]
].merge(
    ordinary_benchmark_df[["modeling_track", "metric", "horizon", "baseline_id"]],
    on=["modeling_track", "metric", "horizon"],
    how="left",
    suffixes=("_scope", "_registry"),
    validate="one_to_one",
)

assert benchmark_alignment_df["baseline_id_registry"].notna().all()
assert (
    benchmark_alignment_df["baseline_id_scope"]
    == benchmark_alignment_df["baseline_id_registry"]
).all()

# The final holdout dates are retained only as a boundary guard. No holdout
# rows are loaded or scored anywhere before the designated final evaluation.
assert ordinary_track_contract_df["final_holdout_target_start_date"].notna().all()

### Resolve the pooled model-input contract

Confirm that the actual 4.2.3 matrices expose the same target-specific predictor surface and categorical roles in both ordinary forecasting tracks.

The output is intentionally compact: one row per forecast target rather than a dump of all 85 input columns.

In [7]:
# ---------------------------------------------------------------------
# 4.4.1.1B — Resolve the global panel-tree predictor contract
# ---------------------------------------------------------------------

matrix_required_metadata = [
    "modeling_track", "stage", "forecast_sequence_id", "taxi_zone_id", "metric",
    "origin_observation_sequence_id", "target_observation_sequence_id", "horizon",
    "target_date", "target_observed_value", "target_label_available",
]

predictor_names_by_target = {}
categorical_predictors_by_target = {}

# Validate the physical matrices once at the notebook boundary. Subsequent
# model cells can then rely on the approved schema rather than repeatedly
# rediscovering columns.
for matrix_path in FORECASTING_MATRIX_PATHS.values():
    if not matrix_path.exists():
        raise FileNotFoundError(matrix_path)

    missing_metadata = sorted(set(matrix_required_metadata) - set(parquet_columns(matrix_path)))
    if missing_metadata:
        raise ValueError(
            f"{matrix_path.name} is missing required modeling metadata: {missing_metadata}"
        )

predictor_contract_rows = []

for target_metric in FORECAST_TARGETS:
    predictor_names = target_predictor_names(forecast_feature_catalog_df, target_metric)
    predictor_names_by_target[target_metric] = predictor_names

    # Both ordinary tracks must expose every approved logical predictor.
    for track, matrix_path in FORECASTING_MATRIX_PATHS.items():
        missing_predictors = sorted(set(predictor_names) - set(parquet_columns(matrix_path)))
        if missing_predictors:
            raise ValueError(
                f"{track} · {target_metric}: missing predictors {missing_predictors}"
            )

    post_categories = categorical_predictor_names(
        FORECASTING_MATRIX_PATHS["post_cp_primary"], predictor_names
    )
    full_categories = categorical_predictor_names(
        FORECASTING_MATRIX_PATHS["full_history_regime_aware"], predictor_names
    )

    if post_categories != full_categories:
        raise ValueError(
            f"{target_metric}: categorical predictor roles differ between ordinary tracks."
        )

    categorical_predictors_by_target[target_metric] = post_categories

    predictor_contract_rows.append(
        {
            "Target": TARGET_LABELS[target_metric],
            "Inherited predictors": 84,
            "Taxi Zone identity": 1,
            "Logical inputs": len(predictor_names),
            "Categorical inputs": len(post_categories),
            "Categorical fields": ", ".join(post_categories),
        }
    )

predictor_contract_display_df = pd.DataFrame(predictor_contract_rows)
display(predictor_contract_display_df)

,Target,Inherited predictors,Taxi Zone identity,Logical inputs,Categorical inputs,Categorical fields
0,Taxi trip count,84,1,85,9,"origin_daypart, origin_temporal_bucket, origin..."
1,Taxi average trip speed,84,1,85,9,"origin_daypart, origin_temporal_bucket, origin..."
2,FHVHV trip count,84,1,85,9,"origin_daypart, origin_temporal_bucket, origin..."
3,FHVHV average trip speed,84,1,85,9,"origin_daypart, origin_temporal_bucket, origin..."
4,Subway ridership,84,1,85,9,"origin_daypart, origin_temporal_bucket, origin..."


**Findings.** All five forecasting targets use the expected **85 model inputs**: 84 predictors inherited from the completed forecasting matrices, plus Taxi Zone identity.

The same nine categorical fields are available in both modeling tracks. This means the Post-CP and Full-History models can treat categories in the same way, while still allowing the model to recognize that different Taxi Zones can behave differently.

> ✅ Decision — Train one citywide tree model while preserving Taxi Zone identity\.

For each Modeling Track × Target × Horizon job, train one model across all eligible Taxi Zones and include \`taxi\_zone\_id\` as a categorical predictor\.

This lets the model learn patterns shared across the city without treating every Taxi Zone as interchangeable\. Keep \`forecast\_sequence\_id\` as an identifier only, not as a model input\.

The diagram below shows what this global panel design means in practice, how it differs from fitting one model per Taxi Zone, and why pooling across zones is useful while still preserving local identity.

<img src="image-20260907-171819.png" width="" align="" />

### Section Summary

4.4.1 begins from exactly **30 ordinary forecasting jobs**: two modeling tracks × five targets × three horizons. The tree family consumes the completed leakage-safe forecasting matrices rather than recreating upstream support, feature-availability, leakage, or benchmark analyses.

The next task is narrower: define a sufficiently broad but computationally bounded experiment that can distinguish promising tree architectures before expensive refit and tuning work begins.

## 4.4.1.2 Define the Tree-Family Screening Experiment

The first new modeling question is architectural:

> **Which tree families are promising enough to deserve deeper tuning and full rolling validation?**

Running every provisional family across the eventual production workload would answer that question at unnecessarily high cost. The first screen instead uses one fixed **28-day slice from the middle of the shared rolling-validation period**. Every model is trained only on target observations that would already have been known before that slice begins, then asked to forecast the same fixed future period without refitting inside the slice.

This is intentionally **not** the later production refit experiment. Holding the model fixed during the screen prevents family selection from becoming entangled with a refit-cadence decision that has not yet been made.

The controlled screen contains:

- all **15 Post-CP Primary Target × Horizon jobs**, giving every target and all three horizons direct representation;
- the **five h=5 Full-History jobs**, testing whether candidate rankings remain plausible when much longer historical training is available and the forecast horizon is most demanding.

That produces 20 representative jobs rather than immediately expanding every provisional family across all 30 final jobs.

For this architectural screen only, training surfaces larger than 250,000 rows are thinned by selecting **complete forecast origins evenly across the legal training history**. This preserves the cross-zone panel at every retained origin and avoids a random row sample that could distort Taxi Zone representation. The final rolling-validation experiment will not inherit this screening cap.

Targets remain on their original scale during family screening. Alternative count-target representations are model-definition questions for later tuning; branching into them now would confound architecture selection with another modeling choice.

### Freeze the representative workload

Derive the common validation slice directly from the inherited rolling-validation calendar and identify the three PyCaret sentinel problems that will provide the independent AutoML cross-check.

In [8]:
# ---------------------------------------------------------------------
# 4.4.1.2A — Derive the fixed temporal screen and representative workload
# ---------------------------------------------------------------------

validation_start = ordinary_track_contract_df["rolling_validation_target_start_date"].iloc[0]
validation_end = ordinary_track_contract_df["rolling_validation_target_end_date"].iloc[0]
validation_dates = pd.date_range(validation_start, validation_end, freq="D")

if SCREEN_VALIDATION_DAYS > len(validation_dates):
    raise ValueError(
        "SCREEN_VALIDATION_DAYS exceeds the inherited validation calendar."
    )

# Center the architectural screen within rolling validation so family selection
# is not dominated by either the initial validation boundary or its final edge.
screen_offset = (len(validation_dates) - SCREEN_VALIDATION_DAYS) // 2
SCREEN_START_DATE = validation_dates[screen_offset]
SCREEN_END_DATE = validation_dates[
    screen_offset + SCREEN_VALIDATION_DAYS - 1
]


# The primary track receives complete Target × Horizon representation so the
# first family screen observes every production forecasting question directly.
post_cp_screen_jobs_df = panel_tree_scope_df.loc[
    panel_tree_scope_df["modeling_track"].eq("post_cp_primary")
].copy()
post_cp_screen_jobs_df["screen_role"] = (
    "complete_post_cp_target_horizon_screen"
)

# Full-History contributes only the most demanding h=5 jobs here. Its role is
# to test whether family rankings remain plausible under a much longer training
# history without duplicating the entire 30-job production workload.
full_history_screen_jobs_df = panel_tree_scope_df.loc[
    panel_tree_scope_df["modeling_track"].eq("full_history_regime_aware")
    & panel_tree_scope_df["horizon"].isin(FULL_HISTORY_SCREEN_HORIZONS)
].copy()
full_history_screen_jobs_df["screen_role"] = (
    "full_history_long_horizon_transfer_check"
)

screen_jobs_df = pd.concat(
    [post_cp_screen_jobs_df, full_history_screen_jobs_df],
    ignore_index=True,
)
screen_jobs_df["horizon"] = screen_jobs_df["horizon"].astype(int)

metric_order = {metric: i for i, metric in enumerate(FORECAST_TARGETS)}
screen_jobs_df["_metric_order"] = screen_jobs_df["metric"].map(metric_order)

screen_jobs_df = (
    screen_jobs_df
    .sort_values(["modeling_track", "_metric_order", "horizon"])
    .drop(columns="_metric_order")
    .reset_index(drop=True)
)

screen_jobs_df["job_id"] = (
    screen_jobs_df["modeling_track"]
    + "__"
    + screen_jobs_df["metric"]
    + "__h"
    + screen_jobs_df["horizon"].astype(str)
)

assert len(screen_jobs_df) == 20
assert len(post_cp_screen_jobs_df) == 15
assert len(full_history_screen_jobs_df) == 5
assert screen_jobs_df["job_id"].is_unique


# PyCaret receives three deliberately different sentinel problems rather than
# all 20 jobs because its role is independent corroboration, not production
# model selection.
PYCARET_SENTINEL_KEYS = [
    ("post_cp_primary", "taxi_trip_count", 1),
    ("post_cp_primary", "taxi_avg_trip_speed", 5),
    ("full_history_regime_aware", "subway_ridership", 5),
]

screen_job_lookup = screen_jobs_df.set_index(
    ["modeling_track", "metric", "horizon"]
)
pycaret_screen_jobs_df = (
    screen_job_lookup.loc[PYCARET_SENTINEL_KEYS]
    .reset_index()
)

assert len(pycaret_screen_jobs_df) == 3


# ---------------------------------------------------------------------
# Reader-facing screening contract
# ---------------------------------------------------------------------

screen_contract_summary_df = pd.DataFrame(
    [
        {
            "Validation window": (
                f"{SCREEN_START_DATE:%Y-%m-%d} → {SCREEN_END_DATE:%Y-%m-%d}"
            ),
            "Days": SCREEN_VALIDATION_DAYS,
            "Controlled jobs": len(screen_jobs_df),
            "Post-CP jobs": len(post_cp_screen_jobs_df),
            "Full-History jobs": len(full_history_screen_jobs_df),
            "Training-row cap": f"{SCREEN_TRAIN_ROW_CAP:,}",
            "PyCaret sentinels": len(pycaret_screen_jobs_df),
        }
    ]
)

screen_workload_summary_df = (
    screen_jobs_df
    .groupby(["modeling_track", "screen_role"], as_index=False)
    .agg(
        Targets=("metric", "nunique"),
        Horizons=("horizon", "nunique"),
        Jobs=("job_id", "nunique"),
    )
)

screen_workload_summary_df["Track"] = screen_workload_summary_df[
    "modeling_track"
].map(TRACK_SHORT_LABELS)

screen_workload_summary_df["Role"] = screen_workload_summary_df[
    "screen_role"
].map(
    {
        "complete_post_cp_target_horizon_screen":
            "Complete Target × Horizon screen",
        "full_history_long_horizon_transfer_check":
            "h=5 transfer check",
    }
)

screen_workload_summary_df = screen_workload_summary_df[
    ["Track", "Role", "Targets", "Horizons", "Jobs"]
]

pycaret_sentinel_display_df = pycaret_screen_jobs_df[
    ["modeling_track", "metric", "horizon"]
].copy()

pycaret_sentinel_display_df["Sentinel"] = (
    pycaret_sentinel_display_df["modeling_track"].map(TRACK_SHORT_LABELS)
    + " · "
    + pycaret_sentinel_display_df["metric"].map(TARGET_SHORT_LABELS)
    + " · h="
    + pycaret_sentinel_display_df["horizon"].astype(str)
)

pycaret_sentinel_display_df = pycaret_sentinel_display_df[["Sentinel"]]

display(screen_contract_summary_df)
display(screen_workload_summary_df)
display(pycaret_sentinel_display_df)

,Validation window,Days,Controlled jobs,Post-CP jobs,Full-History jobs,Training-row cap,PyCaret sentinels
0,2025-08-06 → 2025-09-02,28,20,15,5,"250,000",3


,Track,Role,Targets,Horizons,Jobs
0,Full-History,h=5 transfer check,5,1,5
1,Post-CP,Complete Target × Horizon screen,5,3,15


,Sentinel
0,Post-CP · Taxi trips · h=1
1,Post-CP · Taxi speed · h=5
2,Full-History · Subway · h=5


**Findings.** The first architecture screen is fixed to the 28-day period **2025-08-06 through 2025-09-02**, centered within the shared rolling-validation window. The controlled workload contains **20 representative forecasting jobs**: all 15 Post-CP Target × Horizon combinations plus five Full-History h=5 jobs that test whether model-family rankings remain credible with much longer training histories. Training surfaces are capped at 250,000 rows only for this screening stage. The independent PyCaret check is deliberately narrower, using three contrasting sentinels—Post-CP Taxi trips h=1, Post-CP Taxi speed h=5, and Full-History Subway ridership h=5—so AutoML serves as corroboration rather than a second full model tournament.

### Preserve the frozen benchmark on the exact rows being screened

Every candidate will be compared with the **benchmark already frozen in 4.2.4**. The intermediate baseline-prediction files are used only to recover that fixed rule's prediction on the exact validation rows being scored.

This does not reopen benchmark selection or baseline support analysis. It simply ensures that a model's MAE and the frozen benchmark's MAE are calculated on the same observations, so **skill vs. benchmark** has a clean interpretation:

- positive skill means the tree removes error relative to the frozen benchmark;
- zero means it ties;
- negative skill means the tree performs worse.

MAE remains the only ranking metric. RMSE, NMAE, bias, runtime, and failure behavior provide supporting evidence for interpretation and pruning.

In [9]:
# ---------------------------------------------------------------------
# 4.4.1.2B — Verify access to the frozen row-level benchmark predictions
# ---------------------------------------------------------------------

baseline_required_metadata = [
    "modeling_track", "stage", "forecast_sequence_id",
    "origin_observation_sequence_id", "target_observation_sequence_id",
    "horizon", "metric", "target_date", "target_observed_value",
    "target_label_available",
]

for track in ORDINARY_TRACKS:
    baseline_path = BASELINE_PREDICTION_PATHS[track]
    if not baseline_path.exists():
        raise FileNotFoundError(baseline_path)

    baseline_columns = set(parquet_columns(baseline_path))
    required_baseline_ids = (
        panel_tree_scope_df.loc[
            panel_tree_scope_df["modeling_track"].eq(track), "baseline_id"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    required_prediction_columns = [
        f"baseline__{baseline_id}" for baseline_id in required_baseline_ids
    ]
    missing = sorted(
        set(baseline_required_metadata + required_prediction_columns) - baseline_columns
    )

    if missing:
        raise ValueError(
            f"{baseline_path.name} is missing required benchmark fields: {missing}"
        )

> ✅ Decision — Use a bounded architecture screen before production validation

Screen candidate families on a fixed 28\-day mid\-validation period using all 15 Post\-CP Target × Horizon jobs plus five Full\-History h=5 transfer checks\. Cap only the screening training surface at 250,000 rows by retaining complete forecast origins evenly through legal history\. Keep the target on its raw scale and hold the model fixed inside the screen window so family selection remains separate from the later refit\-cadence decision\.

Run PyCaret independently on three sentinel jobs using the same temporal boundary and model\-input contract\.

## 4.4.1.3 Screen Candidate Tree Families

The candidate screen is designed around **architectural diversity**, not library coverage.

Five controlled candidates represent distinct ways of constructing nonlinear tree ensembles:

- **Random Forest** provides the conventional bagged-tree reference.
- **Extra Trees** increases randomization in both observations and split construction, providing a meaningfully different bagging variant.
- **HistGradientBoosting** provides a dependency-light histogram-based boosting reference from scikit-learn.
- **LightGBM** represents high-performance leaf-wise gradient boosting.
- **CatBoost** is particularly relevant to the global panel setting because categorical variables such as Taxi Zone identity can be handled natively.

XGBoost is not added automatically as a sixth controlled candidate because it occupies much of the same gradient-boosting design space as LightGBM. The independent PyCaret screen will include XGBoost; if it clearly outperforms the controlled boosting candidates, that evidence can justify adding it by the end of this chapter.

Categorical treatment follows each architecture rather than forcing every model through one artificial representation. Random Forest, Extra Trees, and HistGradientBoosting receive training-fitted one-hot encodings of nominal fields. LightGBM and CatBoost retain categorical semantics through their native interfaces. Numeric imputation and any categorical encoding are fitted using training rows only.

These are intentionally **screening-capacity definitions**, not final hyperparameters. The purpose is to decide which model families deserve tuning—not to declare any individual configuration production-ready.

### Build the reusable screening machinery

The helpers below exist only for this chapter. They construct legal train/validation surfaces, preserve complete cross-zone forecast origins when thinning a large screen, align every model to the same frozen benchmark rows, and give each candidate family an appropriate categorical-data interface.

#### Keep the training sample representative over time

Some screening jobs contain more rows than we need for an initial model-family comparison. Rather than randomly dropping individual rows, keep a smaller set of **complete forecast origins** spread across the available history.

A forecast origin is one point in the canonical time sequence. Keeping the whole origin means the model still sees all available Taxi Zones for that moment instead of an arbitrary subset of zones.

This gives the screen a smaller training set while preserving both the panel structure and broad coverage of the historical period.

In [10]:
# ---------------------------------------------------------------------
# 4.4.1.3A.1 — Thin large training surfaces by complete forecast origin
# ---------------------------------------------------------------------

def thin_training_by_origin(df, max_rows):
    """Reduce a screening surface while preserving complete forecast origins.

    When a row cap is needed, retain origins spread across the legal history
    rather than sampling individual rows. This keeps the available cross-zone
    panel intact at every selected point in time.
    """
    if max_rows is None or len(df) <= max_rows:
        return df.copy()

    # Estimate how many complete origins can fit under the requested row cap.
    # Using the largest observed origin prevents the selected panel from
    # unexpectedly exceeding the cap because some origins contain more zones.
    origin_counts = df.groupby("origin_observation_sequence_id", sort=True).size()
    max_rows_per_origin = int(origin_counts.max())

    origins_to_keep = max(
        1,
        min(len(origin_counts), max_rows // max_rows_per_origin),
    )

    # Spread retained origins approximately evenly across the full legal
    # history so the screen does not become concentrated in one time period.
    available_origins = origin_counts.index.to_numpy()
    selected_positions = np.linspace(
        0,
        len(available_origins) - 1,
        num=origins_to_keep,
    )
    selected_positions = np.unique(np.round(selected_positions).astype(int))
    selected_origins = available_origins[selected_positions]

    return df.loc[
        df["origin_observation_sequence_id"].isin(selected_origins)
    ].copy()

#### Build one fair forecasting job

Each candidate model family should solve the **same forecasting problem**.

For one Modeling Track × Target × Horizon combination, the next helpers:

1. identify the already-frozen benchmark for that job;
2. load only rows that belong to the requested target and horizon;
3. separate legal training history from the fixed future screening window;
4. align the tree-model validation rows to the benchmark on the exact same forecasts;
5. return predictors and observed targets in a common format.

The important idea is simple: differences in model performance should come from the **model family**, not from different training dates, different validation rows, or different benchmark support.

In [11]:
# ---------------------------------------------------------------------
# 4.4.1.3A.2 — Assemble one shared family-screen forecasting job
# ---------------------------------------------------------------------

def prepare_tree_dtypes(X, categorical_names):
    """Standardize basic dtypes before model-family-specific preprocessing.

    Categorical predictors remain explicit nominal values, while numeric
    predictors are converted to compact floating-point columns. This step does
    not one-hot encode categories or otherwise favor any model family.
    """
    prepared = X.copy()
    categorical_set = set(categorical_names)

    for column in prepared.columns:
        if column in categorical_set:
            prepared[column] = (
                prepared[column]
                .astype("string")
                .fillna("__MISSING__")
                .astype(object)
            )
        else:
            prepared[column] = (
                pd.to_numeric(prepared[column], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float32")
            )

    return prepared


def frozen_benchmark_row(track, metric, horizon):
    """Return the single frozen benchmark assigned to one forecasting job."""
    match = ordinary_benchmark_df.loc[
        ordinary_benchmark_df["modeling_track"].eq(track)
        & ordinary_benchmark_df["metric"].eq(metric)
        & ordinary_benchmark_df["horizon"].astype(int).eq(int(horizon))
    ]

    if len(match) != 1:
        raise ValueError(
            f"Expected one frozen benchmark for {track} · {metric} · h={horizon}; "
            f"found {len(match)}."
        )

    return match.iloc[0]


def load_family_screen_job(
    track,
    metric,
    horizon,
    train_row_cap=SCREEN_TRAIN_ROW_CAP,
):
    """Build one legal train/validation surface on exact benchmark support.

    Training labels must already be known before the fixed screening window.
    Validation rows are then intersected with the frozen benchmark so model
    and baseline performance are measured on identical forecast observations.
    """
    matrix_path = FORECASTING_MATRIX_PATHS[track]
    predictor_names = predictor_names_by_target[metric]
    categorical_names = categorical_predictors_by_target[metric]

    matrix_columns = list(
        dict.fromkeys(
            [
                "modeling_track",
                "stage",
                "forecast_sequence_id",
                "taxi_zone_id",
                "metric",
                "origin_observation_sequence_id",
                "target_observation_sequence_id",
                "horizon",
                "target_date",
                "target_observed_value",
                "target_label_available",
            ]
            + predictor_names
        )
    )

    # Read only the requested Target × Horizon surface from the large Parquet
    # matrix. This keeps the screen memory-efficient without changing its rows.
    matrix_df = pd.read_parquet(
        matrix_path,
        columns=matrix_columns,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
        ],
    )

    if not matrix_df["modeling_track"].eq(track).all():
        raise ValueError(
            f"{matrix_path.name} returned rows outside {track}."
        )

    matrix_df["target_date"] = pd.to_datetime(
        matrix_df["target_date"]
    ).dt.normalize()

    # A row can train the model only when its outcome is available and occurs
    # before the fixed future window used for family comparison.
    labels_available = (
        matrix_df["target_label_available"].fillna(False).astype(bool)
        & matrix_df["target_observed_value"].notna()
    )

    training_mask = (
        matrix_df["stage"].isin(["initial_training", "rolling_validation"])
        & matrix_df["target_date"].lt(SCREEN_START_DATE)
        & labels_available
    )

    validation_mask = (
        matrix_df["stage"].eq("rolling_validation")
        & matrix_df["target_date"].between(
            SCREEN_START_DATE,
            SCREEN_END_DATE,
            inclusive="both",
        )
        & labels_available
    )

    train_df = matrix_df.loc[training_mask].copy()
    validation_df = matrix_df.loc[validation_mask].copy()

    del matrix_df
    gc.collect()

    if train_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: empty training surface."
        )
    if validation_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: empty validation surface."
        )

    # Apply the architecture-screen row cap only after the legal training
    # surface has been defined. Complete origins are retained rather than
    # individual rows so the available cross-zone panel stays intact.
    train_df = thin_training_by_origin(
        train_df,
        max_rows=train_row_cap,
    )


    # -----------------------------------------------------------------
    # Align the future slice to the already-frozen benchmark.
    # -----------------------------------------------------------------

    benchmark_info = frozen_benchmark_row(track, metric, horizon)
    benchmark_id = str(benchmark_info["baseline_id"])
    benchmark_label = str(benchmark_info["baseline_label"])
    benchmark_column = f"baseline__{benchmark_id}"

    baseline_df = pd.read_parquet(
        BASELINE_PREDICTION_PATHS[track],
        columns=[
            "stage",
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
            "horizon",
            "metric",
            "target_date",
            "target_observed_value",
            "target_label_available",
            benchmark_column,
        ],
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
        ],
    )

    baseline_df["target_date"] = pd.to_datetime(
        baseline_df["target_date"]
    ).dt.normalize()

    baseline_df = baseline_df.loc[
        baseline_df["stage"].eq("rolling_validation")
        & baseline_df["target_date"].between(
            SCREEN_START_DATE,
            SCREEN_END_DATE,
            inclusive="both",
        )
        & baseline_df["target_label_available"].fillna(False).astype(bool)
        & baseline_df[benchmark_column].notna()
    ].copy()

    # These explicit forecast keys identify the same prediction opportunity in
    # both the modeling matrix and the frozen baseline prediction surface.
    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
    ]

    if validation_df.duplicated(join_keys).any():
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "validation keys are not unique."
        )

    if baseline_df.duplicated(join_keys).any():
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "baseline keys are not unique."
        )

    baseline_df = baseline_df[
        join_keys + ["target_observed_value", benchmark_column]
    ].rename(
        columns={
            "target_observed_value": "benchmark_target_observed_value"
        }
    )

    validation_df = validation_df.merge(
        baseline_df,
        on=join_keys,
        how="inner",
        validate="one_to_one",
    )

    del baseline_df
    gc.collect()

    if validation_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "no common model/benchmark validation rows."
        )

    # The modeling matrix and benchmark file must refer to the same observed
    # outcome before their prediction errors can be compared legitimately.
    if not np.allclose(
        validation_df["target_observed_value"].to_numpy(dtype=float),
        validation_df["benchmark_target_observed_value"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "matrix and baseline targets disagree."
        )

    benchmark_metrics = regression_metrics(
        validation_df["target_observed_value"],
        validation_df[benchmark_column],
    )

    X_train = prepare_tree_dtypes(
        train_df[predictor_names],
        categorical_names,
    )
    X_valid = prepare_tree_dtypes(
        validation_df[predictor_names],
        categorical_names,
    )

    y_train = pd.to_numeric(
        train_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    y_valid = pd.to_numeric(
        validation_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_valid": X_valid,
        "y_valid": y_valid,
        "predictor_names": predictor_names,
        "categorical_names": categorical_names,
        "train_rows": int(len(train_df)),
        "train_origins": int(
            train_df["origin_observation_sequence_id"].nunique()
        ),
        "train_forecast_sequences": int(
            train_df["forecast_sequence_id"].nunique()
        ),
        "valid_rows": int(len(validation_df)),
        "valid_origins": int(
            validation_df["origin_observation_sequence_id"].nunique()
        ),
        "valid_forecast_sequences": int(
            validation_df["forecast_sequence_id"].nunique()
        ),
        "benchmark_id": benchmark_id,
        "benchmark_label": benchmark_label,
        "benchmark_metrics": benchmark_metrics,
    }

#### Give each model family the form of data it expects

The five candidates use the same logical predictors, but their software libraries handle categorical variables differently.

**Random Forest, Extra Trees, and HistGradientBoosting** use a scikit-learn preprocessing pipeline. Numeric missing values are filled with the training median, and categories such as Taxi Zone are one-hot encoded so their stored labels are not mistaken for numerical quantities.

**LightGBM and CatBoost** can work with categorical predictors more directly, so their helpers preserve explicit category values instead.

This changes only the **representation of the same predictors**. No model family receives extra information.

In [12]:
# ---------------------------------------------------------------------
# 4.4.1.3A.3 — Prepare family-specific categorical representations
# ---------------------------------------------------------------------

def make_one_hot_pipeline(
    model_id,
    estimator,
    predictor_names,
    categorical_names,
):
    """Build training-fitted preprocessing for the sklearn tree families.

    Numeric missing values are median-imputed from training data. Categorical
    predictors are one-hot encoded so nominal labels such as Taxi Zone ID do
    not acquire an artificial numeric ordering.
    """
    categorical_set = set(categorical_names)
    numeric_names = [
        column
        for column in predictor_names
        if column not in categorical_set
    ]

    # Random Forest and Extra Trees can operate efficiently on sparse one-hot
    # matrices. HistGradientBoosting requires dense input, so only that family
    # receives a dense encoded surface.
    sparse_output = model_id in {"random_forest", "extra_trees"}

    numeric_pipeline = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline = Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                ),
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=sparse_output,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        [
            ("numeric", numeric_pipeline, numeric_names),
            (
                "categorical",
                categorical_pipeline,
                categorical_names,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0 if sparse_output else 0.0,
    )

    return Pipeline(
        [
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )


def prepare_lightgbm_frames(
    X_train,
    X_valid,
    categorical_names,
):
    """Give LightGBM categorical columns with training-defined category levels.

    Validation values not observed during training are mapped to an explicit
    unseen category rather than being assigned an arbitrary numeric code.
    """
    train = X_train.copy()
    valid = X_valid.copy()

    for column in categorical_names:
        train_values = (
            train[column]
            .astype("string")
            .fillna("__MISSING__")
        )
        valid_values = (
            valid[column]
            .astype("string")
            .fillna("__MISSING__")
        )

        # Category levels come from training data only. This preserves the
        # temporal experiment and gives future unseen labels a safe fallback.
        categories = train_values.drop_duplicates().tolist()

        if "__UNSEEN__" not in categories:
            categories.append("__UNSEEN__")

        valid_values = valid_values.where(
            valid_values.isin(categories),
            "__UNSEEN__",
        )

        train[column] = pd.Categorical(
            train_values,
            categories=categories,
        )
        valid[column] = pd.Categorical(
            valid_values,
            categories=categories,
        )

    return train, valid


def prepare_catboost_frames(
    X_train,
    X_valid,
    categorical_names,
):
    """Represent CatBoost categorical predictors explicitly as strings.

    CatBoost performs its own categorical treatment, so the notebook preserves
    nominal values instead of one-hot encoding them beforehand.
    """
    train = X_train.copy()
    valid = X_valid.copy()

    for column in categorical_names:
        train[column] = (
            train[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )
        valid[column] = (
            valid[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return train, valid

#### Fit one candidate under the shared screening rules

The final helper is the common entry point for the controlled model-family screen.

It takes the shared training and validation data, routes them through the preprocessing required by the selected family, fits the model, records fitting and prediction time, and returns its forecasts.

The model definitions are intentionally fixed at this stage. This is a comparison of **tree families**, not a hyperparameter tournament.

In [13]:
# ---------------------------------------------------------------------
# 4.4.1.3A.4 — Fit and score one controlled tree-family candidate
# ---------------------------------------------------------------------

def fit_screen_candidate(
    model_id,
    config,
    X_train,
    y_train,
    X_valid,
    predictor_names,
    categorical_names,
):
    """Fit one fixed screening candidate and return predictions plus runtime.

    Every family receives the same logical predictors and observed target.
    Only the representation required by that model library differs.
    """
    if model_id in {
        "random_forest",
        "extra_trees",
        "hist_gradient_boosting",
    }:
        estimator_classes = {
            "random_forest": RandomForestRegressor,
            "extra_trees": ExtraTreesRegressor,
            "hist_gradient_boosting": HistGradientBoostingRegressor,
        }

        estimator = estimator_classes[model_id](
            **config["params"]
        )

        model = make_one_hot_pipeline(
            model_id,
            estimator,
            predictor_names,
            categorical_names,
        )

        fit_start = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - fit_start

        predict_start = perf_counter()
        predictions = model.predict(X_valid)
        predict_seconds = perf_counter() - predict_start

    elif model_id == "lightgbm":
        # LightGBM receives categorical columns directly so its native
        # categorical handling can be used instead of one-hot encoding.
        train_frame, valid_frame = prepare_lightgbm_frames(
            X_train,
            X_valid,
            categorical_names,
        )

        model = LGBMRegressor(**config["params"])

        fit_start = perf_counter()
        model.fit(
            train_frame,
            y_train,
            categorical_feature=categorical_names,
        )
        fit_seconds = perf_counter() - fit_start

        predict_start = perf_counter()
        predictions = model.predict(valid_frame)
        predict_seconds = perf_counter() - predict_start

        del train_frame, valid_frame

    elif model_id == "catboost":
        # CatBoost also receives explicit categorical values and performs its
        # own category-aware encoding internally during model fitting.
        train_frame, valid_frame = prepare_catboost_frames(
            X_train,
            X_valid,
            categorical_names,
        )

        model = CatBoostRegressor(**config["params"])

        fit_start = perf_counter()
        model.fit(
            train_frame,
            y_train,
            cat_features=categorical_names,
        )
        fit_seconds = perf_counter() - fit_start

        predict_start = perf_counter()
        predictions = model.predict(valid_frame)
        predict_seconds = perf_counter() - predict_start

        del train_frame, valid_frame

    else:
        raise ValueError(
            f"Unknown screen candidate: {model_id}"
        )

    predictions = np.asarray(
        predictions,
        dtype=float,
    )

    del model
    gc.collect()

    return predictions, fit_seconds, predict_seconds

### Run the controlled family screen

The five candidates below deliberately use moderate, fixed-capacity definitions. These settings are large enough for each architecture to express meaningful nonlinear structure but small enough to support a broad first-pass comparison.

No attempt is made to optimize depth, learning rate, number of estimators, leaf size, feature sampling, or regularization here. Allowing one family a large search while another receives a single default model would make the first architectural comparison harder to interpret.

XGBoost is deliberately left outside the controlled roster for this first pass. PyCaret will test it independently against the same kinds of forecasting problems, giving us a direct opportunity to add it if that omission appears costly.

In [14]:
# ---------------------------------------------------------------------
# 4.4.1.3B — Freeze and show the first-pass candidate definitions
# ---------------------------------------------------------------------

# LightGBM and CatBoost are needed only for the completed architecture screen.
# Keep these optional model-family dependencies local to this section rather
# than making them notebook-wide requirements.
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


FAMILY_SCREEN_CONFIGS = {
    "random_forest": {
        "label": "Random Forest",
        "architecture": "Bagged trees",
        "params": {
            "n_estimators": 120,
            "max_depth": 16,
            "min_samples_leaf": 10,
            "max_features": 0.70,
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
        },
    },
    "extra_trees": {
        "label": "Extra Trees",
        "architecture": "Highly randomized bagged trees",
        "params": {
            "n_estimators": 120,
            "max_depth": 16,
            "min_samples_leaf": 10,
            "max_features": 0.70,
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
        },
    },
    "hist_gradient_boosting": {
        "label": "HistGradientBoosting",
        "architecture": "Histogram gradient boosting",
        "params": {
            "learning_rate": 0.07,
            "max_iter": 220,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0,
            "early_stopping": False,
            "random_state": RANDOM_SEED,
        },
    },
    "lightgbm": {
        "label": "LightGBM",
        "architecture": "Leaf-wise gradient boosting",
        "params": {
            "objective": "regression",
            "n_estimators": 250,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "min_child_samples": 30,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_lambda": 1.0,
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
            "verbosity": -1,
        },
    },
    "catboost": {
        "label": "CatBoost",
        "architecture": "Categorical-aware gradient boosting",
        "params": {
            "loss_function": "RMSE",
            "iterations": 250,
            "depth": 8,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3.0,
            "random_seed": RANDOM_SEED,
            "thread_count": -1,
            "allow_writing_files": False,
            "verbose": False,
        },
    },
}

expected_family_screen_fits = len(screen_jobs_df) * len(FAMILY_SCREEN_CONFIGS)

candidate_roster_df = pd.DataFrame(
    [
        {
            "Model family": config["label"],
            "Architecture": config["architecture"],
            "Controlled jobs": len(screen_jobs_df),
            "Planned fits": len(screen_jobs_df),
        }
        for config in FAMILY_SCREEN_CONFIGS.values()
    ]
)

display(candidate_roster_df)

,Model family,Architecture,Controlled jobs,Planned fits
0,Random Forest,Bagged trees,20,20
1,Extra Trees,Highly randomized bagged trees,20,20
2,HistGradientBoosting,Histogram gradient boosting,20,20
3,LightGBM,Leaf-wise gradient boosting,20,20
4,CatBoost,Categorical-aware gradient boosting,20,20


**Findings.** The first-pass architecture screen contains five deliberately distinct tree families—Random Forest, Extra Trees, HistGradientBoosting, LightGBM, and CatBoost—evaluated across the same **20 representative Track × Target × Horizon jobs**. With one fixed screening definition per family, this produces **100 planned fits** before any hyperparameter tuning. The workload is broad enough to compare architectural behavior across all Post-CP Target × Horizon combinations plus Full-History h=5 transfer checks while remaining bounded enough to prune model families before more expensive cadence and tuning experiments.

> 💡 Explainer — Tree families\. Random Forest and Extra Trees are both bagged ensembles: they average many independently varied decision trees, with Extra Trees adding more randomness to how split points are chosen\. HistGradientBoosting, LightGBM, and CatBoost are boosting methods: they build trees sequentially so each new tree focuses on errors left by the current ensemble\. LightGBM is optimized for fast training on large tabular datasets, while CatBoost is especially well suited to categorical predictors such as Taxi Zone identity\. The later PyCaret cross\-check also includes XGBoost as another boosting implementation and Ridge regression as a simple linear reference\.

**Execute or load the controlled family screen.** The controlled family screen is restartable at the individual **Model Family × Track × Target × Horizon** fit.

Each successful logical fit is checkpointed immediately under the frozen screening contract. Completion is determined from the contract and logical experiment identity stored inside the checkpoint—not from filename existence alone. Failed or missing fits therefore remain eligible for retry, while successful fits are reused.

A complete consolidated intermediate is the normal fast path. If that artifact is absent or incomplete, the notebook reconstructs the current experiment state from durable checkpoints and computes only the unfinished logical fits. `RUN_FAMILY_SCREEN=True` forces that reconciliation path, while `RESET_FAMILY_SCREEN_CHECKPOINTS=True` is the only control that deliberately discards completed work.

Predictions are not clipped. In particular, a count model that produces negative forecasts retains those values for audit rather than receiving an artificial accuracy improvement from post-processing.

In [15]:
# ---------------------------------------------------------------------
# 4.4.1.3C — Execute or recover the controlled family screen
# ---------------------------------------------------------------------

FAMILY_SCREEN_KEY_COLUMNS = ["job_id", "model_id"]

family_expected_keys = {
    (job.job_id, model_id)
    for job in screen_jobs_df.itertuples(index=False)
    for model_id in FAMILY_SCREEN_CONFIGS
}
expected_family_fits = len(screen_jobs_df) * len(FAMILY_SCREEN_CONFIGS)

if len(family_expected_keys) != expected_family_fits:
    raise ValueError(
        "Family-screen plan is not unique by Job × Model Family."
    )


# ---------------------------------------------------------------------
# Reset is the only control that deliberately discards completed work
# ---------------------------------------------------------------------

if RESET_FAMILY_SCREEN_CHECKPOINTS:
    reset_checkpoint_dir(FAMILY_SCREEN_PARTS_DIR)
    FAMILY_SCREEN_RESULTS_PATH.unlink(missing_ok=True)

FAMILY_SCREEN_PARTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Prefer the compact consolidated artifact when it is already complete
# ---------------------------------------------------------------------

# RUN=True deliberately bypasses the fast path so the checkpoint surface is
# reconciled. It still does not refit successful logical units; RESET controls
# destructive rebuilding.
family_screen_results_df = None

if not RUN_FAMILY_SCREEN:
    family_screen_results_df = load_complete_consolidated_result(
        FAMILY_SCREEN_RESULTS_PATH,
        contract_column="screen_contract_id",
        contract_id=SCREEN_CONTRACT_ID,
        key_columns=FAMILY_SCREEN_KEY_COLUMNS,
        expected_keys=family_expected_keys,
    )

if family_screen_results_df is not None:
    print(
        f"Family screen complete: loaded {len(family_screen_results_df):,} "
        f"successful fits from {FAMILY_SCREEN_RESULTS_PATH.name}."
    )

else:
    # -----------------------------------------------------------------
    # Recover durable completion by logical key, not physical filename
    # -----------------------------------------------------------------

    family_checkpoint_df = canonical_checkpoint_rows(
        FAMILY_SCREEN_PARTS_DIR,
        contract_column="screen_contract_id",
        contract_id=SCREEN_CONTRACT_ID,
        key_columns=FAMILY_SCREEN_KEY_COLUMNS,
    )

    successful_family_df = (
        family_checkpoint_df.loc[family_checkpoint_df["status"].eq("success")].copy()
        if not family_checkpoint_df.empty
        else pd.DataFrame()
    )

    completed_family_keys = logical_key_set(
        successful_family_df,
        FAMILY_SCREEN_KEY_COLUMNS,
    )
    missing_family_keys = family_expected_keys - completed_family_keys

    print(
        f"Family screen restart state: {len(completed_family_keys):,}/"
        f"{expected_family_fits:,} successful; "
        f"{len(missing_family_keys):,} unfinished."
    )

    # Long-running work must be durable. If unfinished fits exist, refusing to
    # checkpoint would make an interruption unnecessarily expensive.
    if missing_family_keys and not WRITE_CHECKPOINTS:
        raise ValueError(
            "The family screen has unfinished fits and requires "
            "WRITE_CHECKPOINTS=True."
        )


    # -----------------------------------------------------------------
    # Fit only unfinished Model Family × Forecasting Job definitions
    # -----------------------------------------------------------------

    if missing_family_keys:
        from tqdm import tqdm as tqdm_text

        family_progress = tqdm_text(
            total=expected_family_fits,
            initial=len(completed_family_keys),
            desc="Panel-tree family fits",
            unit="fit",
            dynamic_ncols=True,
        )

        failed_this_pass = 0

        for job in screen_jobs_df.itertuples(index=False):
            track = job.modeling_track
            metric = job.metric
            horizon = int(job.horizon)
            job_id = job.job_id

            pending_model_ids = [
                model_id
                for model_id in FAMILY_SCREEN_CONFIGS
                if (job_id, model_id) in missing_family_keys
            ]

            # All families for one forecasting job share the same legal train
            # and validation surface. Load that surface once rather than once
            # per estimator.
            if not pending_model_ids:
                continue

            try:
                payload = load_family_screen_job(
                    track=track,
                    metric=metric,
                    horizon=horizon,
                    train_row_cap=SCREEN_TRAIN_ROW_CAP,
                )
                payload_error = None
            except Exception as exc:
                payload = None
                payload_error = exc

            for model_id in pending_model_ids:
                config = FAMILY_SCREEN_CONFIGS[model_id]

                family_progress.set_postfix_str(
                    f"{TRACK_SHORT_LABELS[track]} · "
                    f"{TARGET_SHORT_LABELS[metric]} · h={horizon} · "
                    f"{config['label']} · failures={failed_this_pass}"
                )

                checkpoint_name = slugify(
                    f"{SCREEN_CONTRACT_ID}__{job_id}__{model_id}"
                ) + ".parquet"
                checkpoint_path = FAMILY_SCREEN_PARTS_DIR / checkpoint_name

                # Every row carries its own contract and logical identity so
                # restart safety does not depend on the filename convention.
                base_record = {
                    "screen_contract_id": SCREEN_CONTRACT_ID,
                    "modeling_track": track,
                    "metric": metric,
                    "horizon": horizon,
                    "job_id": job_id,
                    "model_id": model_id,
                    "model_label": config["label"],
                }

                if payload_error is not None:
                    record = {
                        **base_record,
                        "status": "failed",
                        "failure_phase": "load_job",
                        "error_type": type(payload_error).__name__,
                        "error_message": str(payload_error),
                        "fit_seconds": np.nan,
                        "predict_seconds": np.nan,
                    }
                    failed_this_pass += 1

                else:
                    base_record.update(
                        {
                            "train_rows": payload["train_rows"],
                            "train_origins": payload["train_origins"],
                            "train_forecast_sequences": payload["train_forecast_sequences"],
                            "valid_rows": payload["valid_rows"],
                            "valid_origins": payload["valid_origins"],
                            "valid_forecast_sequences": payload["valid_forecast_sequences"],
                            "predictor_count": len(payload["predictor_names"]),
                            "categorical_predictor_count": len(payload["categorical_names"]),
                            "benchmark_id": payload["benchmark_id"],
                            "benchmark_label": payload["benchmark_label"],
                            "benchmark_mae": payload["benchmark_metrics"]["mae"],
                            "benchmark_rmse": payload["benchmark_metrics"]["rmse"],
                            "benchmark_nmae": payload["benchmark_metrics"]["nmae"],
                            "benchmark_bias": payload["benchmark_metrics"]["bias"],
                        }
                    )

                    try:
                        predictions, fit_seconds, predict_seconds = fit_screen_candidate(
                            model_id=model_id,
                            config=config,
                            X_train=payload["X_train"],
                            y_train=payload["y_train"],
                            X_valid=payload["X_valid"],
                            predictor_names=payload["predictor_names"],
                            categorical_names=payload["categorical_names"],
                        )

                        model_metrics = regression_metrics(payload["y_valid"], predictions)
                        benchmark_mae = float(payload["benchmark_metrics"]["mae"])

                        skill_vs_benchmark_pct = (
                            100.0 * (benchmark_mae - model_metrics["mae"]) / benchmark_mae
                            if benchmark_mae > 0
                            else np.nan
                        )

                        record = {
                            **base_record,
                            "status": "success",
                            "failure_phase": None,
                            "error_type": None,
                            "error_message": None,
                            **model_metrics,
                            "skill_vs_benchmark_pct": skill_vs_benchmark_pct,
                            "negative_prediction_count": int(np.sum(predictions < 0)),
                            "fit_seconds": float(fit_seconds),
                            "predict_seconds": float(predict_seconds),
                        }

                        # The progress bar represents successful durable logical
                        # units. Failed fits remain visibly unfinished and are
                        # retried on the next reconciliation pass.
                        family_progress.update(1)
                        del predictions

                    except Exception as exc:
                        record = {
                            **base_record,
                            "status": "failed",
                            "failure_phase": "fit_or_score",
                            "error_type": type(exc).__name__,
                            "error_message": str(exc),
                            "mae": np.nan,
                            "rmse": np.nan,
                            "nmae": np.nan,
                            "bias": np.nan,
                            "skill_vs_benchmark_pct": np.nan,
                            "negative_prediction_count": np.nan,
                            "fit_seconds": np.nan,
                            "predict_seconds": np.nan,
                        }
                        failed_this_pass += 1

                write_checkpoint_frame(pd.DataFrame([record]), checkpoint_path)

            if payload is not None:
                del payload

            gc.collect()

        family_progress.set_postfix_str(f"failures={failed_this_pass}")
        family_progress.close()


    # -----------------------------------------------------------------
    # Rebuild one canonical experiment table from durable checkpoints
    # -----------------------------------------------------------------

    family_screen_results_df = canonical_checkpoint_rows(
        FAMILY_SCREEN_PARTS_DIR,
        contract_column="screen_contract_id",
        contract_id=SCREEN_CONTRACT_ID,
        key_columns=FAMILY_SCREEN_KEY_COLUMNS,
    )

    if family_screen_results_df.empty:
        raise RuntimeError(
            "Family-screen execution produced no current-contract results."
        )

    observed_family_keys = logical_key_set(
        family_screen_results_df,
        FAMILY_SCREEN_KEY_COLUMNS,
    )

    if observed_family_keys != family_expected_keys:
        missing = family_expected_keys - observed_family_keys
        extra = observed_family_keys - family_expected_keys
        raise RuntimeError(
            "Family-screen result roster does not match the planned experiment. "
            f"Missing={len(missing)}, extra={len(extra)}."
        )

    if WRITE_CONSOLIDATED_INTERMEDIATES:
        atomic_parquet_write(family_screen_results_df, FAMILY_SCREEN_RESULTS_PATH)


# ---------------------------------------------------------------------
# Final contract QA and compact reader-facing status
# ---------------------------------------------------------------------

require_columns(
    family_screen_results_df,
    ["screen_contract_id", "job_id", "model_id", "model_label", "status"],
    "Family-screen results",
)

if not family_screen_results_df["screen_contract_id"].eq(SCREEN_CONTRACT_ID).all():
    raise ValueError("Family-screen results contain another experiment contract.")

if family_screen_results_df.duplicated(FAMILY_SCREEN_KEY_COLUMNS).any():
    raise ValueError(
        "Family-screen results are not unique by Job × Model Family."
    )

if len(family_screen_results_df) != expected_family_fits:
    raise RuntimeError(
        f"Expected {expected_family_fits} family-screen fits; "
        f"found {len(family_screen_results_df)}."
    )

family_screen_status_df = (
    family_screen_results_df
    .groupby("model_label", as_index=False)
    .agg(
        successful_fits=("status", lambda s: int(s.eq("success").sum())),
        failed_fits=("status", lambda s: int(s.eq("failed").sum())),
    )
    .rename(
        columns={
            "model_label": "Model family",
            "successful_fits": "Successful fits",
            "failed_fits": "Failed fits",
        }
    )
)

display(family_screen_status_df)

Family screen complete: loaded 100 successful fits from panel_tree_family_screen_results__family_screen_v1_28d_250k.parquet.


,Model family,Successful fits,Failed fits
0,CatBoost,20,0
1,Extra Trees,20,0
2,HistGradientBoosting,20,0
3,LightGBM,20,0
4,Random Forest,20,0


**Findings.** The controlled tree-family screen completed successfully with **100 of 100 planned fits**. Each of the five candidate families completed all 20 representative jobs, with **zero fit, prediction, or data-loading failures**. Operational reliability therefore does not distinguish the candidates at this stage; the next decision can be based primarily on predictive breadth, benchmark skill, architectural redundancy, and runtime rather than excluding any family because it failed to execute.

### Cross-check the candidate roster independently with PyCaret

The controlled screen reflects a deliberately chosen tree roster. PyCaret provides a useful independent check against our own model-family assumptions.

PyCaret does **not** define the forecasting experiment. Each sentinel receives the same legal training history, the same fixed 28-day future slice, the same 85 logical model inputs, and the same exact-row frozen benchmark used by the controlled screen. The externally supplied validation frame is used as PyCaret's holdout, and cross-validation inside `compare_models()` is disabled so PyCaret cannot replace the project's temporal split with ordinary K-fold validation.

The comparison deliberately includes:

- Ridge regression as a simple non-tree reference;
- Random Forest;
- Extra Trees;
- XGBoost;
- LightGBM;
- CatBoost.

The high-cardinality Taxi Zone ID remains nominal. `max_encoding_ohe=500` keeps its 263 categories within PyCaret's ordinary one-hot encoding path rather than allowing the stored numeric IDs to behave like a continuous measurement.

PyCaret's leaderboard is secondary evidence. It can corroborate the controlled screen or identify an overlooked candidate—especially XGBoost—but it cannot override the frozen forecasting contract or select the final production model.

> 💡 What is a sentinel?
A sentinel is a deliberately chosen representative forecasting job used as a compact check on a broader modeling decision\. Instead of rerunning an expensive comparison across every Track × Target × Horizon combination, the notebook uses a small set of contrasting cases that are meant to expose whether the conclusion holds across different kinds of forecasting problems\.

Here, the three sentinels span both modeling tracks, short and long forecast horizons, demand and speed targets, and more than one transport mode\. They are \*\*not\*\* a substitute for the broader controlled screen; they are an independent cross\-check\.

In [16]:
# ---------------------------------------------------------------------
# 4.4.1.3D — Define the PyCaret cross-check
# ---------------------------------------------------------------------

# PyCaret is used only for the small independent AutoML cross-check. Keep the
# dependency local to this section so an unavailable or version-incompatible
# PyCaret installation cannot block ordinary notebook execution that only
# consumes the already-persisted cross-check results.
try:
    from pycaret.regression import (
        compare_models as pycaret_compare_models,
        models as pycaret_models,
        pull as pycaret_pull,
        setup as pycaret_setup,
    )

    PYCARET_AVAILABLE = True
    PYCARET_IMPORT_ERROR = None

except Exception as exc:
    PYCARET_AVAILABLE = False
    PYCARET_IMPORT_ERROR = exc


# The PyCaret comparison is intentionally small and secondary. It tests whether
# an independent AutoML workflow points toward the same family ranking as the
# controlled screen, while also checking XGBoost and a simple linear reference.
PYCARET_MODEL_IDS = [
    "ridge",
    "rf",
    "et",
    "xgboost",
    "lightgbm",
    "catboost",
]


def run_pycaret_crosscheck(job):
    """Run one bounded PyCaret comparison on the project's temporal holdout.

    The function reuses the exact same screening job construction as the
    controlled family screen, then asks PyCaret to score a small comparison
    roster on that explicit future validation slice. PyCaret is used only as
    an independent cross-check; it does not redefine the forecasting split,
    benchmark, or primary MAE decision metric.
    """

    # PyCaret is required only when this completed secondary experiment is
    # deliberately recomputed. A normal notebook rerun should be able to load
    # persisted PyCaret evidence without requiring a compatible AutoML runtime.
    if not PYCARET_AVAILABLE:
        raise RuntimeError(
            "PyCaret is unavailable in the current environment. "
            f"Import error: {PYCARET_IMPORT_ERROR}"
        )

    track = job.modeling_track
    metric = job.metric
    horizon = int(job.horizon)

    # Reuse the project's own leakage-safe job builder so PyCaret sees the same
    # forecasting problem as the controlled family screen. Only the row cap
    # differs, keeping this secondary comparison computationally bounded.
    payload = load_family_screen_job(
        track=track,
        metric=metric,
        horizon=horizon,
        train_row_cap=PYCARET_TRAIN_ROW_CAP,
    )

    target_column = "target_observed_value"

    # PyCaret expects predictors and target in one tabular frame. Reconstruct
    # that interface without changing feature values or the target scale.
    train_data = payload["X_train"].copy()
    train_data[target_column] = payload["y_train"].to_numpy(dtype=float)

    validation_data = payload["X_valid"].copy()
    validation_data[target_column] = payload["y_valid"].to_numpy(dtype=float)

    # Forecast identity lives in explicit matrix columns rather than the Pandas
    # row index. Reset both frames so PyCaret cannot interpret repeated source
    # indices as part of the experiment definition.
    train_data = train_data.reset_index(drop=True)
    validation_data = validation_data.reset_index(drop=True)

    # PyCaret must respect the project's temporal experiment. The future slice
    # is supplied explicitly as test_data, so PyCaret is not allowed to create
    # a random holdout or substitute ordinary cross-validation.
    #
    # Preprocessing remains deliberately conservative:
    # - median numeric imputation;
    # - mode categorical imputation;
    # - no feature selection;
    # - no scaling/transformation/PCA;
    # - no polynomial expansion.
    #
    # This keeps the cross-check focused on model-family behavior rather than
    # introducing a second automated feature-engineering experiment.
    pycaret_setup(
        data=train_data,
        target=target_column,
        test_data=validation_data,
        index=False,
        categorical_features=payload["categorical_names"],
        preprocess=True,
        numeric_imputation="median",
        categorical_imputation="mode",
        max_encoding_ohe=500,
        feature_selection=False,
        remove_multicollinearity=False,
        polynomial_features=False,
        transformation=False,
        normalize=False,
        pca=False,
        data_split_shuffle=False,
        session_id=RANDOM_SEED,
        n_jobs=-1,
        html=False,
        system_log=False,
        verbose=False,
    )

    # Model availability can vary by installed PyCaret/package versions.
    # Restrict the requested roster to models actually exposed at runtime rather
    # than assuming every optional backend is present.
    available_models_df = pycaret_models()
    available_ids = set(available_models_df.index.astype(str))

    requested_ids = [
        model_id
        for model_id in PYCARET_MODEL_IDS
        if model_id in available_ids
    ]

    # With fewer than two surviving candidates this would no longer be a useful
    # independent comparison, so fail explicitly rather than emit a misleading
    # one-model "leaderboard."
    if len(requested_ids) < 2:
        raise RuntimeError(
            "PyCaret exposes fewer than two requested comparison models."
        )

    # Score each requested candidate once on the project's supplied future
    # holdout. cross_validation=False is essential because the Chapter 4
    # temporal validation contract, not random K-fold CV, defines the question.
    pycaret_compare_models(
        include=requested_ids,
        sort="MAE",
        n_select=len(requested_ids),
        cross_validation=False,
        turbo=False,
        errors="ignore",
        verbose=False,
    )

    # compare_models() renders internally; pull() retrieves its leaderboard so
    # the results can be normalized into the project's own schema.
    leaderboard = pycaret_pull().copy().reset_index(drop=True)

    if "Model" not in leaderboard.columns or "MAE" not in leaderboard.columns:
        raise ValueError(
            "PyCaret comparison output is missing Model or MAE."
        )

    # PyCaret reports human-readable display names while the project uses compact
    # model IDs. Build an explicit mapping so downstream joins and rankings use
    # stable identifiers.
    model_name_to_id = {
        str(row["Name"]): str(model_id)
        for model_id, row in available_models_df.iterrows()
        if "Name" in row.index
    }

    result = pd.DataFrame(
        {
            "pycaret_model_label": leaderboard["Model"].astype(str),
            "pycaret_model_id": (
                leaderboard["Model"].astype(str).map(model_name_to_id)
            ),
            "mae": pd.to_numeric(leaderboard["MAE"], errors="coerce"),
        }
    )

    # Preserve useful supporting diagnostics when the installed PyCaret version
    # exposes them, but do not make downstream logic depend on optional columns.
    if "RMSE" in leaderboard.columns:
        result["rmse"] = pd.to_numeric(
            leaderboard["RMSE"],
            errors="coerce",
        )

    if "R2" in leaderboard.columns:
        result["r2"] = pd.to_numeric(
            leaderboard["R2"],
            errors="coerce",
        )

    if "TT (Sec)" in leaderboard.columns:
        result["tt_seconds"] = pd.to_numeric(
            leaderboard["TT (Sec)"],
            errors="coerce",
        )

    # Fall back to a stable slug only when PyCaret's display name cannot be
    # mapped back to its registered model ID. Preserve the result rather than
    # silently dropping an otherwise valid scored model.
    result["pycaret_model_id"] = result["pycaret_model_id"].fillna(
        result["pycaret_model_label"].map(
            lambda value: slugify(value).lower()
        )
    )

    # Attach forecasting-job identity and experiment contract so every result
    # remains self-describing after checkpoint consolidation.
    result.insert(0, "screen_contract_id", PYCARET_SCREEN_CONTRACT_ID)
    result.insert(1, "modeling_track", track)
    result.insert(2, "metric", metric)
    result.insert(3, "horizon", horizon)

    result["job_id"] = f"{track}__{metric}__h{horizon}"
    result["status"] = "success"

    # Carry the exact support and benchmark context used by the sentinel so the
    # later synthesis can verify that each candidate was scored on the intended
    # temporal experiment.
    result["train_rows"] = payload["train_rows"]
    result["valid_rows"] = payload["valid_rows"]
    result["benchmark_id"] = payload["benchmark_id"]
    result["benchmark_label"] = payload["benchmark_label"]
    result["benchmark_mae"] = payload["benchmark_metrics"]["mae"]

    # Positive skill means lower MAE than the frozen benchmark. Percentage skill
    # allows the comparison to remain interpretable across targets with very
    # different raw units.
    result["skill_vs_benchmark_pct"] = (
        100.0
        * (result["benchmark_mae"] - result["mae"])
        / result["benchmark_mae"]
    )

    # Release the comparatively large sentinel surfaces before processing the
    # next job so this secondary comparison does not retain avoidable memory.
    del payload, train_data, validation_data
    gc.collect()

    return result

**Execute the restartable PyCaret cross-check.** The PyCaret cross-check is restartable at the **sentinel leaderboard** level rather than the individual estimator level because one `compare_models()` execution produces the complete candidate roster for that sentinel.

Each successful sentinel leaderboard is checkpointed under the frozen PyCaret screening contract. Completion is validated from the sentinel identity, contract, and expected model roster stored inside the checkpoint. A failed or incomplete sentinel therefore remains eligible for retry.

A complete consolidated intermediate is the normal fast path. If it is absent or incomplete, the notebook reconstructs the experiment state from durable sentinel checkpoints and reruns only unfinished sentinels. `RUN_PYCARET_SCREEN=True` forces that reconciliation path; `RESET_PYCARET_SCREEN_CHECKPOINTS=True` is the only control that deliberately discards completed work.

In [17]:
# ---------------------------------------------------------------------
# 4.4.1.3E — Execute or recover the PyCaret cross-check
# ---------------------------------------------------------------------

PYCARET_RESULT_KEY_COLUMNS = ["job_id", "pycaret_model_id"]

pycaret_expected_keys = {
    (job.job_id, model_id)
    for job in pycaret_screen_jobs_df.itertuples(index=False)
    for model_id in PYCARET_MODEL_IDS
}
expected_pycaret_rows = len(pycaret_screen_jobs_df) * len(PYCARET_MODEL_IDS)

if len(pycaret_expected_keys) != expected_pycaret_rows:
    raise ValueError(
        "PyCaret plan is not unique by Sentinel Job × Model."
    )


# ---------------------------------------------------------------------
# Reset is the only destructive control
# ---------------------------------------------------------------------

if RESET_PYCARET_SCREEN_CHECKPOINTS:
    reset_checkpoint_dir(PYCARET_SCREEN_PARTS_DIR)
    PYCARET_SCREEN_RESULTS_PATH.unlink(missing_ok=True)

PYCARET_SCREEN_PARTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Use the consolidated artifact when it is complete and reconciliation
# has not been explicitly requested
# ---------------------------------------------------------------------

pycaret_screen_results_df = None

if not RUN_PYCARET_SCREEN:
    pycaret_screen_results_df = load_complete_consolidated_result(
        PYCARET_SCREEN_RESULTS_PATH,
        contract_column="screen_contract_id",
        contract_id=PYCARET_SCREEN_CONTRACT_ID,
        key_columns=PYCARET_RESULT_KEY_COLUMNS,
        expected_keys=pycaret_expected_keys,
    )

if pycaret_screen_results_df is not None:
    print(
        f"PyCaret cross-check complete: loaded "
        f"{len(pycaret_screen_results_df):,} model rows from "
        f"{PYCARET_SCREEN_RESULTS_PATH.name}."
    )

else:
    # -----------------------------------------------------------------
    # A sentinel is complete only when every expected PyCaret model row
    # is represented successfully under the current contract
    # -----------------------------------------------------------------

    pycaret_checkpoint_df = canonical_checkpoint_rows(
        PYCARET_SCREEN_PARTS_DIR,
        contract_column="screen_contract_id",
        contract_id=PYCARET_SCREEN_CONTRACT_ID,
        key_columns=PYCARET_RESULT_KEY_COLUMNS,
    )

    successful_pycaret_df = (
        pycaret_checkpoint_df.loc[pycaret_checkpoint_df["status"].eq("success")].copy()
        if not pycaret_checkpoint_df.empty
        else pd.DataFrame()
    )

    completed_pycaret_jobs = set()

    if not successful_pycaret_df.empty:
        for job_id, group in successful_pycaret_df.groupby("job_id"):
            if set(group["pycaret_model_id"]) == set(PYCARET_MODEL_IDS):
                completed_pycaret_jobs.add(job_id)

    expected_pycaret_jobs = set(pycaret_screen_jobs_df["job_id"])
    missing_pycaret_jobs = expected_pycaret_jobs - completed_pycaret_jobs

    print(
        f"PyCaret restart state: {len(completed_pycaret_jobs)}/"
        f"{len(expected_pycaret_jobs)} sentinels complete; "
        f"{len(missing_pycaret_jobs)} unfinished."
    )

    if missing_pycaret_jobs and not WRITE_CHECKPOINTS:
        raise ValueError(
            "The PyCaret cross-check has unfinished sentinels and requires "
            "WRITE_CHECKPOINTS=True."
        )


    # -----------------------------------------------------------------
    # compare_models() is atomic at the sentinel level, so only complete
    # sentinels are reused; an incomplete sentinel is rerun as one unit
    # -----------------------------------------------------------------

    if missing_pycaret_jobs:
        from tqdm import tqdm as tqdm_text

        pending_jobs_df = pycaret_screen_jobs_df.loc[
            pycaret_screen_jobs_df["job_id"].isin(missing_pycaret_jobs)
        ].copy()

        progress = tqdm_text(
            pending_jobs_df.itertuples(index=False),
            total=len(pending_jobs_df),
            desc="PyCaret sentinels",
            unit="sentinel",
            dynamic_ncols=True,
        )

        for job in progress:
            job_id = job.job_id

            progress.set_postfix_str(
                f"{TRACK_SHORT_LABELS[job.modeling_track]} · "
                f"{TARGET_SHORT_LABELS[job.metric]} · h={int(job.horizon)}"
            )

            checkpoint_name = slugify(
                f"{PYCARET_SCREEN_CONTRACT_ID}__{job_id}"
            ) + ".parquet"
            checkpoint_path = PYCARET_SCREEN_PARTS_DIR / checkpoint_name

            try:
                result = run_pycaret_crosscheck(job)

                require_columns(
                    result,
                    ["screen_contract_id", "job_id", "pycaret_model_id"],
                    f"PyCaret result for {job_id}",
                )

                # A successful sentinel must contain exactly the frozen candidate
                # roster once. Partial leaderboards are not reusable.
                if not result["screen_contract_id"].eq(PYCARET_SCREEN_CONTRACT_ID).all():
                    raise ValueError(f"{job_id}: wrong PyCaret experiment contract.")

                if not result["job_id"].eq(job_id).all():
                    raise ValueError(f"{job_id}: wrong PyCaret sentinel identity.")

                observed_model_ids = set(result["pycaret_model_id"])
                if observed_model_ids != set(PYCARET_MODEL_IDS):
                    raise ValueError(
                        f"{job_id}: expected PyCaret models "
                        f"{sorted(PYCARET_MODEL_IDS)}, found "
                        f"{sorted(observed_model_ids)}."
                    )

                if result.duplicated(PYCARET_RESULT_KEY_COLUMNS).any():
                    raise ValueError(
                        f"{job_id}: duplicate PyCaret model rows were returned."
                    )

                result["status"] = "success"
                result["error_type"] = None
                result["error_message"] = None

            except Exception as exc:
                # Store one explicit sentinel-level failure row. Because it is
                # not a complete expected model roster, the next run retries
                # this sentinel automatically.
                result = pd.DataFrame(
                    [
                        {
                            "screen_contract_id": PYCARET_SCREEN_CONTRACT_ID,
                            "modeling_track": job.modeling_track,
                            "metric": job.metric,
                            "horizon": int(job.horizon),
                            "job_id": job_id,
                            "pycaret_model_id": "__pycaret_run__",
                            "pycaret_model_label": "PyCaret run",
                            "status": "failed",
                            "error_type": type(exc).__name__,
                            "error_message": str(exc),
                        }
                    ]
                )

            write_checkpoint_frame(result, checkpoint_path)
            gc.collect()


    # -----------------------------------------------------------------
    # Reconstruct the current sentinel evidence from durable checkpoints
    # -----------------------------------------------------------------

    pycaret_screen_results_df = canonical_checkpoint_rows(
        PYCARET_SCREEN_PARTS_DIR,
        contract_column="screen_contract_id",
        contract_id=PYCARET_SCREEN_CONTRACT_ID,
        key_columns=PYCARET_RESULT_KEY_COLUMNS,
    )

    if pycaret_screen_results_df.empty:
        raise RuntimeError(
            "PyCaret cross-check produced no current-contract results."
        )

    if WRITE_CONSOLIDATED_INTERMEDIATES:
        atomic_parquet_write(
            pycaret_screen_results_df,
            PYCARET_SCREEN_RESULTS_PATH,
        )


# ---------------------------------------------------------------------
# Compact completion status
# ---------------------------------------------------------------------

require_columns(
    pycaret_screen_results_df,
    ["screen_contract_id", "job_id", "pycaret_model_id", "status"],
    "PyCaret screen results",
)

pycaret_success_df = pycaret_screen_results_df.loc[
    pycaret_screen_results_df["status"].eq("success")
].copy()

completed_pycaret_jobs = 0

for job_id in pycaret_screen_jobs_df["job_id"]:
    model_ids = set(
        pycaret_success_df.loc[
            pycaret_success_df["job_id"].eq(job_id),
            "pycaret_model_id",
        ]
    )
    completed_pycaret_jobs += int(model_ids == set(PYCARET_MODEL_IDS))

pycaret_status_df = pd.DataFrame(
    [
        {
            "Sentinels planned": len(pycaret_screen_jobs_df),
            "Sentinels complete": completed_pycaret_jobs,
            "Successful model rows": len(pycaret_success_df),
            "Expected model rows": expected_pycaret_rows,
            "Failed sentinel rows": int(
                pycaret_screen_results_df["status"].eq("failed").sum()
            ),
        }
    ]
)

display(pycaret_status_df)

PyCaret cross-check complete: loaded 18 model rows from panel_tree_pycaret_screen_results__pycaret_screen_v1_28d_100k.parquet.


,Sentinels planned,Sentinels complete,Successful model rows,Expected model rows,Failed sentinel rows
0,3,3,18,18,0


**Findings.** The independent PyCaret cross-check completed successfully for all three sentinel forecasting problems. Each sentinel scored all **six requested models**—Ridge, Random Forest, Extra Trees, XGBoost, LightGBM, and CatBoost—and no sentinel run failed. The cross-check therefore provides a complete secondary leaderboard for the intended blind-spot test, including XGBoost as the major tree family omitted from the controlled five-family screen.

### Put predictive performance, breadth, reliability, and runtime on one page

The model decision should not be based on one winning MAE.

The controlled screen therefore summarizes each family across all jobs it successfully completed:

- **MAE wins** show how often it is the best candidate on a particular problem;
- **median MAE rank** shows whether it stays near the front even when it does not win;
- **jobs beating the frozen benchmark** measure breadth rather than isolated success;
- **median and worst skill vs. benchmark** distinguish consistently useful models from volatile ones;
- **runtime** shows the computational cost of carrying a family into later refit and tuning experiments;
- **failures** remain visible rather than disappearing from the leaderboard.

PyCaret is summarized independently across its three sentinel problems. Its purpose is corroboration and blind-spot detection, not direct pooling with the controlled experiment.

The visual summaries emphasize **skill relative to the frozen benchmark** because its zero point has a direct meaning across targets measured in very different physical units.

In [18]:
# ---------------------------------------------------------------------
# 4.4.1.3F — Consolidate and visualize controlled family-screen evidence
# ---------------------------------------------------------------------

# The synthesis stage assumes the restartable screen has already produced at
# least some saved fit results. Failing here is preferable to displaying an
# empty or misleading architecture summary.
if family_screen_results_df.empty:
    raise RuntimeError(
        "Run or load the controlled family screen before synthesizing evidence."
    )

# Family-selection evidence is based on successful fits only. Failed fits are
# preserved separately and reintroduced later as a reliability diagnostic
# rather than being silently dropped from the experiment.
family_success_df = family_screen_results_df.loc[
    family_screen_results_df["status"].eq("success")
].copy()

if family_success_df.empty:
    raise RuntimeError(
        "The controlled family screen contains no successful fits."
    )


# ---------------------------------------------------------------------
# Compare candidate families within the same forecasting job
# ---------------------------------------------------------------------

# Raw MAE values cannot be compared directly across different targets because
# Taxi trips, speed, and Subway ridership live on very different numerical
# scales. Rank each family only against its competitors on the same
# Track × Target × Horizon job.
family_success_df["mae_rank"] = (
    family_success_df
    .groupby("job_id")["mae"]
    .rank(method="min", ascending=True)
)

# A rank of 1 identifies the lowest-MAE family for that forecasting job.
# method="min" allows true ties to count as shared wins instead of forcing an
# arbitrary ordering between numerically identical results.
family_success_df["is_mae_winner"] = (
    family_success_df["mae_rank"].eq(1.0)
)

# Positive skill means the candidate lowered MAE relative to the already-frozen
# baseline on the exact same forecast rows.
family_success_df["beats_benchmark"] = (
    family_success_df["skill_vs_benchmark_pct"].gt(0)
)


# ---------------------------------------------------------------------
# Preserve operational reliability alongside predictive performance
# ---------------------------------------------------------------------

# Count failed fits separately by family. A candidate that performs well when
# it runs but repeatedly fails is less attractive for later full validation.
family_failure_counts_df = (
    family_screen_results_df.loc[
        family_screen_results_df["status"].eq("failed")
    ]
    .groupby(["model_id", "model_label"])
    .size()
    .rename("failed_fits")
    .reset_index()
)


# ---------------------------------------------------------------------
# Reduce the 100 controlled fits to one comparable summary row per family
# ---------------------------------------------------------------------

# The summary deliberately combines several kinds of evidence:
# - MAE wins and median rank describe relative accuracy against other families;
# - benchmark skill shows whether the family improves on the inherited baseline;
# - worst skill exposes weak-tail behavior hidden by a median;
# - runtime shows operational cost;
# - failed fits capture reliability.
family_screen_summary_df = (
    family_success_df
    .groupby(["model_id", "model_label"], as_index=False)
    .agg(
        screen_jobs=("job_id", "nunique"),
        mae_wins=("is_mae_winner", "sum"),
        median_mae_rank=("mae_rank", "median"),
        jobs_beating_benchmark=("beats_benchmark", "sum"),
        median_skill_vs_benchmark_pct=(
            "skill_vs_benchmark_pct",
            "median",
        ),
        worst_skill_vs_benchmark_pct=(
            "skill_vs_benchmark_pct",
            "min",
        ),
        median_fit_seconds=("fit_seconds", "median"),
        total_fit_seconds=("fit_seconds", "sum"),
    )
    .merge(
        family_failure_counts_df,
        on=["model_id", "model_label"],
        how="left",
    )
)

# Families with no recorded failures are genuine zeroes, not missing values.
family_screen_summary_df["failed_fits"] = (
    family_screen_summary_df["failed_fits"]
    .fillna(0)
    .astype(int)
)

# Convert the benchmark win count into a percentage so families remain directly
# comparable even if a future screen ever contains unequal completed job counts.
family_screen_summary_df["jobs_beating_benchmark_pct"] = (
    100.0
    * family_screen_summary_df["jobs_beating_benchmark"]
    / family_screen_summary_df["screen_jobs"]
)

# Sort primarily by broad MAE performance, then by benchmark improvement, and
# finally by speed. This ordering is descriptive rather than an automatic
# selection rule; the reader still evaluates the full evidence below.
family_screen_summary_df = (
    family_screen_summary_df
    .sort_values(
        [
            "median_mae_rank",
            "median_skill_vs_benchmark_pct",
            "median_fit_seconds",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Reader-facing family summary
# ---------------------------------------------------------------------

# Keep the visible table compact: one row per family and only the statistics
# needed to judge accuracy, benchmark improvement, runtime, and reliability.
family_screen_display_df = (
    family_screen_summary_df[
        [
            "model_label",
            "screen_jobs",
            "mae_wins",
            "median_mae_rank",
            "jobs_beating_benchmark_pct",
            "median_skill_vs_benchmark_pct",
            "worst_skill_vs_benchmark_pct",
            "median_fit_seconds",
            "failed_fits",
        ]
    ]
    .rename(
        columns={
            "model_label": "Model family",
            "screen_jobs": "Jobs scored",
            "mae_wins": "MAE wins",
            "median_mae_rank": "Median MAE rank",
            "jobs_beating_benchmark_pct": "Beat benchmark %",
            "median_skill_vs_benchmark_pct": "Median skill %",
            "worst_skill_vs_benchmark_pct": "Worst skill %",
            "median_fit_seconds": "Median fit sec",
            "failed_fits": "Failed fits",
        }
    )
)

for column in [
    "Median MAE rank",
    "Beat benchmark %",
    "Median skill %",
    "Worst skill %",
    "Median fit sec",
]:
    family_screen_display_df[column] = (
        family_screen_display_df[column].round(2)
    )

display(family_screen_display_df)


# ---------------------------------------------------------------------
# Visual 1 — Show where each family beats or trails the frozen benchmark
# ---------------------------------------------------------------------

# A readable job label lets the heatmap show the full 20-job pattern rather
# than collapsing architecture performance into one average.
family_success_df["job_label"] = family_success_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · h={int(row['horizon'])}"
    ),
    axis=1,
)

# Preserve the planned screen order on the x-axis so the visual follows the
# experiment design rather than an arbitrary alphabetical pivot order.
screen_job_labels = [
    (
        f"{TRACK_SHORT_LABELS[row.modeling_track]} · "
        f"{TARGET_SHORT_LABELS[row.metric]} · h={int(row.horizon)}"
    )
    for row in screen_jobs_df.itertuples(index=False)
]

if RUN_VISUALS:
    # Rows are model families, columns are forecasting jobs, and each cell shows
    # percentage MAE skill relative to the frozen benchmark.
    #
    # Centering the diverging scale at zero makes the central question visually
    # immediate: positive cells beat the benchmark; negative cells trail it.
    skill_heatmap_df = (
        family_success_df
        .pivot(
            index="model_label",
            columns="job_label",
            values="skill_vs_benchmark_pct",
        )
        .reindex(
            index=[
                config["label"]
                for config in FAMILY_SCREEN_CONFIGS.values()
            ],
            columns=screen_job_labels,
        )
    )

    skill_heatmap_fig = go.Figure(
        data=go.Heatmap(
            z=skill_heatmap_df.to_numpy(),
            x=skill_heatmap_df.columns.tolist(),
            y=skill_heatmap_df.index.tolist(),
            colorscale=BRAND_DIVERGING_SEQUENCE,
            zmid=0,
            colorbar={"title": "Skill vs<br>benchmark (%)"},
            hovertemplate=(
                "Model: %{y}<br>"
                "Job: %{x}<br>"
                "Skill vs benchmark: %{z:.2f}%"
                "<extra></extra>"
            ),
        )
    )

    skill_heatmap_fig.update_layout(
        title=(
            "Controlled Tree-Family Screen — "
            "Skill Relative to the Frozen Benchmark"
        ),
        xaxis_title=None,
        yaxis_title=None,
        width=950,
        height=450,
        margin=dict(l=170, r=80, t=80, b=180),
    )
    skill_heatmap_fig.update_xaxes(tickangle=-55)

    apply_branding(skill_heatmap_fig)
    skill_heatmap_fig.show()


    # -----------------------------------------------------------------
    # Visual 2 — Put predictive improvement beside computational cost
    # -----------------------------------------------------------------

    # The first visual asks "where does each family work?" This second one asks
    # a different operational question: does additional predictive skill require
    # substantially more fitting time?
    #
    # Fit time is shown on a logarithmic x-axis because the families differ by
    # orders of magnitude in training speed.
    runtime_fig = px.scatter(
        family_screen_summary_df,
        x="median_fit_seconds",
        y="median_skill_vs_benchmark_pct",
        text="model_label",
        color="model_label",
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        log_x=True,
        labels={
            "median_fit_seconds":
                "Median fit time per screen job (seconds)",
            "median_skill_vs_benchmark_pct":
                "Median skill vs benchmark (%)",
            "model_label": "Model family",
        },
        title=(
            "Tree-Family Screening Tradeoff — "
            "Predictive Skill vs Fit Time"
        ),
    )

    runtime_fig.update_traces(
        marker={"size": 13},
        textposition="top center",
    )

    # The horizontal zero line separates families that improve on the benchmark
    # from those that are worse on median.
    runtime_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    runtime_fig.update_layout(
        showlegend=False,
        width=900,
        height=550,
        margin=dict(l=80, r=50, t=80, b=70),
    )

    apply_branding(runtime_fig)
    runtime_fig.show()

,Model family,Jobs scored,MAE wins,Median MAE rank,Beat benchmark %,Median skill %,Worst skill %,Median fit sec,Failed fits
0,Extra Trees,20,12,1.0,90.0,11.31,-7.03,208.69,0
1,Random Forest,20,6,2.0,90.0,10.97,-9.59,318.65,0
2,LightGBM,20,2,3.0,75.0,6.02,-8.27,1.96,0
3,HistGradientBoosting,20,0,4.0,60.0,4.44,-22.75,11.97,0
4,CatBoost,20,0,5.0,50.0,1.18,-16.44,12.97,0


**Findings.** Extra Trees is the strongest controlled candidate across the 20 representative forecasting jobs. It records **12 MAE wins**, a **median MAE rank of 1**, beats the frozen benchmark on **90% of jobs**, and delivers **11.31% median skill** with a worst case of **−7.03%**. Random Forest is the closest competitor, with 6 wins, the same 90% benchmark-beat rate, and 10.97% median skill, but it is slower and has a somewhat weaker worst case. LightGBM is by far the fastest candidate—about **1.96 seconds per median fit** versus 208.69 seconds for Extra Trees—but its 6.02% median skill and two wins show a meaningful accuracy tradeoff. HistGradientBoosting and CatBoost are weaker across both rank and benchmark breadth, while all five families remain operationally reliable with zero failed fits. The screen therefore favors Extra Trees on predictive performance rather than merely on isolated wins, while making the computational advantage of boosting explicit for the next human decision.

### Compare the PyCaret evidence with the controlled roster

PyCaret's three sentinel leaderboards answer a narrower question than the controlled screen: **does an independently assembled comparison point toward the same strong model families, or expose a candidate we should not prune yet?**

The table and heatmap below remain separate from the 20-job controlled rankings because the experiments have different breadth. Their role is triangulation, not numerical pooling.

In [19]:
# ---------------------------------------------------------------------
# 4.4.1.3G — Consolidate and visualize the independent PyCaret evidence
# ---------------------------------------------------------------------

# The PyCaret cross-check is only interpretable after at least one sentinel has
# produced saved model results. Fail explicitly rather than presenting an empty
# leaderboard as if the independent check had completed.
if pycaret_screen_results_df.empty:
    raise RuntimeError(
        "Run or load the PyCaret screen before synthesizing its evidence."
    )

# Architecture evidence comes from successfully scored models only. Any failed
# sentinel runs remain preserved in the checkpoint surface and are summarized
# operationally in the preceding completion table.
pycaret_success_df = pycaret_screen_results_df.loc[
    pycaret_screen_results_df["status"].eq("success")
].copy()

if pycaret_success_df.empty:
    raise RuntimeError(
        "The PyCaret cross-check contains no successful model results."
    )


# ---------------------------------------------------------------------
# Compare models only within the same sentinel forecasting problem
# ---------------------------------------------------------------------

# Raw MAE cannot be compared directly across Taxi trips, Taxi speed, and Subway
# ridership because those targets use different physical units. Rank candidate
# models only against one another within the same sentinel.
pycaret_success_df["mae_rank"] = (
    pycaret_success_df
    .groupby("job_id")["mae"]
    .rank(method="min", ascending=True)
)

# A rank of 1 identifies the lowest-MAE model on that sentinel. True numerical
# ties are retained as shared wins rather than being broken arbitrarily.
pycaret_success_df["is_mae_winner"] = (
    pycaret_success_df["mae_rank"].eq(1.0)
)

# Positive skill means the PyCaret model improved on the already-frozen
# benchmark using the same validation rows.
pycaret_success_df["beats_benchmark"] = (
    pycaret_success_df["skill_vs_benchmark_pct"].gt(0)
)


# ---------------------------------------------------------------------
# Reduce the sentinel leaderboards to one summary row per PyCaret model
# ---------------------------------------------------------------------

# This summary asks whether the independent AutoML check tells the same broad
# story as the controlled family screen:
# - how often each model wins on MAE,
# - its typical within-sentinel rank,
# - how often it beats the frozen benchmark,
# - its typical and weakest benchmark-relative skill.
#
# Because only three sentinels are used, these results are corroborating
# evidence rather than a replacement for the broader 20-job controlled screen.
pycaret_summary_df = (
    pycaret_success_df
    .groupby(
        ["pycaret_model_id", "pycaret_model_label"],
        as_index=False,
    )
    .agg(
        sentinel_jobs=("job_id", "nunique"),
        mae_wins=("is_mae_winner", "sum"),
        median_mae_rank=("mae_rank", "median"),
        jobs_beating_benchmark=("beats_benchmark", "sum"),
        median_skill_vs_benchmark_pct=(
            "skill_vs_benchmark_pct",
            "median",
        ),
        worst_skill_vs_benchmark_pct=(
            "skill_vs_benchmark_pct",
            "min",
        ),
    )
)

# Convert benchmark wins to a percentage so the summary remains interpretable
# even if a future PyCaret run contains an unequal number of completed
# sentinels across models.
pycaret_summary_df["jobs_beating_benchmark_pct"] = (
    100.0
    * pycaret_summary_df["jobs_beating_benchmark"]
    / pycaret_summary_df["sentinel_jobs"]
)

# Order the table by broad predictive performance: lower median MAE rank first,
# then stronger typical benchmark skill. This is descriptive ordering, not an
# automatic model-selection rule.
pycaret_summary_df = (
    pycaret_summary_df
    .sort_values(
        [
            "median_mae_rank",
            "median_skill_vs_benchmark_pct",
        ],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Reader-facing PyCaret summary
# ---------------------------------------------------------------------

# Keep the visible table compact. The goal is to show whether the independent
# three-sentinel comparison reinforces or challenges the controlled family
# screen, not to reproduce the full PyCaret leaderboard output.
pycaret_display_df = (
    pycaret_summary_df[
        [
            "pycaret_model_label",
            "sentinel_jobs",
            "mae_wins",
            "median_mae_rank",
            "jobs_beating_benchmark_pct",
            "median_skill_vs_benchmark_pct",
            "worst_skill_vs_benchmark_pct",
        ]
    ]
    .rename(
        columns={
            "pycaret_model_label": "PyCaret model",
            "sentinel_jobs": "Sentinels scored",
            "mae_wins": "MAE wins",
            "median_mae_rank": "Median MAE rank",
            "jobs_beating_benchmark_pct": "Beat benchmark %",
            "median_skill_vs_benchmark_pct": "Median skill %",
            "worst_skill_vs_benchmark_pct": "Worst skill %",
        }
    )
)

for column in [
    "Median MAE rank",
    "Beat benchmark %",
    "Median skill %",
    "Worst skill %",
]:
    pycaret_display_df[column] = (
        pycaret_display_df[column].round(2)
    )

display(pycaret_display_df)


# ---------------------------------------------------------------------
# Visual — Show the three sentinel results without hiding disagreement
# ---------------------------------------------------------------------

if RUN_VISUALS:
    # Human-readable labels make it possible to see exactly which Track ×
    # Target × Horizon sentinel each model is succeeding or struggling on.
    pycaret_success_df["job_label"] = pycaret_success_df.apply(
        lambda row: (
            f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[row['metric']]} · "
            f"h={int(row['horizon'])}"
        ),
        axis=1,
    )

    # Preserve the deliberate sentinel order defined earlier in the notebook
    # instead of accepting the arbitrary alphabetical order produced by pivot().
    pycaret_job_labels = [
        (
            f"{TRACK_SHORT_LABELS[row.modeling_track]} · "
            f"{TARGET_SHORT_LABELS[row.metric]} · h={int(row.horizon)}"
        )
        for row in pycaret_screen_jobs_df.itertuples(index=False)
    ]

    # Rows are PyCaret models, columns are the three sentinel forecasting jobs,
    # and each cell reports percentage MAE skill relative to the frozen
    # benchmark. The heatmap therefore exposes isolated wins or weak spots that
    # a three-number summary could conceal.
    pycaret_heatmap_df = (
        pycaret_success_df
        .pivot(
            index="pycaret_model_label",
            columns="job_label",
            values="skill_vs_benchmark_pct",
        )
        .reindex(columns=pycaret_job_labels)
    )

    # Center the diverging scale at zero so the main interpretation is visual:
    # positive cells beat the benchmark; negative cells trail it.
    pycaret_heatmap_fig = go.Figure(
        data=go.Heatmap(
            z=pycaret_heatmap_df.to_numpy(),
            x=pycaret_heatmap_df.columns.tolist(),
            y=pycaret_heatmap_df.index.tolist(),
            colorscale=BRAND_DIVERGING_SEQUENCE,
            zmid=0,
            colorbar={"title": "Skill vs<br>benchmark (%)"},
            hovertemplate=(
                "Model: %{y}<br>"
                "Sentinel: %{x}<br>"
                "Skill vs benchmark: %{z:.2f}%"
                "<extra></extra>"
            ),
        )
    )

    pycaret_heatmap_fig.update_layout(
        title=(
            "Independent PyCaret Cross-Check — "
            "Skill Relative to the Frozen Benchmark"
        ),
        xaxis_title=None,
        yaxis_title=None,
        width=950,
        height=500,
        margin=dict(l=190, r=80, t=80, b=130),
    )
    pycaret_heatmap_fig.update_xaxes(tickangle=-25)

    apply_branding(pycaret_heatmap_fig)
    pycaret_heatmap_fig.show()

,PyCaret model,Sentinels scored,MAE wins,Median MAE rank,Beat benchmark %,Median skill %,Worst skill %
0,Extra Trees Regressor,3,2,1.0,100.00,13.07,5.13
1,Random Forest Regressor,3,0,2.0,66.67,6.20,-1.93
2,CatBoost Regressor,3,1,3.0,33.33,-1.96,-2.62
3,Light Gradient Boosting Machine,3,0,4.0,33.33,-7.74,-12.84
4,Extreme Gradient Boosting,3,0,5.0,33.33,-11.32,-12.14
5,Ridge Regression,3,0,6.0,33.33,-45.85,-103.99


**Findings.** PyCaret independently reinforces the controlled-screen result. Extra Trees ranks first across the three sentinels, wins **2 of 3**, beats the frozen benchmark on **all three**, and posts **13.07% median skill** with even its worst sentinel remaining positive at **5.13%**. Random Forest again ranks second but provides weaker benchmark breadth and no sentinel wins. CatBoost wins one sentinel but does not show broad enough performance to overturn its weak controlled-screen ranking. Most importantly, **XGBoost does not reveal a blind spot in the controlled roster**: it records no wins, a median rank of 5, and negative median skill of −11.32%, so there is no evidence to add it before pruning. The Ridge reference is substantially less competitive, confirming that the nonlinear tree ensembles are providing material predictive value on these sentinel problems.

> ✅ Decision — Carry Extra Trees as the accuracy leader and LightGBM as the production\-efficiency challenger

Extra Trees is the strongest model in the bounded architecture screen and remains the accuracy\-first candidate\. LightGBM is less accurate on this screen, but its median fitting time is approximately two orders of magnitude lower while it still improves on the frozen benchmark across most representative jobs\. The next stages will therefore evaluate whether Extra Trees' additional accuracy remains operationally feasible at larger training scale\. If production\-scale fitting cost becomes disproportionate, LightGBM provides the already\-screened fallback rather than reopening the full family roster\.

🌲 Why did Extra Trees lead the accuracy screen? Extra Trees is short for Extremely Randomized Trees\. Like Random Forest, it does not trust a single decision tree: it builds many different trees and averages their forecasts\. The important difference is how much randomness it deliberately introduces while building them\.

A Random Forest typically gives each tree a somewhat different sample of the training data and, at each node, considers a random subset of predictors—but it still searches those predictors for relatively good split points\. Extra Trees goes further: for the candidate predictors, it proposes random split thresholds and chooses among those randomized possibilities\. In effect, Random Forest says, “among these randomly selected features, find a strong split,” while Extra Trees says, “also randomize where we consider splitting them\.”

That sounds like it should make the model worse, but the ensemble is the key\. Any individual Extra Tree may be somewhat noisy, yet the trees make different errors\. Averaging many deliberately diverse trees can cancel much of that noise and reduce the tendency to overfit peculiarities of the training data\. The tradeoff is roughly a little more bias in each tree in exchange for less variance in the ensemble\. Random split generation can also make Extra Trees faster than Random Forest because it performs less exhaustive searching for the optimal threshold at every node\.

🧠 What it means for this forecasting problem\. Extra Trees is not a time\-series model that internally “remembers” previous observations\. The temporal information has already been expressed through our leakage\-safe lag, seasonal, calendar, policy, multimodal, weather, spatial, and static features\. Extra Trees then learns nonlinear rules and interactions across those predictors—for example, whether the meaning of recent Taxi demand differs by daypart, weather, surrounding network conditions, or Taxi Zone identity\. Because one global model pools examples across many zones, the ensemble can learn patterns shared across the city while its randomized trees are less dependent on any single set of zone\-specific quirks\.

Intuition: a single tree is one opinion; a Random Forest is a committee whose members see somewhat different evidence; Extra Trees is a committee whose members are also forced to consider somewhat different decision boundaries\. Our screening evidence suggests that this additional diversity is useful for the NYC mobility forecasting panel\.

<img src="image-20260906-004829.png" width="" align="" />

## 4.4.1.4 Test Extra Trees Refit Cadence and Operational Cost

With Extra Trees established as the accuracy leader in the controlled family screen, the next question is operational:

> **Does Extra Trees benefit enough from learning newly observed validation-period outcomes to justify repeated refitting, and what does that cadence cost?**

Two cadence policies are compared while holding the model definition, predictors, evaluation rows, and frozen benchmark constant.

**Validation-start only** fits Extra Trees once at the first legal rolling-validation forecast origin. Its parameters then remain fixed throughout validation, although the leakage-safe predictor values supplied at each later forecast origin continue to update normally.

**Monthly refresh** begins from the same first fit, then rebuilds Extra Trees at the first available forecast origin of each new calendar month using expanding legal history. Outcomes enter training only after their target observation would already have been observed.

The comparison stays deliberately small. It uses three contrasting sentinel jobs: Post-CP Taxi trips at h=1, Post-CP Taxi speed at h=5, and Full-History Subway ridership at h=5. The same 250,000-row origin-balanced training cap used for architecture screening keeps this cadence test bounded; it is not a production training-size decision.

This experiment therefore answers two questions at once: whether repeated retraining improves Extra Trees accuracy, and whether its fitting cost remains reasonable enough to support that cadence at larger scale.

The final holdout remains untouched.

============================================================

<img src="image-20260906-031126.png" width="" align="" />

In [20]:
# ---------------------------------------------------------------------
# 4.4.1.4A — Resolve the cadence sentinels, schedules, and workload
# ---------------------------------------------------------------------

# The architecture decision is already frozen: only Extra Trees advances into
# the cadence experiment. These assertions guard against accidentally running
# this section with a different model family or a missing screen definition.
assert ADVANCING_TREE_MODEL_ID == "extra_trees"
assert ADVANCING_TREE_MODEL_ID in FAMILY_SCREEN_CONFIGS


# ---------------------------------------------------------------------
# Resolve the three representative cadence jobs
# ---------------------------------------------------------------------

# panel_tree_scope_df contains the inherited production job universe. Indexing
# by Track × Target × Horizon lets us pull the exact sentinel definitions
# without rebuilding or manually duplicating their metadata.
scope_lookup = panel_tree_scope_df.set_index(
    ["modeling_track", "metric", "horizon"]
)

cadence_jobs_df = scope_lookup.loc[
    CADENCE_SENTINEL_KEYS
].reset_index()

# Normalize horizon to a stable integer representation and create one compact
# job identifier that can be reused in schedules, checkpoints, and summaries.
cadence_jobs_df["horizon"] = cadence_jobs_df["horizon"].astype(int)
cadence_jobs_df["job_id"] = (
    cadence_jobs_df["modeling_track"]
    + "__"
    + cadence_jobs_df["metric"]
    + "__h"
    + cadence_jobs_df["horizon"].astype(str)
)

# The experiment contract expects exactly the requested sentinels, with no
# accidental duplication after the scope lookup.
assert len(cadence_jobs_df) == len(CADENCE_SENTINEL_KEYS)
assert cadence_jobs_df["job_id"].is_unique


def build_cadence_schedule(origin_surface_df, cadence):
    """Return the Extra Trees refit origins for one cadence policy.

    Both policies begin at the first legal rolling-validation forecast origin.

    - validation_start_only: fit once and reuse that model throughout validation.
    - monthly: fit at validation start, then refresh at the first available
      forecast origin in each new calendar month.

    The schedule is based only on forecast origins, not outcomes, so deciding
    when a refit occurs cannot leak future target information.
    """
    # Reduce the validation surface to one row per canonical forecast origin.
    # Sorting by observation-sequence ID preserves the project's within-day
    # ordering across canonical dayparts.
    origins = (
        origin_surface_df[
            ["origin_observation_sequence_id", "origin_date"]
        ]
        .drop_duplicates()
        .sort_values("origin_observation_sequence_id")
        .reset_index(drop=True)
    )

    if origins.empty:
        raise ValueError(
            "Cadence scheduling received an empty origin surface."
        )

    # One canonical origin must correspond to exactly one calendar date.
    # Violating this would make monthly boundaries ambiguous.
    if (
        origins
        .groupby("origin_observation_sequence_id")["origin_date"]
        .nunique()
        .gt(1)
        .any()
    ):
        raise ValueError(
            "A validation origin maps to more than one origin_date."
        )

    if cadence == "validation_start_only":
        # The fixed policy fits once at the first legal validation origin.
        refits = origins.iloc[[0]].copy()

    elif cadence == "monthly":
        # Monthly refreshing uses the first available canonical forecast origin
        # in each month. If the first calendar day has no usable origin, the
        # schedule naturally advances to the first one that actually exists.
        origins["origin_month"] = origins["origin_date"].dt.to_period("M")

        refits = (
            origins
            .groupby("origin_month", sort=True, as_index=False)
            .first()
            .drop(columns="origin_month")
        )

    else:
        raise ValueError(
            f"Unknown cadence policy: {cadence}"
        )

    # refit_step gives each model refresh a stable sequential identity for later
    # checkpointing and for assigning validation rows to the correct model.
    refits = refits.reset_index(drop=True)
    refits["refit_step"] = np.arange(len(refits), dtype=int)
    refits["cadence"] = cadence

    return refits


# ---------------------------------------------------------------------
# Build the actual fixed and monthly schedules for every sentinel
# ---------------------------------------------------------------------

# Refit timing depends only on legal rolling-validation forecast origins.
# Outcome values and final-holdout rows are unnecessary for this step and are
# intentionally not read.
cadence_schedule_parts = []

for job in cadence_jobs_df.itertuples(index=False):
    matrix_path = FORECASTING_MATRIX_PATHS[job.modeling_track]

    # origin_date is required to identify calendar-month boundaries. Fail
    # loudly if the upstream matrix contract no longer exposes it.
    if "origin_date" not in parquet_columns(matrix_path):
        raise ValueError(
            f"{matrix_path.name} does not expose origin_date, "
            "which is required for cadence scheduling."
        )

    # Read only the small set of columns needed to determine legal validation
    # origins for this one Target × Horizon sentinel.
    origin_surface_df = pd.read_parquet(
        matrix_path,
        columns=[
            "stage",
            "origin_observation_sequence_id",
            "origin_date",
            "metric",
            "horizon",
        ],
        filters=[
            ("metric", "==", job.metric),
            ("horizon", "==", int(job.horizon)),
            ("stage", "==", "rolling_validation"),
        ],
    )

    origin_surface_df["origin_date"] = pd.to_datetime(
        origin_surface_df["origin_date"]
    ).dt.normalize()

    # Build both policies from the same validation-origin surface so the only
    # difference between them is how often Extra Trees is refreshed.
    for cadence in CADENCE_POLICIES:
        schedule = build_cadence_schedule(
            origin_surface_df,
            cadence,
        )

        # Attach job identity now so all schedules can be concatenated into one
        # restartable execution plan.
        schedule.insert(0, "job_id", job.job_id)
        schedule.insert(1, "modeling_track", job.modeling_track)
        schedule.insert(2, "metric", job.metric)
        schedule.insert(3, "horizon", int(job.horizon))

        cadence_schedule_parts.append(schedule)

    del origin_surface_df
    gc.collect()


cadence_schedule_df = pd.concat(
    cadence_schedule_parts,
    ignore_index=True,
)

# Each Job × Cadence × Refit Step must identify exactly one model refresh.
assert not cadence_schedule_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any()


# ---------------------------------------------------------------------
# Reader-facing workload before committing to the long-running fits
# ---------------------------------------------------------------------

# Count how many Extra Trees fits each policy requires for each sentinel.
# The fixed policy should contribute one fit; monthly contributes one per
# calendar month represented in rolling validation.
refit_counts_df = (
    cadence_schedule_df
    .groupby(["job_id", "cadence"], as_index=False)
    .size()
    .pivot(
        index="job_id",
        columns="cadence",
        values="size",
    )
    .reset_index()
    .rename(
        columns={
            "validation_start_only": "Fixed refits",
            "monthly": "Monthly refits",
        }
    )
)

cadence_workload_df = cadence_jobs_df[
    ["job_id", "modeling_track", "metric", "horizon"]
].merge(
    refit_counts_df,
    on="job_id",
    how="left",
    validate="one_to_one",
)

# Human-readable labels keep the workload table understandable without exposing
# internal IDs to the reader.
cadence_workload_df["Sentinel"] = cadence_workload_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · "
        f"h={int(row['horizon'])}"
    ),
    axis=1,
)

cadence_workload_df["Planned fits"] = (
    cadence_workload_df["Fixed refits"]
    + cadence_workload_df["Monthly refits"]
)


# ---------------------------------------------------------------------
# Estimate runtime using measured Extra Trees screening performance
# ---------------------------------------------------------------------

# This is only an operational estimate. The cadence experiment later records
# actual fit time for each refresh. Reusing the measured architecture-screen
# median is more informative than inventing a generic timing assumption.
extra_trees_runtime_match = family_screen_summary_df.loc[
    family_screen_summary_df["model_id"].eq(
        ADVANCING_TREE_MODEL_ID
    ),
    "median_fit_seconds",
]

if len(extra_trees_runtime_match) != 1:
    raise ValueError(
        "Expected one Extra Trees runtime estimate from the family screen."
    )

extra_trees_screen_median_seconds = float(
    extra_trees_runtime_match.iloc[0]
)

cadence_workload_df["Estimated fit minutes"] = (
    cadence_workload_df["Planned fits"]
    * extra_trees_screen_median_seconds
    / 60.0
)


# ---------------------------------------------------------------------
# Keep the visible table compact and reader-facing
# ---------------------------------------------------------------------

cadence_workload_display_df = cadence_workload_df[
    [
        "Sentinel",
        "Fixed refits",
        "Monthly refits",
        "Planned fits",
        "Estimated fit minutes",
    ]
].copy()

cadence_workload_display_df["Estimated fit minutes"] = (
    cadence_workload_display_df["Estimated fit minutes"].round(1)
)

# Add one total row so the reader can see the full computational commitment
# before launching the restartable cadence experiment.
total_row = pd.DataFrame(
    [
        {
            "Sentinel": "TOTAL",
            "Fixed refits": (
                cadence_workload_display_df["Fixed refits"].sum()
            ),
            "Monthly refits": (
                cadence_workload_display_df["Monthly refits"].sum()
            ),
            "Planned fits": (
                cadence_workload_display_df["Planned fits"].sum()
            ),
            "Estimated fit minutes": (
                cadence_workload_display_df[
                    "Estimated fit minutes"
                ]
                .sum()
                .round(1)
            ),
        }
    ]
)

cadence_workload_display_df = pd.concat(
    [cadence_workload_display_df, total_row],
    ignore_index=True,
)

display(cadence_workload_display_df)

,Sentinel,Fixed refits,Monthly refits,Planned fits,Estimated fit minutes
0,Post-CP · Taxi trips · h=1,1,10,11,38.3
1,Post-CP · Taxi speed · h=5,1,10,11,38.3
2,Full-History · Subway · h=5,1,10,11,38.3
3,TOTAL,3,30,33,114.9


**Findings.** The cadence experiment resolves to three deliberately contrasting sentinel jobs: Post-CP Taxi trips at h=1, Post-CP Taxi speed at h=5, and Full-History Subway ridership at h=5. Each sentinel requires one validation-start-only fit and 10 monthly fits, producing **33 planned Extra Trees fits in total**. Using the measured Extra Trees screening runtime, the pre-run estimate was approximately **114.9 minutes**. This keeps the cadence question tightly bounded while still testing short- and longer-horizon forecasts, demand and speed behavior, both modeling tracks, and more than one transport mode.

### Preserve exact metrics across multiple refit windows

Monthly refreshing divides rolling validation into several forecast segments, one for each model refresh. Those segments are not all the same size, so simply averaging their MAEs would give a short segment the same weight as a long one.

Instead, each segment stores the underlying error totals — absolute error, squared error, signed error, and the number of evaluated rows. Those totals can later be added together and converted back into exact full-period MAE, RMSE, NMAE, and bias.

This keeps the final cadence comparison mathematically equivalent to scoring all validation predictions at once.

In [21]:
# ---------------------------------------------------------------------
# 4.4.1.4B — Preserve exact metrics across refit windows
# ---------------------------------------------------------------------

def regression_metric_sums(y_true, y_pred, prefix):
    """Store additive error totals that can be recombined across refit windows.

    Monthly refreshing creates several differently sized validation segments.
    Saving error sums rather than segment-level averages lets the notebook
    reconstruct exact full-period MAE, RMSE, NMAE, and bias later.
    """
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)

    # Evaluate only rows where both the observed value and prediction are
    # finite. The same support rule is applied to model and benchmark results.
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        return {
            f"{prefix}_evaluated_rows": 0,
            f"{prefix}_absolute_error_sum": 0.0,
            f"{prefix}_squared_error_sum": 0.0,
            f"{prefix}_error_sum": 0.0,
            f"{prefix}_absolute_actual_sum": 0.0,
        }

    error = predicted - actual

    # Each quantity below is additive across refit windows. The later synthesis
    # cell sums them first, then computes the final validation-period metrics.
    return {
        f"{prefix}_evaluated_rows": int(len(actual)),
        f"{prefix}_absolute_error_sum": float(np.abs(error).sum()),
        f"{prefix}_squared_error_sum": float(np.square(error).sum()),
        f"{prefix}_error_sum": float(error.sum()),
        f"{prefix}_absolute_actual_sum": float(np.abs(actual).sum()),
    }

### Load one cadence job on the same benchmark support

Each cadence sentinel needs two related data surfaces.

The first is the **legal history** that may become training evidence as validation moves forward. This includes initial-training rows plus rolling-validation rows whose outcomes may become known after earlier forecast origins have passed.

The second is the **evaluation surface**: rolling-validation forecasts that can be matched exactly to the already-frozen baseline. Extra Trees and the benchmark must be judged on the same forecast rows, otherwise a difference in MAE could come from different coverage rather than better predictions.

Final-holdout rows are not loaded.

In [22]:
# ---------------------------------------------------------------------
# 4.4.1.4C — Load one cadence job on exact benchmark support
# ---------------------------------------------------------------------

def load_cadence_job(track, metric, horizon):
    """Load one sentinel's legal history and exact benchmark evaluation surface.

    The returned payload contains:
    - all initial-training and rolling-validation rows that may become legal
      training evidence as the simulated forecast origin advances; and
    - rolling-validation rows shared with the frozen benchmark.

    Final-holdout rows are excluded at the Parquet read boundary.
    """
    matrix_path = FORECASTING_MATRIX_PATHS[track]
    predictor_names = predictor_names_by_target[metric]
    categorical_names = categorical_predictors_by_target[metric]

    # Load only the metadata, target, and predictors needed for this one
    # Target × Horizon cadence job rather than the full forecasting matrix.
    required_columns = list(
        dict.fromkeys(
            [
                "modeling_track",
                "stage",
                "forecast_sequence_id",
                "taxi_zone_id",
                "metric",
                "origin_observation_sequence_id",
                "origin_date",
                "target_observation_sequence_id",
                "horizon",
                "target_date",
                "target_observed_value",
                "target_label_available",
            ]
            + predictor_names
        )
    )

    # Treat the upstream matrix schema as a contract. Missing columns should
    # fail loudly rather than being guessed or reconstructed in this notebook.
    missing_columns = sorted(
        set(required_columns) - set(parquet_columns(matrix_path))
    )

    if missing_columns:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            f"matrix is missing {missing_columns}"
        )

    # Only initial-training and rolling-validation rows are relevant here.
    # The final holdout remains outside the cadence experiment entirely.
    matrix_df = pd.read_parquet(
        matrix_path,
        columns=required_columns,
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("stage", "in", ["initial_training", "rolling_validation"]),
        ],
    )

    if not matrix_df["modeling_track"].eq(track).all():
        raise ValueError(
            f"{track} · {metric} · h={horizon}: matrix returned another track."
        )

    # Calendar dates support monthly scheduling, while canonical sequence IDs
    # preserve the exact within-day ordering of forecast and target positions.
    matrix_df["origin_date"] = pd.to_datetime(
        matrix_df["origin_date"]
    ).dt.normalize()

    matrix_df["target_date"] = pd.to_datetime(
        matrix_df["target_date"]
    ).dt.normalize()

    matrix_df["origin_observation_sequence_id"] = pd.to_numeric(
        matrix_df["origin_observation_sequence_id"],
        errors="raise",
    ).astype("int64")

    matrix_df["target_observation_sequence_id"] = pd.to_numeric(
        matrix_df["target_observation_sequence_id"],
        errors="raise",
    ).astype("int64")

    # A row can become training evidence only when its target is both marked
    # available and actually observed. Whether it is legal at a particular
    # refit origin is determined later from its target sequence position.
    matrix_df["_label_available"] = (
        matrix_df["target_label_available"].fillna(False).astype(bool)
        & matrix_df["target_observed_value"].notna()
    )

    # Evaluation uses observed rolling-validation outcomes only.
    validation_df = matrix_df.loc[
        matrix_df["stage"].eq("rolling_validation")
        & matrix_df["_label_available"]
    ].copy()

    if validation_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: empty validation surface."
        )


    # -----------------------------------------------------------------
    # Align validation to the already-frozen benchmark
    # -----------------------------------------------------------------

    # The baseline winner was selected upstream. This experiment compares
    # cadence policies, so it reuses that frozen benchmark rather than reopening
    # baseline selection.
    benchmark_info = frozen_benchmark_row(
        track,
        metric,
        horizon,
    )

    benchmark_id = str(benchmark_info["baseline_id"])
    benchmark_label = str(benchmark_info["baseline_label"])
    benchmark_column = f"baseline__{benchmark_id}"

    baseline_df = pd.read_parquet(
        BASELINE_PREDICTION_PATHS[track],
        columns=[
            "stage",
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
            "horizon",
            "metric",
            "target_observed_value",
            "target_label_available",
            benchmark_column,
        ],
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("stage", "==", "rolling_validation"),
        ],
    )

    baseline_df = baseline_df.loc[
        baseline_df["target_label_available"].fillna(False).astype(bool)
        & baseline_df["target_observed_value"].notna()
        & baseline_df[benchmark_column].notna()
    ].copy()

    # These keys identify the same individual forecast opportunity in both
    # the model matrix and the frozen baseline prediction surface.
    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
    ]

    if validation_df.duplicated(join_keys).any():
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "validation keys are not unique."
        )

    if baseline_df.duplicated(join_keys).any():
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "benchmark keys are not unique."
        )

    baseline_df = baseline_df[
        join_keys + ["target_observed_value", benchmark_column]
    ].rename(
        columns={
            "target_observed_value": "benchmark_target_observed_value"
        }
    )

    # Inner joining forces model and benchmark onto exact common support.
    validation_df = validation_df.merge(
        baseline_df,
        on=join_keys,
        how="inner",
        validate="one_to_one",
    )

    del baseline_df
    gc.collect()

    if validation_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "no common model/benchmark validation rows."
        )

    # Matching keys are not enough. Verify that both sources point to the same
    # observed target value before comparing their forecast errors.
    if not np.allclose(
        validation_df["target_observed_value"].to_numpy(dtype=float),
        validation_df[
            "benchmark_target_observed_value"
        ].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "matrix and benchmark targets disagree."
        )

    # Keep the full legal-history matrix because each later refit has a
    # different temporal cutoff. The same benchmark-aligned validation surface
    # can be reused by both cadence policies.
    return {
        "matrix_df": matrix_df,
        "validation_df": validation_df,
        "predictor_names": predictor_names,
        "categorical_names": categorical_names,
        "benchmark_id": benchmark_id,
        "benchmark_label": benchmark_label,
        "benchmark_column": benchmark_column,
    }

### Fit one model version and score the forecasts it owns

A refit cadence creates one or more versions of the same Extra Trees model.

Under the **validation-start-only** policy, there is one model version and it is responsible for the entire rolling-validation period.

Under the **monthly refresh** policy, each refreshed model is responsible only for forecasts from its refit origin up to, but not including, the next refresh.

At every refit, training expands only through outcomes that would already have been observed by that exact canonical sequence position. This is stricter than using calendar dates alone because several ordered forecast positions can occur within the same day.

In [23]:
# ---------------------------------------------------------------------
# 4.4.1.4D — Fit and score one legal refit window
# ---------------------------------------------------------------------

def fit_cadence_refit_segment(
    payload,
    policy_schedule_df,
    refit_step,
):
    """Fit Extra Trees once and score the validation window owned by that refit.

    A fixed cadence has one refit and therefore one model version covering all
    validation origins. Monthly cadence has several model versions, each
    responsible for forecasts until immediately before the next refresh.
    """
    # Sort explicitly so the next scheduled refit always has a clear temporal
    # meaning even if the input frame arrives in a different row order.
    schedule = (
        policy_schedule_df
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    row = schedule.loc[
        schedule["refit_step"].eq(int(refit_step))
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one cadence row for refit step {refit_step}; "
            f"found {len(row)}."
        )

    row = row.iloc[0]

    # The refit origin acts as simulated "current time." No outcome whose target
    # sequence position lies after this point may enter the training surface.
    refit_origin_id = int(
        row["origin_observation_sequence_id"]
    )
    refit_origin_date = pd.Timestamp(
        row["origin_date"]
    )

    # The next scheduled refresh defines the end of this model version's
    # scoring responsibility. The last monthly model, and the single fixed
    # model, have no later boundary.
    later_refits = schedule.loc[
        schedule["refit_step"].gt(int(refit_step)),
        "origin_observation_sequence_id",
    ]

    next_refit_origin_id = (
        int(later_refits.iloc[0])
        if len(later_refits)
        else None
    )

    matrix_df = payload["matrix_df"]
    validation_df = payload["validation_df"]


    # -----------------------------------------------------------------
    # Build the legal expanding-history training surface
    # -----------------------------------------------------------------

    # A row becomes legal training evidence only after its target observation
    # has already occurred by the refit origin. Sequence IDs preserve the
    # canonical ordering across multiple forecast positions on the same day.
    training_mask = (
        matrix_df["_label_available"]
        & matrix_df[
            "target_observation_sequence_id"
        ].le(refit_origin_id)
    )

    train_df = matrix_df.loc[
        training_mask
    ].copy()

    if train_df.empty:
        raise ValueError(
            f"Refit {refit_step} has no legal training rows."
        )

    # Keep the uncapped row count for auditability. The cadence experiment still
    # uses the bounded training cap so the question remains refit frequency,
    # not training-surface size.
    uncapped_train_rows = len(train_df)

    train_df = thin_training_by_origin(
        train_df,
        max_rows=CADENCE_TRAIN_ROW_CAP,
    )


    # -----------------------------------------------------------------
    # Assign this model version its validation forecast window
    # -----------------------------------------------------------------

    # Begin scoring at the refit origin itself. For monthly cadence, stop just
    # before the next refresh so every validation origin belongs to exactly one
    # model version.
    score_mask = validation_df[
        "origin_observation_sequence_id"
    ].ge(refit_origin_id)

    if next_refit_origin_id is not None:
        score_mask &= validation_df[
            "origin_observation_sequence_id"
        ].lt(next_refit_origin_id)

    score_df = validation_df.loc[
        score_mask
    ].copy()

    if score_df.empty:
        raise ValueError(
            f"Refit {refit_step} has no validation rows assigned to it."
        )


    # -----------------------------------------------------------------
    # Fit the frozen Extra Trees definition
    # -----------------------------------------------------------------

    # Cadence is the only experimental variable here. Predictor handling and
    # Extra Trees model capacity remain frozen from the architecture screen.
    X_train = prepare_tree_dtypes(
        train_df[payload["predictor_names"]],
        payload["categorical_names"],
    )

    X_score = prepare_tree_dtypes(
        score_df[payload["predictor_names"]],
        payload["categorical_names"],
    )

    y_train = pd.to_numeric(
        train_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    y_score = pd.to_numeric(
        score_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    predictions, fit_seconds, predict_seconds = fit_screen_candidate(
        ADVANCING_TREE_MODEL_ID,
        FAMILY_SCREEN_CONFIGS[ADVANCING_TREE_MODEL_ID],
        X_train,
        y_train,
        X_score,
        payload["predictor_names"],
        payload["categorical_names"],
    )


    # -----------------------------------------------------------------
    # Store additive model and benchmark errors on identical support
    # -----------------------------------------------------------------

    # Store underlying error totals rather than only segment-level metrics so
    # monthly windows can later be recombined into exact full-validation scores.
    model_stats = regression_metric_sums(
        y_score,
        predictions,
        prefix="model",
    )

    benchmark_stats = regression_metric_sums(
        y_score,
        score_df[payload["benchmark_column"]],
        prefix="benchmark",
    )

    if (
        model_stats["model_evaluated_rows"]
        != benchmark_stats["benchmark_evaluated_rows"]
    ):
        raise ValueError(
            "Model and benchmark were not evaluated on identical row support."
        )


    # -----------------------------------------------------------------
    # Return one self-contained checkpoint record
    # -----------------------------------------------------------------

    result = {
        "status": "success",
        "cadence_contract_id": CADENCE_CONTRACT_ID,
        "cadence": str(row["cadence"]),
        "refit_step": int(refit_step),

        # Temporal identity of this model version.
        "refit_origin_id": refit_origin_id,
        "refit_origin_date": refit_origin_date,
        "next_refit_origin_id": next_refit_origin_id,

        # Training support before and after the bounded cadence cap.
        "train_rows_uncapped": int(uncapped_train_rows),
        "train_rows": int(len(train_df)),
        "train_origins": int(
            train_df["origin_observation_sequence_id"].nunique()
        ),

        # Validation region owned by this model version.
        "score_rows": int(len(score_df)),
        "score_origin_start": int(
            score_df["origin_observation_sequence_id"].min()
        ),
        "score_origin_end": int(
            score_df["origin_observation_sequence_id"].max()
        ),

        # Preserve unrealistic forecasts rather than silently clipping them.
        "negative_prediction_count": int(
            (predictions < 0).sum()
        ),

        # Runtime is part of the cadence tradeoff because monthly refreshing
        # repeats the same model-fitting process many more times.
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),

        # Keep benchmark identity attached to every checkpoint row.
        "benchmark_id": payload["benchmark_id"],
        "benchmark_label": payload["benchmark_label"],

        **model_stats,
        **benchmark_stats,
    }

    # Release the large per-refit frames before the next model version is fit.
    del (
        train_df,
        score_df,
        X_train,
        X_score,
        y_train,
        y_score,
        predictions,
    )
    gc.collect()

    return result

### Run, recover, or load the cadence experiment

The helper above defines how **one legal refit window** is trained and scored. This block applies that logic across the complete cadence schedule for the three sentinel forecasting jobs.

Each planned **Sentinel × Cadence × Refit Step** is a durable logical experiment unit under the frozen cadence contract. Completion is determined from that logical identity and the advancing Extra Trees model definition stored inside the checkpoint, not from the checkpoint filename.

A complete consolidated intermediate is the normal fast path. If it is absent or incomplete, the notebook reconstructs the cadence experiment from durable checkpoints and computes only unfinished refit windows. `RUN_CADENCE_SCREEN=True` forces that reconciliation path; `RESET_CADENCE_SCREEN_CHECKPOINTS=True` is the only control that deliberately discards completed cadence evidence.

In [24]:
# ---------------------------------------------------------------------
# 4.4.1.4D.1 — Execute or recover the Extra Trees cadence experiment
# ---------------------------------------------------------------------

CADENCE_RESULT_KEY_COLUMNS = ["job_id", "cadence", "refit_step"]

cadence_expected_keys = set(
    cadence_schedule_df[CADENCE_RESULT_KEY_COLUMNS].itertuples(
        index=False,
        name=None,
    )
)
expected_cadence_fits = len(cadence_schedule_df)

if len(cadence_expected_keys) != expected_cadence_fits:
    raise ValueError(
        "Cadence schedule is not unique by Job × Cadence × Refit Step."
    )


def filter_current_cadence_model(df):
    """Keep only results for the Extra Trees definition advanced in Section 3."""
    if df.empty or "model_id" not in df.columns:
        return df.iloc[0:0].copy()

    return df.loc[df["model_id"].eq(ADVANCING_TREE_MODEL_ID)].copy()


# ---------------------------------------------------------------------
# Reset is the only destructive control
# ---------------------------------------------------------------------

if RESET_CADENCE_SCREEN_CHECKPOINTS:
    reset_checkpoint_dir(CADENCE_SCREEN_PARTS_DIR)
    CADENCE_SCREEN_RESULTS_PATH.unlink(missing_ok=True)

CADENCE_SCREEN_PARTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Prefer the exact completed consolidated experiment
# ---------------------------------------------------------------------

cadence_screen_results_df = None

if not RUN_CADENCE_SCREEN:
    cadence_screen_results_df = load_complete_consolidated_result(
        CADENCE_SCREEN_RESULTS_PATH,
        contract_column="cadence_contract_id",
        contract_id=CADENCE_CONTRACT_ID,
        key_columns=CADENCE_RESULT_KEY_COLUMNS,
        expected_keys=cadence_expected_keys,
        row_filter=filter_current_cadence_model,
    )

if cadence_screen_results_df is not None:
    print(
        f"Cadence experiment complete: loaded "
        f"{len(cadence_screen_results_df):,} successful refits from "
        f"{CADENCE_SCREEN_RESULTS_PATH.name}."
    )

else:
    # -----------------------------------------------------------------
    # Recover successful refits from checkpoint contents by logical key
    # -----------------------------------------------------------------

    cadence_checkpoint_df = canonical_checkpoint_rows(
        CADENCE_SCREEN_PARTS_DIR,
        contract_column="cadence_contract_id",
        contract_id=CADENCE_CONTRACT_ID,
        key_columns=CADENCE_RESULT_KEY_COLUMNS,
        row_filter=filter_current_cadence_model,
    )

    successful_cadence_df = (
        cadence_checkpoint_df.loc[cadence_checkpoint_df["status"].eq("success")].copy()
        if not cadence_checkpoint_df.empty
        else pd.DataFrame()
    )

    completed_cadence_keys = logical_key_set(
        successful_cadence_df,
        CADENCE_RESULT_KEY_COLUMNS,
    )
    missing_cadence_keys = cadence_expected_keys - completed_cadence_keys

    print(
        f"Cadence restart state: {len(completed_cadence_keys):,}/"
        f"{expected_cadence_fits:,} successful refits; "
        f"{len(missing_cadence_keys):,} unfinished."
    )

    if missing_cadence_keys and not WRITE_CHECKPOINTS:
        raise ValueError(
            "The cadence experiment has unfinished refits and requires "
            "WRITE_CHECKPOINTS=True."
        )


    # -----------------------------------------------------------------
    # Process only unfinished legal refit windows
    # -----------------------------------------------------------------

    if missing_cadence_keys:
        from tqdm import tqdm as tqdm_text

        cadence_progress = tqdm_text(
            total=expected_cadence_fits,
            initial=len(completed_cadence_keys),
            desc="Extra Trees cadence refits",
            unit="refit",
            dynamic_ncols=True,
        )

        active_job_id = None
        payload = None
        failed_this_pass = 0

        for schedule_row in cadence_schedule_df.itertuples(index=False):
            job_id = schedule_row.job_id
            cadence = schedule_row.cadence
            refit_step = int(schedule_row.refit_step)
            logical_key = (job_id, cadence, refit_step)

            if logical_key not in missing_cadence_keys:
                continue

            job_match = cadence_jobs_df.loc[cadence_jobs_df["job_id"].eq(job_id)]

            if len(job_match) != 1:
                raise ValueError(
                    f"Expected one cadence job definition for {job_id}; "
                    f"found {len(job_match)}."
                )

            job_row = job_match.iloc[0]

            cadence_progress.set_postfix_str(
                f"{TRACK_SHORT_LABELS[job_row['modeling_track']]} · "
                f"{TARGET_SHORT_LABELS[job_row['metric']]} · "
                f"h={int(job_row['horizon'])} · "
                f"{CADENCE_POLICY_LABELS[cadence]} · refit={refit_step} · "
                f"failures={failed_this_pass}"
            )

            # All refit windows for one sentinel use the same underlying
            # forecasting surface. Keep it resident until the sentinel changes.
            if active_job_id != job_id:
                if payload is not None:
                    del payload
                    gc.collect()

                payload = load_family_screen_job(
                    job_row["modeling_track"],
                    job_row["metric"],
                    int(job_row["horizon"]),
                    train_row_cap=CADENCE_TRAIN_ROW_CAP,
                )
                active_job_id = job_id

            policy_schedule_df = cadence_schedule_df.loc[
                cadence_schedule_df["job_id"].eq(job_id)
                & cadence_schedule_df["cadence"].eq(cadence)
            ].copy()

            checkpoint_name = slugify(
                f"{CADENCE_CONTRACT_ID}__{job_id}__"
                f"{cadence}__refit_{refit_step}"
            ) + ".parquet"
            checkpoint_path = CADENCE_SCREEN_PARTS_DIR / checkpoint_name

            try:
                result = fit_cadence_refit_segment(
                    payload=payload,
                    policy_schedule_df=policy_schedule_df,
                    refit_step=refit_step,
                )

                result.update(
                    {
                        "cadence_contract_id": CADENCE_CONTRACT_ID,
                        "model_id": ADVANCING_TREE_MODEL_ID,
                        "model_label": ADVANCING_TREE_MODEL_LABEL,
                        "job_id": job_id,
                        "modeling_track": job_row["modeling_track"],
                        "metric": job_row["metric"],
                        "horizon": int(job_row["horizon"]),
                        "cadence": cadence,
                        "refit_step": refit_step,
                        "error_type": None,
                        "error_message": None,
                    }
                )

            except Exception as exc:
                result = {
                    "status": "failed",
                    "cadence_contract_id": CADENCE_CONTRACT_ID,
                    "model_id": ADVANCING_TREE_MODEL_ID,
                    "model_label": ADVANCING_TREE_MODEL_LABEL,
                    "job_id": job_id,
                    "modeling_track": job_row["modeling_track"],
                    "metric": job_row["metric"],
                    "horizon": int(job_row["horizon"]),
                    "cadence": cadence,
                    "refit_step": refit_step,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                    "fit_seconds": np.nan,
                    "predict_seconds": np.nan,
                }
                failed_this_pass += 1

            write_checkpoint_frame(pd.DataFrame([result]), checkpoint_path)

            if result["status"] == "success":
                cadence_progress.update(1)

        if payload is not None:
            del payload

        gc.collect()
        cadence_progress.set_postfix_str(f"failures={failed_this_pass}")
        cadence_progress.close()


    # -----------------------------------------------------------------
    # Consolidate exactly one canonical row per planned cadence refit
    # -----------------------------------------------------------------

    cadence_screen_results_df = canonical_checkpoint_rows(
        CADENCE_SCREEN_PARTS_DIR,
        contract_column="cadence_contract_id",
        contract_id=CADENCE_CONTRACT_ID,
        key_columns=CADENCE_RESULT_KEY_COLUMNS,
        row_filter=filter_current_cadence_model,
    )

    if cadence_screen_results_df.empty:
        raise RuntimeError(
            "Cadence execution produced no current-contract Extra Trees results."
        )

    observed_cadence_keys = logical_key_set(
        cadence_screen_results_df,
        CADENCE_RESULT_KEY_COLUMNS,
    )

    if observed_cadence_keys != cadence_expected_keys:
        missing = cadence_expected_keys - observed_cadence_keys
        extra = observed_cadence_keys - cadence_expected_keys
        raise RuntimeError(
            "Cadence result roster does not match the planned experiment. "
            f"Missing={len(missing)}, extra={len(extra)}."
        )

    if WRITE_CONSOLIDATED_INTERMEDIATES:
        atomic_parquet_write(
            cadence_screen_results_df,
            CADENCE_SCREEN_RESULTS_PATH,
        )


# ---------------------------------------------------------------------
# Final contract QA
# ---------------------------------------------------------------------

require_columns(
    cadence_screen_results_df,
    [
        "cadence_contract_id",
        "model_id",
        "job_id",
        "cadence",
        "refit_step",
        "status",
    ],
    "Cadence-screen results",
)

if not cadence_screen_results_df["cadence_contract_id"].eq(CADENCE_CONTRACT_ID).all():
    raise ValueError("Cadence results contain another experiment contract.")

if not cadence_screen_results_df["model_id"].eq(ADVANCING_TREE_MODEL_ID).all():
    raise ValueError("Cadence results contain another model definition.")

if cadence_screen_results_df.duplicated(CADENCE_RESULT_KEY_COLUMNS).any():
    raise ValueError(
        "Cadence results are not unique by Job × Cadence × Refit Step."
    )

if len(cadence_screen_results_df) != expected_cadence_fits:
    raise RuntimeError(
        f"Expected {expected_cadence_fits} cadence refits; "
        f"found {len(cadence_screen_results_df)}."
    )

cadence_status_df = (
    cadence_screen_results_df
    .groupby(["cadence", "status"], as_index=False)
    .size()
    .pivot(index="cadence", columns="status", values="size")
    .fillna(0)
    .astype(int)
    .reset_index()
)

display(cadence_status_df)

Cadence experiment complete: loaded 33 successful refits from panel_tree_cadence_screen_results__extra_trees_cadence_v1_3sentinel_250k.parquet.


status,cadence,success
0,monthly,30
1,validation_start_only,3


### Run the restartable cadence comparison

The next run performs the actual **fixed-versus-monthly** experiment. Each progress unit is one Extra Trees refit, not an entire sentinel. A validation-start-only sentinel therefore requires one fit, while a monthly sentinel requires one fit for each calendar month represented in rolling validation.

The run is checkpointed at **Sentinel × Cadence × Refit Step**. If execution is interrupted, completed refits remain available and only missing units are recomputed. This is especially useful for the monthly policy, where a single sentinel contains several independent model fits.

A completed fit records both its forecast-error components and the corresponding frozen-baseline error on the exact same rows. Those additive quantities allow the monthly segments to be recombined into one exact validation-period MAE, RMSE, NMAE, and bias without averaging incompatible segment-level metrics.

No hyperparameters are tuned in this run, and no final-holdout rows are loaded.


============================================================

In [25]:
# ---------------------------------------------------------------------
# 4.4.1.4E — Synthesize cadence accuracy, benchmark skill, and runtime
# ---------------------------------------------------------------------

# The checkpoint directory currently contains two complete executions of the
# same 33 logical cadence refits. Retain the original Extra Trees generation,
# identified by the frozen model_id used by this experiment.
cadence_screen_results_df = cadence_screen_results_df.loc[
    cadence_screen_results_df["model_id"].eq(ADVANCING_TREE_MODEL_ID)
].copy()

expected_units = len(cadence_schedule_df)

if len(cadence_screen_results_df) != expected_units:
    raise RuntimeError(
        f"Expected {expected_units} retained Extra Trees cadence refits; "
        f"found {len(cadence_screen_results_df)}."
    )

if cadence_screen_results_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any():
    raise ValueError(
        "Retained Extra Trees cadence results are not unique by "
        "Job × Cadence × Refit Step."
    )

print(
    f"Retained Extra Trees cadence generation: "
    f"{len(cadence_screen_results_df):,} refits."
)


# ---------------------------------------------------------------------
# Validate cadence-run completeness
# ---------------------------------------------------------------------

if cadence_screen_results_df.empty:
    raise RuntimeError(
        "Run or load the cadence screen before synthesizing its evidence."
    )

failed_cadence_refits_df = cadence_screen_results_df.loc[
    cadence_screen_results_df["status"].eq("failed")
].copy()

if not failed_cadence_refits_df.empty:
    display(
        failed_cadence_refits_df[
            [
                "modeling_track", "metric", "horizon", "cadence",
                "refit_step", "error_type", "error_message",
            ]
        ]
    )
    raise RuntimeError(
        "The cadence screen contains failed refits. "
        "Resolve them before finalizing the cadence decision."
    )

cadence_success_df = cadence_screen_results_df.loc[
    cadence_screen_results_df["status"].eq("success")
].copy()

if len(cadence_success_df) != expected_units:
    raise RuntimeError(
        f"Expected {expected_units} successful cadence refits; "
        f"found {len(cadence_success_df)}."
    )

if cadence_success_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any():
    raise ValueError(
        "Cadence checkpoints are not unique by "
        "Job × Cadence × Refit Step."
    )


# ---------------------------------------------------------------------
# Recombine monthly refit windows into exact validation-period metrics
# ---------------------------------------------------------------------

# Monthly cadence divides validation into differently sized windows. Averaging
# window-level MAEs would incorrectly give a short window the same weight as a
# long one, so the checkpoints store additive error totals instead.
cadence_aggregate_df = (
    cadence_success_df
    .groupby(
        ["job_id", "modeling_track", "metric", "horizon", "cadence"],
        as_index=False,
    )
    .agg(
        refits=("refit_step", "nunique"),
        evaluated_rows=("model_evaluated_rows", "sum"),
        model_absolute_error_sum=("model_absolute_error_sum", "sum"),
        model_squared_error_sum=("model_squared_error_sum", "sum"),
        model_error_sum=("model_error_sum", "sum"),
        model_absolute_actual_sum=("model_absolute_actual_sum", "sum"),
        benchmark_evaluated_rows=("benchmark_evaluated_rows", "sum"),
        benchmark_absolute_error_sum=("benchmark_absolute_error_sum", "sum"),
        benchmark_squared_error_sum=("benchmark_squared_error_sum", "sum"),
        benchmark_error_sum=("benchmark_error_sum", "sum"),
        benchmark_absolute_actual_sum=("benchmark_absolute_actual_sum", "sum"),
        fit_seconds=("fit_seconds", "sum"),
        predict_seconds=("predict_seconds", "sum"),
        negative_prediction_count=("negative_prediction_count", "sum"),
        min_train_rows=("train_rows", "min"),
        max_train_rows=("train_rows", "max"),
    )
)

# Benchmark skill is meaningful only when model and baseline cover the same
# observations. Different support could masquerade as better forecasting.
if not (
    cadence_aggregate_df["evaluated_rows"]
    == cadence_aggregate_df["benchmark_evaluated_rows"]
).all():
    raise ValueError("Model and benchmark aggregate support differ.")


# Reconstruct the familiar metrics only after all refit-window error totals have
# been combined. This yields the same result as scoring the full validation
# period at once.
cadence_aggregate_df["mae"] = (
    cadence_aggregate_df["model_absolute_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["rmse"] = np.sqrt(
    cadence_aggregate_df["model_squared_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["nmae"] = (
    cadence_aggregate_df["model_absolute_error_sum"]
    / cadence_aggregate_df["model_absolute_actual_sum"]
)

cadence_aggregate_df["bias"] = (
    cadence_aggregate_df["model_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["benchmark_mae"] = (
    cadence_aggregate_df["benchmark_absolute_error_sum"]
    / cadence_aggregate_df["benchmark_evaluated_rows"]
)

# Positive skill means Extra Trees lowered MAE relative to the frozen benchmark
# on the same validation rows.
cadence_aggregate_df["skill_vs_benchmark_pct"] = (
    100.0
    * (cadence_aggregate_df["benchmark_mae"] - cadence_aggregate_df["mae"])
    / cadence_aggregate_df["benchmark_mae"]
)


# ---------------------------------------------------------------------
# Compare fixed and monthly cadence within each sentinel
# ---------------------------------------------------------------------

# Each sentinel serves as its own controlled comparison. Raw MAE should not be
# pooled across Taxi trips, Taxi speed, and Subway ridership because those
# targets live on different numerical scales.
fixed_df = cadence_aggregate_df.loc[
    cadence_aggregate_df["cadence"].eq("validation_start_only")
].copy()

monthly_df = cadence_aggregate_df.loc[
    cadence_aggregate_df["cadence"].eq("monthly")
].copy()

fixed_compare_df = fixed_df[
    [
        "job_id", "modeling_track", "metric", "horizon",
        "mae", "skill_vs_benchmark_pct", "refits", "fit_seconds",
    ]
].rename(
    columns={
        "mae": "fixed_mae",
        "skill_vs_benchmark_pct": "fixed_skill_pct",
        "refits": "fixed_refits",
        "fit_seconds": "fixed_fit_seconds",
    }
)

monthly_compare_df = monthly_df[
    [
        "job_id", "mae", "skill_vs_benchmark_pct",
        "refits", "fit_seconds",
    ]
].rename(
    columns={
        "mae": "monthly_mae",
        "skill_vs_benchmark_pct": "monthly_skill_pct",
        "refits": "monthly_refits",
        "fit_seconds": "monthly_fit_seconds",
    }
)

comparison_df = fixed_compare_df.merge(
    monthly_compare_df,
    on="job_id",
    how="inner",
    validate="one_to_one",
)

if len(comparison_df) != len(cadence_jobs_df):
    raise ValueError(
        "Fixed and monthly cadence results do not cover every sentinel."
    )

# Positive values mean monthly refreshing reduced MAE relative to fitting once
# at validation start. Negative values favor the fixed model.
comparison_df["monthly_vs_fixed_mae_improvement_pct"] = (
    100.0
    * (comparison_df["fixed_mae"] - comparison_df["monthly_mae"])
    / comparison_df["fixed_mae"]
)

# Monthly cadence repeats fitting many more times, so express that extra cost as
# a multiplier rather than forcing the reader to compare raw seconds manually.
comparison_df["fit_time_multiplier"] = (
    comparison_df["monthly_fit_seconds"]
    / comparison_df["fixed_fit_seconds"]
)

comparison_df["Sentinel"] = comparison_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · h={int(row['horizon'])}"
    ),
    axis=1,
)


# ---------------------------------------------------------------------
# Reader-facing sentinel comparison
# ---------------------------------------------------------------------

cadence_comparison_display_df = comparison_df[
    [
        "Sentinel",
        "fixed_mae",
        "monthly_mae",
        "monthly_vs_fixed_mae_improvement_pct",
        "fixed_skill_pct",
        "monthly_skill_pct",
        "fixed_refits",
        "monthly_refits",
        "fit_time_multiplier",
    ]
].rename(
    columns={
        "fixed_mae": "Fixed MAE",
        "monthly_mae": "Monthly MAE",
        "monthly_vs_fixed_mae_improvement_pct":
            "Monthly vs fixed MAE improvement %",
        "fixed_skill_pct": "Fixed skill %",
        "monthly_skill_pct": "Monthly skill %",
        "fixed_refits": "Fixed refits",
        "monthly_refits": "Monthly refits",
        "fit_time_multiplier": "Fit-time multiplier",
    }
)

for column in ["Fixed MAE", "Monthly MAE"]:
    cadence_comparison_display_df[column] = (
        cadence_comparison_display_df[column].round(4)
    )

for column in [
    "Monthly vs fixed MAE improvement %",
    "Fixed skill %",
    "Monthly skill %",
    "Fit-time multiplier",
]:
    cadence_comparison_display_df[column] = (
        cadence_comparison_display_df[column].round(2)
    )

display(cadence_comparison_display_df)


# ---------------------------------------------------------------------
# Summarize the overall accuracy-versus-compute tradeoff
# ---------------------------------------------------------------------

# Cross-sentinel synthesis uses comparable quantities—percentage improvement,
# benchmark skill, wins, and runtime—rather than averaging raw MAE across
# incompatible target units.
cadence_policy_summary_df = (
    cadence_aggregate_df
    .groupby("cadence", as_index=False)
    .agg(
        sentinels=("job_id", "nunique"),
        total_refits=("refits", "sum"),
        median_skill_pct=("skill_vs_benchmark_pct", "median"),
        worst_skill_pct=("skill_vs_benchmark_pct", "min"),
        total_fit_seconds=("fit_seconds", "sum"),
    )
)

monthly_wins = int(
    comparison_df["monthly_vs_fixed_mae_improvement_pct"].gt(0).sum()
)

monthly_median_improvement = float(
    comparison_df["monthly_vs_fixed_mae_improvement_pct"].median()
)

cadence_policy_summary_df["Cadence"] = (
    cadence_policy_summary_df["cadence"].map(CADENCE_POLICY_LABELS)
)

# Fixed is the reference policy, so only monthly can meaningfully be described
# as "beating fixed."
cadence_policy_summary_df["Sentinels beating fixed"] = (
    cadence_policy_summary_df["cadence"].map(
        {
            "validation_start_only": np.nan,
            "monthly": monthly_wins,
        }
    )
)

cadence_policy_summary_df["Median MAE improvement vs fixed %"] = (
    cadence_policy_summary_df["cadence"].map(
        {
            "validation_start_only": 0.0,
            "monthly": monthly_median_improvement,
        }
    )
)

cadence_policy_summary_df["Total fit minutes"] = (
    cadence_policy_summary_df["total_fit_seconds"] / 60.0
)

cadence_policy_display_df = cadence_policy_summary_df[
    [
        "Cadence",
        "sentinels",
        "Sentinels beating fixed",
        "total_refits",
        "Median MAE improvement vs fixed %",
        "median_skill_pct",
        "worst_skill_pct",
        "Total fit minutes",
    ]
].rename(
    columns={
        "sentinels": "Sentinels",
        "total_refits": "Total refits",
        "median_skill_pct": "Median skill %",
        "worst_skill_pct": "Worst skill %",
    }
)

for column in [
    "Median MAE improvement vs fixed %",
    "Median skill %",
    "Worst skill %",
    "Total fit minutes",
]:
    cadence_policy_display_df[column] = (
        cadence_policy_display_df[column].round(2)
    )

display(cadence_policy_display_df)


# ---------------------------------------------------------------------
# Visual — show the cadence effect directly for each sentinel
# ---------------------------------------------------------------------

if RUN_VISUALS:
    # Positive bars favor monthly refreshing; negative bars favor the fixed
    # validation-start model. The zero line marks equal MAE.
    cadence_fig = go.Figure(
        go.Bar(
            x=comparison_df["Sentinel"],
            y=comparison_df["monthly_vs_fixed_mae_improvement_pct"],
            marker_color=BRAND_COLORS["dark_teal"],
            hovertemplate=(
                "%{x}<br>"
                "Monthly vs fixed MAE improvement: %{y:.2f}%"
                "<extra></extra>"
            ),
        )
    )

    cadence_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    cadence_fig.update_layout(
        title=(
            "Extra Trees Refit Cadence — "
            "Monthly MAE Improvement Relative to a Fixed Model"
        ),
        xaxis_title=None,
        yaxis_title="Monthly vs fixed MAE improvement (%)",
        width=900,
        height=500,
        margin=dict(l=80, r=50, t=80, b=110),
    )

    cadence_fig.update_xaxes(tickangle=-15)

    apply_branding(cadence_fig)
    cadence_fig.show()

Retained Extra Trees cadence generation: 33 refits.


,Sentinel,Fixed MAE,Monthly MAE,Monthly vs fixed MAE improvement %,Fixed skill %,Monthly skill %,Fixed refits,Monthly refits,Fit-time multiplier
0,Full-History · Subway · h=5,426.3976,419.1066,1.71,19.82,21.19,1,10,10.83
1,Post-CP · Taxi speed · h=5,2.2099,2.1857,1.10,25.11,25.93,1,10,35.14
2,Post-CP · Taxi trips · h=1,16.2888,14.7061,9.72,19.55,27.36,1,10,26.43


,Cadence,Sentinels,Sentinels beating fixed,Total refits,Median MAE improvement vs fixed %,Median skill %,Worst skill %,Total fit minutes
0,Monthly refresh,3,3.0,30,1.71,25.93,21.19,88.36
1,Validation-start only,3,NaN,3,0.00,19.82,19.55,4.99


**Findings.** The cadence comparison completed cleanly: all **33 of 33 planned Extra Trees fits** succeeded, with zero failed refits across all three sentinels. Each validation-start-only policy completed its single fit, while every monthly policy completed all 10 scheduled refits. The full restartable run finished in about **85 minutes**, somewhat faster than the pre-run estimate. Operational reliability therefore does not distinguish the two cadence policies; the choice can be made from forecast accuracy and computational cost rather than from failed or incomplete model runs.

### Measure whether monthly refreshing earns its additional computational cost

MAE remains the decision metric. Because each cadence scores the same sentinel on the same frozen benchmark support, the cleanest comparison is the percentage change in MAE from the validation-start-only model to monthly refreshing.

A positive **Monthly vs fixed MAE improvement** means monthly refitting lowered forecast error. Supporting evidence retains benchmark skill, total fit time, and refit count so a small accuracy gain can be judged against its operational cost.

Raw MAE values should not be averaged across targets because Taxi trips, Taxi speed, and Subway ridership live on very different numerical scales. Cross-sentinel synthesis therefore uses **percentage improvement and benchmark skill** rather than pooling raw errors across incompatible units.


============================================================

In [26]:
# ---------------------------------------------------------------------
# 4.4.1.4F — Synthesize cadence accuracy, benchmark skill, and runtime
# ---------------------------------------------------------------------

# The synthesis assumes the restartable cadence experiment has already produced
# saved results. Fail explicitly rather than presenting an incomplete summary.
if cadence_screen_results_df.empty:
    raise RuntimeError(
        "Run or load the cadence screen before synthesizing its evidence."
    )


# ---------------------------------------------------------------------
# Verify that the experiment completed cleanly before comparing policies
# ---------------------------------------------------------------------

# Keep failed refits visible. A cadence comparison is not trustworthy if one
# policy silently has missing months or missing sentinel fits.
failed_cadence_refits_df = cadence_screen_results_df.loc[
    cadence_screen_results_df["status"].eq("failed")
].copy()

if not failed_cadence_refits_df.empty:
    display(
        failed_cadence_refits_df[
            [
                "modeling_track",
                "metric",
                "horizon",
                "cadence",
                "refit_step",
                "error_type",
                "error_message",
            ]
        ]
    )

    raise RuntimeError(
        "The cadence screen contains failed refits. "
        "Resolve them before finalizing the cadence decision."
    )


# Restrict the synthesis to successful checkpoint units only after confirming
# that no explicit failures remain.
cadence_success_df = cadence_screen_results_df.loc[
    cadence_screen_results_df["status"].eq("success")
].copy()

# Every planned Job × Cadence × Refit Step should produce one successful
# checkpoint. This guards against missing files that are neither success nor
# explicit failure records.
expected_units = len(cadence_schedule_df)

if len(cadence_success_df) != expected_units:
    raise RuntimeError(
        f"Expected {expected_units} successful cadence refits; "
        f"found {len(cadence_success_df)}."
    )

# The checkpoint grain is one Job × Cadence × Refit Step. Duplicate units would
# double-count both errors and runtime during aggregation.
if cadence_success_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any():
    raise ValueError(
        "Cadence checkpoints are not unique by Job × Cadence × Refit Step."
    )


# ---------------------------------------------------------------------
# Recombine refit segments into exact full-validation metrics
# ---------------------------------------------------------------------

# Monthly refreshing divides validation into several differently sized forecast
# segments. Averaging each segment's MAE would incorrectly give a small segment
# the same weight as a large one.
#
# Instead, each refit checkpoint stored additive error totals. Summing those
# quantities first and computing MAE/RMSE/bias afterward reproduces the metric
# that would have been obtained from all validation predictions at once.
cadence_aggregate_df = (
    cadence_success_df
    .groupby(
        [
            "job_id",
            "modeling_track",
            "metric",
            "horizon",
            "cadence",
        ],
        as_index=False,
    )
    .agg(
        refits=("refit_step", "nunique"),
        evaluated_rows=("model_evaluated_rows", "sum"),
        model_absolute_error_sum=(
            "model_absolute_error_sum",
            "sum",
        ),
        model_squared_error_sum=(
            "model_squared_error_sum",
            "sum",
        ),
        model_error_sum=("model_error_sum", "sum"),
        model_absolute_actual_sum=(
            "model_absolute_actual_sum",
            "sum",
        ),
        benchmark_evaluated_rows=(
            "benchmark_evaluated_rows",
            "sum",
        ),
        benchmark_absolute_error_sum=(
            "benchmark_absolute_error_sum",
            "sum",
        ),
        benchmark_squared_error_sum=(
            "benchmark_squared_error_sum",
            "sum",
        ),
        benchmark_error_sum=(
            "benchmark_error_sum",
            "sum",
        ),
        benchmark_absolute_actual_sum=(
            "benchmark_absolute_actual_sum",
            "sum",
        ),
        fit_seconds=("fit_seconds", "sum"),
        predict_seconds=("predict_seconds", "sum"),
        negative_prediction_count=(
            "negative_prediction_count",
            "sum",
        ),
        min_train_rows=("train_rows", "min"),
        max_train_rows=("train_rows", "max"),
    )
)

# Model and benchmark must cover identical validation rows. Otherwise benchmark
# skill could reflect different support rather than better forecasting.
if not (
    cadence_aggregate_df["evaluated_rows"]
    == cadence_aggregate_df["benchmark_evaluated_rows"]
).all():
    raise ValueError(
        "Model and benchmark aggregate support differ."
    )


# Reconstruct the standard project metrics from their additive components.
# These are now exact over the complete validation period for each cadence.
cadence_aggregate_df["mae"] = (
    cadence_aggregate_df["model_absolute_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["rmse"] = np.sqrt(
    cadence_aggregate_df["model_squared_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["nmae"] = (
    cadence_aggregate_df["model_absolute_error_sum"]
    / cadence_aggregate_df["model_absolute_actual_sum"]
)

cadence_aggregate_df["bias"] = (
    cadence_aggregate_df["model_error_sum"]
    / cadence_aggregate_df["evaluated_rows"]
)

cadence_aggregate_df["benchmark_mae"] = (
    cadence_aggregate_df["benchmark_absolute_error_sum"]
    / cadence_aggregate_df["benchmark_evaluated_rows"]
)

# Positive skill means Extra Trees achieved lower MAE than the frozen benchmark
# on the exact same forecast rows.
cadence_aggregate_df["skill_vs_benchmark_pct"] = (
    100.0
    * (
        cadence_aggregate_df["benchmark_mae"]
        - cadence_aggregate_df["mae"]
    )
    / cadence_aggregate_df["benchmark_mae"]
)


# ---------------------------------------------------------------------
# Compare fixed and monthly cadence within each sentinel
# ---------------------------------------------------------------------

# Each sentinel serves as its own controlled experiment. This is important
# because Taxi trips, speed, and Subway ridership use incompatible raw units;
# their MAEs should never be pooled directly.
fixed_df = cadence_aggregate_df.loc[
    cadence_aggregate_df["cadence"].eq(
        "validation_start_only"
    )
].copy()

monthly_df = cadence_aggregate_df.loc[
    cadence_aggregate_df["cadence"].eq("monthly")
].copy()

# Place the two policies side by side on the same job so the cadence effect can
# be expressed as a percentage change rather than a cross-target raw-MAE average.
comparison_df = (
    fixed_df[
        [
            "job_id",
            "modeling_track",
            "metric",
            "horizon",
            "mae",
            "skill_vs_benchmark_pct",
            "refits",
            "fit_seconds",
        ]
    ]
    .rename(
        columns={
            "mae": "fixed_mae",
            "skill_vs_benchmark_pct": "fixed_skill_pct",
            "refits": "fixed_refits",
            "fit_seconds": "fixed_fit_seconds",
        }
    )
    .merge(
        monthly_df[
            [
                "job_id",
                "mae",
                "skill_vs_benchmark_pct",
                "refits",
                "fit_seconds",
            ]
        ].rename(
            columns={
                "mae": "monthly_mae",
                "skill_vs_benchmark_pct": "monthly_skill_pct",
                "refits": "monthly_refits",
                "fit_seconds": "monthly_fit_seconds",
            }
        ),
        on="job_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(comparison_df) != len(cadence_jobs_df):
    raise ValueError(
        "Fixed and monthly cadence results do not cover every sentinel."
    )

# Positive values mean monthly refreshing reduced MAE relative to fitting once
# at validation start. Negative values mean the repeated refits made forecasts
# worse on that sentinel.
comparison_df["monthly_vs_fixed_mae_improvement_pct"] = (
    100.0
    * (
        comparison_df["fixed_mae"]
        - comparison_df["monthly_mae"]
    )
    / comparison_df["fixed_mae"]
)

# Runtime cost matters because monthly refreshing repeats an otherwise identical
# Extra Trees fit many times. A multiplier makes that operational difference
# easier to interpret than raw seconds alone.
comparison_df["fit_time_multiplier"] = (
    comparison_df["monthly_fit_seconds"]
    / comparison_df["fixed_fit_seconds"]
)

comparison_df["Sentinel"] = comparison_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · "
        f"h={int(row['horizon'])}"
    ),
    axis=1,
)


# ---------------------------------------------------------------------
# Reader-facing sentinel comparison
# ---------------------------------------------------------------------

# Keep the table focused on the actual cadence question:
# accuracy under each policy, benchmark skill, refit count, and runtime cost.
cadence_comparison_display_df = comparison_df[
    [
        "Sentinel",
        "fixed_mae",
        "monthly_mae",
        "monthly_vs_fixed_mae_improvement_pct",
        "fixed_skill_pct",
        "monthly_skill_pct",
        "fixed_refits",
        "monthly_refits",
        "fit_time_multiplier",
    ]
].rename(
    columns={
        "fixed_mae": "Fixed MAE",
        "monthly_mae": "Monthly MAE",
        "monthly_vs_fixed_mae_improvement_pct":
            "Monthly vs fixed MAE improvement %",
        "fixed_skill_pct": "Fixed skill %",
        "monthly_skill_pct": "Monthly skill %",
        "fixed_refits": "Fixed refits",
        "monthly_refits": "Monthly refits",
        "fit_time_multiplier": "Fit-time multiplier",
    }
)

for column in ["Fixed MAE", "Monthly MAE"]:
    cadence_comparison_display_df[column] = (
        cadence_comparison_display_df[column].round(4)
    )

for column in [
    "Monthly vs fixed MAE improvement %",
    "Fixed skill %",
    "Monthly skill %",
    "Fit-time multiplier",
]:
    cadence_comparison_display_df[column] = (
        cadence_comparison_display_df[column].round(2)
    )

display(cadence_comparison_display_df)


# ---------------------------------------------------------------------
# Summarize the overall operational tradeoff
# ---------------------------------------------------------------------

# Do not average raw MAE across different targets. Instead summarize quantities
# that are comparable across sentinels: benchmark skill, number of wins, and
# percentage MAE improvement relative to each sentinel's own fixed model.
cadence_policy_summary_df = (
    cadence_aggregate_df
    .groupby("cadence", as_index=False)
    .agg(
        sentinels=("job_id", "nunique"),
        total_refits=("refits", "sum"),
        median_skill_pct=(
            "skill_vs_benchmark_pct",
            "median",
        ),
        worst_skill_pct=(
            "skill_vs_benchmark_pct",
            "min",
        ),
        total_fit_seconds=("fit_seconds", "sum"),
    )
)

# Count how many sentinels actually favor monthly refreshing.
monthly_wins = int(
    comparison_df[
        "monthly_vs_fixed_mae_improvement_pct"
    ]
    .gt(0)
    .sum()
)

# The median percentage improvement gives one compact measure of the typical
# cadence effect without mixing incompatible target units.
monthly_median_improvement = float(
    comparison_df[
        "monthly_vs_fixed_mae_improvement_pct"
    ].median()
)

cadence_policy_summary_df["Cadence"] = (
    cadence_policy_summary_df["cadence"]
    .map(CADENCE_POLICY_LABELS)
)

# Only monthly can "beat fixed"; the fixed policy is the reference condition.
cadence_policy_summary_df["Sentinels beating fixed"] = (
    cadence_policy_summary_df["cadence"].map(
        {
            "validation_start_only": np.nan,
            "monthly": monthly_wins,
        }
    )
)

cadence_policy_summary_df["Median MAE improvement vs fixed %"] = (
    cadence_policy_summary_df["cadence"].map(
        {
            "validation_start_only": 0.0,
            "monthly": monthly_median_improvement,
        }
    )
)

cadence_policy_summary_df["Total fit minutes"] = (
    cadence_policy_summary_df["total_fit_seconds"] / 60.0
)


# Present one compact row per policy rather than exposing the lower-level
# per-refit checkpoint records.
cadence_policy_display_df = cadence_policy_summary_df[
    [
        "Cadence",
        "sentinels",
        "Sentinels beating fixed",
        "total_refits",
        "Median MAE improvement vs fixed %",
        "median_skill_pct",
        "worst_skill_pct",
        "Total fit minutes",
    ]
].rename(
    columns={
        "sentinels": "Sentinels",
        "total_refits": "Total refits",
        "median_skill_pct": "Median skill %",
        "worst_skill_pct": "Worst skill %",
    }
)

for column in [
    "Median MAE improvement vs fixed %",
    "Median skill %",
    "Worst skill %",
    "Total fit minutes",
]:
    cadence_policy_display_df[column] = (
        cadence_policy_display_df[column].round(2)
    )

display(cadence_policy_display_df)


# ---------------------------------------------------------------------
# Visual — show the cadence effect directly for each sentinel
# ---------------------------------------------------------------------

if RUN_VISUALS:
    # Positive bars mean monthly refreshing lowered MAE. Negative bars mean
    # fitting once at validation start performed better.
    cadence_fig = go.Figure(
        go.Bar(
            x=comparison_df["Sentinel"],
            y=comparison_df[
                "monthly_vs_fixed_mae_improvement_pct"
            ],
            marker_color=BRAND_COLORS["dark_teal"],
            hovertemplate=(
                "%{x}<br>"
                "Monthly vs fixed MAE improvement: %{y:.2f}%"
                "<extra></extra>"
            ),
        )
    )

    # The zero line marks the point of indifference between the two policies.
    cadence_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    cadence_fig.update_layout(
        title=(
            "Extra Trees Refit Cadence — "
            "Monthly MAE Improvement Relative to a Fixed Model"
        ),
        xaxis_title=None,
        yaxis_title="Monthly vs fixed MAE improvement (%)",
        width=900,
        height=500,
        margin=dict(l=80, r=50, t=80, b=110),
    )

    cadence_fig.update_xaxes(
        tickangle=-15
    )

    apply_branding(cadence_fig)
    cadence_fig.show()

,Sentinel,Fixed MAE,Monthly MAE,Monthly vs fixed MAE improvement %,Fixed skill %,Monthly skill %,Fixed refits,Monthly refits,Fit-time multiplier
0,Full-History · Subway · h=5,426.3976,419.1066,1.71,19.82,21.19,1,10,10.83
1,Post-CP · Taxi speed · h=5,2.2099,2.1857,1.10,25.11,25.93,1,10,35.14
2,Post-CP · Taxi trips · h=1,16.2888,14.7061,9.72,19.55,27.36,1,10,26.43


,Cadence,Sentinels,Sentinels beating fixed,Total refits,Median MAE improvement vs fixed %,Median skill %,Worst skill %,Total fit minutes
0,Monthly refresh,3,3.0,30,1.71,25.93,21.19,88.36
1,Validation-start only,3,NaN,3,0.00,19.82,19.55,4.99


**Findings.** Rebuilding Extra Trees once per month improved forecast accuracy for all three representative forecasting problems. The improvement was small for Full-History Subway ridership at h=5 (**1.71% lower MAE**) and Post-CP Taxi speed at h=5 (**1.10%**), but much larger for Post-CP Taxi trips at h=1 (**9.72%**). The typical improvement across the three cases was therefore modest, but the direction was consistent: every monthly-refresh model produced lower forecast error than the model that was fit only once at the start of validation.

Both approaches still performed substantially better than the simple seasonal benchmark. In other words, the fixed Extra Trees model was already useful; monthly refreshing added another layer of improvement rather than rescuing a weak model.

The tradeoff is computation. Fitting once at validation start required about **5 total minutes** across the three sentinels, while monthly refreshing required about **88 minutes** because a completely new Extra Trees model was rebuilt each month using the larger set of outcomes that had become available. The cadence experiment therefore shows a real but mostly modest accuracy benefit from repeated retraining, at a much higher computational cost.

The remaining decision is whether that additional accuracy is large enough to justify monthly retraining in the final forecasting workflow.

> ✅ Decision — Retain monthly refreshing as the preferred Extra Trees cadence, conditional on production\-scale feasibility\.

Monthly rebuilding reduced MAE for all three cadence sentinels, so Extra Trees benefits from incorporating newly observed outcomes\. However, the gain came at substantial computational cost because every monthly update rebuilds the forest from scratch\.

Carry monthly refreshing forward as the preferred cadence if Extra Trees remains computationally viable after final model\-definition tuning and production\-scale testing\. The cadence result does not by itself guarantee that Extra Trees is practical at the uncapped production training scale\.

## 4.4.1.5 Finalize and Stress-Test the Extra Trees Model Definition

Extra Trees remains the accuracy-leading candidate, and the cadence experiment shows that monthly refreshing improves its forecasts. Before committing that architecture to full rolling validation, two remaining questions must be resolved:

> **What Extra Trees definition provides the best accuracy/cost tradeoff, and does that final definition remain operationally viable at production scale?**

The family screen used one reasonable Extra Trees configuration only to determine whether the architecture was promising. It was not intended to prove that those exact settings were optimal. This section therefore tests a small set of deliberately different Extra Trees capacity profiles and then confirms count-target representation.

The purpose is not only to maximize validation accuracy. The selected model must also remain practical enough to support the forecasting workflow established earlier. A capacity improvement that makes production-scale retraining prohibitively expensive is not automatically a viable production choice.

The final holdout remains untouched.

In [27]:
# ---------------------------------------------------------------------
# 4.4.1.5A — Define the bounded Extra Trees tuning experiment
# ---------------------------------------------------------------------

# The family and cadence decisions are complete. This section changes only
# the Extra Trees model definition; it does not reopen either earlier choice.
assert ADVANCING_TREE_MODEL_ID == "extra_trees"

COUNT_TARGET_METRICS = {
    "taxi_trip_count",
    "fhvhv_trip_count",
    "subway_ridership",
}

TUNING_TARGET_REPRESENTATIONS = {
    "raw": "Raw target",
    "log1p": "log1p target",
}

# Rather than searching dozens of loosely related settings, compare three
# intentionally different model shapes around the configuration that already
# performed well in the family screen.
#
# - screen_reference preserves the original Extra Trees screen definition.
# - more_regularized builds shallower trees with larger leaves and considers
#   fewer predictors at each split, producing smoother rules.
# - more_flexible allows deeper trees, smaller leaves, and more candidate
#   predictors, giving the model more room to learn detailed interactions.
TUNING_CAPACITY_PROFILES = {
    "screen_reference": {
        "label": "Screen reference",
        "params": dict(
            n_estimators=120,
            max_depth=16,
            min_samples_leaf=10,
            max_features=0.70,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    },
    "more_regularized": {
        "label": "More regularized",
        "params": dict(
            n_estimators=180,
            max_depth=12,
            min_samples_leaf=20,
            max_features=0.60,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    },
    "more_flexible": {
        "label": "More flexible",
        "params": dict(
            n_estimators=180,
            max_depth=24,
            min_samples_leaf=5,
            max_features=0.85,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    },
}

TUNING_TRAIN_ROW_CAP = 250_000
TUNING_CONTRACT_ID = "extra_trees_definition_v1_3capacity_raw_log1p_250k"

TUNING_PARTS_DIR = INTERMEDIATE_DIR / f"tuning_parts__{TUNING_CONTRACT_ID}"
TUNING_RESULTS_PATH = INTERMEDIATE_DIR / f"panel_tree_tuning_results__{TUNING_CONTRACT_ID}.parquet"

RUN_TUNING_SCREEN = False
RESET_TUNING_SCREEN_CHECKPOINTS = False


# Build the exact list of fits before launching anything expensive.
# Every representative job receives all three raw-target capacity profiles.
# Only the three count targets receive the additional log1p versions.
tuning_plan_rows = []

for job in screen_jobs_df.itertuples(index=False):
    representations = ["raw", "log1p"] if job.metric in COUNT_TARGET_METRICS else ["raw"]

    for representation in representations:
        for profile_id, profile in TUNING_CAPACITY_PROFILES.items():
            tuning_plan_rows.append({
                "modeling_track": job.modeling_track,
                "metric": job.metric,
                "horizon": int(job.horizon),
                "target_representation": representation,
                "capacity_profile": profile_id,
                "capacity_label": profile["label"],
            })

tuning_plan_df = pd.DataFrame(tuning_plan_rows)

expected_tuning_fits = len(tuning_plan_df)
raw_fit_count = int(tuning_plan_df["target_representation"].eq("raw").sum())
log_fit_count = int(tuning_plan_df["target_representation"].eq("log1p").sum())

assert expected_tuning_fits == 96, f"Expected 96 tuning fits; found {expected_tuning_fits}."
assert raw_fit_count == 60, f"Expected 60 raw-target fits; found {raw_fit_count}."
assert log_fit_count == 36, f"Expected 36 log1p fits; found {log_fit_count}."

tuning_workload_df = pd.DataFrame([
    {
        "Representative jobs": screen_jobs_df.shape[0],
        "Capacity profiles": len(TUNING_CAPACITY_PROFILES),
        "Raw-target fits": raw_fit_count,
        "Additional log1p fits": log_fit_count,
        "Total planned fits": expected_tuning_fits,
        "Training-row cap": TUNING_TRAIN_ROW_CAP,
    }
])

display(tuning_workload_df)

,Representative jobs,Capacity profiles,Raw-target fits,Additional log1p fits,Total planned fits,Training-row cap
0,20,3,60,36,96,250000


**Findings.** The bounded Extra Trees tuning plan contains **96 planned fits**: 60 raw-target fits across all 20 representative jobs, plus 36 additional `log1p` fits for the 12 count-target jobs. The **250,000-row training cap** applies only to this bounded tuning experiment, keeping the capacity and representation comparison computationally manageable before broader validation.

### A small search by design

This is not intended to find a mathematically perfect combination of Extra Trees settings. A large grid would multiply runtime quickly while encouraging us to choose among many nearly interchangeable configurations based on one validation slice.

Instead, the three candidates ask a simpler and more useful question:

- Does the **screen reference** already sit in a sensible middle ground?
- Does making the trees **more conservative** improve generalization?
- Does giving the trees **more flexibility** uncover useful structure that the reference model is missing?

For count targets, each of those three shapes is tested twice: once on the original count and once after `log1p` compression. The transformation is applied only to the training target. Forecasts from a log-trained model are converted back with `expm1` before any error is measured.

No prediction is clipped after conversion. If a model produces an implausible negative value, that behavior remains visible in the evidence rather than being silently corrected.

In [28]:
# ---------------------------------------------------------------------
# 4.4.1.5B — Fit one Extra Trees tuning definition
# ---------------------------------------------------------------------

def fit_extra_trees_tuning_candidate(payload, profile_id, target_representation):
    """Fit one bounded Extra Trees definition and score it in original units.

    The target transformation changes only what the model sees during fitting.
    Evaluation always happens after predictions have been returned to the
    target's original scale, keeping MAE directly comparable across candidates.
    """
    if profile_id not in TUNING_CAPACITY_PROFILES:
        raise ValueError(f"Unknown tuning capacity profile: {profile_id}")

    if target_representation not in TUNING_TARGET_REPRESENTATIONS:
        raise ValueError(f"Unknown target representation: {target_representation}")

    y_train = payload["y_train"].to_numpy(dtype=float)
    y_valid = payload["y_valid"].to_numpy(dtype=float)

    # log1p is meaningful only when the training target is nonnegative.
    # Fail loudly rather than silently changing or clipping an invalid target.
    if target_representation == "log1p":
        if np.any(y_train < 0):
            raise ValueError("log1p tuning requires nonnegative training targets.")
        fit_target = np.log1p(y_train)
    else:
        fit_target = y_train

    params = TUNING_CAPACITY_PROFILES[profile_id]["params"]
    estimator = ExtraTreesRegressor(**params)

    # Reuse the same training-fitted preprocessing contract as the family
    # screen so capacity and target representation are the only new variables.
    model = make_one_hot_pipeline(
        "extra_trees",
        estimator,
        payload["predictor_names"],
        payload["categorical_names"],
    )

    fit_started = perf_counter()
    model.fit(payload["X_train"], fit_target)
    fit_seconds = perf_counter() - fit_started

    predict_started = perf_counter()
    predictions = np.asarray(model.predict(payload["X_valid"]), dtype=float)
    predict_seconds = perf_counter() - predict_started

    # Convert log-trained forecasts back to real-world target units before
    # computing any metric. Do not clip negative values after inversion.
    if target_representation == "log1p":
        predictions = np.expm1(predictions)

    metrics = regression_metrics(y_valid, predictions)

    del model, estimator, fit_target
    gc.collect()

    return predictions, metrics, fit_seconds, predict_seconds

### 4.4.1.5C — Run staged Extra Trees capacity tuning

**Question.** How much Extra Trees capacity is justified before full rolling validation?

Capacity tuning is intentionally staged rather than treated as a full Cartesian search. The first stage compares three predeclared capacity profiles across nine representative Track × Target × Horizon jobs. This provides broad evidence about whether additional tree flexibility improves validation accuracy and whether stronger regularization remains competitive.

The second stage then concentrates compute where the capacity decision is still unresolved. The two viable profiles are compared on three additional Post-CP jobs covering Taxi speed, Taxi demand, and Subway demand, where the first-stage evidence provides less direct coverage. This produces a **33-fit capacity roster**: 27 three-profile screening fits plus 6 targeted confirmation fits.

The purpose of this stage is to obtain enough evidence to freeze Extra Trees capacity, not to exhaust every possible hyperparameter combination. If the 33 fits support a clear accuracy/runtime decision, capacity tuning stops here. If the surviving profiles remain materially difficult to distinguish, only narrowly targeted follow-up comparisons will be added. Count-target representation is held raw in this stage and evaluated separately in the next section.

In [29]:
# ---------------------------------------------------------------------
# 4.4.1.5C — Execute or recover staged Extra Trees capacity tuning
# ---------------------------------------------------------------------
# Capacity is resolved before target representation. Stage 1 retains broad
# three-profile evidence across nine representative forecasting jobs. Stage 2
# adds targeted Post-CP coverage using only the two profiles that remain
# relevant to the final accuracy/runtime decision.

TUNING_RESULT_KEY_COLUMNS = [
    "modeling_track",
    "metric",
    "horizon",
    "capacity_profile",
    "target_representation",
]


# ---------------------------------------------------------------------
# Stage 1 — Broad three-profile capacity screen
# ---------------------------------------------------------------------
# These nine jobs form the representative capacity evidence already targeted
# by the staged experiment. Each is evaluated under all three predeclared
# Extra Trees capacity profiles.

BASE_CAPACITY_JOBS = [
    ("full_history_regime_aware", "fhvhv_avg_trip_speed", 5),
    ("full_history_regime_aware", "fhvhv_trip_count", 5),
    ("full_history_regime_aware", "subway_ridership", 5),
    ("full_history_regime_aware", "taxi_avg_trip_speed", 5),
    ("full_history_regime_aware", "taxi_trip_count", 5),
    ("post_cp_primary", "fhvhv_avg_trip_speed", 1),
    ("post_cp_primary", "fhvhv_avg_trip_speed", 2),
    ("post_cp_primary", "fhvhv_avg_trip_speed", 5),
    ("post_cp_primary", "fhvhv_trip_count", 1),
]

BASE_CAPACITY_PROFILES = [
    "screen_reference",
    "more_regularized",
    "more_flexible",
]


# ---------------------------------------------------------------------
# Stage 2 — Focused Post-CP confirmation
# ---------------------------------------------------------------------
# The confirmation surface fills the remaining modality/target coverage gaps
# without spending additional compute on the more-regularized profile.

CAPACITY_CONFIRMATION_JOBS = [
    ("post_cp_primary", "taxi_avg_trip_speed", 1),
    ("post_cp_primary", "taxi_trip_count", 2),
    ("post_cp_primary", "subway_ridership", 5),
]

CAPACITY_CONFIRMATION_PROFILES = [
    "screen_reference",
    "more_flexible",
]


# ---------------------------------------------------------------------
# Freeze the 33-fit logical capacity roster
# ---------------------------------------------------------------------

capacity_plan_rows = []

for track, metric, horizon in BASE_CAPACITY_JOBS:
    for profile_id in BASE_CAPACITY_PROFILES:
        capacity_plan_rows.append(
            {
                "modeling_track": track,
                "metric": metric,
                "horizon": int(horizon),
                "capacity_profile": profile_id,
                "capacity_label": TUNING_CAPACITY_PROFILES[profile_id]["label"],
                "target_representation": "raw",
                "capacity_stage": "three_profile_screen",
            }
        )

for track, metric, horizon in CAPACITY_CONFIRMATION_JOBS:
    for profile_id in CAPACITY_CONFIRMATION_PROFILES:
        capacity_plan_rows.append(
            {
                "modeling_track": track,
                "metric": metric,
                "horizon": int(horizon),
                "capacity_profile": profile_id,
                "capacity_label": TUNING_CAPACITY_PROFILES[profile_id]["label"],
                "target_representation": "raw",
                "capacity_stage": "post_cp_confirmation",
            }
        )

capacity_plan_df = (
    pd.DataFrame(capacity_plan_rows)
    .sort_values(
        [
            "capacity_stage",
            "modeling_track",
            "metric",
            "horizon",
            "capacity_profile",
        ]
    )
    .reset_index(drop=True)
)

expected_capacity_fits = len(capacity_plan_df)

if expected_capacity_fits != 33:
    raise RuntimeError(
        f"Expected 33 staged capacity definitions; found {expected_capacity_fits}."
    )

capacity_expected_keys = set(
    capacity_plan_df[TUNING_RESULT_KEY_COLUMNS].itertuples(index=False, name=None)
)

if len(capacity_expected_keys) != expected_capacity_fits:
    raise ValueError(
        "Capacity roster is not unique by "
        "Track × Target × Horizon × Capacity × Representation."
    )

capacity_plan_summary_df = (
    capacity_plan_df
    .groupby("capacity_stage", as_index=False)
    .agg(
        forecasting_jobs=(
            "metric",
            lambda s: len(
                capacity_plan_df.loc[s.index, ["modeling_track", "metric", "horizon"]]
                .drop_duplicates()
            ),
        ),
        planned_fits=("capacity_profile", "size"),
    )
)

display(capacity_plan_summary_df)


# ---------------------------------------------------------------------
# Capacity-specific consolidated artifact
# ---------------------------------------------------------------------
# The underlying checkpoint namespace remains the existing tuning contract.
# This preserves native continuity with already-completed logical fits while
# allowing 5C to consolidate only the 33 raw capacity definitions. Other
# compatible tuning evidence remains available for the representation stage.

CAPACITY_TUNING_RESULTS_PATH = (
    INTERMEDIATE_DIR
    / f"panel_tree_capacity_results__{TUNING_CONTRACT_ID}.parquet"
)


# ---------------------------------------------------------------------
# Reset is the only destructive control
# ---------------------------------------------------------------------
# RESET=True deliberately rebuilds the entire tuning checkpoint namespace.
# Without reset, successful logical definitions are always recovered and reused.

if RESET_TUNING_SCREEN_CHECKPOINTS:
    reset_checkpoint_dir(TUNING_PARTS_DIR)
    TUNING_RESULTS_PATH.unlink(missing_ok=True)
    CAPACITY_TUNING_RESULTS_PATH.unlink(missing_ok=True)

TUNING_PARTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Prefer a complete 33-fit capacity artifact on ordinary reruns
# ---------------------------------------------------------------------

capacity_results_df = None

if not RUN_TUNING_SCREEN and not RESET_TUNING_SCREEN_CHECKPOINTS:
    capacity_results_df = load_complete_consolidated_result(
        CAPACITY_TUNING_RESULTS_PATH,
        contract_column="tuning_contract_id",
        contract_id=TUNING_CONTRACT_ID,
        key_columns=TUNING_RESULT_KEY_COLUMNS,
        expected_keys=capacity_expected_keys,
    )

if capacity_results_df is not None:
    print(
        f"Extra Trees capacity tuning complete: loaded "
        f"{len(capacity_results_df):,} successful definitions from "
        f"{CAPACITY_TUNING_RESULTS_PATH.name}."
    )

else:
    # -----------------------------------------------------------------
    # Recover successful checkpoint evidence by logical definition
    # -----------------------------------------------------------------

    capacity_checkpoint_df = canonical_checkpoint_rows(
        TUNING_PARTS_DIR,
        contract_column="tuning_contract_id",
        contract_id=TUNING_CONTRACT_ID,
        key_columns=TUNING_RESULT_KEY_COLUMNS,
    )

    if not capacity_checkpoint_df.empty:
        checkpoint_keys = capacity_checkpoint_df.apply(
            lambda row: (
                row["modeling_track"],
                row["metric"],
                int(row["horizon"]),
                row["capacity_profile"],
                row["target_representation"],
            ),
            axis=1,
        )

        # The shared tuning checkpoint namespace can contain representation
        # evidence outside 5C. Only the frozen 33 capacity definitions count
        # toward completion of this stage.
        capacity_checkpoint_df = capacity_checkpoint_df.loc[
            checkpoint_keys.isin(capacity_expected_keys)
        ].copy()

    successful_capacity_df = (
        capacity_checkpoint_df.loc[
            capacity_checkpoint_df["status"].eq("success")
        ].copy()
        if not capacity_checkpoint_df.empty
        else pd.DataFrame()
    )

    completed_capacity_keys = logical_key_set(
        successful_capacity_df,
        TUNING_RESULT_KEY_COLUMNS,
    )
    completed_capacity_keys &= capacity_expected_keys

    missing_capacity_keys = capacity_expected_keys - completed_capacity_keys

    print(
        f"Extra Trees capacity restart state: "
        f"{len(completed_capacity_keys):,}/{expected_capacity_fits:,} "
        f"successful definitions; "
        f"{len(missing_capacity_keys):,} unfinished."
    )

    if missing_capacity_keys and not WRITE_CHECKPOINTS:
        raise ValueError(
            "Capacity tuning has unfinished definitions and requires "
            "WRITE_CHECKPOINTS=True."
        )


    # -----------------------------------------------------------------
    # Execute only unfinished logical fits
    # -----------------------------------------------------------------
    # Each forecasting job is loaded once so all pending capacity profiles for
    # that job share the same legal development surface. Progress advances only
    # after a successful checkpoint is durably written.

    if missing_capacity_keys:
        from tqdm import tqdm as tqdm_text

        capacity_progress = tqdm_text(
            total=expected_capacity_fits,
            initial=len(completed_capacity_keys),
            desc="Extra Trees capacity fits",
            unit="fit",
            dynamic_ncols=True,
        )

        capacity_jobs_df = (
            capacity_plan_df[["modeling_track", "metric", "horizon"]]
            .drop_duplicates()
            .sort_values(["modeling_track", "metric", "horizon"])
            .reset_index(drop=True)
        )

        failed_this_pass = 0

        for job in capacity_jobs_df.itertuples(index=False):
            track = job.modeling_track
            metric = job.metric
            horizon = int(job.horizon)

            job_plan_df = capacity_plan_df.loc[
                capacity_plan_df["modeling_track"].eq(track)
                & capacity_plan_df["metric"].eq(metric)
                & capacity_plan_df["horizon"].astype(int).eq(horizon)
            ].copy()

            pending_rows = [
                row
                for row in job_plan_df.itertuples(index=False)
                if (
                    row.modeling_track,
                    row.metric,
                    int(row.horizon),
                    row.capacity_profile,
                    row.target_representation,
                )
                in missing_capacity_keys
            ]

            if not pending_rows:
                continue

            try:
                payload = load_family_screen_job(
                    track=track,
                    metric=metric,
                    horizon=horizon,
                    train_row_cap=TUNING_TRAIN_ROW_CAP,
                )
                payload_error = None
            except Exception as exc:
                payload = None
                payload_error = exc

            for row in pending_rows:
                profile_id = row.capacity_profile
                capacity_label = TUNING_CAPACITY_PROFILES[profile_id]["label"]
                target_representation = row.target_representation

                capacity_progress.set_postfix_str(
                    f"{TRACK_SHORT_LABELS[track]} · "
                    f"{TARGET_SHORT_LABELS[metric]} · h={horizon} · "
                    f"{capacity_label} · failures={failed_this_pass}"
                )

                checkpoint_name = slugify(
                    f"{TUNING_CONTRACT_ID}__{track}__{metric}__h{horizon}__"
                    f"{profile_id}__{target_representation}"
                ) + ".parquet"
                checkpoint_path = TUNING_PARTS_DIR / checkpoint_name

                base_record = {
                    "tuning_contract_id": TUNING_CONTRACT_ID,
                    "modeling_track": track,
                    "metric": metric,
                    "horizon": horizon,
                    "capacity_profile": profile_id,
                    "capacity_label": capacity_label,
                    "target_representation": target_representation,
                }

                if payload_error is not None:
                    record = {
                        **base_record,
                        "status": "failed",
                        "failure_phase": "load_job",
                        "error_type": type(payload_error).__name__,
                        "error_message": str(payload_error),
                        "fit_seconds": np.nan,
                        "predict_seconds": np.nan,
                    }

                    failed_this_pass += 1
                    write_checkpoint_frame(
                        pd.DataFrame([record]),
                        checkpoint_path,
                    )
                    continue

                base_record.update(
                    {
                        "train_rows": payload["train_rows"],
                        "train_origins": payload["train_origins"],
                        "train_forecast_sequences": payload[
                            "train_forecast_sequences"
                        ],
                        "valid_rows": payload["valid_rows"],
                        "valid_origins": payload["valid_origins"],
                        "valid_forecast_sequences": payload[
                            "valid_forecast_sequences"
                        ],
                        "benchmark_id": payload["benchmark_id"],
                        "benchmark_label": payload["benchmark_label"],
                        "benchmark_mae": payload["benchmark_metrics"]["mae"],
                        "benchmark_rmse": payload["benchmark_metrics"]["rmse"],
                        "benchmark_nmae": payload["benchmark_metrics"]["nmae"],
                        "benchmark_bias": payload["benchmark_metrics"]["bias"],
                    }
                )

                try:
                    predictions, metrics, fit_seconds, predict_seconds = (
                        fit_extra_trees_tuning_candidate(
                            payload,
                            profile_id=profile_id,
                            target_representation=target_representation,
                        )
                    )

                    benchmark_mae = float(payload["benchmark_metrics"]["mae"])
                    skill_vs_benchmark_pct = (
                        100.0 * (benchmark_mae - metrics["mae"]) / benchmark_mae
                        if benchmark_mae > 0
                        else np.nan
                    )

                    record = {
                        **base_record,
                        "status": "success",
                        "failure_phase": None,
                        "error_type": None,
                        "error_message": None,
                        **metrics,
                        "skill_vs_benchmark_pct": skill_vs_benchmark_pct,
                        "negative_prediction_count": int(np.sum(predictions < 0)),
                        "fit_seconds": float(fit_seconds),
                        "predict_seconds": float(predict_seconds),
                    }

                    write_checkpoint_frame(
                        pd.DataFrame([record]),
                        checkpoint_path,
                    )

                    capacity_progress.update(1)
                    del predictions

                except Exception as exc:
                    record = {
                        **base_record,
                        "status": "failed",
                        "failure_phase": "fit_or_score",
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                        "mae": np.nan,
                        "rmse": np.nan,
                        "nmae": np.nan,
                        "bias": np.nan,
                        "skill_vs_benchmark_pct": np.nan,
                        "negative_prediction_count": np.nan,
                        "fit_seconds": np.nan,
                        "predict_seconds": np.nan,
                    }

                    failed_this_pass += 1
                    write_checkpoint_frame(
                        pd.DataFrame([record]),
                        checkpoint_path,
                    )

            if payload is not None:
                del payload

            gc.collect()

        capacity_progress.set_postfix_str(f"failures={failed_this_pass}")
        capacity_progress.close()


    # -----------------------------------------------------------------
    # Reconstruct the exact successful 33-fit capacity roster
    # -----------------------------------------------------------------

    capacity_checkpoint_df = canonical_checkpoint_rows(
        TUNING_PARTS_DIR,
        contract_column="tuning_contract_id",
        contract_id=TUNING_CONTRACT_ID,
        key_columns=TUNING_RESULT_KEY_COLUMNS,
    )

    if capacity_checkpoint_df.empty:
        raise RuntimeError(
            "Capacity tuning produced no current-contract checkpoint rows."
        )

    checkpoint_keys = capacity_checkpoint_df.apply(
        lambda row: (
            row["modeling_track"],
            row["metric"],
            int(row["horizon"]),
            row["capacity_profile"],
            row["target_representation"],
        ),
        axis=1,
    )

    capacity_checkpoint_df = capacity_checkpoint_df.loc[
        checkpoint_keys.isin(capacity_expected_keys)
    ].copy()

    capacity_results_df = capacity_checkpoint_df.loc[
        capacity_checkpoint_df["status"].eq("success")
    ].copy()

    successful_capacity_keys = logical_key_set(
        capacity_results_df,
        TUNING_RESULT_KEY_COLUMNS,
    )

    missing_success_keys = capacity_expected_keys - successful_capacity_keys
    extra_success_keys = successful_capacity_keys - capacity_expected_keys

    if missing_success_keys or extra_success_keys:
        raise RuntimeError(
            "Successful capacity roster does not match the planned experiment. "
            f"Missing={len(missing_success_keys)}, "
            f"extra={len(extra_success_keys)}."
        )

    capacity_results_df = (
        capacity_results_df
        .merge(
            capacity_plan_df[
                TUNING_RESULT_KEY_COLUMNS + ["capacity_stage"]
            ],
            on=TUNING_RESULT_KEY_COLUMNS,
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            [
                "capacity_stage",
                "modeling_track",
                "metric",
                "horizon",
                "capacity_profile",
            ]
        )
        .reset_index(drop=True)
    )

    if WRITE_CONSOLIDATED_INTERMEDIATES:
        atomic_parquet_write(
            capacity_results_df,
            CAPACITY_TUNING_RESULTS_PATH,
        )


# ---------------------------------------------------------------------
# Final contract QA and compact execution summary
# ---------------------------------------------------------------------

require_columns(
    capacity_results_df,
    [
        "tuning_contract_id",
        "modeling_track",
        "metric",
        "horizon",
        "capacity_profile",
        "target_representation",
        "capacity_stage",
        "status",
    ],
    "Extra Trees capacity results",
)

if not capacity_results_df["tuning_contract_id"].eq(TUNING_CONTRACT_ID).all():
    raise ValueError("Capacity results contain another tuning contract.")

if capacity_results_df.duplicated(TUNING_RESULT_KEY_COLUMNS).any():
    raise ValueError(
        "Capacity results are not unique by "
        "Track × Target × Horizon × Capacity × Representation."
    )

if len(capacity_results_df) != expected_capacity_fits:
    raise RuntimeError(
        f"Expected {expected_capacity_fits} successful capacity definitions; "
        f"found {len(capacity_results_df)}."
    )

if not capacity_results_df["status"].eq("success").all():
    raise RuntimeError(
        "The consolidated capacity artifact must contain successful "
        "definitions only."
    )

capacity_status_df = (
    capacity_results_df
    .groupby(["capacity_stage", "status"], as_index=False)
    .size()
    .rename(columns={"size": "definitions"})
)

print(
    f"Focused capacity roster complete: "
    f"{len(capacity_results_df):,}/{expected_capacity_fits:,} successful fits."
)
display(capacity_status_df)

,capacity_stage,forecasting_jobs,planned_fits
0,post_cp_confirmation,3,6
1,three_profile_screen,9,27


Extra Trees capacity tuning complete: loaded 33 successful definitions from panel_tree_capacity_results__extra_trees_definition_v1_3capacity_raw_log1p_250k.parquet.
Focused capacity roster complete: 33/33 successful fits.


,capacity_stage,status,definitions
0,post_cp_confirmation,success,6
1,three_profile_screen,success,27


**Findings.** The staged capacity experiment completed cleanly: all **33 of 33 planned fits succeeded**, comprising the 27-fit three-profile screen and all 6 targeted Post-CP confirmation fits, with no failed definitions. The restart contract also behaved as intended, recovering the 27 previously completed logical fits and executing only the six unfinished confirmations. Across the retained experiment, total model-fitting time was approximately **6.75 hours**. The complete evidence surface is therefore available for capacity comparison without reopening the broader Cartesian tuning plan.

### 4.4.1.5D — Synthesize Extra Trees capacity evidence

**Question.** What does the completed 33-fit capacity experiment say about the three candidate Extra Trees profiles, and does the evidence narrow the final choice?

The staged design produces two complementary evidence sets. First, nine representative forecasting jobs compare all three capacity profiles, providing a fully matched basis for pruning a clearly weaker definition. Second, three targeted Post-CP jobs extend coverage for Taxi speed, Taxi demand, and Subway demand using only the two profiles that remain plausible after the broad screen.

This section therefore summarizes capacity in two layers. The three-profile stage tests whether any candidate can be retired. The survivor comparison then evaluates `screen_reference` against `more_flexible` across every job where both were intentionally run, pairing validation MAE improvement with the corresponding fit-time cost.

The objective is not to reward the most flexible model automatically. A final capacity definition should improve forecast accuracy broadly enough to justify its additional computation before full rolling validation.

In [30]:
# ---------------------------------------------------------------------
# 4.4.1.5D — Synthesize Extra Trees capacity evidence
# ---------------------------------------------------------------------
# The staged design answers two related questions with different evidence:
# the nine three-profile jobs test whether any capacity profile can be pruned,
# while all twelve jobs provide a matched comparison between the two profiles
# still relevant to the final accuracy/runtime decision.

capacity_success_df = capacity_results_df.loc[
    capacity_results_df["status"].eq("success")
].copy()

if capacity_success_df.empty:
    raise ValueError("No successful Extra Trees capacity fits are available to summarize.")

job_keys = ["modeling_track", "metric", "horizon"]

capacity_success_df["mae_rank"] = (
    capacity_success_df
    .groupby(job_keys)["mae"]
    .rank(method="min", ascending=True)
)

capacity_success_df["is_mae_winner"] = capacity_success_df["mae_rank"].eq(1)
capacity_success_df["beats_benchmark"] = (
    capacity_success_df["skill_vs_benchmark_pct"] > 0
)


# ---------------------------------------------------------------------
# Broad profile summary
# ---------------------------------------------------------------------
# Jobs scored differs by design: the more-regularized profile participates
# only in the nine-job three-profile stage, while the two surviving profiles
# also participate in the three targeted Post-CP confirmation jobs.

capacity_profile_summary_df = (
    capacity_success_df
    .groupby(["capacity_profile", "capacity_label"], as_index=False)
    .agg(
        jobs_scored=("mae", "size"),
        mae_wins=("is_mae_winner", "sum"),
        median_mae_rank=("mae_rank", "median"),
        jobs_beating_benchmark=("beats_benchmark", "sum"),
        median_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "median"),
        worst_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "min"),
        median_fit_seconds=("fit_seconds", "median"),
        max_fit_seconds=("fit_seconds", "max"),
        negative_predictions=("negative_prediction_count", "sum"),
    )
)

capacity_profile_summary_df["jobs_beating_benchmark_pct"] = (
    100.0
    * capacity_profile_summary_df["jobs_beating_benchmark"]
    / capacity_profile_summary_df["jobs_scored"]
)

capacity_profile_summary_df = (
    capacity_profile_summary_df
    .sort_values(
        ["median_mae_rank", "mae_wins", "median_skill_vs_benchmark_pct"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

capacity_profile_display_df = capacity_profile_summary_df[
    [
        "capacity_label",
        "jobs_scored",
        "mae_wins",
        "median_mae_rank",
        "jobs_beating_benchmark_pct",
        "median_skill_vs_benchmark_pct",
        "worst_skill_vs_benchmark_pct",
        "median_fit_seconds",
        "max_fit_seconds",
        "negative_predictions",
    ]
].rename(columns={
    "capacity_label": "Capacity",
    "jobs_scored": "Jobs",
    "mae_wins": "MAE wins",
    "median_mae_rank": "Median rank",
    "jobs_beating_benchmark_pct": "Beat benchmark %",
    "median_skill_vs_benchmark_pct": "Median skill %",
    "worst_skill_vs_benchmark_pct": "Worst skill %",
    "median_fit_seconds": "Median fit sec",
    "max_fit_seconds": "Max fit sec",
    "negative_predictions": "Negative predictions",
})

for column in [
    "Median rank",
    "Beat benchmark %",
    "Median skill %",
    "Worst skill %",
    "Median fit sec",
    "Max fit sec",
]:
    capacity_profile_display_df[column] = (
        capacity_profile_display_df[column].round(2)
    )

display(capacity_profile_display_df)


# ---------------------------------------------------------------------
# Stage-1 pruning evidence
# ---------------------------------------------------------------------
# Restrict this comparison to the nine jobs where all three capacity profiles
# were evaluated. This keeps the pruning evidence fully matched.

three_profile_df = capacity_success_df.loc[
    capacity_success_df["capacity_stage"].eq("three_profile_screen")
].copy()

three_profile_job_count = (
    three_profile_df[job_keys]
    .drop_duplicates()
    .shape[0]
)

if three_profile_job_count != len(BASE_CAPACITY_JOBS):
    raise RuntimeError(
        f"Expected {len(BASE_CAPACITY_JOBS)} complete three-profile jobs; "
        f"found {three_profile_job_count}."
    )

three_profile_summary_df = (
    three_profile_df
    .groupby(["capacity_profile", "capacity_label"], as_index=False)
    .agg(
        jobs_scored=("mae", "size"),
        mae_wins=("is_mae_winner", "sum"),
        median_mae_rank=("mae_rank", "median"),
        median_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "median"),
        worst_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "min"),
        median_fit_seconds=("fit_seconds", "median"),
    )
    .sort_values(
        ["median_mae_rank", "mae_wins", "median_skill_vs_benchmark_pct"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

three_profile_display_df = three_profile_summary_df[
    [
        "capacity_label",
        "jobs_scored",
        "mae_wins",
        "median_mae_rank",
        "median_skill_vs_benchmark_pct",
        "worst_skill_vs_benchmark_pct",
        "median_fit_seconds",
    ]
].rename(columns={
    "capacity_label": "Capacity",
    "jobs_scored": "Matched jobs",
    "mae_wins": "MAE wins",
    "median_mae_rank": "Median rank",
    "median_skill_vs_benchmark_pct": "Median skill %",
    "worst_skill_vs_benchmark_pct": "Worst skill %",
    "median_fit_seconds": "Median fit sec",
})

for column in [
    "Median rank",
    "Median skill %",
    "Worst skill %",
    "Median fit sec",
]:
    three_profile_display_df[column] = three_profile_display_df[column].round(2)

display(three_profile_display_df)


# ---------------------------------------------------------------------
# Matched survivor comparison
# ---------------------------------------------------------------------
# The final capacity question is not three-way. Compare screen_reference and
# more_flexible directly across every job where both were intentionally run.

survivor_profiles = ["screen_reference", "more_flexible"]

survivor_df = capacity_success_df.loc[
    capacity_success_df["capacity_profile"].isin(survivor_profiles)
].copy()

survivor_mae_wide_df = (
    survivor_df
    .pivot(
        index=job_keys,
        columns="capacity_profile",
        values="mae",
    )
    .reset_index()
)

survivor_fit_wide_df = (
    survivor_df
    .pivot(
        index=job_keys,
        columns="capacity_profile",
        values="fit_seconds",
    )
    .reset_index()
)

survivor_compare_df = survivor_mae_wide_df.merge(
    survivor_fit_wide_df,
    on=job_keys,
    how="inner",
    suffixes=("_mae", "_fit_seconds"),
    validate="one_to_one",
)

required_survivor_columns = [
    "screen_reference_mae",
    "more_flexible_mae",
    "screen_reference_fit_seconds",
    "more_flexible_fit_seconds",
]

require_columns(
    survivor_compare_df,
    required_survivor_columns,
    "Matched capacity survivor comparison",
)

# Positive MAE improvement means more_flexible reduced error relative to the
# screening reference. Runtime multiple measures the corresponding compute cost.
survivor_compare_df["flexible_mae_improvement_pct"] = (
    100.0
    * (
        survivor_compare_df["screen_reference_mae"]
        - survivor_compare_df["more_flexible_mae"]
    )
    / survivor_compare_df["screen_reference_mae"]
)

survivor_compare_df["flexible_fit_time_multiple"] = (
    survivor_compare_df["more_flexible_fit_seconds"]
    / survivor_compare_df["screen_reference_fit_seconds"]
)

survivor_compare_df["winning_capacity"] = np.where(
    survivor_compare_df["more_flexible_mae"]
    < survivor_compare_df["screen_reference_mae"],
    "More flexible",
    "Screen reference",
)

expected_survivor_jobs = (
    capacity_plan_df[job_keys]
    .drop_duplicates()
    .shape[0]
)

if len(survivor_compare_df) != expected_survivor_jobs:
    raise RuntimeError(
        f"Expected {expected_survivor_jobs} matched survivor jobs; "
        f"found {len(survivor_compare_df)}."
    )

survivor_summary_df = pd.DataFrame(
    {
        "Matched jobs": [len(survivor_compare_df)],
        "More flexible MAE wins": [
            int((survivor_compare_df["winning_capacity"] == "More flexible").sum())
        ],
        "Screen reference MAE wins": [
            int((survivor_compare_df["winning_capacity"] == "Screen reference").sum())
        ],
        "Median flexible MAE improvement %": [
            survivor_compare_df["flexible_mae_improvement_pct"].median()
        ],
        "Worst flexible MAE improvement %": [
            survivor_compare_df["flexible_mae_improvement_pct"].min()
        ],
        "Best flexible MAE improvement %": [
            survivor_compare_df["flexible_mae_improvement_pct"].max()
        ],
        "Median flexible fit-time multiple": [
            survivor_compare_df["flexible_fit_time_multiple"].median()
        ],
    }
)

for column in [
    "Median flexible MAE improvement %",
    "Worst flexible MAE improvement %",
    "Best flexible MAE improvement %",
    "Median flexible fit-time multiple",
]:
    survivor_summary_df[column] = survivor_summary_df[column].round(2)

display(survivor_summary_df)

,Capacity,Jobs,MAE wins,Median rank,Beat benchmark %,Median skill %,Worst skill %,Median fit sec,Max fit sec,Negative predictions
0,More flexible,12,12,1.0,100.00,18.34,1.22,1429.02,2885.74,0.0
1,Screen reference,12,0,2.0,91.67,12.89,-7.03,257.82,445.78,0.0
2,More regularized,9,0,3.0,55.56,7.73,-10.75,133.77,203.08,0.0


,Capacity,Matched jobs,MAE wins,Median rank,Median skill %,Worst skill %,Median fit sec
0,More flexible,9,9,1.0,18.42,8.82,1523.99
1,Screen reference,9,0,2.0,13.73,5.74,271.30
2,More regularized,9,0,3.0,7.73,-10.75,133.77


,Matched jobs,More flexible MAE wins,Screen reference MAE wins,Median flexible MAE improvement %,Worst flexible MAE improvement %,Best flexible MAE improvement %,Median flexible fit-time multiple
0,12,12,0,4.01,0.46,10.78,6.03


**Findings.** The capacity evidence strongly narrows the Extra Trees definition. In the nine fully matched three-profile jobs, **More flexible wins all 9**, with a median rank of 1, **18.42% median skill versus the frozen benchmark**, and even its weakest job remaining **8.82% better than the benchmark**. Screen reference ranks second on all nine, while More regularized ranks third, wins none, and falls below the benchmark in its worst case, so the stronger-regularization branch does not warrant further tuning. The expanded survivor comparison is even clearer: across all **12 matched jobs**, More flexible beats Screen reference **12 of 12 times**, reducing MAE by a median **4.01%**, with improvements ranging from **0.46% to 10.78%**. That consistency comes at a meaningful computational cost—the flexible profile takes about **6.03×** as long to fit at the median—but the added capacity produces a repeatable accuracy gain rather than isolated wins.

### 4.4.1.5E — Inspect job-level capacity stability

**Question.** Is the capacity result broad and stable enough to support a single Extra Trees definition, or is the apparent advantage concentrated in only a few forecasting cases?

Aggregate averages can hide important heterogeneity across modeling tracks, targets, and horizons. This section therefore inspects the winning capacity profile for each staged job and then focuses on the direct `screen_reference` versus `more_flexible` comparison.

For each matched survivor job, the evidence reports both the MAE change from using the more-flexible profile and its fit-time multiple relative to the screening reference. Track- and target-level summaries then show whether the accuracy/runtime tradeoff is consistent or fragmented.

If the survivor comparison is sufficiently clear and stable, the next step can freeze Extra Trees capacity. If the evidence remains materially mixed, any additional tuning should be limited to the specific unresolved Track × Target × Horizon cases rather than reopening a broad hyperparameter grid.

In [31]:
# ---------------------------------------------------------------------
# 4.4.1.5E — Inspect job-level capacity winners and stability
# ---------------------------------------------------------------------
# Aggregate summaries can hide fragmented behavior. Inspect every matched job
# directly, then summarize whether the flexible profile's accuracy advantage is
# concentrated in particular tracks or targets or remains broadly consistent.

job_winners_df = (
    capacity_success_df.loc[capacity_success_df["mae_rank"].eq(1)]
    .sort_values(job_keys + ["mae"])
    .drop_duplicates(job_keys, keep="first")
    .reset_index(drop=True)
)

expected_capacity_jobs = (
    capacity_plan_df[job_keys]
    .drop_duplicates()
    .shape[0]
)

if len(job_winners_df) != expected_capacity_jobs:
    raise RuntimeError(
        f"Expected winners for {expected_capacity_jobs} capacity jobs; "
        f"found {len(job_winners_df)}."
    )

job_winner_display_df = job_winners_df[
    [
        "modeling_track",
        "metric",
        "horizon",
        "capacity_stage",
        "capacity_label",
        "mae",
        "skill_vs_benchmark_pct",
        "fit_seconds",
    ]
].rename(columns={
    "modeling_track": "Track",
    "metric": "Target",
    "horizon": "Horizon",
    "capacity_stage": "Stage",
    "capacity_label": "Winning capacity",
    "mae": "MAE",
    "skill_vs_benchmark_pct": "Skill vs benchmark %",
    "fit_seconds": "Fit sec",
})

for column in ["MAE", "Skill vs benchmark %", "Fit sec"]:
    job_winner_display_df[column] = job_winner_display_df[column].round(3)

display(job_winner_display_df)


# ---------------------------------------------------------------------
# Final survivor comparison by individual job
# ---------------------------------------------------------------------
# This table is the decision-facing view: it shows the accuracy gain from
# choosing more_flexible instead of screen_reference alongside its runtime cost.

survivor_job_display_df = survivor_compare_df[
    [
        "modeling_track",
        "metric",
        "horizon",
        "screen_reference_mae",
        "more_flexible_mae",
        "flexible_mae_improvement_pct",
        "flexible_fit_time_multiple",
        "winning_capacity",
    ]
].copy()

survivor_job_display_df = survivor_job_display_df.rename(columns={
    "modeling_track": "Track",
    "metric": "Target",
    "horizon": "Horizon",
    "screen_reference_mae": "Reference MAE",
    "more_flexible_mae": "Flexible MAE",
    "flexible_mae_improvement_pct": "Flexible MAE improvement %",
    "flexible_fit_time_multiple": "Flexible fit-time multiple",
    "winning_capacity": "MAE winner",
})

for column in [
    "Reference MAE",
    "Flexible MAE",
    "Flexible MAE improvement %",
    "Flexible fit-time multiple",
]:
    survivor_job_display_df[column] = survivor_job_display_df[column].round(3)

display(
    survivor_job_display_df.sort_values(
        ["Track", "Target", "Horizon"]
    ).reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Stability by modeling track
# ---------------------------------------------------------------------

survivor_track_summary_df = (
    survivor_compare_df
    .groupby("modeling_track", as_index=False)
    .agg(
        matched_jobs=("metric", "size"),
        flexible_wins=(
            "winning_capacity",
            lambda x: int((x == "More flexible").sum()),
        ),
        median_flexible_mae_improvement_pct=(
            "flexible_mae_improvement_pct",
            "median",
        ),
        worst_flexible_mae_improvement_pct=(
            "flexible_mae_improvement_pct",
            "min",
        ),
        median_flexible_fit_time_multiple=(
            "flexible_fit_time_multiple",
            "median",
        ),
    )
)

survivor_track_summary_df["flexible_win_pct"] = (
    100.0
    * survivor_track_summary_df["flexible_wins"]
    / survivor_track_summary_df["matched_jobs"]
)

survivor_track_display_df = survivor_track_summary_df[
    [
        "modeling_track",
        "matched_jobs",
        "flexible_wins",
        "flexible_win_pct",
        "median_flexible_mae_improvement_pct",
        "worst_flexible_mae_improvement_pct",
        "median_flexible_fit_time_multiple",
    ]
].rename(columns={
    "modeling_track": "Track",
    "matched_jobs": "Matched jobs",
    "flexible_wins": "Flexible wins",
    "flexible_win_pct": "Flexible win %",
    "median_flexible_mae_improvement_pct": "Median MAE improvement %",
    "worst_flexible_mae_improvement_pct": "Worst MAE improvement %",
    "median_flexible_fit_time_multiple": "Median fit-time multiple",
})

for column in [
    "Flexible win %",
    "Median MAE improvement %",
    "Worst MAE improvement %",
    "Median fit-time multiple",
]:
    survivor_track_display_df[column] = survivor_track_display_df[column].round(2)

display(survivor_track_display_df)


# ---------------------------------------------------------------------
# Stability by target
# ---------------------------------------------------------------------

survivor_target_summary_df = (
    survivor_compare_df
    .groupby("metric", as_index=False)
    .agg(
        matched_jobs=("horizon", "size"),
        flexible_wins=(
            "winning_capacity",
            lambda x: int((x == "More flexible").sum()),
        ),
        median_flexible_mae_improvement_pct=(
            "flexible_mae_improvement_pct",
            "median",
        ),
        worst_flexible_mae_improvement_pct=(
            "flexible_mae_improvement_pct",
            "min",
        ),
        median_flexible_fit_time_multiple=(
            "flexible_fit_time_multiple",
            "median",
        ),
    )
)

survivor_target_summary_df["flexible_win_pct"] = (
    100.0
    * survivor_target_summary_df["flexible_wins"]
    / survivor_target_summary_df["matched_jobs"]
)

survivor_target_display_df = survivor_target_summary_df[
    [
        "metric",
        "matched_jobs",
        "flexible_wins",
        "flexible_win_pct",
        "median_flexible_mae_improvement_pct",
        "worst_flexible_mae_improvement_pct",
        "median_flexible_fit_time_multiple",
    ]
].rename(columns={
    "metric": "Target",
    "matched_jobs": "Matched jobs",
    "flexible_wins": "Flexible wins",
    "flexible_win_pct": "Flexible win %",
    "median_flexible_mae_improvement_pct": "Median MAE improvement %",
    "worst_flexible_mae_improvement_pct": "Worst MAE improvement %",
    "median_flexible_fit_time_multiple": "Median fit-time multiple",
})

for column in [
    "Flexible win %",
    "Median MAE improvement %",
    "Worst MAE improvement %",
    "Median fit-time multiple",
]:
    survivor_target_display_df[column] = survivor_target_display_df[column].round(2)

display(survivor_target_display_df)


# ---------------------------------------------------------------------
# Decision-facing completeness summary
# ---------------------------------------------------------------------

print(
    f"Three-profile capacity jobs:       "
    f"{len(BASE_CAPACITY_JOBS):,}/{len(BASE_CAPACITY_JOBS):,}"
)
print(
    f"Targeted confirmation jobs:       "
    f"{len(CAPACITY_CONFIRMATION_JOBS):,}/{len(CAPACITY_CONFIRMATION_JOBS):,}"
)
print(
    f"Matched survivor comparisons:     "
    f"{len(survivor_compare_df):,}/{expected_capacity_jobs:,}"
)
print(
    f"Successful capacity fits:         "
    f"{len(capacity_success_df):,}/{expected_capacity_fits:,}"
)
print(
    f"Total retained capacity fit hours: "
    f"{capacity_success_df['fit_seconds'].sum() / 3600:.2f}"
)

,Track,Target,Horizon,Stage,Winning capacity,MAE,Skill vs benchmark %,Fit sec
0,full_history_regime_aware,fhvhv_avg_trip_speed,5,three_profile_screen,More flexible,0.891,18.570,1372.802
1,full_history_regime_aware,fhvhv_trip_count,5,three_profile_screen,More flexible,42.097,12.680,1166.038
2,full_history_regime_aware,subway_ridership,5,three_profile_screen,More flexible,310.210,18.419,1102.273
3,full_history_regime_aware,taxi_avg_trip_speed,5,three_profile_screen,More flexible,1.644,24.633,1523.992
4,full_history_regime_aware,taxi_trip_count,5,three_profile_screen,More flexible,12.125,8.822,875.702
5,post_cp_primary,fhvhv_avg_trip_speed,1,three_profile_screen,More flexible,0.859,20.643,2588.755
6,post_cp_primary,fhvhv_avg_trip_speed,2,three_profile_screen,More flexible,0.900,16.839,2555.828
7,post_cp_primary,fhvhv_avg_trip_speed,5,three_profile_screen,More flexible,0.872,19.454,2885.742
8,post_cp_primary,fhvhv_trip_count,1,three_profile_screen,More flexible,39.527,18.269,1943.998
9,post_cp_primary,subway_ridership,5,post_cp_confirmation,More flexible,319.461,17.790,935.099


capacity_profile,Track,Target,Horizon,Reference MAE,Flexible MAE,Flexible MAE improvement %,Flexible fit-time multiple,MAE winner
0,full_history_regime_aware,fhvhv_avg_trip_speed,5,0.901,0.891,1.063,5.619,More flexible
1,full_history_regime_aware,fhvhv_trip_count,5,44.735,42.097,5.896,5.505,More flexible
2,full_history_regime_aware,subway_ridership,5,339.245,310.210,8.559,5.690,More flexible
3,full_history_regime_aware,taxi_avg_trip_speed,5,1.653,1.644,0.573,5.617,More flexible
4,full_history_regime_aware,taxi_trip_count,5,12.534,12.125,3.265,6.690,More flexible
5,post_cp_primary,fhvhv_avg_trip_speed,1,0.899,0.859,4.424,6.949,More flexible
6,post_cp_primary,fhvhv_avg_trip_speed,2,0.934,0.900,3.602,7.131,More flexible
7,post_cp_primary,fhvhv_avg_trip_speed,5,0.876,0.872,0.465,6.473,More flexible
8,post_cp_primary,fhvhv_trip_count,1,44.304,39.527,10.783,5.889,More flexible
9,post_cp_primary,subway_ridership,5,341.769,319.461,6.527,6.162,More flexible


,Track,Matched jobs,Flexible wins,Flexible win %,Median MAE improvement %,Worst MAE improvement %,Median fit-time multiple
0,full_history_regime_aware,5,5,100.0,3.27,0.57,5.62
1,post_cp_primary,7,7,100.0,4.42,0.46,6.25


,Target,Matched jobs,Flexible wins,Flexible win %,Median MAE improvement %,Worst MAE improvement %,Median fit-time multiple
0,fhvhv_avg_trip_speed,4,4,100.0,2.33,0.46,6.71
1,fhvhv_trip_count,2,2,100.0,8.34,5.90,5.70
2,subway_ridership,2,2,100.0,7.54,6.53,5.93
3,taxi_avg_trip_speed,2,2,100.0,0.95,0.57,5.94
4,taxi_trip_count,2,2,100.0,5.49,3.27,5.95


Three-profile capacity jobs:       9/9
Targeted confirmation jobs:       3/3
Matched survivor comparisons:     12/12
Successful capacity fits:         33/33
Total retained capacity fit hours: 6.75


**Findings.** The More flexible advantage is broad rather than concentrated in one modeling track, target, or horizon. It wins **all 5 Full-History comparisons and all 7 Post-CP comparisons**, with median MAE improvements of **3.27%** and **4.42%**, respectively. The same pattern holds across every target family: flexible wins every matched Taxi, FHVHV, and Subway job. The gains are largest for FHVHV trip count (**8.34% median improvement**) and Subway ridership (**7.54%**), remain material for Taxi trip count (**5.49%**), and are smaller but still consistently positive for the two speed targets. No target or track provides evidence for a separate capacity rule, and the screening reference never wins a matched comparison. The capacity result is therefore sufficiently stable to support one shared Extra Trees capacity definition; additional capacity tuning is not warranted unless a later production-scale validation exposes a new problem.

> ✅ Decision — Retire Extra Trees from production forecasting and advance LightGBM\.

Extra Trees remains the strongest bounded\-screen accuracy model, but the tuning evidence exposes a poor fit with the intended forecasting workflow\. The selected More flexible profile improves MAE, yet requires substantially more fitting time than the already\-slow Extra Trees screening reference\. Earlier cadence testing also showed that the model benefits from monthly refreshing, which would require repeatedly rebuilding the forest from scratch as the legal training history expands\.

This creates an unfavorable production tradeoff: the architecture that performs best on bounded accuracy would also be the most expensive to retrain repeatedly across targets, horizons, and modeling tracks\.

LightGBM offers a materially different tradeoff already established in the controlled family screen\. It remained competitive with the frozen benchmark while fitting approximately two orders of magnitude faster than Extra Trees\. That efficiency advantage is large enough to justify advancing LightGBM as the production tree\-family candidate rather than continuing to optimize an increasingly expensive Extra Trees specification\.

No additional Extra Trees tuning or production\-scale validation will be performed\.

### Section Summary

Extra Trees began this stage as the strongest accuracy candidate from the controlled family screen. Subsequent experiments clarified its production tradeoff.

Monthly refreshing improved forecast accuracy, indicating that the model benefits from incorporating newly observed outcomes rather than remaining fixed throughout validation. Capacity tuning also found a more accurate Extra Trees definition. However, both findings increase the computational burden of the intended workflow: monthly refreshing requires repeated full forest rebuilds, while the selected higher-capacity definition is substantially slower than the screening reference.

The result is a clear distinction between **bounded predictive strength** and **production suitability**. Extra Trees remains the strongest accuracy result from the initial screen, but its retraining cost is disproportionate to the forecasting workflow required here.

LightGBM therefore advances as the production tree-family candidate because it combines competitive benchmark performance with dramatically lower fitting cost.

### 4.4.1.5F — Define the target-representation confirmation surface

The Extra Trees capacity choice is complete. The remaining question applies only to the three count targets:

> Should the model learn the original counts directly, or should it fit on log1p-compressed counts and convert its forecasts back before scoring?

Log1p compresses very large counts during fitting, which can keep a few high-volume observations from dominating the model. Every forecast is converted back to the original units before MAE, RMSE, NMAE, bias, benchmark skill, or negative-prediction checks are calculated. Speed targets stay on their raw scale.

This is a matched comparison: both versions use the same data, predictors, h=5 horizon, and frozen More flexible capacity profile. The only changing ingredient is the target representation. Both ordinary tracks are included, creating 2 tracks × 3 count targets × 2 representations = 12 logical fits.

Existing compatible fits may already be recoverable from the shared tuning checkpoints. If both tracks agree for a target, that supports one simple target-level representation rule rather than separate rules for each track.

## 4.4.1.6 — Establish the LightGBM Production Definition

LightGBM now advances as the tree-model candidate for full validation.

The earlier family comparison showed two things: LightGBM can beat the simple seasonal benchmark, and it can do so much faster than the heavier bagged-tree models.

Before freezing the final LightGBM setup, three questions remain:

1. **Model size:** can a somewhat more flexible LightGBM improve accuracy without giving up its speed advantage?
2. **Count-target scale:** should Taxi trips, FHVHV trips, and Subway ridership be modeled in their original units or after a `log1p` transformation?
3. **Retraining frequency:** does rebuilding LightGBM each month improve forecasts enough to justify the extra work?

Everything else stays fixed: the forecasting data, predictors, modeling tracks, horizons, validation periods, leakage rules, and seasonal benchmark.

The goal is not to reopen a broad model search. It is to make a few focused choices about the LightGBM model that already performed well.

<img src="image-20260907-193745.png" width="" align="" />

### 4.4.1.6A — Define a bounded LightGBM capacity experiment

LightGBM already has one useful reference definition from the family screen. That configuration established that the architecture can beat the frozen benchmark while fitting dramatically faster than the bagged-tree alternatives, but it was chosen as a reasonable screening configuration rather than a final production definition.

The first LightGBM-specific question is therefore:

> **Can modestly changing model capacity recover additional forecast accuracy without giving up LightGBM's computational advantage?**

Three deliberately different profiles are compared:

- **Screen reference** preserves the exact LightGBM definition used in the family screen.
- **More expressive** allows more leaves and boosting rounds, giving the model additional capacity to represent nonlinear interactions.
- **More regularized** constrains leaf complexity and strengthens regularization, testing whether the reference model is fitting unnecessary detail.

The comparison reuses the same **20 representative Track × Target × Horizon jobs** from the original family screen and the same 250,000-row origin-balanced training cap. That surface contains every Post-CP Target × Horizon combination plus five Full-History h=5 transfer checks, so it provides broad evidence without approaching production-scale cost.

Targets remain on their **raw scale** here. Count representation is a separate question and will be tested only after one LightGBM capacity profile is selected.

The final holdout remains untouched.

In [32]:
# ---------------------------------------------------------------------
# 4.4.1.6A — Freeze the bounded LightGBM capacity experiment
# ---------------------------------------------------------------------

LGBM_CAPACITY_CONTRACT_ID = "lgbm_capacity_v1"
LGBM_CAPACITY_TRAIN_ROW_CAP = SCREEN_TRAIN_ROW_CAP

# The reference profile is copied directly from the controlled family screen.
# Keeping it unchanged gives every tuned profile an exact paired benchmark
# against the LightGBM result that originally justified this production pivot.
LGBM_SCREEN_REFERENCE_PARAMS = dict(
    FAMILY_SCREEN_CONFIGS["lightgbm"]["params"]
)

LGBM_CAPACITY_PROFILES = {
    "screen_reference": {
        "label": "Screen reference",
        "rationale": "Exact LightGBM definition used in the family screen.",
        "params": dict(LGBM_SCREEN_REFERENCE_PARAMS),
    },

    # More leaves and boosting rounds allow finer interactions among lagged,
    # spatial, contextual, and Taxi Zone effects. The smaller learning rate
    # tempers each individual boosting step as total capacity increases.
    "more_expressive": {
        "label": "More expressive",
        "rationale": "More leaves and boosting rounds with a slightly smaller learning rate.",
        "params": {
            **LGBM_SCREEN_REFERENCE_PARAMS,
            "n_estimators": 400,
            "learning_rate": 0.04,
            "num_leaves": 63,
            "min_child_samples": 20,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
        },
    },

    # This profile asks the opposite question: whether the screen reference
    # benefits from stronger smoothing. Larger leaves and stronger penalties
    # can help when a very flexible leaf-wise model starts fitting local noise.
    "more_regularized": {
        "label": "More regularized",
        "rationale": "Fewer leaves, larger child support, and stronger L1/L2 penalties.",
        "params": {
            **LGBM_SCREEN_REFERENCE_PARAMS,
            "n_estimators": 350,
            "learning_rate": 0.04,
            "num_leaves": 24,
            "min_child_samples": 60,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.50,
            "reg_lambda": 3.00,
        },
    },
}

LGBM_CAPACITY_PROFILE_ORDER = [
    "screen_reference",
    "more_expressive",
    "more_regularized",
]

# Reuse the exact 20-job screen roster rather than constructing another
# partially overlapping tuning surface.
lgbm_capacity_jobs_df = (
    screen_jobs_df[
        ["modeling_track", "metric", "horizon", "job_id"]
    ]
    .drop_duplicates()
    .copy()
)

if len(lgbm_capacity_jobs_df) != 20:
    raise RuntimeError(
        f"Expected the frozen 20-job family-screen roster; "
        f"found {len(lgbm_capacity_jobs_df)} jobs."
    )

if lgbm_capacity_jobs_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise ValueError(
        "LightGBM capacity jobs are not unique by Track × Target × Horizon."
    )

lgbm_capacity_plan_df = pd.DataFrame(
    [
        {
            "modeling_track": job.modeling_track,
            "metric": job.metric,
            "horizon": int(job.horizon),
            "job_id": job.job_id,
            "capacity_profile": profile_id,
            "capacity_label": LGBM_CAPACITY_PROFILES[profile_id]["label"],
        }
        for job in lgbm_capacity_jobs_df.itertuples(index=False)
        for profile_id in LGBM_CAPACITY_PROFILE_ORDER
    ]
)

expected_lgbm_capacity_fits = (
    len(lgbm_capacity_jobs_df)
    * len(LGBM_CAPACITY_PROFILE_ORDER)
)

if expected_lgbm_capacity_fits != 60:
    raise RuntimeError(
        f"Expected 60 bounded LightGBM fits; "
        f"found {expected_lgbm_capacity_fits}."
    )

lgbm_capacity_workload_df = pd.DataFrame(
    [
        {
            "Representative jobs": len(lgbm_capacity_jobs_df),
            "Capacity profiles": len(LGBM_CAPACITY_PROFILE_ORDER),
            "Planned fits": expected_lgbm_capacity_fits,
            "Training-row cap": LGBM_CAPACITY_TRAIN_ROW_CAP,
            "Target representation": "Raw",
        }
    ]
)

display(lgbm_capacity_workload_df)

,Representative jobs,Capacity profiles,Planned fits,Training-row cap,Target representation
0,20,3,60,250000,Raw


**Findings.** The LightGBM capacity experiment is intentionally compact but broad: 20 representative Track × Target × Horizon jobs are each evaluated under three capacity profiles, producing 60 planned fits. The experiment retains the same 250,000-row training cap and raw target representation used for the family screen so that the capacity comparison isolates model flexibility rather than introducing a new training-size or transformation decision. This gives the tuning stage coverage across both modeling tracks, all five forecasting targets, and all three horizons represented in the original screening surface while keeping the final holdout untouched.

### 4.4.1.6B — Run or recover the bounded LightGBM capacity fits

Each of the 60 logical units fits one LightGBM capacity profile on one representative forecasting job.

The experiment uses the same leakage-safe training and validation surfaces as the original family screen. Taxi Zone identity remains categorical and is passed through LightGBM's native categorical interface rather than being converted into an artificial numeric ordering or one-hot expansion.

The run is checkpointed by forecasting job. After all three profiles for one job finish, that job's results are written immediately. If execution is interrupted, rerunning the cell reuses completed jobs and computes only the missing ones.

MAE remains the sole selection metric. RMSE, NMAE, bias, benchmark skill, negative predictions, and fitting time remain supporting diagnostics.

In [33]:
# ---------------------------------------------------------------------
# 4.4.1.6B — Execute or recover bounded LightGBM capacity tuning
# ---------------------------------------------------------------------

RUN_LGBM_CAPACITY = True
RESET_LGBM_CAPACITY = False

LGBM_CAPACITY_ROOT = (
    INTERMEDIATE_DIR
    / LGBM_CAPACITY_CONTRACT_ID
)
LGBM_CAPACITY_PARTS_DIR = (
    LGBM_CAPACITY_ROOT
    / "parts"
)
LGBM_CAPACITY_RESULTS_PATH = (
    LGBM_CAPACITY_ROOT
    / "results.parquet"
)

if RESET_LGBM_CAPACITY:
    reset_checkpoint_dir(LGBM_CAPACITY_ROOT)

LGBM_CAPACITY_PARTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def lgbm_capacity_job_path(track, metric, horizon):
    """Return the durable checkpoint owned by one three-profile tuning job."""
    job_slug = slugify(
        f"{track}__{metric}__h{int(horizon)}"
    )
    return LGBM_CAPACITY_PARTS_DIR / f"{job_slug}.parquet"


def lgbm_capacity_job_is_complete(track, metric, horizon):
    """A job is complete only when all three current-contract profiles succeed."""
    path = lgbm_capacity_job_path(
        track,
        metric,
        horizon,
    )

    if not path.exists():
        return False

    try:
        result_df = pd.read_parquet(path)
    except Exception:
        return False

    if len(result_df) != len(LGBM_CAPACITY_PROFILE_ORDER):
        return False

    if not result_df["status"].eq("success").all():
        return False

    if not result_df["capacity_contract_id"].eq(
        LGBM_CAPACITY_CONTRACT_ID
    ).all():
        return False

    return (
        set(result_df["capacity_profile"])
        == set(LGBM_CAPACITY_PROFILE_ORDER)
    )


def fit_lgbm_capacity_profile(
    payload,
    profile_id,
):
    """Fit one frozen LightGBM profile on one already-legal screen surface."""
    config = LGBM_CAPACITY_PROFILES[profile_id]
    categorical_names = payload["categorical_names"]

    # load_family_screen_job already applies numeric typing, legal target
    # availability, exact frozen-benchmark support, and the origin-balanced
    # training cap. LightGBM-specific preparation therefore only needs to
    # restore training-defined categorical levels.
    train_frame, valid_frame = prepare_lightgbm_frames(
        payload["X_train"],
        payload["X_valid"],
        categorical_names,
    )

    model = LGBMRegressor(
        **config["params"]
    )

    fit_start = perf_counter()
    model.fit(
        train_frame,
        payload["y_train"],
        categorical_feature=categorical_names,
    )
    fit_seconds = perf_counter() - fit_start

    predict_start = perf_counter()
    predictions = model.predict(valid_frame)
    predict_seconds = perf_counter() - predict_start

    predictions = np.asarray(
        predictions,
        dtype=float,
    )

    if not np.isfinite(predictions).all():
        raise ValueError(
            "LightGBM produced non-finite validation predictions."
        )

    model_metrics = regression_metrics(
        payload["y_valid"],
        predictions,
    )
    benchmark_metrics = payload["benchmark_metrics"]

    benchmark_mae = float(
        benchmark_metrics["mae"]
    )
    skill_vs_benchmark_pct = (
        100.0
        * (benchmark_mae - model_metrics["mae"])
        / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    result = {
        "capacity_contract_id": LGBM_CAPACITY_CONTRACT_ID,
        "capacity_profile": profile_id,
        "capacity_label": config["label"],
        "status": "success",
        "train_rows": payload["train_rows"],
        "train_origins": payload["train_origins"],
        "validation_rows": payload["valid_rows"],
        "validation_origins": payload["valid_origins"],
        "benchmark_id": payload["benchmark_id"],
        "benchmark_label": payload["benchmark_label"],
        "mae": float(model_metrics["mae"]),
        "rmse": float(model_metrics["rmse"]),
        "nmae": float(model_metrics["nmae"]),
        "bias": float(model_metrics["bias"]),
        "benchmark_mae": benchmark_mae,
        "skill_vs_benchmark_pct": float(
            skill_vs_benchmark_pct
        ),
        "beats_benchmark": bool(
            skill_vs_benchmark_pct > 0
        ),
        "negative_prediction_count": int(
            (predictions < 0).sum()
        ),
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
    }

    del (
        train_frame,
        valid_frame,
        model,
        predictions,
    )
    gc.collect()

    return result


completed_lgbm_capacity_jobs = {
    (
        row.modeling_track,
        row.metric,
        int(row.horizon),
    )
    for row in lgbm_capacity_jobs_df.itertuples(index=False)
    if lgbm_capacity_job_is_complete(
        row.modeling_track,
        row.metric,
        int(row.horizon),
    )
}

print(
    "LightGBM capacity restart state: "
    f"{len(completed_lgbm_capacity_jobs)}/20 jobs complete."
)

if RUN_LGBM_CAPACITY:
    jobs_to_run_df = (
        lgbm_capacity_jobs_df.loc[
            ~lgbm_capacity_jobs_df.apply(
                lambda row: (
                    row["modeling_track"],
                    row["metric"],
                    int(row["horizon"]),
                )
                in completed_lgbm_capacity_jobs,
                axis=1,
            )
        ]
        .reset_index(drop=True)
    )

    for job_number, job in enumerate(
        jobs_to_run_df.itertuples(index=False),
        start=1,
    ):
        track = job.modeling_track
        metric = job.metric
        horizon = int(job.horizon)

        print(
            f"\n[{job_number}/{len(jobs_to_run_df)}] "
            f"{TRACK_SHORT_LABELS[track]} · "
            f"{TARGET_SHORT_LABELS[metric]} · "
            f"h={horizon}",
            flush=True,
        )

        payload = load_family_screen_job(
            track=track,
            metric=metric,
            horizon=horizon,
            train_row_cap=LGBM_CAPACITY_TRAIN_ROW_CAP,
        )

        job_results = []

        for profile_id in LGBM_CAPACITY_PROFILE_ORDER:
            print(
                f"    fitting "
                f"{LGBM_CAPACITY_PROFILES[profile_id]['label']}...",
                flush=True,
            )

            try:
                result = fit_lgbm_capacity_profile(
                    payload,
                    profile_id,
                )

                result.update(
                    {
                        "modeling_track": track,
                        "metric": metric,
                        "horizon": horizon,
                        "job_id": job.job_id,
                    }
                )

                print(
                    f"      MAE={result['mae']:.4f} · "
                    f"skill={result['skill_vs_benchmark_pct']:+.2f}% · "
                    f"fit={result['fit_seconds']:.2f}s",
                    flush=True,
                )

            except Exception as exc:
                result = {
                    "capacity_contract_id": LGBM_CAPACITY_CONTRACT_ID,
                    "capacity_profile": profile_id,
                    "capacity_label": (
                        LGBM_CAPACITY_PROFILES[profile_id]["label"]
                    ),
                    "status": "failed",
                    "modeling_track": track,
                    "metric": metric,
                    "horizon": horizon,
                    "job_id": job.job_id,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }

                print(
                    f"      failed · "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )

            job_results.append(result)

        # Persist all three matched profiles together. A restart therefore
        # never accepts a job where only one side of the comparison survived.
        atomic_parquet_write(
            pd.DataFrame(job_results),
            lgbm_capacity_job_path(
                track,
                metric,
                horizon,
            ),
        )

        del payload
        gc.collect()


# ---------------------------------------------------------------------
# Reconstruct the full experiment from durable job checkpoints
# ---------------------------------------------------------------------

capacity_parts = []

for job in lgbm_capacity_jobs_df.itertuples(index=False):
    path = lgbm_capacity_job_path(
        job.modeling_track,
        job.metric,
        int(job.horizon),
    )

    if path.exists():
        capacity_parts.append(
            pd.read_parquet(path)
        )

if capacity_parts:
    lgbm_capacity_results_df = pd.concat(
        capacity_parts,
        ignore_index=True,
    )
else:
    lgbm_capacity_results_df = pd.DataFrame()

if not lgbm_capacity_results_df.empty:
    atomic_parquet_write(
        lgbm_capacity_results_df,
        LGBM_CAPACITY_RESULTS_PATH,
    )

successful_fit_count = (
    int(
        lgbm_capacity_results_df["status"]
        .eq("success")
        .sum()
    )
    if not lgbm_capacity_results_df.empty
    else 0
)

failed_fit_count = (
    int(
        lgbm_capacity_results_df["status"]
        .eq("failed")
        .sum()
    )
    if not lgbm_capacity_results_df.empty
    else 0
)

print(
    "\nLightGBM capacity result state: "
    f"{successful_fit_count}/{expected_lgbm_capacity_fits} successful fits; "
    f"{failed_fit_count} failed."
)

LightGBM capacity restart state: 20/20 jobs complete.

LightGBM capacity result state: 60/60 successful fits; 0 failed.


**Findings.** All 60 planned LightGBM capacity fits completed successfully with no failures. Runtime remained extremely small across the entire experiment: the Screen reference typically fit in roughly 1–2 seconds, More regularized in roughly 2–3 seconds, and the larger More expressive profile generally in roughly 3–5 seconds. This confirms that LightGBM preserves the computational advantage that motivated the family switch even when additional capacity is introduced. The provisional accuracy pattern was also favorable: More expressive frequently improved benchmark-relative skill over the Screen reference, including several jobs where the reference model failed to beat the seasonal benchmark.

### 4.4.1.6C — Compare LightGBM capacity profiles

Capacity selection uses matched evidence: every profile is evaluated on the same training rows, validation forecasts, and frozen benchmark for each forecasting job.

The comparison emphasizes four signals:

1. **MAE wins** across the 20 matched jobs.
2. **Median paired MAE change** relative to the original screen reference.
3. **Benchmark breadth**, including the worst forecast jobs rather than only the median.
4. **Fitting cost**, so an accuracy gain is interpreted alongside the computational advantage that motivated the switch to LightGBM.

A more complex profile should advance only if its improvement is broad enough to justify the added capacity. Conversely, the screen reference should remain frozen if the tuned alternatives produce only marginal or inconsistent gains.

In [34]:
# ---------------------------------------------------------------------
# 4.4.1.6C — Synthesize the matched LightGBM capacity evidence
# ---------------------------------------------------------------------

if lgbm_capacity_results_df.empty:
    raise RuntimeError("No LightGBM capacity results are available.")

failed_capacity_df = lgbm_capacity_results_df.loc[
    lgbm_capacity_results_df["status"].eq("failed")
].copy()

if not failed_capacity_df.empty:
    display(
        failed_capacity_df[
            [
                "modeling_track",
                "metric",
                "horizon",
                "capacity_profile",
                "error_type",
                "error_message",
            ]
        ]
    )
    raise RuntimeError(
        "LightGBM capacity tuning contains failed fits. "
        "Resolve them before selecting a profile."
    )

capacity_success_df = lgbm_capacity_results_df.loc[
    lgbm_capacity_results_df["status"].eq("success")
].copy()

if len(capacity_success_df) != 60:
    raise RuntimeError(
        f"Expected 60 successful capacity fits; found {len(capacity_success_df)}."
    )

# A positive skill value means the model beat the frozen benchmark on MAE.
# We keep this as an explicit boolean because later summaries need a simple
# count of how often each profile cleared that benchmark.
capacity_success_df["beats_benchmark"] = (
    capacity_success_df["skill_vs_benchmark_pct"] > 0
)


# ---------------------------------------------------------------------
# Pair every fitted profile with the exact screen-reference result
# ---------------------------------------------------------------------
# Each forecasting job has three LightGBM profiles. To judge whether the
# extra flexibility is worth it, every result is compared back to the same
# job's Screen reference fit.

job_keys = ["modeling_track", "metric", "horizon"]

capacity_reference_df = (
    capacity_success_df.loc[
        capacity_success_df["capacity_profile"].eq("screen_reference"),
        job_keys + ["mae", "fit_seconds"],
    ]
    .rename(
        columns={
            "mae": "reference_mae",
            "fit_seconds": "reference_fit_seconds",
        }
    )
)

capacity_success_df = capacity_success_df.merge(
    capacity_reference_df,
    on=job_keys,
    how="left",
    validate="many_to_one",
)

capacity_success_df["mae_improvement_vs_reference_pct"] = (
    100.0
    * (capacity_success_df["reference_mae"] - capacity_success_df["mae"])
    / capacity_success_df["reference_mae"]
)

capacity_success_df["fit_time_ratio_vs_reference"] = (
    capacity_success_df["fit_seconds"]
    / capacity_success_df["reference_fit_seconds"]
)

# Rank the three profiles only within the same forecasting job. This avoids
# mixing MAE values across targets that live on very different scales.
capacity_success_df["mae_rank"] = (
    capacity_success_df.groupby(job_keys)["mae"]
    .rank(method="min", ascending=True)
)

capacity_success_df["is_mae_winner"] = capacity_success_df["mae_rank"].eq(1)


# ---------------------------------------------------------------------
# Reader-facing summary table
# ---------------------------------------------------------------------
# This is the main decision table. It answers:
# - which profile wins most often,
# - whether it usually beats the benchmark,
# - how much it improves over the Screen reference,
# - and what runtime price it asks us to pay.

lgbm_capacity_summary_df = (
    capacity_success_df
    .groupby(["capacity_profile", "capacity_label"], as_index=False)
    .agg(
        jobs=("job_id", "nunique"),
        mae_wins=("is_mae_winner", "sum"),
        median_mae_rank=("mae_rank", "median"),
        median_mae_improvement_vs_reference_pct=(
            "mae_improvement_vs_reference_pct",
            "median",
        ),
        worst_mae_improvement_vs_reference_pct=(
            "mae_improvement_vs_reference_pct",
            "min",
        ),
        median_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "median"),
        worst_skill_vs_benchmark_pct=("skill_vs_benchmark_pct", "min"),
        jobs_beating_benchmark=("beats_benchmark", "sum"),
        median_fit_seconds=("fit_seconds", "median"),
        median_fit_time_ratio_vs_reference=("fit_time_ratio_vs_reference", "median"),
        negative_predictions=("negative_prediction_count", "sum"),
    )
)

lgbm_capacity_summary_df["jobs_beating_benchmark_pct"] = (
    100.0
    * lgbm_capacity_summary_df["jobs_beating_benchmark"]
    / lgbm_capacity_summary_df["jobs"]
)

profile_order = {
    profile_id: i
    for i, profile_id in enumerate(LGBM_CAPACITY_PROFILE_ORDER)
}

lgbm_capacity_summary_df["_order"] = (
    lgbm_capacity_summary_df["capacity_profile"].map(profile_order)
)

lgbm_capacity_summary_df = (
    lgbm_capacity_summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

lgbm_capacity_display_df = (
    lgbm_capacity_summary_df[
        [
            "capacity_label",
            "mae_wins",
            "median_mae_rank",
            "median_mae_improvement_vs_reference_pct",
            "worst_mae_improvement_vs_reference_pct",
            "median_skill_vs_benchmark_pct",
            "worst_skill_vs_benchmark_pct",
            "jobs_beating_benchmark_pct",
            "median_fit_seconds",
            "median_fit_time_ratio_vs_reference",
            "negative_predictions",
        ]
    ]
    .rename(
        columns={
            "capacity_label": "Capacity profile",
            "mae_wins": "MAE wins",
            "median_mae_rank": "Median MAE rank",
            "median_mae_improvement_vs_reference_pct": "Median MAE improvement vs ref %",
            "worst_mae_improvement_vs_reference_pct": "Worst MAE improvement vs ref %",
            "median_skill_vs_benchmark_pct": "Median benchmark skill %",
            "worst_skill_vs_benchmark_pct": "Worst benchmark skill %",
            "jobs_beating_benchmark_pct": "Jobs beating benchmark %",
            "median_fit_seconds": "Median fit seconds",
            "median_fit_time_ratio_vs_reference": "Fit-time ratio vs ref",
            "negative_predictions": "Negative predictions",
        }
    )
)

for column in [
    "Median MAE rank",
    "Median MAE improvement vs ref %",
    "Worst MAE improvement vs ref %",
    "Median benchmark skill %",
    "Worst benchmark skill %",
    "Jobs beating benchmark %",
    "Median fit seconds",
    "Fit-time ratio vs ref",
]:
    lgbm_capacity_display_df[column] = lgbm_capacity_display_df[column].round(2)

display(lgbm_capacity_display_df)


# ---------------------------------------------------------------------
# Supporting visual — accuracy gain versus overall skill, with runtime
# shown by bubble size
# ---------------------------------------------------------------------
# Each point is one fitted forecasting job. This view helps us see whether
# a profile's gains are broad and practical rather than coming from only a
# few isolated jobs. Bubble size represents fit time, so larger circles are
# more computationally expensive.

capacity_success_df["Job"] = capacity_success_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · "
        f"h={int(row['horizon'])}"
    ),
    axis=1,
)

if RUN_VISUALS:
    bubble_fig = px.scatter(
        capacity_success_df,
        x="mae_improvement_vs_reference_pct",
        y="skill_vs_benchmark_pct",
        size="fit_seconds",
        color="capacity_label",
        hover_name="Job",
        hover_data={
            "capacity_label": True,
            "mae": ":.4f",
            "reference_mae": ":.4f",
            "mae_improvement_vs_reference_pct": ":.2f",
            "skill_vs_benchmark_pct": ":.2f",
            "fit_seconds": ":.2f",
            "fit_time_ratio_vs_reference": ":.2f",
            "negative_prediction_count": True,
        },
        labels={
            "mae_improvement_vs_reference_pct": "MAE improvement vs Screen reference (%)",
            "skill_vs_benchmark_pct": "Skill vs benchmark (%)",
            "fit_seconds": "Fit time (seconds)",
            "capacity_label": "Capacity profile",
        },
        title=(
            "LightGBM Capacity Tradeoff — Accuracy Gain, Benchmark Skill, "
            "and Runtime"
        ),
    )

    # x=0 means "no MAE gain over the Screen reference." Values to the right
    # are better. y=0 means "did not beat the frozen benchmark."
    bubble_fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )
    bubble_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    bubble_fig.update_traces(
        marker={"sizemin": 10, "line": {"width": 1}},
    )

    bubble_fig.update_layout(
        width=940,
        height=650,
        margin=dict(l=80, r=40, t=80, b=70),
        legend_title_text="Capacity profile",
    )

    apply_branding(bubble_fig)
    bubble_fig.show()

,Capacity profile,MAE wins,Median MAE rank,Median MAE improvement vs ref %,Worst MAE improvement vs ref %,Median benchmark skill %,Worst benchmark skill %,Jobs beating benchmark %,Median fit seconds,Fit-time ratio vs ref,Negative predictions
0,Screen reference,0,2.0,0.00,0.00,6.02,-8.27,75.0,1.89,1.00,1039
1,More expressive,18,1.0,5.38,-0.60,11.77,0.92,100.0,3.52,1.88,2051
2,More regularized,2,3.0,-0.35,-0.94,5.49,-9.08,70.0,2.31,1.18,1513


**Findings.** The More expressive LightGBM profile is the clear capacity winner. It achieved the lowest MAE in 18 of the 20 matched forecasting jobs and a median MAE improvement of 5.38% relative to the Screen reference. Its gains were broad rather than isolated: median benchmark skill increased from 6.02% to 11.77%, and the share of jobs beating the frozen benchmark rose from 75% to 100%. Most importantly, the weakest More expressive job still beat the benchmark by 0.92%, whereas the Screen reference fell as low as -8.27%.

The additional accuracy comes at a modest computational cost. Median fitting time increased from 1.89 seconds to 3.52 seconds, or 1.88× the Screen reference, which remains trivial compared with the runtime profile that forced Extra Trees out of production consideration. The visual reinforces the same conclusion: More expressive jobs are concentrated to the right of zero MAE improvement while remaining above the benchmark-skill threshold across the full forecasting surface.

More regularized provides no competing case. It wins only 2 of 20 jobs, slightly worsens median MAE relative to the reference (-0.35%), beats the benchmark in only 70% of jobs, and reaches a worst benchmark skill of -9.08%. Although it is somewhat faster than More expressive, that runtime saving does not compensate for the loss in predictive performance.

One diagnostic does warrant continued observation: More expressive produces more negative predictions than the other profiles. No predictions are clipped or otherwise corrected here, so this remains visible as a model diagnostic rather than being hidden by post-processing. It does not overturn the MAE result, but negative predictions should continue to be monitored in later representation and full-validation stages.

> ✅ Decision — Freeze the More expressive LightGBM capacity profile\.

More expressive is the clear MAE winner, taking 18 of 20 matched jobs and reducing median MAE by 5\.38% relative to the Screen reference\. The gain is broad rather than isolated: all 20 jobs beat the frozen benchmark, and even the weakest job retains positive benchmark skill\.

The added capacity increases median fitting time from 1\.89 to 3\.52 seconds, but that remains operationally trivial\. More regularized provides no evidence that stronger constraint is beneficial\. Freeze \`more\_expressive\` as the single LightGBM capacity definition for subsequent representation and cadence testing\.

### 4.4.1.6D — Confirm count-target representation

With LightGBM capacity fixed, one model-specific question remains for the three count targets:

> **Should LightGBM learn counts directly in their original units, or fit `log1p`-compressed targets and transform forecasts back before evaluation?**

`log1p` compresses very large count values during model fitting, reducing the influence of extreme high-volume observations. It does **not** change the evaluation scale: every `log1p` forecast is converted back with `expm1` before MAE, RMSE, NMAE, bias, benchmark skill, or negative-prediction diagnostics are calculated. Forecasts are never clipped.

The comparison is deliberately matched. Both representations use the frozen **More expressive** LightGBM profile, identical predictors, identical training and validation rows, and h=5. Both ordinary modeling tracks are included for Taxi trips, FHVHV trips, and Subway ridership, producing:

**2 tracks × 3 count targets × 2 representations = 12 fits.**

Speed targets remain raw because a count-scale compression has no methodological role for speed outcomes. If both tracks agree for a target, one target-level representation rule can be carried forward rather than introducing track-specific transformations.

The final holdout remains untouched.

In [35]:
# ---------------------------------------------------------------------
# 4.4.1.6D — Freeze the LightGBM target-representation experiment
# ---------------------------------------------------------------------

SELECTED_LGBM_CAPACITY_PROFILE = "more_expressive"
LGBM_REPRESENTATION_CONTRACT_ID = "lgbm_representation_v1"

LGBM_REPRESENTATION_TARGETS = [
    "taxi_trip_count",
    "fhvhv_trip_count",
    "subway_ridership",
]
LGBM_REPRESENTATION_HORIZON = 5
LGBM_REPRESENTATION_OPTIONS = ["raw", "log1p"]
LGBM_REPRESENTATION_TRAIN_ROW_CAP = LGBM_CAPACITY_TRAIN_ROW_CAP

if SELECTED_LGBM_CAPACITY_PROFILE not in LGBM_CAPACITY_PROFILES:
    raise KeyError(
        f"Unknown LightGBM capacity profile: "
        f"{SELECTED_LGBM_CAPACITY_PROFILE}"
    )

# Use the two ordinary modeling tracks already represented in the frozen
# capacity-screen roster. The representation question changes only the target
# scale; it does not alter the modeling-track contract.
lgbm_representation_jobs_df = (
    lgbm_capacity_jobs_df.loc[
        lgbm_capacity_jobs_df["metric"].isin(LGBM_REPRESENTATION_TARGETS)
        & lgbm_capacity_jobs_df["horizon"].eq(LGBM_REPRESENTATION_HORIZON),
        ["modeling_track", "metric", "horizon"],
    ]
    .drop_duplicates()
    .sort_values(["modeling_track", "metric"])
    .reset_index(drop=True)
)

if len(lgbm_representation_jobs_df) != 6:
    raise RuntimeError(
        f"Expected 6 Track × count-target h=5 jobs; "
        f"found {len(lgbm_representation_jobs_df)}."
    )

lgbm_representation_plan_df = pd.DataFrame(
    [
        {
            "modeling_track": job.modeling_track,
            "metric": job.metric,
            "horizon": int(job.horizon),
            "target_representation": representation,
            "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
        }
        for job in lgbm_representation_jobs_df.itertuples(index=False)
        for representation in LGBM_REPRESENTATION_OPTIONS
    ]
)

expected_lgbm_representation_fits = len(lgbm_representation_plan_df)

if expected_lgbm_representation_fits != 12:
    raise RuntimeError(
        f"Expected 12 LightGBM representation fits; "
        f"found {expected_lgbm_representation_fits}."
    )

lgbm_representation_workload_df = pd.DataFrame(
    [
        {
            "Modeling tracks": lgbm_representation_jobs_df["modeling_track"].nunique(),
            "Count targets": lgbm_representation_jobs_df["metric"].nunique(),
            "Horizon": LGBM_REPRESENTATION_HORIZON,
            "Representations": len(LGBM_REPRESENTATION_OPTIONS),
            "Planned fits": expected_lgbm_representation_fits,
            "Training-row cap": LGBM_REPRESENTATION_TRAIN_ROW_CAP,
            "Capacity profile": LGBM_CAPACITY_PROFILES[
                SELECTED_LGBM_CAPACITY_PROFILE
            ]["label"],
        }
    ]
)

display(lgbm_representation_workload_df)

,Modeling tracks,Count targets,Horizon,Representations,Planned fits,Training-row cap,Capacity profile
0,2,3,5,2,12,250000,More expressive


**Findings.** The representation experiment covers both modeling tracks and all three count targets at h=5, with raw and `log1p` evaluated under the frozen More expressive LightGBM profile. This produces 12 matched fits while preserving the same 250,000-row training cap used in the capacity experiment. The design therefore isolates target representation as the only modeling change.

### 4.4.1.6E — Run or recover the matched representation fits

Each logical forecasting job is evaluated twice: once with the target left in raw units and once with the training target transformed by `log1p`.

LightGBM receives the same native categorical predictor treatment used throughout the family screen and capacity experiment. For a `log1p` fit, only the training target changes. Predictions are transformed back to original count units before any evaluation metric is calculated, ensuring that raw and transformed models are compared directly on the same MAE scale.

Results are checkpointed by Track × Target job, with the raw and `log1p` pair written together. A restart therefore accepts a job as complete only when both sides of the matched comparison are present and successful.

In [36]:
# ---------------------------------------------------------------------
# 4.4.1.6E — Run or recover matched LightGBM representation fits
# ---------------------------------------------------------------------

RUN_LGBM_REPRESENTATION = True
RESET_LGBM_REPRESENTATION = False

LGBM_REPRESENTATION_ROOT = (
    INTERMEDIATE_DIR
    / LGBM_REPRESENTATION_CONTRACT_ID
)
LGBM_REPRESENTATION_PARTS_DIR = (
    LGBM_REPRESENTATION_ROOT
    / "parts"
)
LGBM_REPRESENTATION_RESULTS_PATH = (
    LGBM_REPRESENTATION_ROOT
    / "results.parquet"
)

if RESET_LGBM_REPRESENTATION:
    reset_checkpoint_dir(LGBM_REPRESENTATION_ROOT)

LGBM_REPRESENTATION_PARTS_DIR.mkdir(parents=True, exist_ok=True)


def lgbm_representation_job_path(track, metric):
    """Return the durable raw/log1p checkpoint for one Track × Target pair."""
    job_slug = slugify(
        f"{track}__{metric}__h{LGBM_REPRESENTATION_HORIZON}"
    )
    return LGBM_REPRESENTATION_PARTS_DIR / f"{job_slug}.parquet"


def lgbm_representation_job_is_complete(track, metric):
    """Require both current-contract representations to succeed together."""
    path = lgbm_representation_job_path(track, metric)

    if not path.exists():
        return False

    try:
        result_df = pd.read_parquet(path)
    except Exception:
        return False

    return (
        len(result_df) == 2
        and result_df["status"].eq("success").all()
        and result_df["representation_contract_id"]
        .eq(LGBM_REPRESENTATION_CONTRACT_ID)
        .all()
        and set(result_df["target_representation"])
        == set(LGBM_REPRESENTATION_OPTIONS)
    )


def fit_lgbm_representation(payload, representation):
    """Fit one frozen LightGBM model while changing only target representation."""
    categorical_names = payload["categorical_names"]

    train_frame, valid_frame = prepare_lightgbm_frames(
        payload["X_train"],
        payload["X_valid"],
        categorical_names,
    )

    y_train_raw = np.asarray(payload["y_train"], dtype=float)
    y_valid_raw = np.asarray(payload["y_valid"], dtype=float)

    if representation == "raw":
        y_train_model = y_train_raw

    elif representation == "log1p":
        # Count targets must be nonnegative for log1p to preserve its intended
        # interpretation. Fail loudly rather than silently altering bad inputs.
        if np.any(y_train_raw < 0):
            raise ValueError(
                "log1p representation received negative training targets."
            )

        y_train_model = np.log1p(y_train_raw)

    else:
        raise ValueError(
            f"Unknown target representation: {representation}"
        )

    model = LGBMRegressor(
        **LGBM_CAPACITY_PROFILES[
            SELECTED_LGBM_CAPACITY_PROFILE
        ]["params"]
    )

    fit_start = perf_counter()
    model.fit(
        train_frame,
        y_train_model,
        categorical_feature=categorical_names,
    )
    fit_seconds = perf_counter() - fit_start

    predict_start = perf_counter()
    predictions_model_scale = np.asarray(
        model.predict(valid_frame),
        dtype=float,
    )
    predict_seconds = perf_counter() - predict_start

    # Evaluate every candidate in the original target units. No clipping is
    # applied: negative forecasts remain visible as a diagnostic.
    if representation == "log1p":
        with np.errstate(over="ignore", invalid="ignore"):
            predictions = np.expm1(predictions_model_scale)
    else:
        predictions = predictions_model_scale

    if not np.isfinite(predictions).all():
        raise ValueError(
            f"{representation} produced non-finite validation predictions."
        )

    model_metrics = regression_metrics(y_valid_raw, predictions)
    benchmark_mae = float(payload["benchmark_metrics"]["mae"])

    skill_vs_benchmark_pct = (
        100.0
        * (benchmark_mae - model_metrics["mae"])
        / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    result = {
        "representation_contract_id": LGBM_REPRESENTATION_CONTRACT_ID,
        "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
        "target_representation": representation,
        "status": "success",
        "train_rows": payload["train_rows"],
        "train_origins": payload["train_origins"],
        "validation_rows": payload["valid_rows"],
        "validation_origins": payload["valid_origins"],
        "benchmark_id": payload["benchmark_id"],
        "benchmark_label": payload["benchmark_label"],
        "mae": float(model_metrics["mae"]),
        "rmse": float(model_metrics["rmse"]),
        "nmae": float(model_metrics["nmae"]),
        "bias": float(model_metrics["bias"]),
        "benchmark_mae": benchmark_mae,
        "skill_vs_benchmark_pct": float(skill_vs_benchmark_pct),
        "beats_benchmark": bool(skill_vs_benchmark_pct > 0),
        "negative_prediction_count": int((predictions < 0).sum()),
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
    }

    del (
        train_frame,
        valid_frame,
        y_train_model,
        predictions_model_scale,
        predictions,
        model,
    )
    gc.collect()

    return result


completed_lgbm_representation_jobs = {
    (row.modeling_track, row.metric)
    for row in lgbm_representation_jobs_df.itertuples(index=False)
    if lgbm_representation_job_is_complete(
        row.modeling_track,
        row.metric,
    )
}

print(
    "LightGBM representation restart state: "
    f"{len(completed_lgbm_representation_jobs)}/6 jobs complete."
)

if RUN_LGBM_REPRESENTATION:
    jobs_to_run_df = (
        lgbm_representation_jobs_df.loc[
            ~lgbm_representation_jobs_df.apply(
                lambda row: (
                    row["modeling_track"],
                    row["metric"],
                )
                in completed_lgbm_representation_jobs,
                axis=1,
            )
        ]
        .reset_index(drop=True)
    )

    for job_number, job in enumerate(
        jobs_to_run_df.itertuples(index=False),
        start=1,
    ):
        track = job.modeling_track
        metric = job.metric

        print(
            f"\n[{job_number}/{len(jobs_to_run_df)}] "
            f"{TRACK_SHORT_LABELS[track]} · "
            f"{TARGET_SHORT_LABELS[metric]} · "
            f"h={LGBM_REPRESENTATION_HORIZON}",
            flush=True,
        )

        payload = load_family_screen_job(
            track=track,
            metric=metric,
            horizon=LGBM_REPRESENTATION_HORIZON,
            train_row_cap=LGBM_REPRESENTATION_TRAIN_ROW_CAP,
        )

        job_results = []

        for representation in LGBM_REPRESENTATION_OPTIONS:
            print(
                f"    fitting {representation}...",
                flush=True,
            )

            try:
                result = fit_lgbm_representation(
                    payload,
                    representation,
                )

                result.update(
                    {
                        "modeling_track": track,
                        "metric": metric,
                        "horizon": LGBM_REPRESENTATION_HORIZON,
                    }
                )

                print(
                    f"      MAE={result['mae']:.4f} · "
                    f"skill={result['skill_vs_benchmark_pct']:+.2f}% · "
                    f"fit={result['fit_seconds']:.2f}s · "
                    f"negative={result['negative_prediction_count']}",
                    flush=True,
                )

            except Exception as exc:
                result = {
                    "representation_contract_id": LGBM_REPRESENTATION_CONTRACT_ID,
                    "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
                    "target_representation": representation,
                    "status": "failed",
                    "modeling_track": track,
                    "metric": metric,
                    "horizon": LGBM_REPRESENTATION_HORIZON,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }

                print(
                    f"      failed · {type(exc).__name__}: {exc}",
                    flush=True,
                )

            job_results.append(result)

        # Commit the matched raw/log1p pair together so restart semantics cannot
        # silently compare one newly fitted side with one incomplete side.
        atomic_parquet_write(
            pd.DataFrame(job_results),
            lgbm_representation_job_path(track, metric),
        )

        del payload
        gc.collect()


representation_parts = []

for job in lgbm_representation_jobs_df.itertuples(index=False):
    path = lgbm_representation_job_path(
        job.modeling_track,
        job.metric,
    )

    if path.exists():
        representation_parts.append(pd.read_parquet(path))

lgbm_representation_results_df = (
    pd.concat(representation_parts, ignore_index=True)
    if representation_parts
    else pd.DataFrame()
)

if not lgbm_representation_results_df.empty:
    atomic_parquet_write(
        lgbm_representation_results_df,
        LGBM_REPRESENTATION_RESULTS_PATH,
    )

successful_representation_fits = (
    int(
        lgbm_representation_results_df["status"]
        .eq("success")
        .sum()
    )
    if not lgbm_representation_results_df.empty
    else 0
)

failed_representation_fits = (
    int(
        lgbm_representation_results_df["status"]
        .eq("failed")
        .sum()
    )
    if not lgbm_representation_results_df.empty
    else 0
)

print(
    "\nLightGBM representation result state: "
    f"{successful_representation_fits}/12 successful fits; "
    f"{failed_representation_fits} failed."
)

LightGBM representation restart state: 6/6 jobs complete.

LightGBM representation result state: 12/12 successful fits; 0 failed.


**Findings.** All 12 planned representation fits completed successfully with no failures. Runtime remained very small, generally around 3–5 seconds per fit, so the representation choice has no meaningful operational consequence for LightGBM.

The immediate pattern already favors raw targets. Raw MAE is lower than `log1p` for all six Track × Target comparisons. `log1p` does reduce negative forecasts for FHVHV trips and eliminates them entirely for Subway ridership, but it sharply increases negative Taxi-trip forecasts and does not compensate with better MAE. Because forecasts are evaluated without clipping, these differences remain visible as diagnostics rather than affecting the selection rule.

### 4.4.1.6F — Compare raw and `log1p` count forecasts

Representation selection is based on paired MAE in the original target units.

For each Track × Target pair, the raw model and `log1p` model are evaluated on the same validation rows against the same frozen benchmark. A positive `log1p` MAE improvement means the transformed model reduced forecast error after its predictions were converted back to ordinary counts.

The primary question is whether both modeling tracks point to the same representation for each target. Agreement supports one simple target-level rule. Disagreement would justify retaining the representation question for a narrowly targeted follow-up rather than prematurely imposing one transformation everywhere.

Negative forecasts remain diagnostic evidence only; they are not clipped or otherwise corrected.

In [37]:
# ---------------------------------------------------------------------
# 4.4.1.6F — Compare raw and log1p count forecasts
# ---------------------------------------------------------------------

if lgbm_representation_results_df.empty:
    raise RuntimeError(
        "No LightGBM representation results are available."
    )

failed_representation_df = lgbm_representation_results_df.loc[
    lgbm_representation_results_df["status"].eq("failed")
].copy()

if not failed_representation_df.empty:
    display(
        failed_representation_df[
            [
                "modeling_track",
                "metric",
                "target_representation",
                "error_type",
                "error_message",
            ]
        ]
    )
    raise RuntimeError(
        "LightGBM representation testing contains failed fits."
    )

representation_success_df = lgbm_representation_results_df.loc[
    lgbm_representation_results_df["status"].eq("success")
].copy()

if len(representation_success_df) != 12:
    raise RuntimeError(
        f"Expected 12 successful representation fits; "
        f"found {len(representation_success_df)}."
    )


# ---------------------------------------------------------------------
# Build exact raw/log1p pairs on the same Track × Target × Horizon rows
# ---------------------------------------------------------------------
# Every comparison remains in original target units because log1p forecasts
# were already transformed back with expm1 before MAE was calculated.

pair_keys = ["modeling_track", "metric", "horizon"]

raw_df = (
    representation_success_df.loc[
        representation_success_df["target_representation"].eq("raw"),
        pair_keys
        + [
            "mae",
            "skill_vs_benchmark_pct",
            "negative_prediction_count",
            "fit_seconds",
        ],
    ]
    .rename(
        columns={
            "mae": "raw_mae",
            "skill_vs_benchmark_pct": "raw_skill_pct",
            "negative_prediction_count": "raw_negative_predictions",
            "fit_seconds": "raw_fit_seconds",
        }
    )
)

log_df = (
    representation_success_df.loc[
        representation_success_df["target_representation"].eq("log1p"),
        pair_keys
        + [
            "mae",
            "skill_vs_benchmark_pct",
            "negative_prediction_count",
            "fit_seconds",
        ],
    ]
    .rename(
        columns={
            "mae": "log1p_mae",
            "skill_vs_benchmark_pct": "log1p_skill_pct",
            "negative_prediction_count": "log1p_negative_predictions",
            "fit_seconds": "log1p_fit_seconds",
        }
    )
)

lgbm_representation_pairs_df = raw_df.merge(
    log_df,
    on=pair_keys,
    how="inner",
    validate="one_to_one",
)

if len(lgbm_representation_pairs_df) != 6:
    raise RuntimeError(
        f"Expected 6 matched raw/log1p pairs; "
        f"found {len(lgbm_representation_pairs_df)}."
    )

lgbm_representation_pairs_df["log1p_mae_improvement_pct"] = (
    100.0
    * (
        lgbm_representation_pairs_df["raw_mae"]
        - lgbm_representation_pairs_df["log1p_mae"]
    )
    / lgbm_representation_pairs_df["raw_mae"]
)

lgbm_representation_pairs_df["mae_winner"] = np.where(
    lgbm_representation_pairs_df["log1p_mae"]
    < lgbm_representation_pairs_df["raw_mae"],
    "log1p",
    "Raw",
)

lgbm_representation_pairs_df["Job"] = lgbm_representation_pairs_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]}"
    ),
    axis=1,
)


# ---------------------------------------------------------------------
# Reader-facing paired comparison table
# ---------------------------------------------------------------------
# This table retains the exact values needed for the representation decision:
# MAE, relative improvement, benchmark skill, and negative-prediction behavior.

lgbm_representation_display_df = (
    lgbm_representation_pairs_df[
        [
            "Job",
            "raw_mae",
            "log1p_mae",
            "log1p_mae_improvement_pct",
            "raw_skill_pct",
            "log1p_skill_pct",
            "raw_negative_predictions",
            "log1p_negative_predictions",
            "mae_winner",
        ]
    ]
    .rename(
        columns={
            "raw_mae": "Raw MAE",
            "log1p_mae": "log1p MAE",
            "log1p_mae_improvement_pct": "log1p MAE improvement %",
            "raw_skill_pct": "Raw benchmark skill %",
            "log1p_skill_pct": "log1p benchmark skill %",
            "raw_negative_predictions": "Raw negative predictions",
            "log1p_negative_predictions": "log1p negative predictions",
            "mae_winner": "MAE winner",
        }
    )
)

for column in [
    "Raw MAE",
    "log1p MAE",
    "log1p MAE improvement %",
    "Raw benchmark skill %",
    "log1p benchmark skill %",
]:
    lgbm_representation_display_df[column] = (
        lgbm_representation_display_df[column].round(2)
    )

display(lgbm_representation_display_df)


# ---------------------------------------------------------------------
# Supporting visual — paired Raw vs log1p MAE
# ---------------------------------------------------------------------
# Absolute MAE cannot be compared across targets because Taxi/FHVHV counts
# and Subway ridership live on very different scales. Normalize each pair to
# its Raw MAE instead: Raw is fixed at 100, and log1p falls below 100 when it
# improves MAE. This preserves the paired decision while making all six jobs
# comparable in one chart.

representation_plot_rows = []

for row in lgbm_representation_pairs_df.itertuples(index=False):
    representation_plot_rows.extend(
        [
            {
                "Job": row.Job,
                "Representation": "Raw",
                "Relative MAE index": 100.0,
                "MAE": row.raw_mae,
                "Benchmark skill %": row.raw_skill_pct,
                "Negative predictions": row.raw_negative_predictions,
            },
            {
                "Job": row.Job,
                "Representation": "log1p",
                "Relative MAE index": (
                    100.0 * row.log1p_mae / row.raw_mae
                ),
                "MAE": row.log1p_mae,
                "Benchmark skill %": row.log1p_skill_pct,
                "Negative predictions": row.log1p_negative_predictions,
            },
        ]
    )

lgbm_representation_plot_df = pd.DataFrame(representation_plot_rows)

job_order = lgbm_representation_pairs_df["Job"].tolist()

if RUN_VISUALS:
    representation_fig = go.Figure()

    # Draw one connector per Track × Target pair so the visual reads as six
    # matched experiments rather than twelve unrelated observations.
    for job in job_order:
        job_df = (
            lgbm_representation_plot_df.loc[
                lgbm_representation_plot_df["Job"].eq(job)
            ]
            .set_index("Representation")
            .loc[["Raw", "log1p"]]
            .reset_index()
        )

        # Pre-format tooltip values before Plotly receives them. This guarantees
        # no tooltip exposes more than three decimal places.
        tooltip_data = np.column_stack(
            [
                job_df["MAE"].map(lambda value: f"{value:.3f}"),
                job_df["Benchmark skill %"].map(
                    lambda value: f"{value:+.3f}%"
                ),
                job_df["Negative predictions"].map(
                    lambda value: f"{int(value):,}"
                ),
            ]
        )

        representation_fig.add_trace(
            go.Scatter(
                x=job_df["Representation"],
                y=job_df["Relative MAE index"].round(3),
                mode="lines+markers",
                name=job,
                customdata=tooltip_data,
                hovertemplate=(
                    f"{job}<br>"
                    "Representation: %{x}<br>"
                    "Relative MAE index: %{y:.3f}<br>"
                    "MAE: %{customdata[0]}<br>"
                    "Benchmark skill: %{customdata[1]}<br>"
                    "Negative predictions: %{customdata[2]}"
                    "<extra></extra>"
                ),
            )
        )

    # Raw is always normalized to 100, so this line is the exact no-change
    # threshold. log1p below 100 improves MAE; above 100 worsens it.
    representation_fig.add_hline(
        y=100,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
        annotation_text="Raw MAE reference",
        annotation_position="top right",
    )

    representation_fig.update_layout(
        title=(
            "LightGBM Count Representation — "
            "Paired MAE Change from Raw to log1p"
        ),
        xaxis_title=None,
        yaxis_title="Relative MAE index (Raw = 100)",
        width=960,
        height=650,
        margin=dict(l=80, r=40, t=85, b=70),
        legend_title_text="Track × Target",
    )

    apply_branding(representation_fig)
    representation_fig.show()

,Job,Raw MAE,log1p MAE,log1p MAE improvement %,Raw benchmark skill %,log1p benchmark skill %,Raw negative predictions,log1p negative predictions,MAE winner
0,Full-History · FHVHV trips,42.67,43.27,-1.40,11.50,10.26,247,82,Raw
1,Full-History · Subway,358.83,397.39,-10.75,5.63,-4.51,41,0,Raw
2,Full-History · Taxi trips,12.44,12.52,-0.63,6.45,5.87,73,1029,Raw
3,Post-CP · FHVHV trips,41.83,42.08,-0.60,13.51,12.99,171,130,Raw
4,Post-CP · Subway,357.43,396.70,-10.99,8.02,-2.09,18,0,Raw
5,Post-CP · Taxi trips,11.90,12.00,-0.91,10.83,10.02,367,919,Raw


**Findings.** Raw target representation wins all six matched MAE comparisons, giving the same result in both modeling tracks for Taxi trips, FHVHV trips, and Subway ridership. The effect is modest for Taxi and FHVHV counts but decisive for Subway. Relative to raw, `log1p` worsens MAE by 0.629%–0.908% for Taxi trips, 0.600%–1.403% for FHVHV trips, and 10.745%–10.987% for Subway ridership.

Benchmark-relative performance tells the same story. Raw remains positive in all six jobs, while `log1p` reduces skill in every comparison and pushes both Subway jobs below the frozen benchmark. The visual makes the consistency especially clear: every `log1p` point lies above the Raw = 100 MAE reference, with the largest deterioration concentrated in Subway ridership.

Negative predictions do not overturn the accuracy result. `log1p` reduces negative FHVHV forecasts and eliminates negative Subway forecasts, but substantially increases negative Taxi-trip forecasts. Since MAE is the primary selector and both tracks agree for every count target, there is no evidence here for retaining target-specific or track-specific transformations.

> ✅ Decision — Use raw target representation for all LightGBM count targets\.

Raw representation wins all six matched MAE comparisons across both modeling tracks and all three count targets\. It also preserves positive benchmark skill in every job, while \`log1p\` reduces skill throughout and pushes both Subway comparisons below the frozen benchmark\.

Freeze raw representation for Taxi trips, FHVHV trips, and Subway ridership\. No target\-specific or track\-specific transformation is needed; proceed to LightGBM cadence testing\.

### 4.4.1.6G — Confirm the LightGBM refit cadence across all targets

The LightGBM model definition is now frozen: the **More expressive** capacity profile advances, and all five forecasting targets are modeled in their original raw units. One family-specific question remains before full rolling validation:

> **Should LightGBM be fit once at the start of validation, or refreshed monthly as new outcomes become legally available?**

The cadence experiment evaluates **all five forecasting targets in both modeling tracks at h=5**. Using one common farther-ahead horizon provides a consistent surface for determining whether newly available history is valuable enough to justify repeated refitting.

Two policies are compared:

- **Validation-start only:** fit once at the first legal validation origin and reuse that model throughout validation.
- **Monthly refresh:** fit at validation start, then rebuild the model at the first legal forecast origin of each subsequent calendar month using the expanding history available at that point.

This produces **10 Track × Target comparisons**. Each receives one validation-start fit and ten monthly fits, for **110 planned LightGBM refits**.

Every model retains the 250,000-row origin-balanced training cap so the experiment isolates refit cadence rather than final production training size. The final holdout remains untouched.

In [38]:
# ---------------------------------------------------------------------
# 4.4.1.6G — Freeze the all-target LightGBM cadence experiment
# ---------------------------------------------------------------------

LGBM_CADENCE_CONTRACT_ID = "lgbm_cadence_all_targets_v1"
LGBM_CADENCE_HORIZON = 5
LGBM_CADENCE_TRAIN_ROW_CAP = 250_000

LGBM_CADENCE_POLICIES = ["validation_start_only", "monthly"]
LGBM_CADENCE_POLICY_LABELS = {
    "validation_start_only": "Validation-start only",
    "monthly": "Monthly refresh",
}

# Capacity testing already contains both tracks × all five metrics at h=5.
# Reuse that known roster rather than defining a separate cadence-specific
# target list by hand.
lgbm_cadence_jobs_df = (
    lgbm_capacity_jobs_df.loc[
        lgbm_capacity_jobs_df["horizon"].eq(LGBM_CADENCE_HORIZON),
        ["modeling_track", "metric", "horizon", "job_id"],
    ]
    .drop_duplicates()
    .sort_values(["modeling_track", "metric"])
    .reset_index(drop=True)
)

if len(lgbm_cadence_jobs_df) != 10:
    raise RuntimeError(
        f"Expected 10 Track × Target h=5 cadence jobs; "
        f"found {len(lgbm_cadence_jobs_df)}."
    )

if lgbm_cadence_jobs_df["metric"].nunique() != 5:
    raise RuntimeError(
        "LightGBM cadence experiment does not cover all five targets."
    )

if lgbm_cadence_jobs_df["modeling_track"].nunique() != 2:
    raise RuntimeError(
        "LightGBM cadence experiment does not cover both modeling tracks."
    )


# ---------------------------------------------------------------------
# Build each job's legal fixed/monthly refit schedule
# ---------------------------------------------------------------------
# The schedule depends only on rolling-validation forecast origins. Loading
# each job here ensures that monthly boundaries reflect its actual benchmark-
# aligned validation surface rather than assuming all targets share identical
# support.

lgbm_cadence_schedule_parts = []

for job_number, job in enumerate(
    lgbm_cadence_jobs_df.itertuples(index=False),
    start=1,
):
    print(
        f"Building cadence schedule [{job_number}/10] · "
        f"{TRACK_SHORT_LABELS[job.modeling_track]} · "
        f"{TARGET_SHORT_LABELS[job.metric]} · "
        f"h={int(job.horizon)}",
        flush=True,
    )

    payload = load_cadence_job(
        job.modeling_track,
        job.metric,
        int(job.horizon),
    )

    for cadence in LGBM_CADENCE_POLICIES:
        schedule_part = build_cadence_schedule(
            payload["validation_df"],
            cadence,
        ).copy()

        schedule_part["job_id"] = job.job_id
        schedule_part["modeling_track"] = job.modeling_track
        schedule_part["metric"] = job.metric
        schedule_part["horizon"] = int(job.horizon)
        schedule_part["cadence"] = cadence

        lgbm_cadence_schedule_parts.append(schedule_part)

    del payload
    gc.collect()

lgbm_cadence_schedule_df = (
    pd.concat(lgbm_cadence_schedule_parts, ignore_index=True)
    .sort_values(["job_id", "cadence", "refit_step"])
    .reset_index(drop=True)
)

if lgbm_cadence_schedule_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any():
    raise ValueError(
        "LightGBM cadence schedule is not unique by "
        "Job × Cadence × Refit Step."
    )

expected_lgbm_cadence_refits = len(lgbm_cadence_schedule_df)

# The common validation window currently spans ten monthly refresh points.
# Fail loudly if the workload no longer matches that established contract.
if expected_lgbm_cadence_refits != 110:
    raise RuntimeError(
        f"Expected 110 LightGBM cadence refits; "
        f"found {expected_lgbm_cadence_refits}."
    )


# ---------------------------------------------------------------------
# Reader-facing workload
# ---------------------------------------------------------------------

lgbm_cadence_workload_df = (
    lgbm_cadence_schedule_df
    .groupby(["job_id", "cadence"], as_index=False)
    .size()
    .pivot(index="job_id", columns="cadence", values="size")
    .reset_index()
    .rename(
        columns={
            "validation_start_only": "Validation-start fits",
            "monthly": "Monthly-refresh fits",
        }
    )
)

cadence_labels_df = lgbm_cadence_jobs_df[
    ["job_id", "modeling_track", "metric", "horizon"]
].copy()

cadence_labels_df["Job"] = cadence_labels_df.apply(
    lambda row: (
        f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
        f"{TARGET_SHORT_LABELS[row['metric']]} · "
        f"h={int(row['horizon'])}"
    ),
    axis=1,
)

lgbm_cadence_workload_df = (
    lgbm_cadence_workload_df
    .merge(
        cadence_labels_df[["job_id", "Job"]],
        on="job_id",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "Job",
            "Validation-start fits",
            "Monthly-refresh fits",
        ]
    ]
)

display(lgbm_cadence_workload_df)

print(
    f"Planned LightGBM cadence refits: "
    f"{expected_lgbm_cadence_refits}",
    flush=True,
)

Building cadence schedule [1/10] · Full-History · FHVHV speed · h=5
Building cadence schedule [2/10] · Full-History · FHVHV trips · h=5
Building cadence schedule [3/10] · Full-History · Subway · h=5
Building cadence schedule [4/10] · Full-History · Taxi speed · h=5
Building cadence schedule [5/10] · Full-History · Taxi trips · h=5
Building cadence schedule [6/10] · Post-CP · FHVHV speed · h=5
Building cadence schedule [7/10] · Post-CP · FHVHV trips · h=5
Building cadence schedule [8/10] · Post-CP · Subway · h=5
Building cadence schedule [9/10] · Post-CP · Taxi speed · h=5
Building cadence schedule [10/10] · Post-CP · Taxi trips · h=5


,Job,Validation-start fits,Monthly-refresh fits
0,Full-History · FHVHV speed · h=5,1,10
1,Full-History · FHVHV trips · h=5,1,10
2,Full-History · Subway · h=5,1,10
3,Full-History · Taxi speed · h=5,1,10
4,Full-History · Taxi trips · h=5,1,10
5,Post-CP · FHVHV speed · h=5,1,10
6,Post-CP · FHVHV trips · h=5,1,10
7,Post-CP · Subway · h=5,1,10
8,Post-CP · Taxi speed · h=5,1,10
9,Post-CP · Taxi trips · h=5,1,10


Planned LightGBM cadence refits: 110


**Findings.** The cadence experiment covers the complete forecasting target set across both modeling tracks at h=5: 10 Track × Target jobs spanning FHVHV speed, FHVHV trips, Subway ridership, Taxi speed, and Taxi trips. Each job receives one validation-start fit and ten monthly-refresh fits, producing 110 planned LightGBM refits in total.

This gives the cadence decision direct evidence for every target in both tracks while keeping the comparison controlled: all jobs use the frozen More expressive LightGBM specification, raw target representation, the same h=5 forecast horizon, and the same 250,000-row bounded training cap. The experiment is therefore positioned to determine whether refit cadence should be shared globally or vary by metric.

### 4.4.1.6H — Run or recover the LightGBM cadence experiment

Each scheduled refit is one durable experimental unit identified by **Sentinel × Cadence × Refit Step**.

At each refit origin, training rows are legal only when their target outcome has already become observable. Monthly models therefore gain access to additional historical outcomes over time without seeing beyond the current forecast origin. Each refreshed model owns the validation forecasts from its refit point until the next scheduled refresh.

The checkpoints store additive forecast-error quantities rather than only segment-level metrics. This matters because monthly segments contain different numbers of rows; their errors can therefore be recombined into exact full-validation MAE, RMSE, NMAE, and bias instead of incorrectly averaging monthly metrics.

Execution is restartable. A completed current-contract checkpoint is never refit on restart; missing or failed units are the only units recomputed. Resetting is the only destructive control.

In [39]:
# ---------------------------------------------------------------------
# 4.4.1.6H — Run or recover the LightGBM cadence experiment
# ---------------------------------------------------------------------

RUN_LGBM_CADENCE = True
RESET_LGBM_CADENCE = False

LGBM_CADENCE_ROOT = INTERMEDIATE_DIR / LGBM_CADENCE_CONTRACT_ID
LGBM_CADENCE_PARTS_DIR = LGBM_CADENCE_ROOT / "parts"
LGBM_CADENCE_RESULTS_PATH = LGBM_CADENCE_ROOT / "results.parquet"

if RESET_LGBM_CADENCE:
    reset_checkpoint_dir(LGBM_CADENCE_ROOT)

LGBM_CADENCE_PARTS_DIR.mkdir(parents=True, exist_ok=True)

LGBM_CADENCE_RESULT_KEYS = ["job_id", "cadence", "refit_step"]

lgbm_cadence_expected_keys = set(
    lgbm_cadence_schedule_df[LGBM_CADENCE_RESULT_KEYS].itertuples(
        index=False,
        name=None,
    )
)


def lgbm_cadence_checkpoint_path(job_id, cadence, refit_step):
    """Return the durable checkpoint path for one legal cadence refit."""
    name = slugify(
        f"{LGBM_CADENCE_CONTRACT_ID}__{job_id}__"
        f"{cadence}__refit_{int(refit_step)}"
    )
    return LGBM_CADENCE_PARTS_DIR / f"{name}.parquet"


def lgbm_cadence_checkpoint_is_complete(job_id, cadence, refit_step):
    """Accept only one successful row from the current cadence contract."""
    path = lgbm_cadence_checkpoint_path(job_id, cadence, refit_step)

    if not path.exists():
        return False

    try:
        checkpoint_df = pd.read_parquet(path)
    except Exception:
        return False

    if len(checkpoint_df) != 1:
        return False

    row = checkpoint_df.iloc[0]

    return (
        row.get("status") == "success"
        and row.get("cadence_contract_id") == LGBM_CADENCE_CONTRACT_ID
        and row.get("job_id") == job_id
        and row.get("cadence") == cadence
        and int(row.get("refit_step")) == int(refit_step)
    )


def fit_lgbm_cadence_refit_segment(payload, policy_schedule_df, refit_step):
    """Fit frozen LightGBM once and score the origins owned by this refit."""
    schedule = (
        policy_schedule_df
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    row = schedule.loc[
        schedule["refit_step"].eq(int(refit_step))
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one cadence row for refit step {refit_step}; "
            f"found {len(row)}."
        )

    row = row.iloc[0]
    refit_origin_id = int(row["origin_observation_sequence_id"])
    refit_origin_date = pd.Timestamp(row["origin_date"])

    later_refits = schedule.loc[
        schedule["refit_step"].gt(int(refit_step)),
        "origin_observation_sequence_id",
    ]

    next_refit_origin_id = (
        int(later_refits.iloc[0])
        if len(later_refits)
        else None
    )

    matrix_df = payload["matrix_df"]
    validation_df = payload["validation_df"]

    # A training row is legal only after its target position is observable at
    # the current refit origin. The canonical sequence ID preserves the proper
    # within-day ordering across dayparts.
    training_mask = (
        matrix_df["_label_available"]
        & matrix_df["target_observation_sequence_id"].le(refit_origin_id)
    )

    train_df = matrix_df.loc[training_mask].copy()

    if train_df.empty:
        raise ValueError(
            f"Refit {refit_step} has no legal training rows."
        )

    uncapped_train_rows = len(train_df)

    train_df = thin_training_by_origin(
        train_df,
        max_rows=LGBM_CADENCE_TRAIN_ROW_CAP,
    )

    # A monthly model owns forecasts until immediately before the next refresh.
    # The validation-start-only model has no later refit and therefore owns the
    # entire rolling-validation surface.
    score_mask = validation_df[
        "origin_observation_sequence_id"
    ].ge(refit_origin_id)

    if next_refit_origin_id is not None:
        score_mask &= validation_df[
            "origin_observation_sequence_id"
        ].lt(next_refit_origin_id)

    score_df = validation_df.loc[score_mask].copy()

    if score_df.empty:
        raise ValueError(
            f"Refit {refit_step} has no validation rows assigned to it."
        )

    categorical_names = payload["categorical_names"]

    X_train, X_score = prepare_lightgbm_frames(
        train_df[payload["predictor_names"]],
        score_df[payload["predictor_names"]],
        categorical_names,
    )

    # Representation testing froze raw units for all count targets; speed
    # targets were already raw. Cadence therefore changes only when the model
    # relearns, not how the target is represented.
    y_train = pd.to_numeric(
        train_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    y_score = pd.to_numeric(
        score_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    model = LGBMRegressor(
        **LGBM_CAPACITY_PROFILES[
            SELECTED_LGBM_CAPACITY_PROFILE
        ]["params"]
    )

    fit_start = perf_counter()
    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_names,
    )
    fit_seconds = perf_counter() - fit_start

    predict_start = perf_counter()
    predictions = np.asarray(
        model.predict(X_score),
        dtype=float,
    )
    predict_seconds = perf_counter() - predict_start

    if not np.isfinite(predictions).all():
        raise ValueError(
            f"Refit {refit_step} produced non-finite predictions."
        )

    model_stats = regression_metric_sums(
        y_score,
        predictions,
        prefix="model",
    )

    benchmark_stats = regression_metric_sums(
        y_score,
        score_df[payload["benchmark_column"]],
        prefix="benchmark",
    )

    if (
        model_stats["model_evaluated_rows"]
        != benchmark_stats["benchmark_evaluated_rows"]
    ):
        raise ValueError(
            "Model and benchmark were not evaluated on identical row support."
        )

    result = {
        "status": "success",
        "cadence_contract_id": LGBM_CADENCE_CONTRACT_ID,
        "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
        "target_representation": "raw",
        "cadence": str(row["cadence"]),
        "refit_step": int(refit_step),
        "refit_origin_id": refit_origin_id,
        "refit_origin_date": refit_origin_date,
        "next_refit_origin_id": next_refit_origin_id,
        "train_rows_uncapped": int(uncapped_train_rows),
        "train_rows": int(len(train_df)),
        "train_origins": int(
            train_df["origin_observation_sequence_id"].nunique()
        ),
        "score_rows": int(len(score_df)),
        "score_origin_start": int(
            score_df["origin_observation_sequence_id"].min()
        ),
        "score_origin_end": int(
            score_df["origin_observation_sequence_id"].max()
        ),
        "negative_prediction_count": int((predictions < 0).sum()),
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "benchmark_id": payload["benchmark_id"],
        "benchmark_label": payload["benchmark_label"],
        **model_stats,
        **benchmark_stats,
    }

    del (
        train_df,
        score_df,
        X_train,
        X_score,
        y_train,
        y_score,
        predictions,
        model,
    )
    gc.collect()

    return result


# ---------------------------------------------------------------------
# Determine current restart state from logical evidence
# ---------------------------------------------------------------------

complete_lgbm_cadence_keys = {
    (
        row.job_id,
        row.cadence,
        int(row.refit_step),
    )
    for row in lgbm_cadence_schedule_df.itertuples(index=False)
    if lgbm_cadence_checkpoint_is_complete(
        row.job_id,
        row.cadence,
        row.refit_step,
    )
}

print(
    "LightGBM cadence restart state: "
    f"{len(complete_lgbm_cadence_keys)}/"
    f"{expected_lgbm_cadence_refits} refits complete.",
    flush=True,
)


# ---------------------------------------------------------------------
# Reconcile only missing or failed logical units
# ---------------------------------------------------------------------

if RUN_LGBM_CADENCE:
    active_job_id = None
    payload = None

    missing_schedule_df = lgbm_cadence_schedule_df.loc[
        ~lgbm_cadence_schedule_df.apply(
            lambda row: (
                row["job_id"],
                row["cadence"],
                int(row["refit_step"]),
            )
            in complete_lgbm_cadence_keys,
            axis=1,
        )
    ].copy()

    total_missing = len(missing_schedule_df)

    for unit_number, schedule_row in enumerate(
        missing_schedule_df.itertuples(index=False),
        start=1,
    ):
        job_id = schedule_row.job_id
        cadence = schedule_row.cadence
        refit_step = int(schedule_row.refit_step)

        job_match = lgbm_cadence_jobs_df.loc[
            lgbm_cadence_jobs_df["job_id"].eq(job_id)
        ]

        if len(job_match) != 1:
            raise ValueError(
                f"Expected one cadence job row for {job_id}; "
                f"found {len(job_match)}."
            )

        job_row = job_match.iloc[0]

        # All refit windows for the same forecasting job share one underlying matrix.
        # Keep it resident until the job changes to avoid repeated Parquet loading
        # during its monthly sequence.
        if active_job_id != job_id:
            if payload is not None:
                del payload
                gc.collect()

            payload = load_cadence_job(
                job_row["modeling_track"],
                job_row["metric"],
                int(job_row["horizon"]),
            )
            active_job_id = job_id

        policy_schedule_df = lgbm_cadence_schedule_df.loc[
            lgbm_cadence_schedule_df["job_id"].eq(job_id)
            & lgbm_cadence_schedule_df["cadence"].eq(cadence)
        ].copy()

        print(
            f"[{unit_number}/{total_missing}] "
            f"{TRACK_SHORT_LABELS[job_row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[job_row['metric']]} · "
            f"h={int(job_row['horizon'])} · "
            f"{LGBM_CADENCE_POLICY_LABELS[cadence]} · "
            f"refit={refit_step}",
            flush=True,
        )

        try:
            result = fit_lgbm_cadence_refit_segment(
                payload,
                policy_schedule_df,
                refit_step,
            )

            result.update(
                {
                    "job_id": job_id,
                    "modeling_track": job_row["modeling_track"],
                    "metric": job_row["metric"],
                    "horizon": int(job_row["horizon"]),
                }
            )

            print(
                f"    rows={result['score_rows']:,} · "
                f"fit={result['fit_seconds']:.3f}s · "
                f"negative={result['negative_prediction_count']:,}",
                flush=True,
            )

        except Exception as exc:
            result = {
                "status": "failed",
                "cadence_contract_id": LGBM_CADENCE_CONTRACT_ID,
                "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
                "target_representation": "raw",
                "job_id": job_id,
                "modeling_track": job_row["modeling_track"],
                "metric": job_row["metric"],
                "horizon": int(job_row["horizon"]),
                "cadence": cadence,
                "refit_step": refit_step,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }

            print(
                f"    failed · {type(exc).__name__}: {exc}",
                flush=True,
            )

        atomic_parquet_write(
            pd.DataFrame([result]),
            lgbm_cadence_checkpoint_path(
                job_id,
                cadence,
                refit_step,
            ),
        )

    if payload is not None:
        del payload
        gc.collect()


# ---------------------------------------------------------------------
# Reconstruct the consolidated result from current-contract checkpoints
# ---------------------------------------------------------------------

lgbm_cadence_parts = []

for schedule_row in lgbm_cadence_schedule_df.itertuples(index=False):
    path = lgbm_cadence_checkpoint_path(
        schedule_row.job_id,
        schedule_row.cadence,
        schedule_row.refit_step,
    )

    if path.exists():
        part_df = pd.read_parquet(path)

        if (
            len(part_df) == 1
            and part_df["cadence_contract_id"]
            .eq(LGBM_CADENCE_CONTRACT_ID)
            .all()
        ):
            lgbm_cadence_parts.append(part_df)

lgbm_cadence_results_df = (
    pd.concat(lgbm_cadence_parts, ignore_index=True)
    if lgbm_cadence_parts
    else pd.DataFrame()
)

if not lgbm_cadence_results_df.empty:
    if lgbm_cadence_results_df.duplicated(
        LGBM_CADENCE_RESULT_KEYS
    ).any():
        raise ValueError(
            "LightGBM cadence checkpoints contain duplicate logical units."
        )

    atomic_parquet_write(
        lgbm_cadence_results_df,
        LGBM_CADENCE_RESULTS_PATH,
    )

successful_lgbm_cadence_refits = (
    int(lgbm_cadence_results_df["status"].eq("success").sum())
    if not lgbm_cadence_results_df.empty
    else 0
)

failed_lgbm_cadence_refits = (
    int(lgbm_cadence_results_df["status"].eq("failed").sum())
    if not lgbm_cadence_results_df.empty
    else 0
)

print(
    "\nLightGBM cadence result state: "
    f"{successful_lgbm_cadence_refits}/"
    f"{expected_lgbm_cadence_refits} successful refits; "
    f"{failed_lgbm_cadence_refits} failed.",
    flush=True,
)

LightGBM cadence restart state: 110/110 refits complete.

LightGBM cadence result state: 110/110 successful refits; 0 failed.


**Findings.** All 110 planned LightGBM cadence refits completed successfully with no failures across both modeling tracks and all five forecasting targets. Fitting remained lightweight throughout: individual fits ranged from 1.661 to 4.606 seconds, with a median of 3.569 seconds and about 374.834 seconds of total model-fitting time across the entire experiment. This confirms that even repeated monthly retraining remains computationally practical for LightGBM.

Negative predictions remained confined to the three count targets and varied materially by target, track, and refit window; both speed targets produced none. These values are retained as diagnostics without clipping or other correction. They do not determine the cadence choice here—the next comparison evaluates each cadence over its complete validation support using reconstructed full-period MAE and benchmark skill.

### 4.4.1.6I — Compare fixed and monthly LightGBM forecasting across all targets

The final cadence comparison reconstructs each policy over the **entire rolling-validation period** for all ten Track × Target h=5 jobs.

Monthly validation is split across multiple refreshed models, so monthly MAE cannot be obtained by simply averaging monthly MAEs. Instead, the additive error totals stored at each refit are summed first and converted back into exact full-period metrics. The validation-start-only and monthly policies therefore cover identical forecast support within each job.

MAE remains the primary selector. With all five targets represented in both modeling tracks, the evidence can support either a single shared cadence or a metric-specific rule where both tracks point consistently to the same target-level choice.

The table reports the exact matched comparisons. The supporting visual shows each Track × Target job on an accuracy-gain versus runtime-cost surface. Positive x-values favor monthly refreshing; negative values favor fitting once at validation start.

In [40]:
# ---------------------------------------------------------------------
# 4.4.1.6I — Compare fixed and monthly LightGBM forecasting across all targets
# ---------------------------------------------------------------------

if lgbm_cadence_results_df.empty:
    raise RuntimeError(
        "No LightGBM cadence results are available."
    )

failed_lgbm_cadence_df = lgbm_cadence_results_df.loc[
    lgbm_cadence_results_df["status"].eq("failed")
].copy()

if not failed_lgbm_cadence_df.empty:
    display(
        failed_lgbm_cadence_df[
            [
                "modeling_track",
                "metric",
                "horizon",
                "cadence",
                "refit_step",
                "error_type",
                "error_message",
            ]
        ]
    )

    raise RuntimeError(
        "LightGBM cadence testing contains failed refits. "
        "Resolve them before selecting a cadence."
    )

lgbm_cadence_success_df = lgbm_cadence_results_df.loc[
    lgbm_cadence_results_df["status"].eq("success")
].copy()

if len(lgbm_cadence_success_df) != expected_lgbm_cadence_refits:
    raise RuntimeError(
        f"Expected {expected_lgbm_cadence_refits} successful cadence refits; "
        f"found {len(lgbm_cadence_success_df)}."
    )

if lgbm_cadence_success_df.duplicated(
    ["job_id", "cadence", "refit_step"]
).any():
    raise ValueError(
        "LightGBM cadence results are not unique by "
        "Job × Cadence × Refit Step."
    )


# ---------------------------------------------------------------------
# Reconstruct exact full-validation metrics for each cadence policy
# ---------------------------------------------------------------------
# Monthly validation is partitioned across multiple refreshed models. The
# segment-level additive error totals must therefore be summed before MAE,
# RMSE, NMAE, and bias are reconstructed. Averaging monthly metrics directly
# would give unequal-sized segments equal weight and would be incorrect.

lgbm_cadence_aggregate_df = (
    lgbm_cadence_success_df
    .groupby(
        [
            "job_id",
            "modeling_track",
            "metric",
            "horizon",
            "cadence",
        ],
        as_index=False,
    )
    .agg(
        refits=("refit_step", "nunique"),
        evaluated_rows=("model_evaluated_rows", "sum"),
        model_absolute_error_sum=("model_absolute_error_sum", "sum"),
        model_squared_error_sum=("model_squared_error_sum", "sum"),
        model_error_sum=("model_error_sum", "sum"),
        model_absolute_actual_sum=("model_absolute_actual_sum", "sum"),
        benchmark_evaluated_rows=("benchmark_evaluated_rows", "sum"),
        benchmark_absolute_error_sum=("benchmark_absolute_error_sum", "sum"),
        fit_seconds=("fit_seconds", "sum"),
        predict_seconds=("predict_seconds", "sum"),
        negative_prediction_count=("negative_prediction_count", "sum"),
        min_train_rows=("train_rows", "min"),
        max_train_rows=("train_rows", "max"),
    )
)

if not (
    lgbm_cadence_aggregate_df["evaluated_rows"]
    == lgbm_cadence_aggregate_df["benchmark_evaluated_rows"]
).all():
    raise ValueError(
        "Model and benchmark aggregate support differ."
    )

lgbm_cadence_aggregate_df["mae"] = (
    lgbm_cadence_aggregate_df["model_absolute_error_sum"]
    / lgbm_cadence_aggregate_df["evaluated_rows"]
)

lgbm_cadence_aggregate_df["rmse"] = np.sqrt(
    lgbm_cadence_aggregate_df["model_squared_error_sum"]
    / lgbm_cadence_aggregate_df["evaluated_rows"]
)

lgbm_cadence_aggregate_df["nmae"] = (
    lgbm_cadence_aggregate_df["model_absolute_error_sum"]
    / lgbm_cadence_aggregate_df["model_absolute_actual_sum"]
)

lgbm_cadence_aggregate_df["bias"] = (
    lgbm_cadence_aggregate_df["model_error_sum"]
    / lgbm_cadence_aggregate_df["evaluated_rows"]
)

lgbm_cadence_aggregate_df["benchmark_mae"] = (
    lgbm_cadence_aggregate_df["benchmark_absolute_error_sum"]
    / lgbm_cadence_aggregate_df["benchmark_evaluated_rows"]
)

lgbm_cadence_aggregate_df["skill_vs_benchmark_pct"] = (
    100.0
    * (
        lgbm_cadence_aggregate_df["benchmark_mae"]
        - lgbm_cadence_aggregate_df["mae"]
    )
    / lgbm_cadence_aggregate_df["benchmark_mae"]
)


# ---------------------------------------------------------------------
# Pair validation-start-only and monthly results within each job
# ---------------------------------------------------------------------

lgbm_fixed_df = lgbm_cadence_aggregate_df.loc[
    lgbm_cadence_aggregate_df["cadence"].eq(
        "validation_start_only"
    )
].copy()

lgbm_monthly_df = lgbm_cadence_aggregate_df.loc[
    lgbm_cadence_aggregate_df["cadence"].eq("monthly")
].copy()

lgbm_cadence_comparison_df = (
    lgbm_fixed_df[
        [
            "job_id",
            "modeling_track",
            "metric",
            "horizon",
            "mae",
            "skill_vs_benchmark_pct",
            "refits",
            "fit_seconds",
            "negative_prediction_count",
        ]
    ]
    .rename(
        columns={
            "mae": "fixed_mae",
            "skill_vs_benchmark_pct": "fixed_skill_pct",
            "refits": "fixed_refits",
            "fit_seconds": "fixed_fit_seconds",
            "negative_prediction_count": "fixed_negative_predictions",
        }
    )
    .merge(
        lgbm_monthly_df[
            [
                "job_id",
                "mae",
                "skill_vs_benchmark_pct",
                "refits",
                "fit_seconds",
                "negative_prediction_count",
            ]
        ].rename(
            columns={
                "mae": "monthly_mae",
                "skill_vs_benchmark_pct": "monthly_skill_pct",
                "refits": "monthly_refits",
                "fit_seconds": "monthly_fit_seconds",
                "negative_prediction_count": "monthly_negative_predictions",
            }
        ),
        on="job_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(lgbm_cadence_comparison_df) != 10:
    raise RuntimeError(
        "Fixed and monthly LightGBM cadence results do not cover "
        "all 10 Track × Target h=5 jobs."
    )

lgbm_cadence_comparison_df["monthly_mae_improvement_pct"] = (
    100.0
    * (
        lgbm_cadence_comparison_df["fixed_mae"]
        - lgbm_cadence_comparison_df["monthly_mae"]
    )
    / lgbm_cadence_comparison_df["fixed_mae"]
)

lgbm_cadence_comparison_df["fit_time_multiplier"] = (
    lgbm_cadence_comparison_df["monthly_fit_seconds"]
    / lgbm_cadence_comparison_df["fixed_fit_seconds"]
)

lgbm_cadence_comparison_df["Job"] = (
    lgbm_cadence_comparison_df.apply(
        lambda row: (
            f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[row['metric']]} · "
            f"h={int(row['horizon'])}"
        ),
        axis=1,
    )
)


# ---------------------------------------------------------------------
# Reader-facing matched comparison table
# ---------------------------------------------------------------------

lgbm_cadence_display_df = (
    lgbm_cadence_comparison_df[
        [
            "Job",
            "fixed_mae",
            "monthly_mae",
            "monthly_mae_improvement_pct",
            "fixed_skill_pct",
            "monthly_skill_pct",
            "fixed_refits",
            "monthly_refits",
            "fixed_fit_seconds",
            "monthly_fit_seconds",
            "fit_time_multiplier",
        ]
    ]
    .rename(
        columns={
            "fixed_mae": "Fixed MAE",
            "monthly_mae": "Monthly MAE",
            "monthly_mae_improvement_pct": "Monthly MAE improvement %",
            "fixed_skill_pct": "Fixed skill %",
            "monthly_skill_pct": "Monthly skill %",
            "fixed_refits": "Fixed refits",
            "monthly_refits": "Monthly refits",
            "fixed_fit_seconds": "Fixed fit seconds",
            "monthly_fit_seconds": "Monthly fit seconds",
            "fit_time_multiplier": "Fit-time multiplier",
        }
    )
    .sort_values("Job")
    .reset_index(drop=True)
)

for column in [
    "Fixed MAE",
    "Monthly MAE",
    "Monthly MAE improvement %",
    "Fixed skill %",
    "Monthly skill %",
    "Fixed fit seconds",
    "Monthly fit seconds",
    "Fit-time multiplier",
]:
    lgbm_cadence_display_df[column] = (
        lgbm_cadence_display_df[column].round(3)
    )

display(lgbm_cadence_display_df)


# ---------------------------------------------------------------------
# Supporting visual — cadence accuracy gain by metric and modeling track
# ---------------------------------------------------------------------
# Color identifies the forecasting metric while marker shape identifies the
# modeling track. Horizon is constant at h=5 for this experiment, so encoding
# it visually would add no information.

metric_order = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

track_symbols = {
    "full_history_regime_aware": "circle",
    "post_cp_primary": "diamond",
}

if RUN_VISUALS:
    lgbm_cadence_fig = go.Figure()

    for metric in metric_order:
        for track in track_symbols:
            plot_df = lgbm_cadence_comparison_df.loc[
                lgbm_cadence_comparison_df["metric"].eq(metric)
                & lgbm_cadence_comparison_df["modeling_track"].eq(track)
            ].copy()

            if plot_df.empty:
                continue

            tooltip_data = np.column_stack(
                [
                    plot_df["monthly_mae_improvement_pct"]
                    .map(lambda value: f"{value:+.3f}%"),
                    plot_df["fixed_mae"]
                    .map(lambda value: f"{value:.3f}"),
                    plot_df["monthly_mae"]
                    .map(lambda value: f"{value:.3f}"),
                    plot_df["fixed_skill_pct"]
                    .map(lambda value: f"{value:+.3f}%"),
                    plot_df["monthly_skill_pct"]
                    .map(lambda value: f"{value:+.3f}%"),
                    plot_df["fixed_fit_seconds"]
                    .map(lambda value: f"{value:.3f}s"),
                    plot_df["monthly_fit_seconds"]
                    .map(lambda value: f"{value:.3f}s"),
                    plot_df["fit_time_multiplier"]
                    .map(lambda value: f"{value:.3f}×"),
                ]
            )

            lgbm_cadence_fig.add_trace(
                go.Scatter(
                    x=plot_df["monthly_mae_improvement_pct"].round(3),
                    y=plot_df["fit_time_multiplier"].round(3),
                    mode="markers",
                    name=(
                        f"{TARGET_SHORT_LABELS[metric]} · "
                        f"{TRACK_SHORT_LABELS[track]}"
                    ),
                    marker={
                        "size": 18,
                        "symbol": track_symbols[track],
                        "line": {"width": 1},
                    },
                    customdata=tooltip_data,
                    hovertext=plot_df["Job"],
                    hovertemplate=(
                        "%{hovertext}<br>"
                        "Monthly MAE improvement: %{customdata[0]}<br>"
                        "Fixed MAE: %{customdata[1]}<br>"
                        "Monthly MAE: %{customdata[2]}<br>"
                        "Fixed benchmark skill: %{customdata[3]}<br>"
                        "Monthly benchmark skill: %{customdata[4]}<br>"
                        "Fixed fit time: %{customdata[5]}<br>"
                        "Monthly fit time: %{customdata[6]}<br>"
                        "Fit-time multiplier: %{customdata[7]}"
                        "<extra></extra>"
                    ),
                )
            )

    lgbm_cadence_fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
        annotation_text="No MAE gain",
        annotation_position="top left",
    )

    lgbm_cadence_fig.update_layout(
        title=(
            "LightGBM Refit Cadence — "
            "Accuracy Gain versus Runtime Cost"
        ),
        xaxis_title="Monthly vs fixed MAE improvement (%)",
        yaxis_title="Monthly / fixed fit-time multiplier",
        width=960,
        height=650,
        margin=dict(l=80, r=50, t=90, b=80),
        legend_title_text="Metric × Track",
    )

    apply_branding(lgbm_cadence_fig)
    lgbm_cadence_fig.show()

,Job,Fixed MAE,Monthly MAE,Monthly MAE improvement %,Fixed skill %,Monthly skill %,Fixed refits,Monthly refits,Fixed fit seconds,Monthly fit seconds,Fit-time multiplier
0,Full-History · FHVHV speed · h=5,0.990,1.025,-3.512,25.507,22.891,1,10,3.899,40.111,10.288
1,Full-History · FHVHV trips · h=5,53.148,86.922,-63.549,23.447,-25.202,1,10,3.635,39.483,10.861
2,Full-History · Subway · h=5,440.748,481.043,-9.142,17.120,9.543,1,10,3.223,36.048,11.185
3,Full-History · Taxi speed · h=5,1.769,1.739,1.698,22.935,24.244,1,10,3.810,36.817,9.664
4,Full-History · Taxi trips · h=5,15.701,19.861,-26.497,22.210,1.598,1,10,3.525,31.398,8.908
5,Post-CP · FHVHV speed · h=5,1.071,1.178,-10.005,18.715,10.583,1,10,2.356,36.888,15.657
6,Post-CP · FHVHV trips · h=5,58.084,86.558,-49.023,16.586,-24.305,1,10,2.331,35.085,15.050
7,Post-CP · Subway · h=5,505.111,577.959,-14.422,6.728,-6.724,1,10,1.661,28.965,17.439
8,Post-CP · Taxi speed · h=5,2.276,2.330,-2.396,22.875,21.027,1,10,1.784,31.208,17.492
9,Post-CP · Taxi trips · h=5,18.599,18.496,0.550,8.138,8.644,1,10,2.049,30.558,14.912


**Findings.** Validation-start-only fitting is the stronger cadence across the full ten-job h=5 comparison. Monthly refreshing lowers MAE in only 2 of 10 Track × Target jobs: Full-History Taxi speed improves by 1.698%, and Post-CP Taxi trips improves by 0.550%. The remaining eight jobs worsen under monthly refreshing, including substantial deterioration for FHVHV trips in both tracks (-63.549% Full-History; -49.023% Post-CP), Taxi trips in Full-History (-26.497%), and Subway in both tracks (-9.142% and -14.422%).

The strongest target-level agreement favors fitting once. FHVHV speed, FHVHV trips, and Subway all prefer validation-start-only in both modeling tracks. Taxi speed and Taxi trips split by track, so neither supports a stable metric-specific monthly rule. Benchmark-relative skill follows the same pattern: monthly refreshing pushes both FHVHV-trip jobs below the frozen benchmark and also makes Post-CP Subway negative. Monthly fitting additionally requires about 8.908×–17.492× the fitting time of a single validation-start model. Although LightGBM remains inexpensive in absolute terms, the added refitting cost is not accompanied by a broad accuracy benefit.

> ✅ Decision — Use validation\-start\-only fitting as the LightGBM refit cadence\.

Validation\-start\-only produces lower MAE in 8 of 10 Track × Target h=5 comparisons\. Monthly refreshing improves only two jobs, while several targets deteriorate materially and benchmark\-relative skill also weakens in multiple cases\. The evidence therefore does not support a stable global or metric\-specific monthly\-refresh advantage\.

Freeze validation\-start\-only as the LightGBM cadence for full rolling validation\. Monthly refreshing will not be carried forward\.

### 4.4.1.6J — Freeze the LightGBM production configuration

The LightGBM tuning work is complete.

The evidence points to one shared setup for all 30 forecasting jobs:

- the **More expressive** LightGBM profile;
- targets modeled in their **original units**;
- one model fit at the **start of validation**, rather than monthly retraining.

Using one shared setup keeps the forecasting system simpler and easier to compare across targets. None of the experiments showed a consistent reason to create separate LightGBM settings for particular targets, horizons, or modeling tracks.

The full workload is:

**2 modeling tracks × 5 targets × 3 horizons = 30 forecasting jobs.**

During validation, LightGBM's raw predictions are preserved exactly as produced. Any rule that prevents physically impossible negative forecasts is tested separately rather than hidden inside model fitting.

In [41]:
# ---------------------------------------------------------------------
# 4.4.1.6J — Freeze and visualize the LightGBM production configuration
# ---------------------------------------------------------------------

LGBM_PRODUCTION_CONFIG_ID = "lgbm_production_config_v1"
LGBM_PRODUCTION_CONFIG_PATH = (
    INTERMEDIATE_DIR / f"{LGBM_PRODUCTION_CONFIG_ID}.parquet"
)

LGBM_PRODUCTION_HORIZONS = [1, 2, 5]
LGBM_PRODUCTION_REFIT_CADENCE = "validation_start_only"
LGBM_PRODUCTION_TARGET_REPRESENTATION = "raw"

# The final configuration is shared across all production jobs. Build the
# executable roster explicitly so Section 7 can consume a frozen contract
# rather than reconstructing tuning decisions inside the validation loop.
production_tracks = sorted(
    lgbm_capacity_jobs_df["modeling_track"].dropna().unique().tolist()
)

if len(production_tracks) != 2:
    raise RuntimeError(
        f"Expected 2 modeling tracks; found {len(production_tracks)}."
    )

if len(FORECAST_TARGETS) != 5:
    raise RuntimeError(
        f"Expected 5 forecasting targets; found {len(FORECAST_TARGETS)}."
    )

selected_lgbm_params = dict(
    LGBM_CAPACITY_PROFILES[
        SELECTED_LGBM_CAPACITY_PROFILE
    ]["params"]
)

lgbm_production_config_rows = []

for track in production_tracks:
    for metric in FORECAST_TARGETS:
        for horizon in LGBM_PRODUCTION_HORIZONS:
            lgbm_production_config_rows.append(
                {
                    "production_config_id": LGBM_PRODUCTION_CONFIG_ID,
                    "modeling_track": track,
                    "metric": metric,
                    "horizon": int(horizon),
                    "model_family": "LightGBM",
                    "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
                    "target_representation": LGBM_PRODUCTION_TARGET_REPRESENTATION,
                    "refit_cadence": LGBM_PRODUCTION_REFIT_CADENCE,
                    "prediction_constraint": "none",
                    "count_support_assessment": (
                        "pending_full_validation"
                        if metric in LGBM_REPRESENTATION_TARGETS
                        else "not_applicable"
                    ),
                }
            )

lgbm_production_config_df = pd.DataFrame(
    lgbm_production_config_rows
)

expected_production_jobs = (
    len(production_tracks)
    * len(FORECAST_TARGETS)
    * len(LGBM_PRODUCTION_HORIZONS)
)

if expected_production_jobs != 30:
    raise RuntimeError(
        f"Expected 30 production jobs; calculated "
        f"{expected_production_jobs}."
    )

if len(lgbm_production_config_df) != expected_production_jobs:
    raise RuntimeError(
        f"Expected {expected_production_jobs} production-contract rows; "
        f"found {len(lgbm_production_config_df)}."
    )

if lgbm_production_config_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise ValueError(
        "Production configuration is not unique by "
        "Track × Target × Horizon."
    )

if set(lgbm_production_config_df["metric"]) != set(FORECAST_TARGETS):
    raise ValueError(
        "Production configuration does not cover the frozen target roster."
    )

if set(lgbm_production_config_df["horizon"]) != set(
    LGBM_PRODUCTION_HORIZONS
):
    raise ValueError(
        "Production configuration does not cover horizons 1, 2, and 5."
    )

atomic_parquet_write(
    lgbm_production_config_df,
    LGBM_PRODUCTION_CONFIG_PATH,
)


# ---------------------------------------------------------------------
# Compact contract QA
# ---------------------------------------------------------------------

lgbm_production_contract_summary_df = pd.DataFrame(
    [
        {
            "Production jobs": len(lgbm_production_config_df),
            "Modeling tracks": lgbm_production_config_df[
                "modeling_track"
            ].nunique(),
            "Targets": lgbm_production_config_df["metric"].nunique(),
            "Horizons": lgbm_production_config_df["horizon"].nunique(),
            "Model family": "LightGBM",
            "Capacity profile": LGBM_CAPACITY_PROFILES[
                SELECTED_LGBM_CAPACITY_PROFILE
            ]["label"],
            "Target representation": "Raw",
            "Refit cadence": "Validation-start only",
        }
    ]
)

display(lgbm_production_contract_summary_df)


# ---------------------------------------------------------------------
# Reader-facing configuration overview
# ---------------------------------------------------------------------
# The visual collapses Track × Horizon because tuning did not establish
# different settings along those dimensions. The matrix therefore emphasizes
# the shared production specification without implying unsupported variation.

metric_labels = [
    TARGET_SHORT_LABELS[metric]
    for metric in FORECAST_TARGETS
]

configuration_rows = [
    "Model family",
    "Capacity",
    "Representation",
    "Refit cadence",
    "Horizons",
]

configuration_text = []

for setting in configuration_rows:
    row_values = []

    for metric in FORECAST_TARGETS:
        if setting == "Model family":
            value = "LightGBM"

        elif setting == "Capacity":
            value = LGBM_CAPACITY_PROFILES[
                SELECTED_LGBM_CAPACITY_PROFILE
            ]["label"]

        elif setting == "Representation":
            value = "Raw"

        elif setting == "Refit cadence":
            value = "Validation-start"

        elif setting == "Horizons":
            value = "1 · 2 · 5"

        row_values.append(value)

    configuration_text.append(row_values)


if RUN_VISUALS:
    configuration_z = np.ones(
        (
            len(configuration_rows),
            len(metric_labels),
        )
    )

    production_config_fig = go.Figure(
        data=go.Heatmap(
            z=configuration_z,
            x=metric_labels,
            y=configuration_rows,
            text=configuration_text,
            texttemplate="%{text}",
            colorscale=[
                [0.0, BRAND_COLORS["ice"]],
                [1.0, BRAND_COLORS["seafoam"]],
            ],
            showscale=False,
            hoverinfo="skip",
            xgap=3,
            ygap=3,
        )
    )

    # Keep the frozen LightGBM parameters visible alongside the matrix so the
    # figure serves as a compact implementation reference as well as a summary.
    parameter_lines = [
        f"n_estimators = {selected_lgbm_params.get('n_estimators')}",
        f"learning_rate = {selected_lgbm_params.get('learning_rate'):.3f}",
        f"num_leaves = {selected_lgbm_params.get('num_leaves')}",
        f"min_child_samples = {selected_lgbm_params.get('min_child_samples')}",
        f"subsample = {selected_lgbm_params.get('subsample'):.3f}",
        f"colsample_bytree = {selected_lgbm_params.get('colsample_bytree'):.3f}",
        f"reg_alpha = {selected_lgbm_params.get('reg_alpha', 0.0):.3f}",
        f"reg_lambda = {selected_lgbm_params.get('reg_lambda', 0.0):.3f}",
    ]

    production_config_fig.add_annotation(
        x=1.02,
        y=0.95,
        xref="paper",
        yref="paper",
        text=(
            "<b>Frozen LightGBM parameters</b><br>"
            + "<br>".join(parameter_lines)
        ),
        showarrow=False,
        align="left",
        xanchor="left",
        yanchor="top",
    )

    production_config_fig.add_annotation(
        x=1.02,
        y=0.22,
        xref="paper",
        yref="paper",
        text=(
            "<b>Prediction constraint</b><br>"
            "None during validation<br><br>"
            "<b>Count support check</b><br>"
            "Pending full-validation assessment"
        ),
        showarrow=False,
        align="left",
        xanchor="left",
        yanchor="top",
    )

    production_config_fig.update_layout(
        title=(
            "LightGBM Production Configuration — "
            "One Shared Specification Across 30 Forecasting Jobs"
        ),
        xaxis_title=None,
        yaxis_title=None,
        width=960,
        height=650,
        margin=dict(l=145, r=255, t=90, b=80),
    )

    apply_branding(production_config_fig)
    production_config_fig.show()

,Production jobs,Modeling tracks,Targets,Horizons,Model family,Capacity profile,Target representation,Refit cadence
0,30,2,5,3,LightGBM,More expressive,Raw,Validation-start only


**Findings.** The final LightGBM configuration consolidates the tuning evidence into one shared production specification for all 30 Track × Target × Horizon jobs. Every job uses LightGBM with 400 estimators, a 0.040 learning rate, 63 leaves, 20 minimum child samples, 0.900 subsampling, 0.900 column subsampling, no L1 penalty, and 1.000 L2 regularization. The same **More expressive** capacity profile, **raw target representation**, and **validation-start-only** cadence apply across both modeling tracks, all five targets, and horizons 1, 2, and 5.

No target-, track-, or horizon-specific exceptions are carried forward because the tuning evidence did not establish a stable need for them. Full rolling validation will therefore evaluate one consistent LightGBM specification across the complete forecasting workload. Predictions remain unconstrained during validation; the effect of enforcing the known zero lower bound shared by all five physical mobility outcomes will be assessed separately from the saved validation forecasts.

> ✅ Decision — Use one shared LightGBM setup across all 30 forecasting jobs\.

Use the More expressive LightGBM profile, raw target values, and validation\-start\-only fitting for both modeling tracks, all five targets, and horizons 1, 2, and 5\.

The tuning experiments did not show a consistent reason to create special settings for individual targets or horizons\.

## 4.4.1.7 — Run Full LightGBM Rolling Validation

The LightGBM model specification is now frozen. The next stage evaluates that configuration across the complete ordinary forecasting workload rather than the bounded tuning surfaces used to select it.

Each production job is one **Modeling Track × Target × Horizon** combination. With two tracks, five targets, and three horizons, full rolling validation contains **30 jobs**.

The validation rules remain unchanged:

- training history must be legally observable at the validation-start origin;
- the model is fit once at validation start and reused throughout that job's rolling-validation period;
- each job is scored only on its frozen benchmark-aligned validation support;
- MAE remains the primary metric, with RMSE, NMAE, bias, benchmark-relative skill, and negative predictions retained as supporting diagnostics;
- forecasts are saved in their original unconstrained form so the nonnegative support constraint can be assessed separately across all five targets;
- the final holdout remains untouched.

Execution will be checkpointed by production job. A completed current-contract job is reused on restart; only missing or failed jobs are recomputed.

In [42]:
# ---------------------------------------------------------------------
# 4.4.1.7A — Freeze the full LightGBM rolling-validation workload
# ---------------------------------------------------------------------

LGBM_FULL_VALIDATION_CONTRACT_ID = "lgbm_full_validation_v1"

lgbm_full_validation_jobs_df = (
    lgbm_production_config_df[
        [
            "modeling_track",
            "metric",
            "horizon",
            "model_family",
            "capacity_profile",
            "target_representation",
            "refit_cadence",
        ]
    ]
    .copy()
    .sort_values(["modeling_track", "metric", "horizon"])
    .reset_index(drop=True)
)

if len(lgbm_full_validation_jobs_df) != 30:
    raise RuntimeError(
        f"Expected 30 full-validation jobs; "
        f"found {len(lgbm_full_validation_jobs_df)}."
    )

if lgbm_full_validation_jobs_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise ValueError(
        "Full-validation workload is not unique by "
        "Track × Target × Horizon."
    )

lgbm_full_validation_jobs_df["job_id"] = (
    lgbm_full_validation_jobs_df.apply(
        lambda row: slugify(
            f"{row['modeling_track']}__"
            f"{row['metric']}__h{int(row['horizon'])}"
        ),
        axis=1,
    )
)

lgbm_full_validation_display_df = pd.DataFrame(
    [
        {
            "Production jobs": len(lgbm_full_validation_jobs_df),
            "Modeling tracks": lgbm_full_validation_jobs_df[
                "modeling_track"
            ].nunique(),
            "Targets": lgbm_full_validation_jobs_df["metric"].nunique(),
            "Horizons": lgbm_full_validation_jobs_df["horizon"].nunique(),
            "Capacity profile": LGBM_CAPACITY_PROFILES[
                SELECTED_LGBM_CAPACITY_PROFILE
            ]["label"],
            "Representation": "Raw",
            "Refit cadence": "Validation-start only",
            "Final holdout": "Untouched",
        }
    ]
)

display(lgbm_full_validation_display_df)

,Production jobs,Modeling tracks,Targets,Horizons,Capacity profile,Representation,Refit cadence,Final holdout
0,30,2,5,3,More expressive,Raw,Validation-start only,Untouched


**Findings.** The full rolling-validation contract contains exactly **30 production jobs**, covering both modeling tracks, all five forecasting targets, and horizons 1, 2, and 5. Every job carries the frozen **More expressive** LightGBM capacity profile, uses targets in **raw units**, and follows the **validation-start-only** refit cadence.

The workload therefore enters full validation with one consistent model specification and complete Track × Target × Horizon coverage. The final holdout remains untouched, preserving it for the later out-of-sample evaluation stage.

> ✅ Decision — Approve the 30\-job LightGBM contract for full rolling validation\.
The workload has complete coverage of both modeling tracks, all five targets, and all three forecast horizons, with the frozen More expressive, raw\-target, validation\-start\-only LightGBM specification applied consistently throughout\. The final holdout remains untouched\.

Proceed to full rolling validation under this frozen contract\.

### 4.4.1.7B — Run or recover the full LightGBM validation jobs

The earlier experiments used smaller training samples so model choices could be compared quickly. Now the frozen LightGBM setup is tested using the full training history available at the start of validation.

Each of the 30 forecasting jobs is fit **once**. The model then makes predictions across that job's full validation period.

While each fitted model is still in memory, the notebook also saves the information needed later to explain what the model learned. This avoids fitting the same model a second time just for feature importance.

Each job saves three things:

- validation metrics;
- row-level predictions;
- feature-importance and SHAP summaries.

SHAP is calculated on a fixed sample of validation rows so the explanation step remains manageable and reproducible.

Raw model predictions are saved without adjustment. This lets the notebook separately test whether physically impossible negative forecasts should be replaced with zero.

The final holdout is not used here.

In [43]:
# ---------------------------------------------------------------------
# 4.4.1.7B — Run or recover full LightGBM rolling validation
# ---------------------------------------------------------------------

RUN_LGBM_FULL_VALIDATION = True
RESET_LGBM_FULL_VALIDATION = False

LGBM_FULL_VALIDATION_CONTRACT_ID = "lgbm_full_validation_v1"
LGBM_EXPLAINABILITY_CONTRACT_ID = "lgbm_explainability_v1"
LGBM_EXPLAINABILITY_SAMPLE_ROWS = 5_000

LGBM_FULL_VALIDATION_ROOT = INTERMEDIATE_DIR / LGBM_FULL_VALIDATION_CONTRACT_ID
LGBM_FULL_VALIDATION_METRICS_DIR = LGBM_FULL_VALIDATION_ROOT / "metrics"
LGBM_FULL_VALIDATION_PREDICTIONS_DIR = LGBM_FULL_VALIDATION_ROOT / "predictions"
LGBM_FULL_VALIDATION_EXPLAINABILITY_DIR = LGBM_FULL_VALIDATION_ROOT / "explainability"
LGBM_FULL_VALIDATION_RESULTS_PATH = LGBM_FULL_VALIDATION_ROOT / "results.parquet"
LGBM_EXPLAINABILITY_RESULTS_PATH = LGBM_FULL_VALIDATION_ROOT / "feature_importance.parquet"

if RESET_LGBM_FULL_VALIDATION:
    reset_checkpoint_dir(LGBM_FULL_VALIDATION_ROOT)

for directory in [
    LGBM_FULL_VALIDATION_METRICS_DIR,
    LGBM_FULL_VALIDATION_PREDICTIONS_DIR,
    LGBM_FULL_VALIDATION_EXPLAINABILITY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

FULL_VALIDATION_JOB_KEYS = [
    "modeling_track",
    "metric",
    "horizon",
]


def lgbm_full_validation_metrics_path(track, metric, horizon):
    """Return the durable one-row metrics checkpoint for one production job."""
    job_slug = slugify(f"{track}__{metric}__h{int(horizon)}")
    return LGBM_FULL_VALIDATION_METRICS_DIR / f"{job_slug}.parquet"


def lgbm_full_validation_predictions_path(track, metric, horizon):
    """Return the row-level prediction checkpoint for one production job."""
    job_slug = slugify(f"{track}__{metric}__h{int(horizon)}")
    return LGBM_FULL_VALIDATION_PREDICTIONS_DIR / f"{job_slug}.parquet"


def lgbm_full_validation_explainability_path(track, metric, horizon):
    """Return the compact explainability checkpoint for one production job."""
    job_slug = slugify(f"{track}__{metric}__h{int(horizon)}")
    return LGBM_FULL_VALIDATION_EXPLAINABILITY_DIR / f"{job_slug}.parquet"


def lgbm_full_validation_job_is_complete(track, metric, horizon):
    """Require metrics, predictions, and explainability from the current contract."""
    metrics_path = lgbm_full_validation_metrics_path(track, metric, horizon)
    predictions_path = lgbm_full_validation_predictions_path(track, metric, horizon)
    explainability_path = lgbm_full_validation_explainability_path(
        track, metric, horizon
    )

    if not all(
        path.exists()
        for path in [metrics_path, predictions_path, explainability_path]
    ):
        return False

    try:
        metrics_df = pd.read_parquet(metrics_path)
        predictions_df = pd.read_parquet(predictions_path)
        explainability_df = pd.read_parquet(explainability_path)
    except Exception:
        return False

    if len(metrics_df) != 1 or predictions_df.empty or explainability_df.empty:
        return False

    row = metrics_df.iloc[0]
    expected_predictors = set(predictor_names_by_target[metric])

    metrics_valid = (
        row.get("status") == "success"
        and row.get("validation_contract_id") == LGBM_FULL_VALIDATION_CONTRACT_ID
        and row.get("modeling_track") == track
        and row.get("metric") == metric
        and int(row.get("horizon")) == int(horizon)
        and int(row.get("validation_rows")) == len(predictions_df)
    )

    explainability_valid = (
        explainability_df["explainability_contract_id"]
        .eq(LGBM_EXPLAINABILITY_CONTRACT_ID)
        .all()
        and explainability_df["modeling_track"].eq(track).all()
        and explainability_df["metric"].eq(metric).all()
        and explainability_df["horizon"].eq(int(horizon)).all()
        and set(explainability_df["feature"]) == expected_predictors
    )

    return metrics_valid and explainability_valid


def fit_lgbm_full_validation_job(track, metric, horizon):
    """Fit one frozen model and capture validation plus explainability evidence."""
    payload = load_cadence_job(track, metric, int(horizon))

    matrix_df = payload["matrix_df"]
    validation_df = payload["validation_df"].copy()

    required_validation_columns = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_observed_value",
        payload["benchmark_column"],
    ]
    require_columns(
        validation_df,
        required_validation_columns,
        "full-validation rows",
    )

    if validation_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon} has no validation rows."
        )

    # Validation-start-only fitting uses only target outcomes observable at the
    # first forecast origin. Later validation outcomes remain unavailable.
    validation_start_origin_id = int(
        validation_df["origin_observation_sequence_id"].min()
    )

    training_mask = (
        matrix_df["_label_available"]
        & matrix_df["target_observation_sequence_id"].le(
            validation_start_origin_id
        )
    )
    train_df = matrix_df.loc[training_mask].copy()

    if train_df.empty:
        raise ValueError(
            f"{track} · {metric} · h={horizon} has no legal training rows."
        )

    predictor_names = payload["predictor_names"]
    categorical_names = payload["categorical_names"]

    # Prepare the complete validation frame for scoring. The categorical levels
    # are defined from training so validation never introduces a new encoding.
    X_train, X_valid = prepare_lightgbm_frames(
        train_df[predictor_names],
        validation_df[predictor_names],
        categorical_names,
    )

    y_train = pd.to_numeric(
        train_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    y_valid = pd.to_numeric(
        validation_df["target_observed_value"],
        errors="raise",
    ).astype(float)

    model = LGBMRegressor(**selected_lgbm_params)

    fit_start = perf_counter()
    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_names,
    )
    fit_seconds = perf_counter() - fit_start

    # -----------------------------------------------------------------
    # Full validation predictions
    # -----------------------------------------------------------------

    predict_start = perf_counter()
    predictions = np.asarray(model.predict(X_valid), dtype=float)
    predict_seconds = perf_counter() - predict_start

    if not np.isfinite(predictions).all():
        raise ValueError(
            f"{track} · {metric} · h={horizon} produced "
            "non-finite validation predictions."
        )

    benchmark_predictions = pd.to_numeric(
        validation_df[payload["benchmark_column"]],
        errors="raise",
    ).astype(float).to_numpy()

    model_metrics = regression_metrics(y_valid, predictions)
    benchmark_metrics = regression_metrics(y_valid, benchmark_predictions)

    benchmark_mae = float(benchmark_metrics["mae"])
    skill_vs_benchmark_pct = (
        100.0 * (benchmark_mae - model_metrics["mae"]) / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    predictions_df = validation_df[
        [
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
        ]
    ].copy()

    predictions_df["validation_contract_id"] = LGBM_FULL_VALIDATION_CONTRACT_ID
    predictions_df["modeling_track"] = track
    predictions_df["metric"] = metric
    predictions_df["horizon"] = int(horizon)
    predictions_df["actual"] = y_valid.to_numpy()
    predictions_df["benchmark_prediction"] = benchmark_predictions
    predictions_df["lgbm_prediction_raw"] = predictions

    # -----------------------------------------------------------------
    # Explainability from the same fitted model
    # -----------------------------------------------------------------
    # Gain and split importance come directly from the fitted tree ensemble.
    # SHAP-style contributions use a deterministic held-out validation sample
    # so interpretation remains compact and repeatable.

    sample_rows = min(LGBM_EXPLAINABILITY_SAMPLE_ROWS, len(validation_df))
    sample_positions = (
        validation_df
        .sample(n=sample_rows, random_state=RANDOM_SEED)
        .index
    )

    X_explain = X_valid.loc[sample_positions].copy()

    booster = model.booster_

    gain_importance = np.asarray(
        booster.feature_importance(importance_type="gain"),
        dtype=float,
    )
    split_importance = np.asarray(
        booster.feature_importance(importance_type="split"),
        dtype=float,
    )

    shap_start = perf_counter()
    shap_values = np.asarray(
        model.predict(X_explain, pred_contrib=True),
        dtype=float,
    )
    shap_seconds = perf_counter() - shap_start

    # LightGBM appends the model's expected-value term as the final SHAP column.
    # Only the predictor contribution columns belong in feature importance.
    if shap_values.shape[1] != len(predictor_names) + 1:
        raise ValueError(
            f"Unexpected SHAP contribution width for "
            f"{track} · {metric} · h={horizon}."
        )

    mean_abs_shap = np.abs(shap_values[:, :-1]).mean(axis=0)
    mean_shap = shap_values[:, :-1].mean(axis=0)

    gain_total = gain_importance.sum()
    shap_total = mean_abs_shap.sum()

    explainability_df = pd.DataFrame(
        {
            "explainability_contract_id": LGBM_EXPLAINABILITY_CONTRACT_ID,
            "validation_contract_id": LGBM_FULL_VALIDATION_CONTRACT_ID,
            "modeling_track": track,
            "metric": metric,
            "horizon": int(horizon),
            "feature": predictor_names,
            "gain_importance": gain_importance,
            "split_importance": split_importance,
            "mean_abs_shap": mean_abs_shap,
            "mean_shap": mean_shap,
            "gain_share": (
                gain_importance / gain_total
                if gain_total > 0
                else np.zeros_like(gain_importance)
            ),
            "shap_share": (
                mean_abs_shap / shap_total
                if shap_total > 0
                else np.zeros_like(mean_abs_shap)
            ),
            "sample_rows": int(sample_rows),
            "train_rows": int(len(train_df)),
            "fit_seconds": float(fit_seconds),
            "shap_seconds": float(shap_seconds),
        }
    )

    metrics_row = {
        "status": "success",
        "validation_contract_id": LGBM_FULL_VALIDATION_CONTRACT_ID,
        "production_config_id": LGBM_PRODUCTION_CONFIG_ID,
        "modeling_track": track,
        "metric": metric,
        "horizon": int(horizon),
        "model_family": "LightGBM",
        "capacity_profile": SELECTED_LGBM_CAPACITY_PROFILE,
        "target_representation": "raw",
        "refit_cadence": LGBM_PRODUCTION_REFIT_CADENCE,
        "training_row_cap": None,
        "validation_start_origin_id": validation_start_origin_id,
        "train_rows": int(len(train_df)),
        "train_origins": int(
            train_df["origin_observation_sequence_id"].nunique()
        ),
        "validation_rows": int(len(validation_df)),
        "validation_origins": int(
            validation_df["origin_observation_sequence_id"].nunique()
        ),
        "benchmark_id": payload["benchmark_id"],
        "benchmark_label": payload["benchmark_label"],
        "mae": float(model_metrics["mae"]),
        "rmse": float(model_metrics["rmse"]),
        "nmae": float(model_metrics["nmae"]),
        "bias": float(model_metrics["bias"]),
        "benchmark_mae": benchmark_mae,
        "benchmark_rmse": float(benchmark_metrics["rmse"]),
        "benchmark_nmae": float(benchmark_metrics["nmae"]),
        "benchmark_bias": float(benchmark_metrics["bias"]),
        "skill_vs_benchmark_pct": float(skill_vs_benchmark_pct),
        "beats_benchmark": bool(skill_vs_benchmark_pct > 0),
        "negative_prediction_count": int((predictions < 0).sum()),
        "negative_prediction_pct": float(100.0 * (predictions < 0).mean()),
        "explainability_sample_rows": int(sample_rows),
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "shap_seconds": float(shap_seconds),
    }

    del (
        matrix_df,
        validation_df,
        train_df,
        X_train,
        X_valid,
        X_explain,
        y_train,
        y_valid,
        predictions,
        benchmark_predictions,
        shap_values,
        model,
        booster,
        payload,
    )
    gc.collect()

    return metrics_row, predictions_df, explainability_df


# ---------------------------------------------------------------------
# Determine restart state from complete logical jobs
# ---------------------------------------------------------------------

complete_full_validation_jobs = {
    (row.modeling_track, row.metric, int(row.horizon))
    for row in lgbm_full_validation_jobs_df.itertuples(index=False)
    if lgbm_full_validation_job_is_complete(
        row.modeling_track,
        row.metric,
        row.horizon,
    )
}

print(
    "LightGBM full-validation restart state: "
    f"{len(complete_full_validation_jobs)}/30 jobs complete.",
    flush=True,
)


# ---------------------------------------------------------------------
# Fit only incomplete jobs
# ---------------------------------------------------------------------

if RUN_LGBM_FULL_VALIDATION:
    jobs_to_run_df = lgbm_full_validation_jobs_df.loc[
        ~lgbm_full_validation_jobs_df.apply(
            lambda row: (
                row["modeling_track"],
                row["metric"],
                int(row["horizon"]),
            )
            in complete_full_validation_jobs,
            axis=1,
        )
    ].reset_index(drop=True)

    total_missing_jobs = len(jobs_to_run_df)

    for job_number, job in enumerate(
        jobs_to_run_df.itertuples(index=False),
        start=1,
    ):
        track = job.modeling_track
        metric = job.metric
        horizon = int(job.horizon)

        print(
            f"\n[{job_number}/{total_missing_jobs}] "
            f"{TRACK_SHORT_LABELS[track]} · "
            f"{TARGET_SHORT_LABELS[metric]} · "
            f"h={horizon}",
            flush=True,
        )

        try:
            (
                metrics_row,
                predictions_df,
                explainability_df,
            ) = fit_lgbm_full_validation_job(
                track,
                metric,
                horizon,
            )

            # A job becomes reusable only after all three artifacts have been
            # written. Atomic writes prevent a partial file from looking valid.
            atomic_parquet_write(
                predictions_df,
                lgbm_full_validation_predictions_path(
                    track,
                    metric,
                    horizon,
                ),
            )

            atomic_parquet_write(
                explainability_df,
                lgbm_full_validation_explainability_path(
                    track,
                    metric,
                    horizon,
                ),
            )

            atomic_parquet_write(
                pd.DataFrame([metrics_row]),
                lgbm_full_validation_metrics_path(
                    track,
                    metric,
                    horizon,
                ),
            )

            print(
                f"    train={metrics_row['train_rows']:,} · "
                f"validation={metrics_row['validation_rows']:,} · "
                f"MAE={metrics_row['mae']:.3f} · "
                f"skill={metrics_row['skill_vs_benchmark_pct']:+.3f}% · "
                f"fit={metrics_row['fit_seconds']:.3f}s · "
                f"SHAP={metrics_row['shap_seconds']:.3f}s · "
                f"negative={metrics_row['negative_prediction_count']:,}",
                flush=True,
            )

        except Exception as exc:
            failed_row = {
                "status": "failed",
                "validation_contract_id": LGBM_FULL_VALIDATION_CONTRACT_ID,
                "production_config_id": LGBM_PRODUCTION_CONFIG_ID,
                "modeling_track": track,
                "metric": metric,
                "horizon": horizon,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }

            atomic_parquet_write(
                pd.DataFrame([failed_row]),
                lgbm_full_validation_metrics_path(
                    track,
                    metric,
                    horizon,
                ),
            )

            print(
                f"    failed · {type(exc).__name__}: {exc}",
                flush=True,
            )


# ---------------------------------------------------------------------
# Reconstruct consolidated validation and explainability artifacts
# ---------------------------------------------------------------------

full_validation_metric_parts = []
full_validation_explainability_parts = []

for job in lgbm_full_validation_jobs_df.itertuples(index=False):
    metrics_path = lgbm_full_validation_metrics_path(
        job.modeling_track,
        job.metric,
        job.horizon,
    )
    explainability_path = lgbm_full_validation_explainability_path(
        job.modeling_track,
        job.metric,
        job.horizon,
    )

    if metrics_path.exists():
        metrics_part_df = pd.read_parquet(metrics_path)

        if (
            len(metrics_part_df) == 1
            and metrics_part_df["validation_contract_id"]
            .eq(LGBM_FULL_VALIDATION_CONTRACT_ID)
            .all()
        ):
            full_validation_metric_parts.append(metrics_part_df)

    if explainability_path.exists():
        explainability_part_df = pd.read_parquet(explainability_path)

        if (
            not explainability_part_df.empty
            and explainability_part_df["explainability_contract_id"]
            .eq(LGBM_EXPLAINABILITY_CONTRACT_ID)
            .all()
        ):
            full_validation_explainability_parts.append(
                explainability_part_df
            )

lgbm_full_validation_results_df = (
    pd.concat(full_validation_metric_parts, ignore_index=True)
    if full_validation_metric_parts
    else pd.DataFrame()
)

lgbm_feature_importance_df = (
    pd.concat(full_validation_explainability_parts, ignore_index=True)
    if full_validation_explainability_parts
    else pd.DataFrame()
)

if not lgbm_full_validation_results_df.empty:
    if lgbm_full_validation_results_df.duplicated(
        FULL_VALIDATION_JOB_KEYS
    ).any():
        raise ValueError(
            "Full-validation metrics contain duplicate production jobs."
        )

    atomic_parquet_write(
        lgbm_full_validation_results_df,
        LGBM_FULL_VALIDATION_RESULTS_PATH,
    )

if not lgbm_feature_importance_df.empty:
    atomic_parquet_write(
        lgbm_feature_importance_df,
        LGBM_EXPLAINABILITY_RESULTS_PATH,
    )

successful_full_validation_jobs = (
    int(lgbm_full_validation_results_df["status"].eq("success").sum())
    if not lgbm_full_validation_results_df.empty
    else 0
)

failed_full_validation_jobs = (
    int(lgbm_full_validation_results_df["status"].eq("failed").sum())
    if not lgbm_full_validation_results_df.empty
    else 0
)

completed_explainability_jobs = (
    lgbm_feature_importance_df[
        ["modeling_track", "metric", "horizon"]
    ]
    .drop_duplicates()
    .shape[0]
    if not lgbm_feature_importance_df.empty
    else 0
)

print(
    "\nLightGBM full-validation result state: "
    f"{successful_full_validation_jobs}/30 successful jobs; "
    f"{failed_full_validation_jobs} failed.",
    flush=True,
)

print(
    "LightGBM explainability artifact state: "
    f"{completed_explainability_jobs}/30 jobs complete.",
    flush=True,
)

LightGBM full-validation restart state: 30/30 jobs complete.

LightGBM full-validation result state: 30/30 successful jobs; 0 failed.
LightGBM explainability artifact state: 30/30 jobs complete.


**Findings.** All **30 LightGBM validation jobs completed successfully**. The models used the full training history available at the start of validation, ranging from about 65,000 rows for the shortest Post-CP jobs to more than 1 million rows for the longest Full-History jobs.

LightGBM remained fast even at this scale. Individual models took roughly 2 to 13 seconds to fit, and all 30 fits together required about 165 seconds.

The same 30 fits also produced the feature-importance evidence used later in the notebook, so no second round of model fitting was needed for explanation.

Negative predictions appeared only for Taxi trips, FHVHV trips, and Subway ridership. Neither speed target produced a negative forecast. The raw predictions are kept unchanged so the effect of replacing impossible negative values with zero can be tested separately.

### 4.4.1.7B.1 — Watch LightGBM learn as trees are added

A LightGBM model is built gradually. It starts with a simple prediction and then adds trees one at a time, with each new tree trying to correct some of the errors left by the trees that came before it.

The final LightGBM specification uses **400 trees**. To see what happens during those 400 steps, the same 30 frozen **Modeling Track × Target × Horizon** jobs are fit again using the same legal training history available at the start of validation.

After every tree, the diagnostic records error on two different surfaces:

- **Training error** measures how well the model fits the historical examples it is learning from.
- **Validation error** measures how well that partially built model predicts the held-out ordinary-validation observations.

The validation rows are used only to **measure** error after each tree. They do not contribute to the tree updates, do not trigger early stopping, and do not change the frozen 400-tree model specification.

Both mean squared error (MSE) and mean absolute error (MAE) are retained. MSE matches the loss LightGBM is trained to reduce, while MAE expresses error in the same original target units used throughout this notebook.

This diagnostic therefore lets us ask a more important question than whether training error keeps falling: **does validation error continue improving too, or does it eventually flatten or rise while training error keeps declining?** The latter pattern would be evidence that additional trees are increasingly fitting the training data rather than improving generalization.

The diagnostic is restartable by forecasting job. Completed jobs are reused rather than fit again.

In [44]:
# ---------------------------------------------------------------------
# 4.4.1.7B.1 — Record training and validation error after every tree
# ---------------------------------------------------------------------

RUN_LGBM_LEARNING_HISTORY = True
RESET_LGBM_LEARNING_HISTORY = False

LGBM_LEARNING_HISTORY_CONTRACT_ID = "lgbm_learning_history_v2"
LGBM_LEARNING_HISTORY_ROOT = INTERMEDIATE_DIR / LGBM_LEARNING_HISTORY_CONTRACT_ID
LGBM_LEARNING_HISTORY_JOBS_DIR = LGBM_LEARNING_HISTORY_ROOT / "jobs"
LGBM_LEARNING_HISTORY_PATH = LGBM_LEARNING_HISTORY_ROOT / "learning_history.parquet"

if RESET_LGBM_LEARNING_HISTORY:
    reset_checkpoint_dir(LGBM_LEARNING_HISTORY_ROOT)

LGBM_LEARNING_HISTORY_JOBS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_LGBM_ROUNDS = int(selected_lgbm_params["n_estimators"])
EXPECTED_LGBM_LEARNING_ROWS = len(lgbm_full_validation_jobs_df) * EXPECTED_LGBM_ROUNDS


def lgbm_learning_history_job_path(track, metric, horizon):
    """Return the tree-by-tree learning checkpoint for one forecasting job."""
    job_id = slugify(f"{track}__{metric}__h{int(horizon)}")
    return LGBM_LEARNING_HISTORY_JOBS_DIR / f"{job_id}.parquet"


def lgbm_learning_history_job_is_complete(track, metric, horizon):
    """Reuse a job only when all 400 training and validation measurements exist."""
    path = lgbm_learning_history_job_path(track, metric, horizon)
    if not path.exists():
        return False

    try:
        history_df = pd.read_parquet(path)
    except Exception:
        return False

    required_columns = {
        "learning_history_contract_id", "production_config_id", "model_family",
        "modeling_track", "metric", "horizon", "boosting_round",
        "train_rows", "validation_rows",
        "train_mae", "validation_mae", "train_mse", "validation_mse",
        "fit_seconds",
    }

    return (
        required_columns.issubset(history_df.columns)
        and len(history_df) == EXPECTED_LGBM_ROUNDS
        and history_df["learning_history_contract_id"].eq(LGBM_LEARNING_HISTORY_CONTRACT_ID).all()
        and history_df["production_config_id"].eq(LGBM_PRODUCTION_CONFIG_ID).all()
        and history_df["modeling_track"].eq(track).all()
        and history_df["metric"].eq(metric).all()
        and history_df["horizon"].eq(int(horizon)).all()
        and history_df["boosting_round"].tolist() == list(range(1, EXPECTED_LGBM_ROUNDS + 1))
        and np.isfinite(
            history_df[
                ["train_mae", "validation_mae", "train_mse", "validation_mse"]
            ].to_numpy(dtype=float)
        ).all()
    )


def fit_lgbm_learning_history_job(track, metric, horizon):
    """Repeat one frozen fit and observe training versus validation learning."""
    payload = load_cadence_job(track, metric, int(horizon))
    matrix_df = payload["matrix_df"]
    validation_df = payload["validation_df"].copy()

    if validation_df.empty:
        raise ValueError(f"{track} · {metric} · h={horizon} has no validation rows.")

    # Match the production information boundary exactly. Training may use only
    # target outcomes already observable when ordinary validation begins.
    validation_start_origin_id = int(validation_df["origin_observation_sequence_id"].min())
    training_mask = (
        matrix_df["_label_available"]
        & matrix_df["target_observation_sequence_id"].le(validation_start_origin_id)
    )
    train_df = matrix_df.loc[training_mask].copy()

    if train_df.empty:
        raise ValueError(f"{track} · {metric} · h={horizon} has no legal training rows.")

    predictor_names = payload["predictor_names"]
    categorical_names = payload["categorical_names"]

    # Prepare training and validation features with the same categorical contract
    # as the production model. Validation outcomes are scored, never trained on.
    X_train, X_validation = prepare_lightgbm_frames(
        train_df[predictor_names],
        validation_df[predictor_names],
        categorical_names,
    )

    y_train = pd.to_numeric(train_df["target_observed_value"], errors="raise").astype(float)
    y_validation = pd.to_numeric(
        validation_df["target_observed_value"], errors="raise"
    ).astype(float)

    # LightGBM reports both surfaces after every added tree. With no early
    # stopping callback, validation scores cannot change the 400-tree fit.
    model = LGBMRegressor(**selected_lgbm_params)

    fit_start = perf_counter()
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_validation, y_validation)],
        eval_names=["train", "validation"],
        eval_metric=["l1", "l2"],
        categorical_feature=categorical_names,
    )
    fit_seconds = perf_counter() - fit_start

    train_history = model.evals_result_.get("train", {})
    validation_history = model.evals_result_.get("validation", {})

    train_mae = train_history.get("l1")
    train_mse = train_history.get("l2")
    validation_mae = validation_history.get("l1")
    validation_mse = validation_history.get("l2")

    histories = {
        "train_mae": train_mae,
        "train_mse": train_mse,
        "validation_mae": validation_mae,
        "validation_mse": validation_mse,
    }

    missing_histories = [name for name, values in histories.items() if values is None]
    if missing_histories:
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: "
            f"missing learning histories: {missing_histories}."
        )

    wrong_lengths = {
        name: len(values)
        for name, values in histories.items()
        if len(values) != EXPECTED_LGBM_ROUNDS
    }
    if wrong_lengths:
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: "
            f"expected {EXPECTED_LGBM_ROUNDS} rounds; found {wrong_lengths}."
        )

    history_df = pd.DataFrame({
        "learning_history_contract_id": LGBM_LEARNING_HISTORY_CONTRACT_ID,
        "production_config_id": LGBM_PRODUCTION_CONFIG_ID,
        "model_family": "LightGBM",
        "modeling_track": track,
        "metric": metric,
        "horizon": int(horizon),
        "boosting_round": np.arange(1, EXPECTED_LGBM_ROUNDS + 1, dtype=int),
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(validation_df)),
        "train_mae": np.asarray(train_mae, dtype=float),
        "validation_mae": np.asarray(validation_mae, dtype=float),
        "train_mse": np.asarray(train_mse, dtype=float),
        "validation_mse": np.asarray(validation_mse, dtype=float),
        "fit_seconds": float(fit_seconds),
    })

    metric_columns = ["train_mae", "validation_mae", "train_mse", "validation_mse"]
    if not np.isfinite(history_df[metric_columns].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{track} · {metric} · h={horizon}: learning history contains invalid values."
        )

    del (
        payload, matrix_df, validation_df, train_df,
        X_train, X_validation, y_train, y_validation, model,
    )
    gc.collect()

    return history_df


# Find only the jobs still missing a complete v2 learning-history checkpoint.
jobs_to_run = [
    job
    for job in lgbm_full_validation_jobs_df.itertuples(index=False)
    if not lgbm_learning_history_job_is_complete(
        job.modeling_track, job.metric, int(job.horizon)
    )
]

completed_jobs = len(lgbm_full_validation_jobs_df) - len(jobs_to_run)
print(
    f"LightGBM learning-history restart state: "
    f"{completed_jobs}/{len(lgbm_full_validation_jobs_df)} jobs complete.",
    flush=True,
)


# Save every job immediately so an interruption loses at most one fit.
if RUN_LGBM_LEARNING_HISTORY:
    for job_number, job in enumerate(jobs_to_run, start=1):
        track = job.modeling_track
        metric = job.metric
        horizon = int(job.horizon)

        print(
            f"[{job_number}/{len(jobs_to_run)}] "
            f"{TRACK_SHORT_LABELS[track]} · {TARGET_SHORT_LABELS[metric]} · h={horizon}",
            flush=True,
        )

        history_df = fit_lgbm_learning_history_job(track, metric, horizon)
        atomic_parquet_write(
            history_df,
            lgbm_learning_history_job_path(track, metric, horizon),
        )

        print(
            f"    {len(history_df):,} trees · "
            f"{int(history_df['train_rows'].iloc[0]):,} training rows · "
            f"{int(history_df['validation_rows'].iloc[0]):,} validation rows · "
            f"{history_df['fit_seconds'].iloc[0]:.3f}s · "
            f"validation MSE "
            f"{history_df['validation_mse'].iloc[0]:.3f} → "
            f"{history_df['validation_mse'].iloc[-1]:.3f}",
            flush=True,
        )


# Combine the 30 small checkpoints into one compact analysis artifact.
learning_parts = []

for job in lgbm_full_validation_jobs_df.itertuples(index=False):
    track, metric, horizon = job.modeling_track, job.metric, int(job.horizon)

    if not lgbm_learning_history_job_is_complete(track, metric, horizon):
        raise RuntimeError(
            f"Learning history is incomplete for {track} · {metric} · h={horizon}."
        )

    learning_parts.append(
        pd.read_parquet(lgbm_learning_history_job_path(track, metric, horizon))
    )

lgbm_learning_history_df = pd.concat(learning_parts, ignore_index=True)

if len(lgbm_learning_history_df) != EXPECTED_LGBM_LEARNING_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_LGBM_LEARNING_ROWS:,} learning-history rows; "
        f"found {len(lgbm_learning_history_df):,}."
    )

job_round_key = ["modeling_track", "metric", "horizon", "boosting_round"]
if lgbm_learning_history_df.duplicated(job_round_key).any():
    raise RuntimeError("Learning history contains duplicate job × tree rows.")

atomic_parquet_write(lgbm_learning_history_df, LGBM_LEARNING_HISTORY_PATH)

total_fit_seconds = (
    lgbm_learning_history_df
    .groupby(["modeling_track", "metric", "horizon"], observed=True)["fit_seconds"]
    .first()
    .sum()
)

print(
    f"\nLightGBM learning history complete: "
    f"{len(lgbm_full_validation_jobs_df)} jobs · "
    f"{len(lgbm_learning_history_df):,} rows · "
    f"{total_fit_seconds:.3f}s total fit time.",
    flush=True,
)

LightGBM learning-history restart state: 30/30 jobs complete.

LightGBM learning history complete: 30 jobs · 12,000 rows · 188.704s total fit time.


**Findings.** The learning-history diagnostic is complete for all **30 LightGBM jobs**, with **400 tree-by-tree observations per job** for a total of **12,000 rows**. Both training and held-out validation MAE and MSE are available throughout the full boosting sequence.

The notebook recovered all **30 of 30** learning-history checkpoints during this run rather than fitting the diagnostic models again. The reported **188.704 seconds** is the cumulative fitting time stored with those completed diagnostic jobs, not additional fitting performed during this restart.

The complete learning history therefore gives the notebook a direct check on whether validation performance continues improving as trees are added while leaving the original validation predictions and final holdout unchanged.

### 4.4.1.7B.2 — Inspect validation loss across the 400-tree budget

Training error is expected to keep falling as LightGBM adds trees, because each new tree is built to correct mistakes remaining in the training data. The more important question is whether those additional trees continue improving performance on **held-out validation observations**.

The five forecasting targets use very different numerical scales, so their raw MSE values cannot share one useful chart. Each Target × Horizon job is therefore indexed to its own best validation MSE:

- **100** marks that job's lowest validation MSE anywhere in the 400-tree sequence;
- values above 100 show how much higher validation error is at another tree count.

The plots retain all **15 individual validation curves within each modeling track** rather than averaging them together. This matters because different forecasting jobs can reach their best validation score at different trees.

The accompanying table summarizes two practical quantities:

- the **best validation tree** for each job;
- the **Tree-400 validation penalty**, or how much higher validation MSE is at Tree 400 than at that job's own minimum.

Because validation error falls dramatically during the earliest trees, small late-stage differences can look visually flat on the full 1-to-400 plots. The table therefore provides the more precise measure of whether continuing to Tree 400 meaningfully worsens validation performance.

This is a retrospective diagnostic of the already selected LightGBM specification. It does not use the final holdout and does not change the frozen model.

In [45]:
# ---------------------------------------------------------------------
# 4.4.1.7B.2 — Plot held-out validation loss as each tree is added
# ---------------------------------------------------------------------

if "lgbm_learning_history_df" not in globals() or lgbm_learning_history_df.empty:
    raise RuntimeError("Run Section 7B.1 before plotting validation loss.")

JOB_KEYS = ["modeling_track", "metric", "horizon"]

# Each forecasting target has a different numerical scale, so raw MSE values
# cannot share one useful y-axis. Normalize each job to its own lowest
# validation MSE. A value of 100 is that job's best observed validation loss;
# values above 100 show how much worse validation performance is at that tree.
best_validation_df = (
    lgbm_learning_history_df
    .groupby(JOB_KEYS, observed=True)["validation_mse"]
    .min()
    .rename("best_validation_mse")
    .reset_index()
)

validation_curve_df = lgbm_learning_history_df.merge(
    best_validation_df,
    on=JOB_KEYS,
    how="left",
    validate="many_to_one",
)

validation_curve_df["validation_mse_index"] = (
    100
    * validation_curve_df["validation_mse"]
    / validation_curve_df["best_validation_mse"]
)


# Summarize the actual minimum for each job so the table and graph describe
# the same underlying validation curves.
job_summary_rows = []

for job_key, job_df in validation_curve_df.groupby(JOB_KEYS, observed=True):
    job_df = job_df.sort_values("boosting_round").reset_index(drop=True)

    best_row = job_df.loc[job_df["validation_mse"].idxmin()]
    final_row = job_df.iloc[-1]

    job_summary_rows.append({
        "modeling_track": job_key[0],
        "metric": job_key[1],
        "horizon": int(job_key[2]),
        "best_validation_tree": int(best_row["boosting_round"]),
        "tree_400_validation_index": float(final_row["validation_mse_index"]),
        "tree_400_penalty_pct": float(final_row["validation_mse_index"] - 100),
    })

validation_job_summary_df = pd.DataFrame(job_summary_rows)

validation_summary_df = (
    validation_job_summary_df
    .groupby("modeling_track", observed=True)
    .agg(
        jobs=("metric", "size"),
        median_best_validation_tree=("best_validation_tree", "median"),
        median_tree_400_penalty_pct=("tree_400_penalty_pct", "median"),
        jobs_best_before_tree_400=(
            "best_validation_tree",
            lambda values: int((values < EXPECTED_LGBM_ROUNDS).sum()),
        ),
    )
    .reset_index()
)

validation_summary_display_df = validation_summary_df.rename(columns={
    "modeling_track": "Modeling track",
    "jobs": "Jobs",
    "median_best_validation_tree": "Median best validation tree",
    "median_tree_400_penalty_pct": "Median Tree-400 validation penalty %",
    "jobs_best_before_tree_400": "Jobs best before Tree 400",
})

validation_summary_display_df["Modeling track"] = (
    validation_summary_display_df["Modeling track"]
    .map(TRACK_SHORT_LABELS)
    .fillna(validation_summary_display_df["Modeling track"])
)

validation_summary_display_df[
    ["Median best validation tree", "Median Tree-400 validation penalty %"]
] = validation_summary_display_df[
    ["Median best validation tree", "Median Tree-400 validation penalty %"]
].round(3)

display(validation_summary_display_df)


# Plot every held-out validation trajectory rather than averaging jobs together.
# This preserves an early minimum even when different jobs overfit at different
# points in the 400-tree sequence.
if RUN_VISUALS:
    for track in ["full_history_regime_aware", "post_cp_primary"]:
        track_df = validation_curve_df.loc[
            validation_curve_df["modeling_track"].eq(track)
        ].copy()

        fig = go.Figure()

        for (metric, horizon), job_df in track_df.groupby(
            ["metric", "horizon"],
            observed=True,
        ):
            job_df = job_df.sort_values("boosting_round")

            target_label = TARGET_SHORT_LABELS.get(metric, metric)
            line_label = f"{target_label} · h={int(horizon)}"

            fig.add_trace(go.Scatter(
                x=job_df["boosting_round"],
                y=job_df["validation_mse_index"],
                mode="lines",
                name=line_label,
                hovertemplate=(
                    f"{line_label}<br>"
                    "Tree %{x}<br>"
                    "Validation MSE index %{y:.3f}"
                    "<extra></extra>"
                ),
            ))

        fig.add_hline(
            y=100,
            line_dash="dot",
            line_color=BRAND_COLORS["seafoam"],
            annotation_text="Best validation loss = 100",
            annotation_position="top right",
        )

        fig.update_layout(
            title=f"{TRACK_SHORT_LABELS[track]} — Validation Loss as Trees Are Added",
            xaxis_title="Tree added to the model",
            yaxis_title="Validation MSE index · each job's best = 100",
            width=960,
            height=650,
            hovermode="x unified",
            legend={
                "orientation": "h",
                "yanchor": "top",
                "y": -0.18,
                "xanchor": "center",
                "x": 0.5,
            },
            margin={"l": 90, "r": 40, "t": 90, "b": 180},
        )

        apply_branding(fig)

        # Branding can reset layout details, so restore the readable legend.
        fig.update_layout(
            legend={
                "orientation": "h",
                "yanchor": "top",
                "y": -0.18,
                "xanchor": "center",
                "x": 0.5,
            },
            margin={"l": 90, "r": 40, "t": 90, "b": 180},
        )

        fig.show()

,Modeling track,Jobs,Median best validation tree,Median Tree-400 validation penalty %,Jobs best before Tree 400
0,Full-History,15,400.0,0.000,6
1,Post-CP,15,173.0,0.955,11


**Findings.** Both modeling tracks show the expected LightGBM learning pattern: validation error falls rapidly as the first trees are added, then changes much more gradually as the model approaches its best fit.

For **Full-History**, the median best validation point occurs at **Tree 400**, with no median loss in validation accuracy from using the full 400-tree budget. Six of the 15 individual jobs reach their minimum somewhat earlier, but the overall Full-History evidence supports retaining the existing tree count.

For **Post-CP**, the median best validation point occurs earlier, at **Tree 173**, and 11 of 15 jobs reach their minimum before Tree 400. The practical difference is small, however: continuing to Tree 400 raises validation MSE by only **0.955% at the median** relative to each job's own best point. This is better interpreted as **diminishing returns near the end of boosting** than as material overfitting. The validation curves largely flatten rather than showing a substantial late-stage increase.

Overall, the diagnostic does not provide a reason to reopen the frozen 400-tree LightGBM specification. Full-History continues to benefit from the full budget at the median, while Post-CP reaches a similar validation plateau somewhat earlier.

### 4.4.1.7C — Compare LightGBM with the seasonal benchmark

The main question is now simple:

> **Does LightGBM predict more accurately than the simple seasonal benchmark?**

MAE remains the primary accuracy measure. Lower MAE is better.

To make results comparable across targets with very different units, the chart reports **improvement over the benchmark as a percentage**:

- positive values mean LightGBM is better;
- zero means the two approaches tie;
- negative values mean the seasonal benchmark is better.

This percentage is called **benchmark skill** in the tables and code.

The chart keeps target, modeling track, and forecast horizon visible so we can see whether LightGBM's advantage is broad or limited to only a few cases.

In [46]:
# ---------------------------------------------------------------------
# 4.4.1.7C — Synthesize full-validation performance
# ---------------------------------------------------------------------

if lgbm_full_validation_results_df.empty:
    raise RuntimeError(
        "No full LightGBM validation results are available."
    )

failed_full_validation_df = lgbm_full_validation_results_df.loc[
    lgbm_full_validation_results_df["status"].eq("failed")
].copy()

if not failed_full_validation_df.empty:
    display(
        failed_full_validation_df[
            [
                "modeling_track",
                "metric",
                "horizon",
                "error_type",
                "error_message",
            ]
        ]
    )

    raise RuntimeError(
        "Full LightGBM validation contains failed jobs. "
        "Resolve them before interpreting model performance."
    )

full_validation_success_df = lgbm_full_validation_results_df.loc[
    lgbm_full_validation_results_df["status"].eq("success")
].copy()

if len(full_validation_success_df) != 30:
    raise RuntimeError(
        f"Expected 30 successful full-validation jobs; "
        f"found {len(full_validation_success_df)}."
    )

full_validation_success_df["Job"] = (
    full_validation_success_df.apply(
        lambda row: (
            f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[row['metric']]} · "
            f"h={int(row['horizon'])}"
        ),
        axis=1,
    )
)


# ---------------------------------------------------------------------
# Reader-facing job table
# ---------------------------------------------------------------------

lgbm_full_validation_display_df = (
    full_validation_success_df[
        [
            "Job",
            "mae",
            "benchmark_mae",
            "skill_vs_benchmark_pct",
            "rmse",
            "nmae",
            "bias",
            "negative_prediction_count",
            "negative_prediction_pct",
            "fit_seconds",
        ]
    ]
    .rename(
        columns={
            "mae": "LightGBM MAE",
            "benchmark_mae": "Benchmark MAE",
            "skill_vs_benchmark_pct": "Benchmark skill %",
            "rmse": "RMSE",
            "nmae": "NMAE",
            "bias": "Bias",
            "negative_prediction_count": "Negative predictions",
            "negative_prediction_pct": "Negative predictions %",
            "fit_seconds": "Fit seconds",
        }
    )
    .sort_values("Job")
    .reset_index(drop=True)
)

for column in [
    "LightGBM MAE",
    "Benchmark MAE",
    "Benchmark skill %",
    "RMSE",
    "NMAE",
    "Bias",
    "Negative predictions %",
    "Fit seconds",
]:
    lgbm_full_validation_display_df[column] = (
        lgbm_full_validation_display_df[column].round(3)
    )

display(lgbm_full_validation_display_df)


# ---------------------------------------------------------------------
# Supporting visual — benchmark skill by target, horizon, and track
# ---------------------------------------------------------------------
# Benchmark-relative skill makes targets with very different raw units
# comparable. Positive values favor LightGBM; negative values favor the
# seasonal benchmark.

metric_order = list(FORECAST_TARGETS)

track_symbols = {
    "full_history_regime_aware": "circle",
    "post_cp_primary": "diamond",
}

if RUN_VISUALS:
    validation_skill_fig = go.Figure()

    for metric in metric_order:
        for track, symbol in track_symbols.items():
            plot_df = (
                full_validation_success_df.loc[
                    full_validation_success_df["metric"].eq(metric)
                    & full_validation_success_df["modeling_track"].eq(track)
                ]
                .sort_values("horizon")
                .copy()
            )

            if plot_df.empty:
                continue

            tooltip_data = np.column_stack(
                [
                    plot_df["mae"].map(
                        lambda value: f"{value:.3f}"
                    ),
                    plot_df["benchmark_mae"].map(
                        lambda value: f"{value:.3f}"
                    ),
                    plot_df["skill_vs_benchmark_pct"].map(
                        lambda value: f"{value:+.3f}%"
                    ),
                    plot_df["rmse"].map(
                        lambda value: f"{value:.3f}"
                    ),
                    plot_df["nmae"].map(
                        lambda value: f"{value:.3f}"
                    ),
                    plot_df["bias"].map(
                        lambda value: f"{value:+.3f}"
                    ),
                    plot_df["negative_prediction_pct"].map(
                        lambda value: f"{value:.3f}%"
                    ),
                ]
            )

            validation_skill_fig.add_trace(
                go.Scatter(
                    x=plot_df["horizon"],
                    y=plot_df["skill_vs_benchmark_pct"].round(3),
                    mode="lines+markers",
                    name=(
                        f"{TARGET_SHORT_LABELS[metric]} · "
                        f"{TRACK_SHORT_LABELS[track]}"
                    ),
                    marker={
                        "size": 12,
                        "symbol": symbol,
                        "line": {"width": 1},
                    },
                    customdata=tooltip_data,
                    hovertext=plot_df["Job"],
                    hovertemplate=(
                        "%{hovertext}<br>"
                        "LightGBM MAE: %{customdata[0]}<br>"
                        "Benchmark MAE: %{customdata[1]}<br>"
                        "Benchmark skill: %{customdata[2]}<br>"
                        "RMSE: %{customdata[3]}<br>"
                        "NMAE: %{customdata[4]}<br>"
                        "Bias: %{customdata[5]}<br>"
                        "Negative predictions: %{customdata[6]}"
                        "<extra></extra>"
                    ),
                )
            )

    validation_skill_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
        annotation_text="Benchmark parity",
        annotation_position="top left",
    )

    validation_skill_fig.update_layout(
        title=(
            "LightGBM Full Validation — "
            "Skill Relative to the Seasonal Benchmark"
        ),
        xaxis_title="Forecast horizon",
        yaxis_title="Benchmark skill (%)",
        width=960,
        height=650,
        margin=dict(l=80, r=50, t=90, b=80),
        legend_title_text="Metric × Track",
    )

    validation_skill_fig.update_xaxes(
        tickmode="array",
        tickvals=LGBM_PRODUCTION_HORIZONS,
        ticktext=[
            f"h={horizon}"
            for horizon in LGBM_PRODUCTION_HORIZONS
        ],
    )

    apply_branding(validation_skill_fig)
    validation_skill_fig.show()


,Job,LightGBM MAE,Benchmark MAE,Benchmark skill %,RMSE,NMAE,Bias,Negative predictions,Negative predictions %,Fit seconds
0,Full-History · FHVHV speed · h=1,0.938,1.329,29.419,1.454,0.056,0.180,0,0.000,12.620
1,Full-History · FHVHV speed · h=2,0.982,1.329,26.107,1.492,0.058,0.238,0,0.000,12.004
2,Full-History · FHVHV speed · h=5,0.978,1.329,26.365,1.487,0.058,0.178,0,0.000,11.173
3,Full-History · FHVHV trips · h=1,47.647,69.426,31.370,89.170,0.092,-2.127,1367,0.386,11.446
4,Full-History · FHVHV trips · h=2,52.459,69.426,24.439,102.228,0.101,-3.527,1713,0.484,11.067
5,Full-History · FHVHV trips · h=5,49.994,69.426,27.989,100.170,0.097,-1.822,1673,0.473,10.919
6,Full-History · Subway · h=1,416.726,531.790,21.637,993.972,0.088,45.226,955,0.457,6.440
7,Full-History · Subway · h=2,478.391,531.790,10.041,1109.883,0.101,44.837,761,0.364,6.652
8,Full-History · Subway · h=5,442.154,531.790,16.855,1094.652,0.093,-148.340,2495,1.195,6.482
9,Full-History · Taxi speed · h=1,1.772,2.296,22.805,2.792,0.131,-0.116,0,0.000,5.882


**Findings.** LightGBM beats the seasonal benchmark in **29 of 30 forecasting jobs**. The typical improvement in MAE is about **21.6%**, with gains reaching as high as **31.4%**.

The strongest result is Full-History FHVHV trips at h=1, where LightGBM reduces MAE by 31.4%. The only loss is Post-CP Subway at h=2, where LightGBM is about 1.8% worse than the benchmark.

The gains appear across both modeling tracks and all three forecast horizons. Taxi speed and FHVHV speed are especially consistent. Subway is the weakest target overall, particularly in the shorter Post-CP history, but only one Subway job actually falls below the benchmark.

The main result is therefore very strong: LightGBM improves on the simple benchmark almost everywhere, with one small exception.

### 4.4.1.7D — Identify robust and weak forecasting regions

A single overall win rate can hide an important distinction. A model might perform well for one horizon but fail for another, or work reliably in one modeling track while remaining weaker in the other.

This final validation summary therefore groups the 30 jobs by forecasting target and asks two practical questions:

1. **How consistently does LightGBM beat the benchmark across horizons and tracks?**
2. **Are any weaknesses concentrated in a particular target, horizon, or modeling track?**

The target-level summary does not replace the individual job evidence. Its purpose is to reveal whether LightGBM's advantage is broad enough to support a simple production model, or whether later forecasting stages should carry forward only selected targets or horizons.


In [47]:
# ---------------------------------------------------------------------
# 4.4.1.7D — Summarize robustness across targets
# ---------------------------------------------------------------------

lgbm_target_validation_summary_df = (
    full_validation_success_df
    .groupby("metric", as_index=False)
    .agg(
        jobs=("horizon", "size"),
        jobs_beating_benchmark=("beats_benchmark", "sum"),
        median_skill_pct=("skill_vs_benchmark_pct", "median"),
        worst_skill_pct=("skill_vs_benchmark_pct", "min"),
        best_skill_pct=("skill_vs_benchmark_pct", "max"),
        median_nmae=("nmae", "median"),
        total_negative_predictions=("negative_prediction_count", "sum"),
    )
)

lgbm_target_validation_summary_df["benchmark_win_rate_pct"] = (
    100.0
    * lgbm_target_validation_summary_df["jobs_beating_benchmark"]
    / lgbm_target_validation_summary_df["jobs"]
)

lgbm_target_validation_summary_df["Target"] = (
    lgbm_target_validation_summary_df["metric"]
    .map(TARGET_SHORT_LABELS)
)

lgbm_target_validation_display_df = (
    lgbm_target_validation_summary_df[
        [
            "Target",
            "jobs",
            "jobs_beating_benchmark",
            "benchmark_win_rate_pct",
            "median_skill_pct",
            "worst_skill_pct",
            "best_skill_pct",
            "median_nmae",
            "total_negative_predictions",
        ]
    ]
    .rename(
        columns={
            "jobs": "Jobs",
            "jobs_beating_benchmark": "Benchmark wins",
            "benchmark_win_rate_pct": "Benchmark win rate %",
            "median_skill_pct": "Median skill %",
            "worst_skill_pct": "Worst skill %",
            "best_skill_pct": "Best skill %",
            "median_nmae": "Median NMAE",
            "total_negative_predictions": "Negative predictions",
        }
    )
)

for column in [
    "Benchmark win rate %",
    "Median skill %",
    "Worst skill %",
    "Best skill %",
    "Median NMAE",
]:
    lgbm_target_validation_display_df[column] = (
        lgbm_target_validation_display_df[column].round(3)
    )

display(lgbm_target_validation_display_df)


# ---------------------------------------------------------------------
# Compact Track × Target × Horizon skill matrix
# ---------------------------------------------------------------------
# This heatmap exposes weak cells directly. Rows distinguish both track and
# target; columns are the three forecast horizons. Positive values favor
# LightGBM, while negative values favor the frozen benchmark.

full_validation_success_df["Track × Target"] = (
    full_validation_success_df.apply(
        lambda row: (
            f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[row['metric']]}"
        ),
        axis=1,
    )
)

skill_matrix_df = (
    full_validation_success_df
    .pivot(
        index="Track × Target",
        columns="horizon",
        values="skill_vs_benchmark_pct",
    )
    .reindex(columns=LGBM_PRODUCTION_HORIZONS)
)

if RUN_VISUALS:
    skill_values = skill_matrix_df.to_numpy()

    validation_matrix_fig = go.Figure(
        data=go.Heatmap(
            z=skill_values,
            x=[
                f"h={horizon}"
                for horizon in skill_matrix_df.columns
            ],
            y=skill_matrix_df.index.tolist(),
            text=np.vectorize(
                lambda value: f"{value:+.3f}%"
            )(skill_values),
            texttemplate="%{text}",
            colorscale=list(reversed(BRAND_DIVERGING_SEQUENCE)),
            zmid=0,
            colorbar={
                "title": "Benchmark<br>skill (%)",
            },
            hovertemplate=(
                "%{y}<br>"
                "%{x}<br>"
                "Benchmark skill: %{text}"
                "<extra></extra>"
            ),
            xgap=3,
            ygap=3,
        )
    )

    validation_matrix_fig.update_layout(
        title=(
            "LightGBM Validation Robustness — "
            "Benchmark Skill Across All 30 Jobs"
        ),
        xaxis_title="Forecast horizon",
        yaxis_title=None,
        width=960,
        height=650,
        margin=dict(l=210, r=90, t=90, b=80),
    )

    apply_branding(validation_matrix_fig)
    validation_matrix_fig.show()


,Target,Jobs,Benchmark wins,Benchmark win rate %,Median skill %,Worst skill %,Best skill %,Median NMAE,Negative predictions
0,FHVHV speed,6,6,100.000,22.411,16.062,29.419,0.061,0
1,FHVHV trips,6,6,100.000,22.226,15.013,31.370,0.104,9327
2,Subway,6,5,83.333,8.385,-1.808,21.637,0.103,11033
3,Taxi speed,6,6,100.000,22.688,21.476,23.989,0.148,0
4,Taxi trips,6,6,100.000,20.825,7.594,30.242,0.153,10086


**Findings.** Four of the five targets beat the seasonal benchmark in **every combination of modeling track and forecast horizon**: Taxi trips, Taxi speed, FHVHV trips, and FHVHV speed.

For those four targets, the typical improvement over the benchmark is about 21% to 23%.

Subway is less consistent. It still beats the benchmark in 5 of 6 cases, but Post-CP h=2 falls slightly below it at about -1.8%.

The heatmap makes the pattern easy to see: the problem is concentrated in one Subway case rather than spread across the LightGBM system as a whole.

> ✅ Decision — Keep both modeling tracks through final interpretation\.

LightGBM performs well enough across both ordinary modeling tracks to keep the complete 30\-job validation evidence through the explainability stage\. The isolated Post\-CP Subway h=2 benchmark loss does not justify dropping that track before we understand whether the two histories learned materially different forecasting logic\.

Carry both tracks into Section 8 for interpretation\. Do not yet treat both tracks as downstream candidates; the final LightGBM modeling\-track winner will be selected after predictive performance and explainability evidence have been considered together\.

### 4.4.1.7E — Test a zero floor for impossible negative forecasts

None of the five forecasting targets can be negative in the real world. Trip counts and Subway ridership cannot fall below zero, and neither can average vehicle speed.

LightGBM does not know this automatically. Like many regression models, it can occasionally predict a value below zero.

The saved validation predictions let us test a simple rule without fitting any new models:

$$
\hat{y}_{\text{final}} = \max\left(0,\hat{y}_{\text{raw}}\right)
$$

In plain language: **keep every positive prediction unchanged, but replace any negative prediction with zero.**

The raw LightGBM prediction is still preserved for auditing.

The test asks three practical questions:

- How often does LightGBM predict an impossible negative value?
- Does replacing those values with zero improve forecast error?
- Does the change meaningfully affect forecast bias?

In [48]:
# ---------------------------------------------------------------------
# 4.4.1.7E — Assess a nonnegative support constraint for all forecasts
# ---------------------------------------------------------------------

NONNEGATIVE_TARGETS = list(FORECAST_TARGETS)

support_constraint_rows = []

for job in lgbm_full_validation_jobs_df.loc[
    lgbm_full_validation_jobs_df["metric"].isin(NONNEGATIVE_TARGETS)
].itertuples(index=False):
    predictions_path = lgbm_full_validation_predictions_path(
        job.modeling_track,
        job.metric,
        job.horizon,
    )

    predictions_df = pd.read_parquet(predictions_path)

    require_columns(
        predictions_df,
        ["actual", "benchmark_prediction", "lgbm_prediction_raw"],
        "full-validation predictions",
    )

    actual = predictions_df["actual"].to_numpy(dtype=float)
    raw_prediction = predictions_df["lgbm_prediction_raw"].to_numpy(dtype=float)

    # Every forecasting target has a known physical lower bound of zero.
    # Preserve the unconstrained model output, then derive a second forecast
    # that enforces only this known support constraint.
    constrained_prediction = np.maximum(raw_prediction, 0.0)

    raw_metrics = regression_metrics(
        actual,
        raw_prediction,
    )
    constrained_metrics = regression_metrics(
        actual,
        constrained_prediction,
    )

    negative_mask = raw_prediction < 0

    support_constraint_rows.append(
        {
            "modeling_track": job.modeling_track,
            "metric": job.metric,
            "horizon": int(job.horizon),
            "validation_rows": int(len(predictions_df)),
            "negative_predictions": int(negative_mask.sum()),
            "negative_prediction_pct": float(
                100.0 * negative_mask.mean()
            ),
            "raw_mae": float(raw_metrics["mae"]),
            "constrained_mae": float(constrained_metrics["mae"]),
            "mae_improvement_pct": float(
                100.0
                * (raw_metrics["mae"] - constrained_metrics["mae"])
                / raw_metrics["mae"]
            ),
            "raw_rmse": float(raw_metrics["rmse"]),
            "constrained_rmse": float(constrained_metrics["rmse"]),
            "raw_bias": float(raw_metrics["bias"]),
            "constrained_bias": float(constrained_metrics["bias"]),
        }
    )

support_constraint_df = pd.DataFrame(
    support_constraint_rows
)

if len(support_constraint_df) != 30:
    raise RuntimeError(
        f"Expected 30 validation jobs for support assessment; "
        f"found {len(support_constraint_df)}."
    )

support_constraint_df["Job"] = (
    support_constraint_df.apply(
        lambda row: (
            f"{TRACK_SHORT_LABELS[row['modeling_track']]} · "
            f"{TARGET_SHORT_LABELS[row['metric']]} · "
            f"h={int(row['horizon'])}"
        ),
        axis=1,
    )
)


# ---------------------------------------------------------------------
# Reader-facing comparison table
# ---------------------------------------------------------------------
# Speed jobs are retained even when the constraint never activates. Showing
# their zero-change rows confirms that the support rule is harmless where the
# model already respects the physical domain.

support_constraint_display_df = (
    support_constraint_df[
        [
            "Job",
            "negative_predictions",
            "negative_prediction_pct",
            "raw_mae",
            "constrained_mae",
            "mae_improvement_pct",
            "raw_rmse",
            "constrained_rmse",
            "raw_bias",
            "constrained_bias",
        ]
    ]
    .rename(
        columns={
            "negative_predictions": "Negative predictions",
            "negative_prediction_pct": "Negative predictions %",
            "raw_mae": "Raw MAE",
            "constrained_mae": "Constrained MAE",
            "mae_improvement_pct": "MAE improvement %",
            "raw_rmse": "Raw RMSE",
            "constrained_rmse": "Constrained RMSE",
            "raw_bias": "Raw bias",
            "constrained_bias": "Constrained bias",
        }
    )
)

for column in [
    "Negative predictions %",
    "Raw MAE",
    "Constrained MAE",
    "MAE improvement %",
    "Raw RMSE",
    "Constrained RMSE",
    "Raw bias",
    "Constrained bias",
]:
    support_constraint_display_df[column] = (
        support_constraint_display_df[column].round(3)
    )

display(support_constraint_display_df)


# ---------------------------------------------------------------------
# Visual — where does enforcing nonnegative support matter?
# ---------------------------------------------------------------------
# The x-axis shows how often the raw model leaves the valid domain. The y-axis
# shows how much enforcing the known zero lower bound improves MAE. Speed jobs
# should appear at the origin because they produced no negative predictions.

if RUN_VISUALS:
    track_symbols = {
        "full_history_regime_aware": "circle",
        "post_cp_primary": "diamond",
    }

    support_constraint_fig = go.Figure()

    for metric in FORECAST_TARGETS:
        for track, symbol in track_symbols.items():
            plot_df = support_constraint_df.loc[
                support_constraint_df["metric"].eq(metric)
                & support_constraint_df["modeling_track"].eq(track)
            ].copy()

            if plot_df.empty:
                continue

            tooltip_data = np.column_stack(
                [
                    plot_df["negative_prediction_pct"]
                    .map(lambda value: f"{value:.3f}%"),
                    plot_df["mae_improvement_pct"]
                    .map(lambda value: f"{value:+.3f}%"),
                    plot_df["raw_mae"]
                    .map(lambda value: f"{value:.3f}"),
                    plot_df["constrained_mae"]
                    .map(lambda value: f"{value:.3f}"),
                    plot_df["raw_bias"]
                    .map(lambda value: f"{value:+.3f}"),
                    plot_df["constrained_bias"]
                    .map(lambda value: f"{value:+.3f}"),
                ]
            )

            support_constraint_fig.add_trace(
                go.Scatter(
                    x=plot_df["negative_prediction_pct"].round(3),
                    y=plot_df["mae_improvement_pct"].round(3),
                    mode="markers",
                    name=(
                        f"{TARGET_SHORT_LABELS[metric]} · "
                        f"{TRACK_SHORT_LABELS[track]}"
                    ),
                    marker={
                        "size": 16,
                        "symbol": symbol,
                        "line": {"width": 1},
                    },
                    customdata=tooltip_data,
                    hovertext=plot_df["Job"],
                    hovertemplate=(
                        "%{hovertext}<br>"
                        "Negative predictions: %{customdata[0]}<br>"
                        "MAE improvement: %{customdata[1]}<br>"
                        "Raw MAE: %{customdata[2]}<br>"
                        "Constrained MAE: %{customdata[3]}<br>"
                        "Raw bias: %{customdata[4]}<br>"
                        "Constrained bias: %{customdata[5]}"
                        "<extra></extra>"
                    ),
                )
            )

    support_constraint_fig.add_vline(
        x=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    support_constraint_fig.add_hline(
        y=0,
        line_dash="dash",
        line_color=BRAND_COLORS["terracotta"],
    )

    support_constraint_fig.update_layout(
        title=(
            "Nonnegative Forecast Constraint — "
            "Validation Impact Across All 30 Jobs"
        ),
        xaxis_title="Negative predictions (%)",
        yaxis_title="MAE improvement from zero floor (%)",
        width=960,
        height=650,
        margin=dict(l=80, r=50, t=90, b=80),
        legend_title_text="Metric × Track",
    )

    apply_branding(support_constraint_fig)
    support_constraint_fig.show()

,Job,Negative predictions,Negative predictions %,Raw MAE,Constrained MAE,MAE improvement %,Raw RMSE,Constrained RMSE,Raw bias,Constrained bias
0,Full-History · FHVHV speed · h=1,0,0.000,0.938,0.938,0.000,1.454,1.454,0.180,0.180
1,Full-History · FHVHV speed · h=2,0,0.000,0.982,0.982,0.000,1.492,1.492,0.238,0.238
2,Full-History · FHVHV speed · h=5,0,0.000,0.978,0.978,0.000,1.487,1.487,0.178,0.178
3,Full-History · FHVHV trips · h=1,1367,0.386,47.647,47.632,0.031,89.170,89.169,-2.127,-2.112
4,Full-History · FHVHV trips · h=2,1713,0.484,52.459,52.439,0.038,102.228,102.227,-3.527,-3.507
5,Full-History · FHVHV trips · h=5,1673,0.473,49.994,49.978,0.033,100.170,100.169,-1.822,-1.806
6,Full-History · Subway · h=1,955,0.457,416.726,416.479,0.059,993.972,993.938,45.226,45.472
7,Full-History · Subway · h=2,761,0.364,478.391,478.138,0.053,1109.883,1109.855,44.837,45.090
8,Full-History · Subway · h=5,2495,1.195,442.154,441.537,0.140,1094.652,1094.594,-148.340,-147.723
9,Full-History · Taxi speed · h=1,0,0.000,1.772,1.772,0.000,2.792,2.792,-0.116,-0.116


**Findings.** The zero-floor rule behaves exactly as expected.

None of the **12 speed models** produces a negative forecast, so the rule changes nothing for them.

Negative values appear only in the **18 count models**, and even there they are uncommon—about **0.6% of validation predictions overall**.

Whenever a negative prediction does occur, replacing it with zero improves both MAE and RMSE. The improvement is usually tiny, although it reaches about **0.6%** for Post-CP Subway h=2, where negative forecasts are most common.

Bias moves slightly upward because negative values are being replaced with zero. That is expected.

The accuracy benefit is modest, but the rule prevents impossible forecasts and does not alter any prediction that is already valid.

> ✅ Decision — Prevent negative final forecasts

For all five targets, use \`max\(0, prediction\)\` when producing the final forecast\.

This changes only predictions below zero\. The original LightGBM output will still be saved alongside the final forecast so the model remains fully auditable\.

### Section Summary

Full rolling validation confirms that the frozen LightGBM system performs strongly across the complete ordinary forecasting workload. The 30 jobs span both modeling tracks, all five targets, and all three horizons, and **29 of 30** improve on the seasonal benchmark. Four targets win across every Track × Horizon combination; only Post-CP Subway h=2 falls slightly below benchmark.

The tree-by-tree validation diagnostic does not provide a reason to reopen the frozen 400-tree specification. **Full-History reaches its median best validation score at Tree 400**, while Post-CP reaches its median minimum earlier at Tree 173 but gives back only **0.955% validation MSE at the median** by continuing to Tree 400. The main pattern is therefore strong early learning followed by diminishing returns rather than material late-stage degradation.

The zero-floor test also behaves safely. Negative predictions occur only among the count outcomes, and replacing impossible negative values with zero never worsens MAE while leaving already valid forecasts unchanged.

Both modeling tracks remain available through Section 8 so their learned feature structure can be compared before the final LightGBM track is selected. Predictive performance remains the primary selection evidence; explainability provides the final sanity check.

## 4.4.1.8 — Explain What the LightGBM Models Learned

Accuracy tells us whether LightGBM forecasts well. This section asks a different question:

> **What information is LightGBM actually using to make those forecasts?**

The 30 validation models are already fitted and frozen. Sections 8A–8E explain their behavior using the feature-importance evidence captured during validation; no models are retrained here.

Two complementary measures are used:

- **Gain importance** asks which features were most useful when LightGBM built its trees.
- **SHAP importance** asks which features most strongly changed predictions on held-out validation examples.

Because the five targets operate on very different scales, importance is converted to a share within each model before results are compared.

The section first asks which features matter most overall, then checks whether **Gain and SHAP tell a consistent story**, how reliance changes across mobility targets, and whether the most important signals remain reasonably stable across forecasting jobs.

The final subsection adds one last sanity check before the tree-model branch closes: **did Full-History and Post-CP learn broadly recognizable versions of the same forecasting problem, or did the better-performing track begin relying on something unexpected?**

Feature importance is not a second accuracy test and does not prove cause and effect. Its role here is to make sure the final track choice is supported by forecasting performance without being contradicted by suspicious model behavior.

<img src="image-20260914-194648.png" width="" align="" />

> 🧠 SHAP Explainer — What does SHAP tell us?

SHAP helps explain a prediction by asking how each input feature moves the forecast away from the model's usual prediction\.

For one forecast, a feature can push the prediction higher or lower\. Add those feature contributions to the model's baseline prediction and they reproduce the final forecast\.

In the charts below, we mostly use mean absolute SHAP\. That ignores whether a feature pushes predictions up or down and instead asks:

How strongly does this feature usually affect the forecast?

A larger value means LightGBM relies more heavily on that feature\.

SHAP explains the model's behavior\. It does not prove that the feature causes the real\-world outcome\.

### 4.4.1.8A — Load and verify the saved explanation data

Feature importance was calculated while each of the 30 validation models was still fitted in memory. No new models need to be trained here.

For every forecasting job, the saved explanation data include:

- the full set of model inputs;
- LightGBM gain importance;
- SHAP importance calculated from 5,000 held-out validation examples.

This step simply checks that all 30 jobs have complete explanation data before comparing their feature importance.

In [49]:
# ---------------------------------------------------------------------
# 4.4.1.8A — Load and verify saved LightGBM explainability evidence
# ---------------------------------------------------------------------

if LGBM_EXPLAINABILITY_RESULTS_PATH.exists():
    lgbm_feature_importance_df = pd.read_parquet(
        LGBM_EXPLAINABILITY_RESULTS_PATH
    )
elif "lgbm_feature_importance_df" not in globals():
    raise FileNotFoundError(
        "The consolidated LightGBM explainability artifact is unavailable."
    )

require_columns(
    lgbm_feature_importance_df,
    [
        "explainability_contract_id",
        "modeling_track",
        "metric",
        "horizon",
        "feature",
        "gain_importance",
        "split_importance",
        "mean_abs_shap",
        "mean_shap",
        "gain_share",
        "shap_share",
        "sample_rows",
        "train_rows",
        "fit_seconds",
        "shap_seconds",
    ],
    "LightGBM explainability artifact",
)

if not lgbm_feature_importance_df[
    "explainability_contract_id"
].eq(LGBM_EXPLAINABILITY_CONTRACT_ID).all():
    raise RuntimeError(
        "Explainability artifact contains rows from a different contract."
    )

explainability_job_summary_df = (
    lgbm_feature_importance_df
    .groupby(
        ["modeling_track", "metric", "horizon"],
        as_index=False,
    )
    .agg(
        predictors=("feature", "nunique"),
        sample_rows=("sample_rows", "first"),
        train_rows=("train_rows", "first"),
        fit_seconds=("fit_seconds", "first"),
        shap_seconds=("shap_seconds", "first"),
    )
)

if len(explainability_job_summary_df) != 30:
    raise RuntimeError(
        f"Expected explainability evidence for 30 production jobs; "
        f"found {len(explainability_job_summary_df)}."
    )

expected_job_keys = set(
    lgbm_full_validation_jobs_df[
        ["modeling_track", "metric", "horizon"]
    ]
    .itertuples(index=False, name=None)
)

observed_job_keys = set(
    explainability_job_summary_df[
        ["modeling_track", "metric", "horizon"]
    ]
    .itertuples(index=False, name=None)
)

if observed_job_keys != expected_job_keys:
    raise RuntimeError(
        "Explainability job coverage does not match the frozen "
        "30-job production workload."
    )

# Predictor counts vary by target only if the upstream target-specific feature
# contract says they should. Check every artifact against that frozen roster.
for job in explainability_job_summary_df.itertuples(index=False):
    job_features = set(
        lgbm_feature_importance_df.loc[
            lgbm_feature_importance_df["modeling_track"].eq(job.modeling_track)
            & lgbm_feature_importance_df["metric"].eq(job.metric)
            & lgbm_feature_importance_df["horizon"].eq(job.horizon),
            "feature",
        ]
    )

    expected_features = set(
        predictor_names_by_target[job.metric]
    )

    if job_features != expected_features:
        raise RuntimeError(
            f"Predictor roster mismatch for "
            f"{job.modeling_track} · {job.metric} · h={job.horizon}."
        )

explainability_overview_df = pd.DataFrame(
    [
        {
            "Production jobs": len(explainability_job_summary_df),
            "Targets": explainability_job_summary_df["metric"].nunique(),
            "Horizons": explainability_job_summary_df["horizon"].nunique(),
            "Modeling tracks": (
                explainability_job_summary_df["modeling_track"].nunique()
            ),
            "SHAP rows per job": int(
                explainability_job_summary_df["sample_rows"].median()
            ),
            "Total SHAP seconds": float(
                explainability_job_summary_df["shap_seconds"].sum()
            ),
        }
    ]
)

explainability_overview_df["Total SHAP seconds"] = (
    explainability_overview_df["Total SHAP seconds"].round(3)
)

display(explainability_overview_df)

,Production jobs,Targets,Horizons,Modeling tracks,SHAP rows per job,Total SHAP seconds
0,30,5,3,2,5000,65.764


**Findings.** Explanation data are complete for all **30 LightGBM models**, covering both modeling tracks, all five targets, and all three horizons.

Each model uses a sample of **5,000 validation rows** for SHAP. The SHAP calculations took about **66 seconds in total** across the full workload.

No models are fitted in this section. The explanation data were saved during the same 30 fits used for validation.

### 4.4.1.8B — Which features matter most overall?

A feature can be very important for one target but barely matter for another. This view looks across all 30 models to find the inputs that LightGBM relies on most often.

The main ranking uses **average absolute SHAP importance**. In simple terms, it measures **how strongly each feature tends to move the model's forecasts**.

**Gain importance** looks at a different part of the same fitted model. Every time LightGBM splits a tree using a feature, that split receives Gain according to how much it improves the model's training objective. Adding those improvements across the trees tells us how useful that feature was while LightGBM constructed its decision rules.

The two measures therefore work together rather than competing:

* **Gain asks:** Which features helped LightGBM build useful tree structure?
* **SHAP asks:** Which features most strongly influence the predictions produced by that structure?

Because they answer different questions, Gain and SHAP are normalized **separately** and are not blended into a single importance score. The SHAP ranking remains the primary view of forecast behavior; the Gain comparison provides a second lens on whether the same features were also important during tree construction.

High importance under either measure should be interpreted as **predictive reliance, not causality**.


In [63]:
# ---------------------------------------------------------------------
# 4.4.1.8B — Aggregate global feature importance
# ---------------------------------------------------------------------

if lgbm_feature_importance_df.empty:
    raise RuntimeError(
        "No LightGBM explainability artifacts are available."
    )

global_feature_importance_df = (
    lgbm_feature_importance_df
    .groupby("feature", as_index=False)
    .agg(
        jobs=("horizon", "size"),
        mean_shap_share=("shap_share", "mean"),
        median_shap_share=("shap_share", "median"),
        mean_gain_share=("gain_share", "mean"),
    )
    .sort_values("mean_shap_share", ascending=False)
    .reset_index(drop=True)
)

TOP_GLOBAL_FEATURES = 20

global_feature_display_df = (
    global_feature_importance_df.head(TOP_GLOBAL_FEATURES)
    .copy()
)

for column in [
    "mean_shap_share",
    "median_shap_share",
    "mean_gain_share",
]:
    global_feature_display_df[column] = (
        100.0 * global_feature_display_df[column]
    ).round(3)

global_feature_display_df = global_feature_display_df.rename(
    columns={
        "feature": "Feature",
        "jobs": "Jobs represented",
        "mean_shap_share": "Mean SHAP share %",
        "median_shap_share": "Median SHAP share %",
        "mean_gain_share": "Mean gain share %",
    }
)

display(global_feature_display_df)


if RUN_VISUALS:
    global_plot_df = (
        global_feature_importance_df
        .head(TOP_GLOBAL_FEATURES)
        .sort_values("mean_shap_share")
        .copy()
    )

    global_importance_fig = go.Figure(
        go.Bar(
            x=(100.0 * global_plot_df["mean_shap_share"]).round(3),
            y=global_plot_df["feature"],
            orientation="h",
            customdata=np.column_stack(
                [
                    global_plot_df["mean_shap_share"]
                    .map(lambda value: f"{100.0 * value:.3f}%"),
                    global_plot_df["mean_gain_share"]
                    .map(lambda value: f"{100.0 * value:.3f}%"),
                    global_plot_df["jobs"]
                    .map(lambda value: f"{int(value)}"),
                ]
            ),
            hovertemplate=(
                "%{y}<br>"
                "Mean SHAP share: %{customdata[0]}<br>"
                "Mean gain share: %{customdata[1]}<br>"
                "Jobs represented: %{customdata[2]}"
                "<extra></extra>"
            ),
        )
    )

    global_importance_fig.update_layout(
        title="LightGBM Explainability — Most Influential Predictors Overall",
        xaxis_title="Mean absolute SHAP contribution share (%)",
        yaxis_title=None,
        width=960,
        height=650,
        margin=dict(l=260, r=50, t=90, b=80),
        showlegend=False,
    )

    apply_branding(global_importance_fig)
    global_importance_fig.show()


,Feature,Jobs represented,Mean SHAP share %,Median SHAP share %,Mean gain share %
0,seq_lag_3,30,14.153,1.171,21.502
1,seq_lag_4,30,14.121,1.573,20.820
2,origin_value,30,12.450,3.393,16.239
3,taxi_zone_id,30,10.519,9.927,9.634
4,target_temporal_bucket,30,9.825,6.881,5.890
5,same_daypart_roll_mean_7d,30,6.784,0.355,8.470
6,origin_temporal_bucket,30,6.105,3.054,3.712
7,same_daypart_roll_mean_28d,30,2.236,0.385,2.430
8,origin_weekday_index,30,2.034,1.410,0.989
9,target_weekday_index,30,1.600,1.273,0.553


**Findings.** LightGBM relies most heavily on three kinds of information:

1. **what the target is doing now and what it did recently;**
2. **which Taxi Zone the forecast belongs to;**
3. **what time of day the forecast represents.**

The strongest individual features are recent lags, the current observed value (`origin_value`), Taxi Zone identity, and the target time bucket.

`origin_value` answers a very simple forecasting question: **where is this series right now?** The lag features add: **where was it in the recent past?**

Taxi Zone identity is also consistently important. This means the citywide model is not treating every part of NYC as if it behaves the same way.

Weather, nearby transportation conditions, and other context features still contribute, but they play a smaller supporting role than recent history, place, and time.

**4.4.1.8B2 — Do Gain and SHAP tell the same story?**

The overall ranking above uses SHAP because our main question is **which features most strongly influence LightGBM's forecasts**. But LightGBM also records **Gain**, which asks a different question: which features were most useful when the trees themselves were being built?

Comparing the two gives us a useful consistency check.

* **Gain** is larger when splits using a feature repeatedly improve the model's training objective.
* **SHAP** is larger when a feature tends to move the fitted model's forecasts farther from their baseline prediction.

The two measures are therefore **not combined into one score**. Instead, the comparison below places their separately normalized importance shares side by side. Features with similar positions under both measures were important both to **building the tree structure** and to **producing forecasts**. Larger gaps reveal features whose role is more pronounced under one lens than the other.

The rank-agreement statistic summarizes that comparison across the full feature set, while the chart makes the largest feature-level agreements and disagreements visible.


In [69]:
# ---------------------------------------------------------------------
# 4.4.1.8B2 — Compare Gain and SHAP importance
# ---------------------------------------------------------------------
# 8B already created the authoritative global importance table.
# Here we reuse it to compare two different views of the same fitted trees:
# Gain describes tree construction; SHAP describes forecast influence.

gain_shap_df = global_feature_importance_df.assign(
    shap_pct=lambda df: 100.0 * df["mean_shap_share"],
    gain_pct=lambda df: 100.0 * df["mean_gain_share"],
)

# Spearman agreement compares the full feature rankings rather than just
# the features selected for display. A higher value means the two lenses
# broadly agree on which predictors matter most.
shap_rank = gain_shap_df["mean_shap_share"].rank(method="average")
gain_rank = gain_shap_df["mean_gain_share"].rank(method="average")
gain_shap_spearman = shap_rank.corr(gain_rank)

# This gap is descriptive only. It helps surface disagreements between
# the two measures; it is not a new or combined importance score.
gain_shap_df["share_gap_pp"] = (
    gain_shap_df["shap_pct"] - gain_shap_df["gain_pct"]
).abs()

gain_shap_agreement_df = pd.DataFrame([{
    "Features compared": len(gain_shap_df),
    "Gain–SHAP rank agreement (Spearman)": gain_shap_spearman,
    "Mean absolute share gap (pp)": gain_shap_df["share_gap_pp"].mean(),
    "Largest absolute share gap (pp)": gain_shap_df["share_gap_pp"].max(),
}]).round(3)

display(gain_shap_agreement_df)


# A feature qualifies for the visual if it is prominent under either lens.
# This avoids hiding a high-Gain feature merely because it falls outside
# the SHAP-first top-N used in the preceding section.
gain_shap_df["max_importance_pct"] = gain_shap_df[
    ["shap_pct", "gain_pct"]
].max(axis=1)

gain_shap_plot_df = (
    gain_shap_df
    .nlargest(TOP_GLOBAL_FEATURES, "max_importance_pct")
    .sort_values("shap_pct")
    .reset_index(drop=True)
)

if RUN_VISUALS:
    gain_shap_fig = go.Figure()

    # Connecting lines make agreement easy to read: shorter lines mean
    # Gain and SHAP assign similar normalized importance to that feature.
    connector_x, connector_y = [], []

    for row in gain_shap_plot_df.itertuples(index=False):
        connector_x.extend([row.gain_pct, row.shap_pct, None])
        connector_y.extend([row.feature, row.feature, None])

    gain_shap_fig.add_trace(
        go.Scatter(
            x=connector_x,
            y=connector_y,
            mode="lines",
            line={"color": "rgba(0, 109, 119, 0.24)", "width": 2},
            hoverinfo="skip",
            showlegend=False,
        )
    )

    gain_shap_fig.add_trace(
        go.Scatter(
            x=gain_shap_plot_df["gain_pct"],
            y=gain_shap_plot_df["feature"],
            mode="markers",
            name="Gain — tree construction",
            marker={
                "color": BRAND_COLORS["terracotta"],
                "size": 10,
                "symbol": "circle",
            },
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Gain share: %{x:.3f}%"
                "<extra></extra>"
            ),
        )
    )

    gain_shap_fig.add_trace(
        go.Scatter(
            x=gain_shap_plot_df["shap_pct"],
            y=gain_shap_plot_df["feature"],
            mode="markers",
            name="Mean |SHAP| — forecast influence",
            marker={
                "color": BRAND_COLORS["dark_teal"],
                "size": 11,
                "symbol": "diamond",
            },
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Mean |SHAP| share: %{x:.3f}%"
                "<extra></extra>"
            ),
        )
    )

    apply_branding(gain_shap_fig)

    gain_shap_fig.update_layout(
        title="Do Gain and SHAP Identify the Same Important Features?",
        xaxis_title="Mean importance share across forecasting jobs (%)",
        yaxis_title=None,
        width=940,
        height=max(620, 24 * len(gain_shap_plot_df) + 180),
        margin={"l": 235, "r": 40, "t": 100, "b": 65},
        legend={
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": 1.02,
            "yanchor": "bottom",
        },
    )

    gain_shap_fig.update_xaxes(rangemode="tozero")
    gain_shap_fig.show()

,Features compared,Gain–SHAP rank agreement (Spearman),Mean absolute share gap (pp),Largest absolute share gap (pp)
0,87,0.895,0.458,7.349


**Findings.** Gain and SHAP tell a strongly consistent overall story. Across all **87 features**, their importance rankings have a **Spearman correlation of 0.895**, while the average absolute difference between their normalized shares is only **0.458 percentage points**. Most predictors therefore occupy similar positions whether we ask which features helped LightGBM construct useful tree splits or which features most strongly influence its finished forecasts.

The largest differences are concentrated among a handful of the most important features rather than spread broadly across the feature set. In particular, **`seq_lag_3` and `seq_lag_4` receive substantially more Gain than SHAP importance**. LightGBM therefore relies heavily on these recent-history features when constructing its trees, but their average influence on final forecasts is somewhat smaller than their tree-building importance alone would suggest. `origin_value` shows the same pattern to a lesser degree.

The reverse pattern is visible for features such as **`target_temporal_bucket` and `origin_temporal_bucket`**, whose SHAP shares exceed their Gain shares. These variables may account for a smaller share of total split improvement during training, yet the splits in which they participate can still have meaningful effects on the forecasts that the fitted trees ultimately produce. **`taxi_zone_id` is comparatively well supported by both lenses**, indicating that geographic context matters both to tree construction and to forecast behavior.

Taken together, the comparison supports using **SHAP as the primary measure of forecast reliance while retaining Gain as a complementary structural check**. The high overall agreement indicates that the SHAP story is not disconnected from what LightGBM learned during training, while the feature-level gaps demonstrate why the two measures should not be collapsed into a single importance score. As with the broader feature-importance analysis, neither measure should be interpreted as evidence of causality.

### 4.4.1.8C — Do different mobility targets rely on different information?

The overall ranking can hide important differences between targets.

Taxi demand, Subway ridership, and vehicle speed do not necessarily depend on the same signals. This heatmap therefore shows which features matter most for each target after combining both modeling tracks and all three forecast horizons.

The goal is to see whether LightGBM learns one generic forecasting recipe or adapts to the type of mobility outcome being predicted.

In [51]:
# ---------------------------------------------------------------------
# 4.4.1.8C — Compare feature importance across forecasting targets
# ---------------------------------------------------------------------

target_feature_importance_df = (
    lgbm_feature_importance_df
    .groupby(["metric", "feature"], as_index=False)
    .agg(
        mean_shap_share=("shap_share", "mean"),
        mean_gain_share=("gain_share", "mean"),
    )
)

top_features_by_target = (
    target_feature_importance_df
    .sort_values(
        ["metric", "mean_shap_share"],
        ascending=[True, False],
    )
    .groupby("metric", as_index=False)
    .head(8)
)

target_feature_roster = (
    top_features_by_target
    .groupby("feature", as_index=False)["mean_shap_share"]
    .max()
    .sort_values("mean_shap_share", ascending=False)
    .head(20)["feature"]
    .tolist()
)

target_feature_matrix_df = (
    target_feature_importance_df.loc[
        target_feature_importance_df["feature"].isin(
            target_feature_roster
        )
    ]
    .pivot(
        index="feature",
        columns="metric",
        values="mean_shap_share",
    )
    .reindex(columns=FORECAST_TARGETS)
    .fillna(0.0)
)

target_feature_matrix_df.columns = [
    TARGET_SHORT_LABELS[metric]
    for metric in target_feature_matrix_df.columns
]

target_feature_matrix_df = target_feature_matrix_df.loc[
    target_feature_matrix_df.max(axis=1)
    .sort_values(ascending=False)
    .index
]


if RUN_VISUALS:
    target_importance_values = (
        100.0 * target_feature_matrix_df.to_numpy()
    )

    target_importance_text = np.vectorize(
        lambda value: f"{value:.3f}%"
    )(target_importance_values)

    target_importance_fig = go.Figure(
        go.Heatmap(
            z=target_importance_values,
            x=target_feature_matrix_df.columns.tolist(),
            y=target_feature_matrix_df.index.tolist(),
            text=target_importance_text,
            texttemplate="%{text}",
            colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
            colorbar={"title": "Mean SHAP<br>share (%)"},
            hovertemplate=(
                "%{y}<br>"
                "%{x}<br>"
                "Mean SHAP share: %{text}"
                "<extra></extra>"
            ),
            xgap=2,
            ygap=2,
        )
    )

    target_importance_fig.update_layout(
        title=(
            "LightGBM Explainability — "
            "Feature Importance by Forecasting Target"
        ),
        xaxis_title=None,
        yaxis_title=None,
        width=960,
        height=650,
        margin=dict(l=260, r=90, t=90, b=80),
    )

    apply_branding(target_importance_fig)
    target_importance_fig.show()


**Findings.** The heatmap shows a clear difference between **demand forecasts** and **speed forecasts**.

Taxi trips, FHVHV trips, and Subway ridership depend heavily on their own recent history. For these targets, the current value and recent lags are among the strongest features. In plain language, **recent demand is a strong guide to near-future demand**.

Speed behaves differently. Taxi speed relies much more on **Taxi Zone and time of day**, while FHVHV speed uses a more balanced mix of location, time, and recent speed.

LightGBM is therefore not using one generic recipe for every outcome. It learns a different balance of information depending on what it is trying to predict.

Weather, multimodal, and spatial features add useful context, but they do not replace the stronger signals from recent history, location, and time.

### 4.4.1.8D — Which important features matter consistently?

A feature may rank highly overall because it is extremely important in only a few models. Another feature may be a little less dominant but matter across almost every forecasting job.

This chart compares those two ideas:

- farther **right** means the feature matters more on average;
- farther **down** means its importance is more consistent from model to model.

Color groups features by their broader feature family.

This helps distinguish broadly useful forecasting signals from features that are especially important only for particular targets or horizons.

In [52]:
# ---------------------------------------------------------------------
# 4.4.1.8D — Measure feature-importance stability across jobs
# ---------------------------------------------------------------------

feature_stability_df = (
    lgbm_feature_importance_df
    .groupby("feature", as_index=False)
    .agg(
        jobs=("horizon", "size"),
        mean_shap_share=("shap_share", "mean"),
        std_shap_share=("shap_share", "std"),
    )
)

feature_stability_df["cv_shap_share"] = np.where(
    feature_stability_df["mean_shap_share"] > 0,
    feature_stability_df["std_shap_share"]
    / feature_stability_df["mean_shap_share"],
    np.nan,
)

feature_stability_df = (
    feature_stability_df
    .sort_values("mean_shap_share", ascending=False)
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Attach the frozen upstream feature family
# ---------------------------------------------------------------------
# Reuse the 4.2.3 feature catalog already loaded at the notebook boundary.
# This keeps the explainability grouping tied to the authoritative upstream
# feature taxonomy rather than inferring categories from feature names.

require_columns(
    forecast_feature_catalog_df,
    ["feature_name", "feature_family"],
    "forecast_feature_catalog_complete",
)

feature_family_lookup_df = (
    forecast_feature_catalog_df[
        ["feature_name", "feature_family"]
    ]
    .drop_duplicates()
    .rename(columns={"feature_name": "feature"})
)

# Taxi Zone identity is the one panel-tree input added in 4.4.1 rather than
# inherited from the 84 target-eligible definitions in the shared catalog.
feature_family_lookup_df = pd.concat(
    [
        feature_family_lookup_df,
        pd.DataFrame(
            {
                "feature": ["taxi_zone_id"],
                "feature_family": ["Series identity"],
            }
        ),
    ],
    ignore_index=True,
).drop_duplicates("feature")

feature_stability_df = feature_stability_df.merge(
    feature_family_lookup_df,
    on="feature",
    how="left",
    validate="one_to_one",
)

if feature_stability_df["feature_family"].isna().any():
    missing_features = feature_stability_df.loc[
        feature_stability_df["feature_family"].isna(),
        "feature",
    ].tolist()

    raise RuntimeError(
        "Feature-family metadata is missing for explainability features: "
        f"{missing_features}"
    )

stability_plot_df = feature_stability_df.head(30).copy()


# ---------------------------------------------------------------------
# Visual — importance versus stability, colored by feature family
# ---------------------------------------------------------------------
# Features farther right matter more on average. Features lower on the chart
# have more consistent importance across the 30 forecasting jobs.

if RUN_VISUALS:
    stability_fig = px.scatter(
        stability_plot_df,
        x=100.0 * stability_plot_df["mean_shap_share"],
        y="cv_shap_share",
        color="feature_family",
        hover_name="feature",
        custom_data=[
            "mean_shap_share",
            "std_shap_share",
            "cv_shap_share",
            "jobs",
            "feature_family",
        ],
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        labels={
            "feature_family": "Feature family",
            "cv_shap_share": "Importance variation ratio",
        },
    )

    stability_fig.update_traces(
        marker={"size": 14, "line": {"width": 1}},
        hovertemplate=(
            "%{hovertext}<br>"
            "Feature family: %{customdata[4]}<br>"
            "Mean SHAP share: %{customdata[0]:.3%}<br>"
            "SHAP-share SD: %{customdata[1]:.3%}<br>"
            "Variation ratio: %{customdata[2]:.3f}<br>"
            "Jobs represented: %{customdata[3]}"
            "<extra></extra>"
        ),
    )

    stability_fig.update_layout(
        title=(
            "LightGBM Explainability — "
            "Importance versus Stability Across Forecasting Jobs"
        ),
        xaxis_title="Mean absolute SHAP contribution share (%)",
        yaxis_title="Importance variation ratio",
        width=960,
        height=650,
        margin=dict(l=80, r=50, t=90, b=80),
        legend_title_text="Feature family",
    )

    apply_branding(stability_fig)
    stability_fig.show()

**Findings.** Taxi Zone identity stands out as one of the strongest **and most consistent** features in the forecasting system. It matters across many different targets, tracks, and horizons rather than becoming important in only a few special cases.

Time-of-day information is also relatively consistent.

The recent lag features tell a different story. They are among the most important features overall, but their importance changes much more from model to model. That makes sense given the previous heatmap: recent history is extremely important for demand forecasts but less dominant for some speed forecasts.

The models therefore rely on two complementary kinds of information:

- **stable context**, especially location and time of day;
- **recent mobility history**, whose importance depends more strongly on what is being forecast.

> Explainability summary
The LightGBM models can now be interpreted at three complementary levels:

\- overall importance identifies predictors that repeatedly contribute to forecasts across the system;
\- target\-specific importance shows where different mobility outcomes rely on different information;
\- importance stability distinguishes broadly reusable forecasting signals from features whose influence is concentrated in particular models\.

These views explain the forecasting system rather than establish causal relationships\. A feature can be highly useful for prediction because it summarizes timing, geography, recent mobility conditions, or correlated context without being a policy lever itself\.

Together with the full\-validation results, the explainability evidence provides a final check that the model's predictive strength is grounded in recognizable forecasting information rather than a small number of opaque or unstable signals\.


> 🧠 Synthesis — What did LightGBM learn?

Across the 30 models, the strongest forecasting signals are remarkably intuitive:

What is happening now?
The current value and recent lags are especially important for Taxi trips, FHVHV trips, and Subway ridership\.

Where is it happening?
Taxi Zone identity is one of the strongest and most consistent features, showing that the citywide model still preserves local differences\.

When is it happening?
Time\-of\-day features matter throughout the system and are especially important for speed forecasts\.

Weather, nearby transportation conditions, and other context features add useful information, but they generally play supporting roles\.

These results help explain how LightGBM makes forecasts\. They do not show that any one feature causes mobility to change\.

### 4.4.1.8E — Compare what the two modeling tracks learned

Section 7 tells us how well the two LightGBM tracks forecast. Before selecting one of them, this final explainability check asks whether the longer Full-History training record changed the model's forecasting logic in a reasonable way.

The two tracks do **not** need identical feature importance. They learned from different amounts of history, so some reweighting is expected.

Instead, three questions provide a useful sanity check:

- **Do the two tracks rank features similarly?** A rank correlation compares the overall ordering of the 85 predictors within each matched Target × Horizon job.
- **Do the same important features stay near the top?** Top-10 overlap counts how many of the ten most influential SHAP features appear in both tracks.
- **Where does the emphasis change most?** The final table and chart show the predictors whose average SHAP contribution differs most between Full-History and Post-CP.

The goal is not to choose the track with the most similar explanation. It is to check whether the stronger forecasting track still relies on recognizable signals such as recent mobility, place, and recurring time patterns rather than developing an unexpected dependence on one unusual predictor.

In [53]:
# ---------------------------------------------------------------------
# 4.4.1.8E — Compare feature reliance between the two modeling tracks
# ---------------------------------------------------------------------

FULL_HISTORY_TRACK = "full_history_regime_aware"
POST_CP_TRACK = "post_cp_primary"
TOP_FEATURE_OVERLAP_K = 10
TOP_FEATURES_TO_PLOT = 20

required_tracks = {FULL_HISTORY_TRACK, POST_CP_TRACK}
observed_tracks = set(lgbm_feature_importance_df["modeling_track"].unique())

if observed_tracks != required_tracks:
    raise RuntimeError(
        f"Expected explainability evidence for {sorted(required_tracks)}; "
        f"found {sorted(observed_tracks)}."
    )


# ---------------------------------------------------------------------
# 1) Compare feature rankings within each matched Target × Horizon job
# ---------------------------------------------------------------------
# We are not looking for identical importance values. The question is whether
# both histories tend to rank broadly similar forecasting signals near the top.

matched_similarity_rows = []

matched_jobs_df = (
    lgbm_feature_importance_df[["metric", "horizon"]]
    .drop_duplicates()
    .sort_values(["metric", "horizon"])
)

for job in matched_jobs_df.itertuples(index=False):
    job_df = lgbm_feature_importance_df.loc[
        lgbm_feature_importance_df["metric"].eq(job.metric)
        & lgbm_feature_importance_df["horizon"].eq(job.horizon),
        ["modeling_track", "feature", "shap_share"],
    ]

    comparison_df = (
        job_df.pivot(index="feature", columns="modeling_track", values="shap_share")
        .rename(columns={
            FULL_HISTORY_TRACK: "full_history_shap_share",
            POST_CP_TRACK: "post_cp_shap_share",
        })
        .dropna()
    )

    expected_features = set(predictor_names_by_target[job.metric])
    if set(comparison_df.index) != expected_features:
        raise RuntimeError(
            f"{job.metric} · h={int(job.horizon)}: matched feature roster is incomplete."
        )

    # Spearman-style rank agreement can be obtained by ranking both SHAP
    # profiles first, then correlating those ranks.
    full_rank = comparison_df["full_history_shap_share"].rank(
        ascending=False, method="average"
    )
    post_rank = comparison_df["post_cp_shap_share"].rank(
        ascending=False, method="average"
    )
    rank_correlation = float(full_rank.corr(post_rank))

    full_top = set(
        comparison_df["full_history_shap_share"].nlargest(TOP_FEATURE_OVERLAP_K).index
    )
    post_top = set(
        comparison_df["post_cp_shap_share"].nlargest(TOP_FEATURE_OVERLAP_K).index
    )
    top_overlap = len(full_top & post_top)

    matched_similarity_rows.append({
        "metric": job.metric,
        "horizon": int(job.horizon),
        "rank_correlation": rank_correlation,
        "top_10_overlap": top_overlap,
    })

track_similarity_df = pd.DataFrame(matched_similarity_rows)

track_similarity_summary_df = pd.DataFrame([{
    "Matched Target × Horizon jobs": len(track_similarity_df),
    "Median SHAP-rank correlation": track_similarity_df["rank_correlation"].median(),
    "Lowest SHAP-rank correlation": track_similarity_df["rank_correlation"].min(),
    "Median Top-10 overlap": track_similarity_df["top_10_overlap"].median(),
    "Lowest Top-10 overlap": track_similarity_df["top_10_overlap"].min(),
}])

for column in ["Median SHAP-rank correlation", "Lowest SHAP-rank correlation"]:
    track_similarity_summary_df[column] = track_similarity_summary_df[column].round(3)

display(track_similarity_summary_df)


# ---------------------------------------------------------------------
# 2) Show where the two tracks place different amounts of emphasis
# ---------------------------------------------------------------------
# Average normalized SHAP share within each track. Because every model's SHAP
# shares sum to one, these percentages can be compared without mixing the raw
# scales of trips, speeds, and ridership.

track_feature_mean_df = (
    lgbm_feature_importance_df
    .groupby(["modeling_track", "feature"], as_index=False)
    .agg(mean_shap_share=("shap_share", "mean"))
)

feature_track_comparison_df = (
    track_feature_mean_df
    .pivot(index="feature", columns="modeling_track", values="mean_shap_share")
    .rename(columns={
        FULL_HISTORY_TRACK: "full_history_shap_share",
        POST_CP_TRACK: "post_cp_shap_share",
    })
    .reset_index()
)

feature_track_comparison_df["full_history_shap_pct"] = (
    100.0 * feature_track_comparison_df["full_history_shap_share"]
)
feature_track_comparison_df["post_cp_shap_pct"] = (
    100.0 * feature_track_comparison_df["post_cp_shap_share"]
)
feature_track_comparison_df["difference_pp"] = (
    feature_track_comparison_df["full_history_shap_pct"]
    - feature_track_comparison_df["post_cp_shap_pct"]
)
feature_track_comparison_df["max_shap_pct"] = feature_track_comparison_df[
    ["full_history_shap_pct", "post_cp_shap_pct"]
].max(axis=1)

largest_shift_display_df = (
    feature_track_comparison_df
    .assign(abs_difference_pp=lambda df: df["difference_pp"].abs())
    .sort_values("abs_difference_pp", ascending=False)
    .head(15)
    .loc[:, ["feature", "full_history_shap_pct", "post_cp_shap_pct", "difference_pp"]]
    .rename(columns={
        "feature": "Feature",
        "full_history_shap_pct": "Full-History mean SHAP share %",
        "post_cp_shap_pct": "Post-CP mean SHAP share %",
        "difference_pp": "Full-History minus Post-CP pp",
    })
)

numeric_columns = [
    "Full-History mean SHAP share %",
    "Post-CP mean SHAP share %",
    "Full-History minus Post-CP pp",
]
largest_shift_display_df[numeric_columns] = (
    largest_shift_display_df[numeric_columns].round(3)
)

display(largest_shift_display_df)


# ---------------------------------------------------------------------
# 3) Reader-facing comparison of the most important features
# ---------------------------------------------------------------------
# Matching feature rows make large shifts easy to see. Similar bar lengths
# indicate that both histories place roughly similar emphasis on that signal.

if RUN_VISUALS:
    plot_df = (
        feature_track_comparison_df.nlargest(TOP_FEATURES_TO_PLOT, "max_shap_pct")
        .sort_values("max_shap_pct")
        .copy()
    )

    track_comparison_fig = go.Figure()

    track_comparison_fig.add_trace(go.Bar(
        x=plot_df["full_history_shap_pct"],
        y=plot_df["feature"],
        orientation="h",
        name=TRACK_SHORT_LABELS[FULL_HISTORY_TRACK],
        marker_color=BRAND_COLORS["dark_teal"],
        hovertemplate=(
            "%{y}<br>"
            "Full-History SHAP share: %{x:.3f}%"
            "<extra></extra>"
        ),
    ))

    track_comparison_fig.add_trace(go.Bar(
        x=plot_df["post_cp_shap_pct"],
        y=plot_df["feature"],
        orientation="h",
        name=TRACK_SHORT_LABELS[POST_CP_TRACK],
        marker_color=BRAND_COLORS["terracotta"],
        hovertemplate=(
            "%{y}<br>"
            "Post-CP SHAP share: %{x:.3f}%"
            "<extra></extra>"
        ),
    ))

    track_comparison_fig.update_layout(
        title="LightGBM Feature Reliance — Full-History vs. Post-CP",
        xaxis_title="Mean absolute SHAP contribution share (%)",
        yaxis_title=None,
        barmode="group",
        width=960,
        height=700,
        margin=dict(l=270, r=40, t=90, b=120),
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.12,
            xanchor="center",
            x=0.5,
        ),
    )

    apply_branding(track_comparison_fig)

    # Preserve the bottom legend even if the branding helper resets layout.
    track_comparison_fig.update_layout(
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.12,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(l=270, r=40, t=90, b=120),
    )

    track_comparison_fig.show()

,Matched Target × Horizon jobs,Median SHAP-rank correlation,Lowest SHAP-rank correlation,Median Top-10 overlap,Lowest Top-10 overlap
0,15,0.799,0.641,8.0,6


modeling_track,Feature,Full-History mean SHAP share %,Post-CP mean SHAP share %,Full-History minus Post-CP pp
48,same_daypart_roll_mean_28d,3.935,0.536,3.399
49,same_daypart_roll_mean_7d,8.345,5.224,3.120
41,origin_value,11.080,13.821,-2.741
36,origin_elapsed_days,1.677,0.038,1.639
74,target_temporal_bucket,9.189,10.460,-1.270
82,weather_target_proxy__temperature,0.400,1.540,-1.140
76,taxi_zone_id,10.020,11.019,-0.999
78,weather_origin__temperature,0.368,0.916,-0.548
1,annual_fourier_sin,0.714,0.204,0.510
57,seq_lag_3,14.386,13.919,0.467


**Findings.** Full-History and Post-CP learn broadly similar forecasting structures rather than two fundamentally different models.

Across the 15 matched Target × Horizon jobs, the median SHAP-rank correlation is **0.799**, and even the lowest matched-job correlation remains **0.641**. The two tracks also share a median **8 of their Top 10 features**, with no comparison sharing fewer than 6. This indicates substantial agreement about which predictors matter most.

The largest differences are shifts in emphasis rather than explainability red flags. Full-History relies more heavily on the same-daypart **28-day rolling mean (+3.399 percentage points)** and **7-day rolling mean (+3.120 pp)**, while Post-CP places somewhat more weight on the immediate **origin value (+2.741 pp for Post-CP)**, target temporal bucket, Taxi Zone identity, and weather. Recent sequence lags remain among the strongest predictors in both tracks.

Overall, the longer Full-History training record changes how LightGBM distributes importance across familiar forecasting signals, but it does not introduce an unexpected or implausible dependency. Nothing in the explainability comparison provides a reason to overturn the stronger predictive evidence for Full-History established in Section 7.

### 4.4.1.8F — Select the LightGBM modeling-track winner

Section 7 provides the primary selection evidence. Across the **15 matched Target × Horizon comparisons**, Full-History Regime-Aware has **higher benchmark-relative skill in all 15** and **lower NMAE in all 15**.

Its median improvement over the seasonal benchmark is **25.290%**, compared with **16.360%** for Post-CP. Its median NMAE is also lower at **0.101 versus 0.114**, indicating better scale-adjusted forecast accuracy. Full-History beats the benchmark in all 15 jobs, while Post-CP wins 14 of 15.

Section 8 then provides the sanity check. The two tracks retain substantially similar feature structures: median SHAP-rank correlation is **0.799**, and the median matched job shares **8 of its Top 10 features** across the two histories. The largest differences are understandable shifts in emphasis among recent mobility, rolling-history, time, place, and weather predictors rather than an unexpected dependency.

Forecast accuracy therefore favors Full-History decisively, and the explainability evidence provides no reason to overturn that result.

> ✅ Decision — Select Full\-History Regime\-Aware as the LightGBM modeling\-track winner\.

Carry forward Full\-History Regime\-Aware only as the LightGBM representative for downstream model\-family comparison in 4\.6\.x\.

Retain the Post\-CP models, predictions, and explainability evidence as supporting validation and audit material, but do not treat Post\-CP as an additional tree\-model candidate downstream\.

In [54]:
# ---------------------------------------------------------------------
# 4.4.1.8F — Freeze the selected LightGBM modeling track
# ---------------------------------------------------------------------

LGBM_SELECTED_MODELING_TRACK = "full_history_regime_aware"
LGBM_SELECTED_MODELING_TRACK_LABEL = TRACK_SHORT_LABELS[
    LGBM_SELECTED_MODELING_TRACK
]

selected_track_jobs = lgbm_full_validation_results_df.loc[
    lgbm_full_validation_results_df["modeling_track"].eq(
        LGBM_SELECTED_MODELING_TRACK
    )
]

if len(selected_track_jobs) != 15:
    raise RuntimeError(
        f"Expected 15 selected Full-History jobs; found {len(selected_track_jobs)}."
    )

print(
    "Selected LightGBM modeling track: "
    f"{LGBM_SELECTED_MODELING_TRACK_LABEL} · "
    f"{len(selected_track_jobs)} Target × Horizon jobs",
    flush=True,
)

Selected LightGBM modeling track: Full-History · 15 Target × Horizon jobs


### Section Summary

LightGBM's forecasting behavior is grounded in recognizable information rather than one mysterious predictor.

Across the complete 30-model explainability study, the strongest recurring signals are **recent mobility behavior, Taxi Zone identity, and recurring time-of-day structure**. Demand forecasts rely more heavily on their own recent history, while speed forecasts place more weight on where and when the forecast occurs.

The final track comparison adds one more check. Full-History and Post-CP can place different weights on individual predictors, but the longer-history models continue to use the same broadly interpretable forecasting structure rather than depending on an unexpected feature.

Combined with the stronger predictive evidence from Section 7, that sanity check supports the final tree-track decision:

**Full-History Regime-Aware is the selected LightGBM modeling track.**

The 15 Full-History Target × Horizon jobs advance to 4.6.x as the single tree-model candidate. Post-CP remains preserved as supporting validation and audit evidence.

## 4.4.1.9 — Package and Close the LightGBM Forecasting Handoff

The tree-model experiment is complete. This final section preserves both the evidence used to reach the decision and the single LightGBM candidate that advances downstream.

No models are fit here.

The package now has two clearly different roles:

**Audit evidence** preserves the complete 30-job experiment across both Full-History and Post-CP. These files retain the validation results, predictions, support-rule evidence, and explainability evidence needed to understand or reproduce the track comparison.

**Selected-candidate evidence** contains only the 15 Full-History Target × Horizon jobs chosen at the end of Section 8. This is the LightGBM surface that 4.6.x should use when comparing tree models with the classical and neural forecasting winners.

The exact predictor roster and forecast-output rule remain shared because both tracks used the same frozen LightGBM input and output contracts.

The final holdout remains untouched.

### 4.4.1.9A — Preserve the core 30-job LightGBM audit package

First preserve the ordinary-validation experiment exactly as evaluated.

The 30-row model registry, support-rule evidence, feature importance, predictor registry, and row-level validation predictions retain both modeling tracks. These files document how the Full-History versus Post-CP decision was reached and remain useful for audit and diagnostic work.

The complete two-track tree-by-tree learning diagnostic remains checkpointed in `4.4.1.intermediate_tables/`. Because only Full-History advances downstream, Section 9A.1 promotes the selected track's 6,000-row learning-history subset into the final candidate package.

The 30-job audit files are **not** the candidate roster for 4.6.x. A separate winner-only handoff is created immediately afterward.

In [55]:
# ---------------------------------------------------------------------
# 4.4.1.9A — Build the final LightGBM forecasting handoff
# ---------------------------------------------------------------------

LGBM_FINAL_MODEL_REGISTRY_PATH = FINAL_DIR / "lgbm_final_model_registry.parquet"
LGBM_FINAL_OUTPUT_CONTRACT_PATH = FINAL_DIR / "lgbm_forecast_output_contract.parquet"
LGBM_FINAL_SUPPORT_PATH = FINAL_DIR / "lgbm_support_constraint_summary.parquet"
LGBM_FINAL_IMPORTANCE_PATH = FINAL_DIR / "lgbm_feature_importance.parquet"
LGBM_FINAL_QA_PATH = FINAL_DIR / "lgbm_final_qa.parquet"
LGBM_FINAL_MANIFEST_PATH = FINAL_DIR / "lgbm_final_manifest.parquet"

FINAL_JOB_KEYS = ["modeling_track", "metric", "horizon"]


# ---------------------------------------------------------------------
# 1) Require the completed evidence used by the handoff
# ---------------------------------------------------------------------

require_columns(
    lgbm_full_validation_results_df,
    [
        "status",
        "validation_contract_id",
        "production_config_id",
        "modeling_track",
        "metric",
        "horizon",
        "model_family",
        "capacity_profile",
        "target_representation",
        "refit_cadence",
        "train_rows",
        "validation_rows",
        "benchmark_id",
        "benchmark_label",
        "mae",
        "rmse",
        "nmae",
        "bias",
        "benchmark_mae",
        "benchmark_rmse",
        "benchmark_nmae",
        "benchmark_bias",
        "skill_vs_benchmark_pct",
        "beats_benchmark",
        "negative_prediction_count",
        "negative_prediction_pct",
        "fit_seconds",
        "predict_seconds",
    ],
    "LightGBM full-validation results",
)

require_columns(
    lgbm_feature_importance_df,
    [
        "explainability_contract_id",
        "modeling_track",
        "metric",
        "horizon",
        "feature",
        "gain_importance",
        "split_importance",
        "mean_abs_shap",
        "mean_shap",
        "gain_share",
        "shap_share",
    ],
    "LightGBM explainability evidence",
)

require_columns(
    support_constraint_df,
    [
        "modeling_track",
        "metric",
        "horizon",
        "validation_rows",
        "negative_predictions",
        "negative_prediction_pct",
        "raw_mae",
        "constrained_mae",
        "mae_improvement_pct",
        "raw_rmse",
        "constrained_rmse",
        "raw_bias",
        "constrained_bias",
    ],
    "LightGBM nonnegative-support assessment",
)


# ---------------------------------------------------------------------
# 2) Build the 30-row final model registry
# ---------------------------------------------------------------------
# Validation remains based on raw LightGBM predictions. The final output rule
# is recorded separately so downstream users can distinguish model behavior
# from the small physical-domain correction applied after prediction.

lgbm_final_model_registry_df = (
    lgbm_full_validation_results_df.loc[
        lgbm_full_validation_results_df["status"].eq("success")
    ]
    .copy()
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

if len(lgbm_final_model_registry_df) != 30:
    raise RuntimeError(
        f"Expected 30 successful LightGBM jobs; "
        f"found {len(lgbm_final_model_registry_df)}."
    )

if lgbm_final_model_registry_df.duplicated(FINAL_JOB_KEYS).any():
    raise RuntimeError(
        "Final LightGBM registry contains duplicate forecasting jobs."
    )

# Preserve the exact selected LightGBM definition in every registry row.
for param_name, param_value in selected_lgbm_params.items():
    lgbm_final_model_registry_df[f"param_{param_name}"] = param_value

lgbm_final_model_registry_df["final_prediction_rule"] = "max(0, raw_prediction)"
lgbm_final_model_registry_df["raw_prediction_preserved"] = True
lgbm_final_model_registry_df["final_holdout_scored"] = False

atomic_parquet_write(
    lgbm_final_model_registry_df,
    LGBM_FINAL_MODEL_REGISTRY_PATH,
)


# ---------------------------------------------------------------------
# 3) Build the five-target forecast-output contract
# ---------------------------------------------------------------------

lgbm_forecast_output_contract_df = pd.DataFrame(
    [
        {
            "model_family": "LightGBM",
            "metric": metric,
            "physical_lower_bound": 0.0,
            "final_prediction_rule": "max(0, raw_prediction)",
            "raw_prediction_preserved": True,
            "validation_selection_uses_raw_prediction": True,
        }
        for metric in FORECAST_TARGETS
    ]
)

atomic_parquet_write(
    lgbm_forecast_output_contract_df,
    LGBM_FINAL_OUTPUT_CONTRACT_PATH,
)


# ---------------------------------------------------------------------
# 4) Promote the support-rule and explainability evidence
# ---------------------------------------------------------------------

lgbm_final_support_df = (
    support_constraint_df
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

atomic_parquet_write(
    lgbm_final_support_df,
    LGBM_FINAL_SUPPORT_PATH,
)

atomic_parquet_write(
    lgbm_feature_importance_df.sort_values(
        FINAL_JOB_KEYS + ["feature"]
    ).reset_index(drop=True),
    LGBM_FINAL_IMPORTANCE_PATH,
)


# ---------------------------------------------------------------------
# 5) Show one compact reader-facing handoff table
# ---------------------------------------------------------------------

lgbm_handoff_overview_df = pd.DataFrame(
    [
        {
            "Artifact": "Final model registry",
            "Rows": len(lgbm_final_model_registry_df),
            "Purpose": "Frozen 30-job LightGBM definition and validation results",
        },
        {
            "Artifact": "Forecast-output contract",
            "Rows": len(lgbm_forecast_output_contract_df),
            "Purpose": "Shared zero-floor rule for the five mobility targets",
        },
        {
            "Artifact": "Support-rule evidence",
            "Rows": len(lgbm_final_support_df),
            "Purpose": "Raw versus zero-floor validation comparison",
        },
        {
            "Artifact": "Feature importance",
            "Rows": len(lgbm_feature_importance_df),
            "Purpose": "Gain and SHAP evidence from the 30 validation models",
        },
    ]
)

display(lgbm_handoff_overview_df)

,Artifact,Rows,Purpose
0,Final model registry,30,Frozen 30-job LightGBM definition and validati...
1,Forecast-output contract,5,Shared zero-floor rule for the five mobility t...
2,Support-rule evidence,30,Raw versus zero-floor validation comparison
3,Feature importance,2550,Gain and SHAP evidence from the 30 validation ...


**Findings.** The compact 30-job audit package is complete.

The model registry contains all **30 LightGBM Track × Target × Horizon jobs**, the forecast-output contract covers all **five targets**, the support-rule evidence contains one row for every job, and the feature-importance artifact contains **2,550 rows = 30 models × 85 predictors**.

These artifacts preserve the complete ordinary-validation comparison across Full-History and Post-CP. They remain audit evidence rather than the candidate roster consumed by 4.6.x.

### 4.4.1.9B — Check the core 30-job LightGBM audit package

The first closeout check verifies that the ordinary-validation experiment has been preserved correctly.

It confirms that both modeling tracks still have all 30 validation jobs, support-rule evidence, explainability evidence, predictor definitions, and row-level ordinary-validation forecasts. It also confirms that raw predictions remain auditable, the zero-floor output rule is reproduced exactly, and the final holdout remains excluded.

These checks protect the **audit layer**. A second check immediately afterward verifies the smaller winner-only candidate—including its selected learning history—that will actually be consumed by 4.6.x.

In [56]:
# ---------------------------------------------------------------------
# 4.4.1.9A — Add downstream-ready prediction and predictor handoffs
# ---------------------------------------------------------------------

# Publish the row-level ordinary-validation handoff under one permanent,
# downstream-facing filename. Raw model output and the final constrained
# forecast remain separate so later evaluation can audit both.
LGBM_FINAL_VALIDATION_PREDICTIONS_PATH = (
    FINAL_DIR / "lgbm_row_level_validation_predictions.parquet"
)
LGBM_FINAL_PREDICTOR_REGISTRY_PATH = FINAL_DIR / "lgbm_predictor_registry.parquet"

PREDICTION_JOIN_KEYS = [
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

PREDICTION_CONTEXT_COLUMNS = [
    "modeling_track",
    "stage",
    "forecast_sequence_id",
    "taxi_zone_id",
    "zone",
    "borough",
    "metric",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "horizon_label",
    "origin_date",
    "origin_daypart",
    "origin_temporal_bucket",
    "origin_pre_post_cp",
    "target_date",
    "target_daypart",
    "target_temporal_bucket",
    "target_pre_post_cp",
    "target_observed_value",
]

# Restrict checkpoint reads to fields actually required by the downstream
# handoff. This also prevents accidental duplicate columns during the context
# merge if a checkpoint later gains additional metadata.
PREDICTION_CHECKPOINT_COLUMNS = [
    "modeling_track",
    "metric",
    "horizon",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "validation_contract_id",
    "actual",
    "benchmark_prediction",
    "lgbm_prediction_raw",
]

PREDICTION_HANDOFF_COLUMNS = [
    "forecast_role",
    "stage",
    "model_family",
    "modeling_track",
    "metric",
    "horizon",
    "horizon_label",
    "forecast_sequence_id",
    "taxi_zone_id",
    "zone",
    "borough",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "origin_date",
    "origin_daypart",
    "origin_temporal_bucket",
    "origin_pre_post_cp",
    "target_date",
    "target_daypart",
    "target_temporal_bucket",
    "target_pre_post_cp",
    "actual",
    "prediction_raw",
    "prediction_final",
    "error_raw",
    "error_final",
    "absolute_error_raw",
    "absolute_error_final",
    "squared_error_raw",
    "squared_error_final",
    "benchmark_id",
    "benchmark_label",
    "benchmark_prediction",
    "benchmark_error",
    "benchmark_absolute_error",
    "benchmark_squared_error",
    "validation_contract_id",
    "production_config_id",
    "capacity_profile",
    "target_representation",
    "refit_cadence",
    "selected_within_family",
    "selection_prediction_rule",
    "prediction_output_rule",
    "raw_prediction_preserved",
    "model_prediction_available",
    "benchmark_prediction_available",
]

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# 1) Freeze the exact predictor roster needed to reproduce LightGBM later
# ---------------------------------------------------------------------
# The final model registry preserves the selected LightGBM parameters. This
# companion table preserves the other half of the model definition: exactly
# which logical predictors enter each target model, in what order, and which
# inputs receive categorical treatment.

predictor_registry_rows = []

for metric in FORECAST_TARGETS:
    categorical_names = set(categorical_predictors_by_target[metric])

    for predictor_order, predictor_name in enumerate(
        predictor_names_by_target[metric],
        start=1,
    ):
        predictor_registry_rows.append(
            {
                "model_family": "LightGBM",
                "metric": metric,
                "predictor_order": predictor_order,
                "predictor_name": predictor_name,
                "is_categorical": predictor_name in categorical_names,
                "is_taxi_zone_identity": predictor_name == "taxi_zone_id",
            }
        )

lgbm_predictor_registry_df = pd.DataFrame(predictor_registry_rows)

if len(lgbm_predictor_registry_df) != 425:
    raise RuntimeError(
        "Expected 425 predictor-registry rows (5 targets × 85 logical inputs)."
    )

if lgbm_predictor_registry_df.duplicated(["metric", "predictor_name"]).any():
    raise RuntimeError(
        "Predictor registry contains duplicate Target × Predictor rows."
    )

for metric in FORECAST_TARGETS:
    metric_registry_df = (
        lgbm_predictor_registry_df.loc[
            lgbm_predictor_registry_df["metric"].eq(metric)
        ]
        .sort_values("predictor_order")
    )

    if metric_registry_df["predictor_name"].tolist() != list(
        predictor_names_by_target[metric]
    ):
        raise RuntimeError(f"{metric}: predictor order changed during export.")

    observed_categorical = set(
        metric_registry_df.loc[
            metric_registry_df["is_categorical"],
            "predictor_name",
        ]
    )

    if observed_categorical != set(categorical_predictors_by_target[metric]):
        raise RuntimeError(
            f"{metric}: categorical predictor roster changed during export."
        )

atomic_parquet_write(
    lgbm_predictor_registry_df,
    LGBM_FINAL_PREDICTOR_REGISTRY_PATH,
)

print(
    "Predictor registry ready: "
    f"{len(lgbm_predictor_registry_df):,} rows.",
    flush=True,
)


# ---------------------------------------------------------------------
# 2) Establish the expected 30-job prediction population
# ---------------------------------------------------------------------
# Counting rows from Parquet metadata is cheap and gives us a deterministic
# completeness target before we decide whether an existing final or staging
# artifact can be reused.

ordered_jobs_df = (
    lgbm_full_validation_jobs_df
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

if len(ordered_jobs_df) != 30:
    raise RuntimeError(
        f"Expected 30 LightGBM validation jobs; found {len(ordered_jobs_df)}."
    )

expected_prediction_rows = 0

for job in ordered_jobs_df.itertuples(index=False):
    source_path = lgbm_full_validation_predictions_path(
        job.modeling_track,
        job.metric,
        int(job.horizon),
    )

    if not source_path.exists():
        raise FileNotFoundError(
            "Missing completed LightGBM prediction checkpoint: "
            f"{source_path}"
        )

    expected_prediction_rows += int(
        pq.ParquetFile(source_path).metadata.num_rows
    )

print(
    "Expected consolidated validation rows: "
    f"{expected_prediction_rows:,}.",
    flush=True,
)


# ---------------------------------------------------------------------
# 3) Helper: recognize a reusable consolidated artifact
# ---------------------------------------------------------------------
# Re-running the notebook should not rebuild a multi-million-row file when an
# already published or staged artifact has the correct row count and schema.

def consolidated_prediction_artifact_is_valid(path):
    """Return True when a consolidated prediction artifact is reusable."""
    if not path.exists():
        return False

    try:
        parquet_file = pq.ParquetFile(path)
        observed_rows = int(parquet_file.metadata.num_rows)
        observed_columns = set(parquet_file.schema_arrow.names)

        return (
            observed_rows == expected_prediction_rows
            and set(PREDICTION_HANDOFF_COLUMNS).issubset(observed_columns)
        )

    except (OSError, ValueError, pa.ArrowInvalid):
        return False


# ---------------------------------------------------------------------
# 4) Prefer an already-valid final artifact
# ---------------------------------------------------------------------

final_artifact_ready = consolidated_prediction_artifact_is_valid(
    LGBM_FINAL_VALIDATION_PREDICTIONS_PATH
)

if final_artifact_ready:
    print(
        "Reusing existing final LightGBM validation-prediction handoff: "
        f"{expected_prediction_rows:,} rows.",
        flush=True,
    )


# ---------------------------------------------------------------------
# 5) Otherwise reuse a valid staging artifact or rebuild it
# ---------------------------------------------------------------------

if not final_artifact_ready:
    # A verified staging artifact can be reused on restart so the
    # multi-million-row context join is not repeated unnecessarily.
    staging_candidates = [
        INTERMEDIATE_DIR / "lgbm_row_level_validation_predictions__staging.parquet",
    ]

    prediction_staging_path = next(
        (
            path
            for path in staging_candidates
            if consolidated_prediction_artifact_is_valid(path)
        ),
        None,
    )

    if prediction_staging_path is not None:
        print(
            "Reusing verified LightGBM prediction staging artifact: "
            f"{prediction_staging_path}",
            flush=True,
        )

    else:
        # If the preferred staging path cannot be replaced, use a uniquely
        # named staging file so packaging can continue without risking an
        # existing artifact.
        preferred_staging_path = staging_candidates[0]

        if preferred_staging_path.exists():
            try:
                preferred_staging_path.unlink()
                prediction_staging_path = preferred_staging_path

            except OSError:
                run_token = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S_%f")
                prediction_staging_path = (
                    INTERMEDIATE_DIR
                    / (
                        "lgbm_row_level_validation_predictions"
                        f"__staging_{run_token}.parquet"
                    )
                )
        else:
            prediction_staging_path = preferred_staging_path

        writer = None
        writer_schema = None
        rows_written = 0

        try:
            for job_number, job in enumerate(
                ordered_jobs_df.itertuples(index=False),
                start=1,
            ):
                track = job.modeling_track
                metric = job.metric
                horizon = int(job.horizon)

                print(
                    f"Packaging validation predictions [{job_number}/30] "
                    f"{TRACK_SHORT_LABELS[track]} · "
                    f"{TARGET_SHORT_LABELS[metric]} · h={horizon}",
                    flush=True,
                )

                predictions_path = lgbm_full_validation_predictions_path(
                    track,
                    metric,
                    horizon,
                )

                predictions_df = pd.read_parquet(predictions_path)

                require_columns(
                    predictions_df,
                    PREDICTION_CHECKPOINT_COLUMNS,
                    (
                        f"{track} · {metric} · h={horizon} "
                        "prediction checkpoint"
                    ),
                )

                # Keep only the explicit source contract. This avoids accidental
                # overlap with geographic/context columns joined below.
                predictions_df = predictions_df[
                    PREDICTION_CHECKPOINT_COLUMNS
                ].copy()

                checkpoint_contract_ok = (
                    predictions_df["modeling_track"].eq(track).all()
                    and predictions_df["metric"].eq(metric).all()
                    and predictions_df["horizon"].eq(horizon).all()
                    and predictions_df["validation_contract_id"]
                    .eq(LGBM_FULL_VALIDATION_CONTRACT_ID)
                    .all()
                )

                if not checkpoint_contract_ok:
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "prediction checkpoint does not match "
                        "the frozen validation contract."
                    )

                if predictions_df.duplicated(PREDICTION_JOIN_KEYS).any():
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "prediction checkpoint contains duplicate rows."
                    )

                sequence_ids = (
                    predictions_df["forecast_sequence_id"]
                    .drop_duplicates()
                    .tolist()
                )

                # Only rolling-validation context is read. Final-holdout rows
                # therefore cannot enter the downstream prediction handoff.
                context_df = pd.read_parquet(
                    FORECASTING_MATRIX_PATHS[track],
                    columns=PREDICTION_CONTEXT_COLUMNS,
                    filters=[
                        ("stage", "==", "rolling_validation"),
                        ("metric", "==", metric),
                        ("horizon", "==", horizon),
                        ("forecast_sequence_id", "in", sequence_ids),
                    ],
                )

                if context_df.duplicated(PREDICTION_JOIN_KEYS).any():
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "4.2.3 context is not unique by forecast row."
                    )

                handoff_df = predictions_df.merge(
                    context_df,
                    on=PREDICTION_JOIN_KEYS,
                    how="left",
                    validate="one_to_one",
                    indicator=True,
                )

                if not handoff_df["_merge"].eq("both").all():
                    missing_rows = int(handoff_df["_merge"].ne("both").sum())

                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        f"{missing_rows:,} prediction rows lack 4.2.3 context."
                    )

                handoff_df = handoff_df.drop(columns="_merge")

                # The checkpoint and authoritative matrix must describe the same
                # observed outcome before any row is exported.
                if not np.allclose(
                    handoff_df["actual"].to_numpy(dtype=float),
                    handoff_df["target_observed_value"].to_numpy(dtype=float),
                    rtol=1e-10,
                    atol=1e-10,
                    equal_nan=True,
                ):
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "saved actuals disagree with the "
                        "authoritative forecasting matrix."
                    )

                registry_matches_df = lgbm_final_model_registry_df.loc[
                    lgbm_final_model_registry_df["modeling_track"].eq(track)
                    & lgbm_final_model_registry_df["metric"].eq(metric)
                    & lgbm_final_model_registry_df["horizon"].eq(horizon)
                ]

                if len(registry_matches_df) != 1:
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "expected exactly one final registry row; "
                        f"found {len(registry_matches_df)}."
                    )

                registry_row = registry_matches_df.iloc[0]

                actual = pd.to_numeric(handoff_df["actual"], errors="raise")
                raw_prediction = pd.to_numeric(
                    handoff_df["lgbm_prediction_raw"],
                    errors="raise",
                )
                benchmark_prediction = pd.to_numeric(
                    handoff_df["benchmark_prediction"],
                    errors="raise",
                )

                # Preserve the original unconstrained LightGBM prediction.
                # The physical zero floor is derived separately.
                handoff_df["forecast_role"] = "ordinary_validation"
                handoff_df["model_family"] = "LightGBM"
                handoff_df["prediction_raw"] = raw_prediction
                handoff_df["prediction_final"] = raw_prediction.clip(lower=0.0)

                handoff_df["error_raw"] = handoff_df["prediction_raw"] - actual
                handoff_df["error_final"] = handoff_df["prediction_final"] - actual
                handoff_df["absolute_error_raw"] = handoff_df["error_raw"].abs()
                handoff_df["absolute_error_final"] = handoff_df["error_final"].abs()
                handoff_df["squared_error_raw"] = handoff_df["error_raw"].pow(2)
                handoff_df["squared_error_final"] = handoff_df["error_final"].pow(2)

                handoff_df["benchmark_id"] = registry_row["benchmark_id"]
                handoff_df["benchmark_label"] = registry_row["benchmark_label"]
                handoff_df["benchmark_prediction"] = benchmark_prediction
                handoff_df["benchmark_error"] = benchmark_prediction - actual
                handoff_df["benchmark_absolute_error"] = (
                    handoff_df["benchmark_error"].abs()
                )
                handoff_df["benchmark_squared_error"] = (
                    handoff_df["benchmark_error"].pow(2)
                )

                handoff_df["production_config_id"] = (
                    registry_row["production_config_id"]
                )
                handoff_df["capacity_profile"] = registry_row["capacity_profile"]
                handoff_df["target_representation"] = (
                    registry_row["target_representation"]
                )
                handoff_df["refit_cadence"] = registry_row["refit_cadence"]

                handoff_df["selected_within_family"] = True
                handoff_df["selection_prediction_rule"] = "raw_unconstrained"
                handoff_df["prediction_output_rule"] = "max(0, raw_prediction)"
                handoff_df["raw_prediction_preserved"] = True

                handoff_df["model_prediction_available"] = np.isfinite(
                    raw_prediction.to_numpy(dtype=float)
                )
                handoff_df["benchmark_prediction_available"] = np.isfinite(
                    benchmark_prediction.to_numpy(dtype=float)
                )

                if not handoff_df["stage"].eq("rolling_validation").all():
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "non-validation rows entered the handoff."
                    )

                if not handoff_df["model_prediction_available"].all():
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "missing LightGBM predictions entered the handoff."
                    )

                if not handoff_df["benchmark_prediction_available"].all():
                    raise RuntimeError(
                        f"{track} · {metric} · h={horizon}: "
                        "missing benchmark predictions entered the handoff."
                    )

                # Normalize categoricals before streaming so every job produces
                # the same Arrow schema.
                for column in handoff_df.select_dtypes(
                    include=["category"]
                ).columns:
                    handoff_df[column] = handoff_df[column].astype("string")

                handoff_table = pa.Table.from_pandas(
                    handoff_df[PREDICTION_HANDOFF_COLUMNS],
                    preserve_index=False,
                )

                if writer is None:
                    writer_schema = handoff_table.schema
                    writer = pq.ParquetWriter(
                        prediction_staging_path,
                        writer_schema,
                        compression="zstd",
                    )

                elif handoff_table.schema != writer_schema:
                    handoff_table = handoff_table.cast(writer_schema)

                writer.write_table(handoff_table)
                rows_written += len(handoff_df)

                del predictions_df, context_df, handoff_df, handoff_table
                gc.collect()

            if writer is None:
                raise RuntimeError(
                    "No LightGBM validation predictions were available to package."
                )

            writer.close()
            writer = None

        except Exception:
            if writer is not None:
                try:
                    writer.close()
                except Exception:
                    pass

            # Only the current staging artifact is disposable. Failure to remove
            # it must not hide the actual packaging exception.
            try:
                prediction_staging_path.unlink(missing_ok=True)
            except OSError:
                pass

            raise

        if not consolidated_prediction_artifact_is_valid(prediction_staging_path):
            raise RuntimeError(
                "The consolidated LightGBM staging artifact "
                "failed row-count or schema verification."
            )

        print(
            "Built and verified LightGBM prediction staging artifact: "
            f"{expected_prediction_rows:,} rows.",
            flush=True,
        )

    # -----------------------------------------------------------------
    # 6) Publish the verified staging artifact
    # -----------------------------------------------------------------
    # Do not unlink the destination first. If a prior incomplete destination
    # exists, shutil.copyfile opens it for replacement. A valid destination is
    # always reused before reaching this branch.

    try:
        shutil.copyfile(
            prediction_staging_path,
            LGBM_FINAL_VALIDATION_PREDICTIONS_PATH,
        )

    except OSError as exc:
        raise RuntimeError(
            "The LightGBM prediction staging artifact is complete and "
            "verified, but it could not be published to "
            f"{LGBM_FINAL_VALIDATION_PREDICTIONS_PATH}. "
            "The verified staging artifact remains available at "
            f"{prediction_staging_path}."
        ) from exc

    if not consolidated_prediction_artifact_is_valid(
        LGBM_FINAL_VALIDATION_PREDICTIONS_PATH
    ):
        raise RuntimeError(
            "The published LightGBM validation-prediction handoff "
            "failed row-count or schema verification."
        )

    print(
        "Published final LightGBM validation-prediction handoff: "
        f"{expected_prediction_rows:,} rows.",
        flush=True,
    )

    final_artifact_ready = True

    # Staging is no longer needed once the final artifact has been verified.
    # Cleanup failure is harmless and should never fail the notebook.
    try:
        prediction_staging_path.unlink(missing_ok=True)
    except OSError:
        print(
            "Warning: final prediction handoff is valid, but the "
            "staging copy could not be removed.",
            flush=True,
        )


# ---------------------------------------------------------------------
# 7) Final downstream-facing summary
# ---------------------------------------------------------------------

if not final_artifact_ready:
    raise RuntimeError(
        "LightGBM validation-prediction handoff is not ready."
    )

prediction_file = pq.ParquetFile(
    LGBM_FINAL_VALIDATION_PREDICTIONS_PATH
)

lgbm_downstream_handoff_df = pd.DataFrame(
    [
        {
            "Artifact": "Validation predictions",
            "Rows": int(prediction_file.metadata.num_rows),
            "Purpose": (
                "Row-level raw, final, and benchmark forecasts for 4.6.x"
            ),
        },
        {
            "Artifact": "Predictor registry",
            "Rows": len(lgbm_predictor_registry_df),
            "Purpose": (
                "Exact LightGBM input roster for later production use"
            ),
        },
    ]
)

del prediction_file
gc.collect()

display(lgbm_downstream_handoff_df)

Predictor registry ready: 425 rows.
Expected consolidated validation rows: 8,846,655.
Reusing existing final LightGBM validation-prediction handoff: 8,846,655 rows.


,Artifact,Rows,Purpose
0,Validation predictions,8846655,"Row-level raw, final, and benchmark forecasts ..."
1,Predictor registry,425,Exact LightGBM input roster for later producti...


**Findings.** The row-level audit handoff is also complete.

The consolidated validation-prediction file contains **8,846,655 rows** across the full 30-job experiment. The existing validated artifact was reused during this run rather than rebuilt. Raw LightGBM predictions, zero-floored final predictions, benchmark forecasts, observed outcomes, forecast identifiers, and geographic and temporal context remain available together for downstream auditing.

The predictor registry contains exactly **425 rows = 5 targets × 85 ordered predictors**, preserving the feature order and categorical-treatment contract needed to reproduce the LightGBM design.

#### 4.4.1.9A.1 — Build the winner-only LightGBM candidate

The complete 30-job package remains available for audit, but the downstream tournament needs only one tree-model candidate.

This step therefore creates a second set of files containing **Full-History Regime-Aware only**:

- the 15 selected Target × Horizon model definitions;
- their support-rule evidence;
- their gain and SHAP feature importance;
- their 400-tree training and validation learning histories;
- their row-level ordinary-validation forecasts;
- a one-row contract explicitly naming Full-History as the selected LightGBM track.

No model is refit. The selected files are filtered from evidence already created and validated earlier in the notebook.

In [57]:
# ---------------------------------------------------------------------
# 4.4.1.9A.1 — Build the winner-only LightGBM candidate for 4.6.x
# ---------------------------------------------------------------------

if "LGBM_SELECTED_MODELING_TRACK" not in globals():
    raise RuntimeError(
        "Select the LightGBM modeling-track winner in Section 8 "
        "before packaging 4.6.x."
    )

if LGBM_SELECTED_MODELING_TRACK != "full_history_regime_aware":
    raise RuntimeError(
        f"Unexpected selected LightGBM track: {LGBM_SELECTED_MODELING_TRACK}"
    )

LGBM_SELECTED_TRACK_CONTRACT_PATH = FINAL_DIR / "lgbm_selected_track_contract.parquet"
LGBM_SELECTED_MODEL_REGISTRY_PATH = FINAL_DIR / "lgbm_selected_model_registry.parquet"
LGBM_SELECTED_SUPPORT_PATH = FINAL_DIR / "lgbm_selected_support_constraint_summary.parquet"
LGBM_SELECTED_IMPORTANCE_PATH = FINAL_DIR / "lgbm_selected_feature_importance.parquet"
LGBM_SELECTED_LEARNING_HISTORY_PATH = FINAL_DIR / "lgbm_selected_learning_history.parquet"
LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH = (
    FINAL_DIR / "lgbm_selected_row_level_validation_predictions.parquet"
)
LGBM_SELECTED_QA_PATH = FINAL_DIR / "lgbm_selected_qa.parquet"
LGBM_SELECTED_MANIFEST_PATH = FINAL_DIR / "lgbm_selected_manifest.parquet"


# ---------------------------------------------------------------------
# 1) Write an explicit one-row downstream selection contract
# ---------------------------------------------------------------------
# The 30-job files remain useful audit evidence. This contract tells 4.6.x
# unambiguously which track represents the tree-model family.

lgbm_selected_track_contract_df = pd.DataFrame([{
    "model_family": "LightGBM",
    "selected_modeling_track": LGBM_SELECTED_MODELING_TRACK,
    "selected_modeling_track_label": LGBM_SELECTED_MODELING_TRACK_LABEL,
    "selected_jobs": 15,
    "selection_stage": "4.4.1.8F",
    "selection_basis": "predictive performance plus explainability sanity check",
    "downstream_role": "sole_tree_candidate_for_4.6.x",
    "nonselected_track_role": "audit_only",
}])

atomic_parquet_write(
    lgbm_selected_track_contract_df,
    LGBM_SELECTED_TRACK_CONTRACT_PATH,
)


# ---------------------------------------------------------------------
# 2) Filter the compact artifacts to the selected 15 jobs
# ---------------------------------------------------------------------

selected_registry_df = (
    lgbm_final_model_registry_df.loc[
        lgbm_final_model_registry_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        )
    ]
    .copy()
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

selected_support_df = (
    lgbm_final_support_df.loc[
        lgbm_final_support_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        )
    ]
    .copy()
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

selected_importance_df = (
    lgbm_feature_importance_df.loc[
        lgbm_feature_importance_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        )
    ]
    .copy()
    .sort_values(FINAL_JOB_KEYS + ["feature"])
    .reset_index(drop=True)
)

if len(selected_registry_df) != 15:
    raise RuntimeError(
        f"Expected 15 selected LightGBM jobs; found {len(selected_registry_df)}."
    )

if len(selected_support_df) != 15:
    raise RuntimeError(
        f"Expected 15 selected support rows; found {len(selected_support_df)}."
    )

if len(selected_importance_df) != 15 * 85:
    raise RuntimeError(
        f"Expected {15 * 85:,} selected importance rows; "
        f"found {len(selected_importance_df):,}."
    )

selected_registry_df["selected_modeling_track"] = True
selected_registry_df["downstream_candidate"] = True

atomic_parquet_write(
    selected_registry_df,
    LGBM_SELECTED_MODEL_REGISTRY_PATH,
)
atomic_parquet_write(
    selected_support_df,
    LGBM_SELECTED_SUPPORT_PATH,
)
atomic_parquet_write(
    selected_importance_df,
    LGBM_SELECTED_IMPORTANCE_PATH,
)


# ---------------------------------------------------------------------
# 3) Preserve the tree-by-tree learning history for the winner
# ---------------------------------------------------------------------

if "lgbm_learning_history_df" not in globals():
    if (
        "LGBM_LEARNING_HISTORY_PATH" in globals()
        and LGBM_LEARNING_HISTORY_PATH.exists()
    ):
        lgbm_learning_history_df = pd.read_parquet(
            LGBM_LEARNING_HISTORY_PATH
        )
    else:
        raise FileNotFoundError(
            "The LightGBM training/validation learning-history "
            "artifact is unavailable."
        )

selected_learning_history_df = (
    lgbm_learning_history_df.loc[
        lgbm_learning_history_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        )
    ]
    .copy()
    .sort_values(FINAL_JOB_KEYS + ["boosting_round"])
    .reset_index(drop=True)
)

expected_rounds = int(selected_lgbm_params["n_estimators"])
expected_learning_rows = 15 * expected_rounds

if len(selected_learning_history_df) != expected_learning_rows:
    raise RuntimeError(
        f"Expected {expected_learning_rows:,} selected learning-history rows; "
        f"found {len(selected_learning_history_df):,}."
    )

atomic_parquet_write(
    selected_learning_history_df,
    LGBM_SELECTED_LEARNING_HISTORY_PATH,
)


# ---------------------------------------------------------------------
# 4) Create a winner-only row-level validation file without refitting
# ---------------------------------------------------------------------
# Stream the existing validated 30-job file by Parquet row group and retain
# only Full-History rows. This avoids loading millions of rows into memory.

import pyarrow.compute as pc

selected_jobs_df = (
    lgbm_full_validation_jobs_df.loc[
        lgbm_full_validation_jobs_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        )
    ]
    .sort_values(FINAL_JOB_KEYS)
    .reset_index(drop=True)
)

expected_selected_prediction_rows = sum(
    int(
        pq.ParquetFile(
            lgbm_full_validation_predictions_path(
                job.modeling_track,
                job.metric,
                int(job.horizon),
            )
        ).metadata.num_rows
    )
    for job in selected_jobs_df.itertuples(index=False)
)


def selected_prediction_artifact_is_valid(path):
    """Return True when the winner-only row-level handoff is complete and pure."""
    if not path.exists():
        return False

    try:
        parquet_file = pq.ParquetFile(path)

        if int(parquet_file.metadata.num_rows) != expected_selected_prediction_rows:
            return False

        if not set(PREDICTION_HANDOFF_COLUMNS).issubset(
            parquet_file.schema_arrow.names
        ):
            return False

        observed_tracks = set()

        for batch in parquet_file.iter_batches(
            columns=["modeling_track"],
            batch_size=250_000,
        ):
            observed_tracks.update(
                pc.unique(batch.column(0)).to_pylist()
            )

        return observed_tracks == {LGBM_SELECTED_MODELING_TRACK}

    except (OSError, ValueError, pa.ArrowInvalid):
        return False


if not selected_prediction_artifact_is_valid(
    LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH
):
    staging_path = (
        INTERMEDIATE_DIR
        / "lgbm_selected_row_level_validation_predictions__staging.parquet"
    )
    staging_path.unlink(missing_ok=True)

    source_parquet = pq.ParquetFile(
        LGBM_FINAL_VALIDATION_PREDICTIONS_PATH
    )

    writer = None
    rows_written = 0

    try:
        for row_group_index in range(source_parquet.num_row_groups):
            track_table = source_parquet.read_row_group(
                row_group_index,
                columns=["modeling_track"],
            )
            track_values = set(
                pc.unique(track_table["modeling_track"]).to_pylist()
            )

            if LGBM_SELECTED_MODELING_TRACK not in track_values:
                continue

            row_group_table = source_parquet.read_row_group(
                row_group_index
            )

            selected_mask = pc.equal(
                row_group_table["modeling_track"],
                LGBM_SELECTED_MODELING_TRACK,
            )
            selected_table = row_group_table.filter(selected_mask)

            if selected_table.num_rows == 0:
                continue

            if writer is None:
                writer = pq.ParquetWriter(
                    staging_path,
                    selected_table.schema,
                    compression="zstd",
                )

            writer.write_table(selected_table)
            rows_written += selected_table.num_rows

        if writer is None:
            raise RuntimeError(
                "No selected LightGBM validation rows were found."
            )

        writer.close()
        writer = None

    except Exception:
        if writer is not None:
            writer.close()

        staging_path.unlink(missing_ok=True)
        raise

    if rows_written != expected_selected_prediction_rows:
        raise RuntimeError(
            f"Selected prediction staging file has {rows_written:,} rows; "
            f"expected {expected_selected_prediction_rows:,}."
        )

    if not selected_prediction_artifact_is_valid(staging_path):
        raise RuntimeError(
            "Selected prediction staging file failed validation."
        )

    shutil.copyfile(
        staging_path,
        LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH,
    )
    staging_path.unlink(missing_ok=True)


if not selected_prediction_artifact_is_valid(
    LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH
):
    raise RuntimeError(
        "Winner-only LightGBM validation handoff is not valid."
    )


# ---------------------------------------------------------------------
# 5) Show exactly what 4.6.x receives
# ---------------------------------------------------------------------

selected_prediction_file = pq.ParquetFile(
    LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH
)

lgbm_selected_handoff_overview_df = pd.DataFrame([
    {
        "Artifact": "Selected track contract",
        "Rows": len(lgbm_selected_track_contract_df),
        "Purpose": "Declares the sole LightGBM candidate for 4.6.x",
    },
    {
        "Artifact": "Selected model registry",
        "Rows": len(selected_registry_df),
        "Purpose": "15 Full-History Target × Horizon model definitions",
    },
    {
        "Artifact": "Selected validation predictions",
        "Rows": int(selected_prediction_file.metadata.num_rows),
        "Purpose": "Winner-only row-level evidence for cross-family evaluation",
    },
    {
        "Artifact": "Selected feature importance",
        "Rows": len(selected_importance_df),
        "Purpose": "Gain and SHAP evidence for the selected 15 jobs",
    },
    {
        "Artifact": "Selected learning history",
        "Rows": len(selected_learning_history_df),
        "Purpose": "Tree-by-tree training and validation diagnostics",
    },
])

del selected_prediction_file
gc.collect()

display(lgbm_selected_handoff_overview_df)

,Artifact,Rows,Purpose
0,Selected track contract,1,Declares the sole LightGBM candidate for 4.6.x
1,Selected model registry,15,15 Full-History Target × Horizon model definit...
2,Selected validation predictions,4311531,Winner-only row-level evidence for cross-famil...
3,Selected feature importance,1275,Gain and SHAP evidence for the selected 15 jobs
4,Selected learning history,6000,Tree-by-tree training and validation diagnostics


**Findings.** The winner-only LightGBM handoff was created successfully from the already validated evidence.

The selected-track contract contains one explicit decision: **Full-History Regime-Aware is the sole LightGBM candidate for 4.6.x**. The selected model registry contains exactly **15 Target × Horizon jobs**, and the winner-only row-level validation file contains **4,311,531 predictions**.

The package also preserves **1,275 feature-importance rows = 15 models × 85 predictors** and **6,000 learning-history rows = 15 models × 400 trees**. No model was refit to create these files; they are filtered from the completed validation and diagnostic evidence.

The broader Post-CP evidence remains available for audit, while the downstream candidate surface is now unambiguous.

### 4.4.1.9B — Check the complete LightGBM audit package

The first closeout check verifies that the complete experiment has been preserved correctly.

It confirms that both modeling tracks still have all 30 validation jobs, support-rule evidence, explainability evidence, predictor definitions, and row-level ordinary-validation forecasts. It also confirms that raw predictions remain auditable, the zero-floor output rule is reproduced exactly, and the final holdout remains excluded.

These checks protect the **audit layer**. A second check immediately afterward verifies the smaller winner-only candidate that will actually be consumed by 4.6.x.

In [58]:
# ---------------------------------------------------------------------
# 4.4.1.9B — Final handoff QA and manifest
# ---------------------------------------------------------------------

expected_final_job_keys = set(
    lgbm_full_validation_jobs_df[FINAL_JOB_KEYS]
    .itertuples(index=False, name=None)
)

observed_final_job_keys = set(
    lgbm_final_model_registry_df[FINAL_JOB_KEYS]
    .itertuples(index=False, name=None)
)

support_job_keys = set(
    lgbm_final_support_df[FINAL_JOB_KEYS]
    .itertuples(index=False, name=None)
)

importance_job_keys = set(
    lgbm_feature_importance_df[FINAL_JOB_KEYS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)


def handoff_qa_row(check, passed, detail):
    """Create one compact final-handoff QA result."""
    return {
        "check": check,
        "passed": bool(passed),
        "detail": str(detail),
    }


# Every explainability artifact should still contain the complete predictor
# roster for its target. This prevents a partially written importance artifact
# from passing merely because the Track × Target × Horizon key exists.
predictor_rosters_complete = True

for job in lgbm_final_model_registry_df.itertuples(index=False):
    observed_features = set(
        lgbm_feature_importance_df.loc[
            lgbm_feature_importance_df["modeling_track"].eq(job.modeling_track)
            & lgbm_feature_importance_df["metric"].eq(job.metric)
            & lgbm_feature_importance_df["horizon"].eq(job.horizon),
            "feature",
        ]
    )

    if observed_features != set(predictor_names_by_target[job.metric]):
        predictor_rosters_complete = False
        break


lgbm_final_qa_df = pd.DataFrame(
    [
        handoff_qa_row(
            "Exactly 30 production jobs",
            len(lgbm_final_model_registry_df) == 30,
            f"{len(lgbm_final_model_registry_df)}/30 rows",
        ),
        handoff_qa_row(
            "Production-job keys match the frozen workload",
            observed_final_job_keys == expected_final_job_keys,
            f"{len(observed_final_job_keys)}/30 unique jobs",
        ),
        handoff_qa_row(
            "All validation jobs succeeded",
            lgbm_final_model_registry_df["status"].eq("success").all(),
            (
                f"{int(lgbm_final_model_registry_df['status'].eq('success').sum())}"
                "/30 successful"
            ),
        ),
        handoff_qa_row(
            "Model family is uniformly LightGBM",
            lgbm_final_model_registry_df["model_family"].eq("LightGBM").all(),
            "LightGBM",
        ),
        handoff_qa_row(
            "Frozen capacity profile is preserved",
            lgbm_final_model_registry_df["capacity_profile"]
            .eq(SELECTED_LGBM_CAPACITY_PROFILE)
            .all(),
            str(SELECTED_LGBM_CAPACITY_PROFILE),
        ),
        handoff_qa_row(
            "Raw target representation is preserved",
            lgbm_final_model_registry_df["target_representation"].eq("raw").all(),
            "raw",
        ),
        handoff_qa_row(
            "Validation-start-only fitting is preserved",
            lgbm_final_model_registry_df["refit_cadence"]
            .eq(LGBM_PRODUCTION_REFIT_CADENCE)
            .all(),
            str(LGBM_PRODUCTION_REFIT_CADENCE),
        ),
        handoff_qa_row(
            "Benchmark comparison exists for every job",
            (
                lgbm_final_model_registry_df["benchmark_id"].notna().all()
                and lgbm_final_model_registry_df["benchmark_mae"].notna().all()
                and lgbm_final_model_registry_df[
                    "skill_vs_benchmark_pct"
                ].notna().all()
            ),
            "30/30 jobs carry benchmark metrics",
        ),
        handoff_qa_row(
            "Support-rule evidence covers every job",
            support_job_keys == expected_final_job_keys,
            f"{len(support_job_keys)}/30 jobs",
        ),
        handoff_qa_row(
            "Zero flooring never worsens MAE",
            (
                lgbm_final_support_df["constrained_mae"]
                <= lgbm_final_support_df["raw_mae"] + 1e-12
            ).all(),
            "30/30 jobs",
        ),
        handoff_qa_row(
            "Explainability covers every job",
            importance_job_keys == expected_final_job_keys,
            f"{len(importance_job_keys)}/30 jobs",
        ),
        handoff_qa_row(
            "Explainability predictor rosters are complete",
            predictor_rosters_complete,
            "All 30 jobs match their frozen predictor roster",
        ),
        handoff_qa_row(
            "Output contract covers all five targets",
            (
                len(lgbm_forecast_output_contract_df) == 5
                and set(lgbm_forecast_output_contract_df["metric"])
                == set(FORECAST_TARGETS)
            ),
            f"{len(lgbm_forecast_output_contract_df)}/5 targets",
        ),
        handoff_qa_row(
            "Raw forecasts remain preserved",
            lgbm_forecast_output_contract_df[
                "raw_prediction_preserved"
            ].all(),
            "Raw and final forecast meanings remain distinct",
        ),
    ]
)


# ---------------------------------------------------------------------
# Extend final QA to the two downstream handoff artifacts
# ---------------------------------------------------------------------

lgbm_predictor_registry_df = pd.read_parquet(
    LGBM_FINAL_PREDICTOR_REGISTRY_PATH
)

prediction_parquet = pq.ParquetFile(
    LGBM_FINAL_VALIDATION_PREDICTIONS_PATH
)

missing_prediction_columns = sorted(
    set(PREDICTION_HANDOFF_COLUMNS)
    - set(prediction_parquet.schema_arrow.names)
)

if missing_prediction_columns:
    raise RuntimeError(
        "Final validation-prediction handoff is missing columns: "
        f"{missing_prediction_columns}"
    )


# Scan the large prediction handoff in bounded batches. This verifies its
# contract without loading the entire row-level file into memory.
prediction_job_keys = set()
prediction_rows_scanned = 0
prediction_contract_exact = True
prediction_rule_exact = True
prediction_nonnegative = True
prediction_holdout_excluded = True

# Validate holdout exclusion against the actual packaged forecast dates rather
# than relying only on descriptive metadata written by this notebook.
holdout_start_by_track = (
    ordinary_track_contract_df
    .set_index("modeling_track")["final_holdout_target_start_date"]
    .to_dict()
)

scan_columns = [
    "forecast_role",
    "stage",
    "model_family",
    "modeling_track",
    "metric",
    "horizon",
    "target_date",
    "prediction_raw",
    "prediction_final",
    "validation_contract_id",
    "raw_prediction_preserved",
]

for batch in prediction_parquet.iter_batches(
    columns=scan_columns,
    batch_size=250_000,
):
    batch_df = batch.to_pandas()
    prediction_rows_scanned += len(batch_df)

    batch_df["target_date"] = pd.to_datetime(
        batch_df["target_date"]
    ).dt.normalize()

    for track, track_df in batch_df.groupby(
        "modeling_track",
        sort=False,
    ):
        if track not in holdout_start_by_track:
            prediction_holdout_excluded = False
            continue

        holdout_start = pd.Timestamp(
            holdout_start_by_track[track]
        ).normalize()

        prediction_holdout_excluded = (
            prediction_holdout_excluded
            and track_df["target_date"].lt(holdout_start).all()
        )

    prediction_job_keys.update(
        batch_df[FINAL_JOB_KEYS]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    prediction_contract_exact = (
        prediction_contract_exact
        and batch_df["forecast_role"].eq("ordinary_validation").all()
        and batch_df["stage"].eq("rolling_validation").all()
        and batch_df["model_family"].eq("LightGBM").all()
        and batch_df["validation_contract_id"]
        .eq(LGBM_FULL_VALIDATION_CONTRACT_ID)
        .all()
        and batch_df["raw_prediction_preserved"].all()
    )

    raw_prediction = batch_df["prediction_raw"].to_numpy(dtype=float)
    final_prediction = batch_df["prediction_final"].to_numpy(dtype=float)

    prediction_rule_exact = (
        prediction_rule_exact
        and np.allclose(
            final_prediction,
            np.maximum(raw_prediction, 0.0),
            rtol=0.0,
            atol=0.0,
            equal_nan=True,
        )
    )

    finite_final = final_prediction[np.isfinite(final_prediction)]

    prediction_nonnegative = (
        prediction_nonnegative
        and bool((finite_final >= 0.0).all())
    )


# The final file is assembled once from each unique validation checkpoint.
# Matching the summed source row count therefore protects against either
# missing or duplicated job material.
expected_prediction_rows = 0
source_prediction_keys_unique = True

for job in lgbm_full_validation_jobs_df.itertuples(index=False):
    source_path = lgbm_full_validation_predictions_path(
        job.modeling_track,
        job.metric,
        int(job.horizon),
    )

    source_parquet = pq.ParquetFile(source_path)

    expected_prediction_rows += int(
        source_parquet.metadata.num_rows
    )

    source_keys_df = pd.read_parquet(
        source_path,
        columns=[
            "forecast_sequence_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
        ],
    )

    if source_keys_df.duplicated().any():
        source_prediction_keys_unique = False
        break


# Reconstruct the frozen target-level predictor rosters from the persisted
# registry rather than trusting only the in-memory objects that created it.
predictor_registry_exact = (
    len(lgbm_predictor_registry_df) == 425
    and not lgbm_predictor_registry_df.duplicated(
        ["metric", "predictor_name"]
    ).any()
)

if predictor_registry_exact:
    for metric in FORECAST_TARGETS:
        metric_registry_df = (
            lgbm_predictor_registry_df.loc[
                lgbm_predictor_registry_df["metric"].eq(metric)
            ]
            .sort_values("predictor_order")
        )

        if metric_registry_df["predictor_name"].tolist() != list(
            predictor_names_by_target[metric]
        ):
            predictor_registry_exact = False
            break

        observed_categorical = set(
            metric_registry_df.loc[
                metric_registry_df["is_categorical"],
                "predictor_name",
            ]
        )

        if observed_categorical != set(
            categorical_predictors_by_target[metric]
        ):
            predictor_registry_exact = False
            break


downstream_handoff_qa_df = pd.DataFrame(
    [
        handoff_qa_row(
            "Validation-prediction handoff covers every job",
            prediction_job_keys == expected_final_job_keys,
            f"{len(prediction_job_keys)}/30 jobs",
        ),
        handoff_qa_row(
            "Validation-prediction rows are complete and unique",
            (
                prediction_rows_scanned == expected_prediction_rows
                and source_prediction_keys_unique
            ),
            (
                f"{prediction_rows_scanned:,} final rows · "
                f"{expected_prediction_rows:,} source rows"
            ),
        ),
        handoff_qa_row(
            "Prediction handoff contains ordinary validation only",
            prediction_contract_exact,
            (
                "rolling_validation · current LightGBM "
                "contract · raw preserved"
            ),
        ),
        handoff_qa_row(
            "Final prediction rule is exact",
            prediction_rule_exact and prediction_nonnegative,
            "prediction_final = max(0, prediction_raw)",
        ),
        handoff_qa_row(
            "Predictor registry matches every frozen target roster",
            predictor_registry_exact,
            "425 rows · 85 ordered inputs per target",
        ),
        handoff_qa_row(
            "Final holdout remains unscored",
            prediction_holdout_excluded,
            (
                "All packaged target dates precede each track's "
                "final-holdout start date"
            ),
        ),
    ]
)

lgbm_final_qa_df = pd.concat(
    [
        lgbm_final_qa_df,
        downstream_handoff_qa_df,
    ],
    ignore_index=True,
)

atomic_parquet_write(
    lgbm_final_qa_df,
    LGBM_FINAL_QA_PATH,
)

if not lgbm_final_qa_df["passed"].all():
    failed_checks = lgbm_final_qa_df.loc[
        ~lgbm_final_qa_df["passed"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Final LightGBM handoff QA failed: "
        f"{failed_checks}"
    )

display(
    lgbm_final_qa_df.assign(
        Status=np.where(
            lgbm_final_qa_df["passed"],
            "PASS",
            "FAIL",
        )
    )[["Status", "check", "detail"]]
)


# ---------------------------------------------------------------------
# Write a compact final manifest
# ---------------------------------------------------------------------

final_artifacts = [
    (
        "Final LightGBM model registry",
        LGBM_FINAL_MODEL_REGISTRY_PATH,
        "Frozen 30-job model definition and validation evidence",
    ),
    (
        "Forecast-output contract",
        LGBM_FINAL_OUTPUT_CONTRACT_PATH,
        "Five-target nonnegative forecast-output rule",
    ),
    (
        "Support-constraint summary",
        LGBM_FINAL_SUPPORT_PATH,
        "Raw versus zero-floor validation evidence",
    ),
    (
        "LightGBM feature importance",
        LGBM_FINAL_IMPORTANCE_PATH,
        "Gain and SHAP explanation evidence",
    ),
    (
        "Final LightGBM QA",
        LGBM_FINAL_QA_PATH,
        "Executable final-handoff integrity checks",
    ),
    (
        "LightGBM validation predictions",
        LGBM_FINAL_VALIDATION_PREDICTIONS_PATH,
        "Row-level ordinary-validation forecasts and error evidence",
    ),
    (
        "LightGBM predictor registry",
        LGBM_FINAL_PREDICTOR_REGISTRY_PATH,
        "Frozen ordered predictor and categorical-treatment contract",
    ),
]

manifest_rows = []

for artifact_name, artifact_path, purpose in final_artifacts:
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Expected final artifact is missing: {artifact_path}"
        )

    artifact_parquet = pq.ParquetFile(artifact_path)

    manifest_rows.append(
        {
            "artifact": artifact_name,
            "purpose": purpose,
            "path": artifact_path.as_posix(),
            "rows": int(artifact_parquet.metadata.num_rows),
            "columns": int(len(artifact_parquet.schema_arrow.names)),
            "size_mb": float(
                artifact_path.stat().st_size / (1024 ** 2)
            ),
        }
    )

lgbm_final_manifest_df = pd.DataFrame(manifest_rows)

atomic_parquet_write(
    lgbm_final_manifest_df,
    LGBM_FINAL_MANIFEST_PATH,
)

lgbm_final_manifest_display_df = lgbm_final_manifest_df.copy()
lgbm_final_manifest_display_df["size_mb"] = (
    lgbm_final_manifest_display_df["size_mb"].round(3)
)

display(lgbm_final_manifest_display_df)

,Status,check,detail
0,PASS,Exactly 30 production jobs,30/30 rows
1,PASS,Production-job keys match the frozen workload,30/30 unique jobs
2,PASS,All validation jobs succeeded,30/30 successful
3,PASS,Model family is uniformly LightGBM,LightGBM
4,PASS,Frozen capacity profile is preserved,more_expressive
5,PASS,Raw target representation is preserved,raw
6,PASS,Validation-start-only fitting is preserved,validation_start_only
7,PASS,Benchmark comparison exists for every job,30/30 jobs carry benchmark metrics
8,PASS,Support-rule evidence covers every job,30/30 jobs
9,PASS,Zero flooring never worsens MAE,30/30 jobs


,artifact,purpose,path,rows,columns,size_mb
0,Final LightGBM model registry,Frozen 30-job model definition and validation ...,pipeline_data/4.4.1.final_tables/lgbm_final_mo...,30,49,0.037
1,Forecast-output contract,Five-target nonnegative forecast-output rule,pipeline_data/4.4.1.final_tables/lgbm_forecast...,5,6,0.005
2,Support-constraint summary,Raw versus zero-floor validation evidence,pipeline_data/4.4.1.final_tables/lgbm_support_...,30,14,0.012
3,LightGBM feature importance,Gain and SHAP explanation evidence,pipeline_data/4.4.1.final_tables/lgbm_feature_...,2550,16,0.119
4,Final LightGBM QA,Executable final-handoff integrity checks,pipeline_data/4.4.1.final_tables/lgbm_final_qa...,20,3,0.003
5,LightGBM validation predictions,Row-level ordinary-validation forecasts and er...,pipeline_data/4.4.1.final_tables/lgbm_row_leve...,8846655,47,744.032
6,LightGBM predictor registry,Frozen ordered predictor and categorical-treat...,pipeline_data/4.4.1.final_tables/lgbm_predicto...,425,6,0.006


**Findings.** Every complete-experiment audit check passes.

All **30 LightGBM validation jobs** remain preserved across the two modeling tracks, together with their benchmark results, support-rule evidence, feature importance, and row-level ordinary-validation forecasts.

The 425-row predictor registry continues to reproduce all 85 ordered logical inputs for each target, and the final prediction rule remains exactly `max(0, prediction_raw)` while preserving the original model output.

This 30-job layer is now treated as the LightGBM **audit record**, not the downstream candidate roster. The winner-only package is checked separately below.

#### 4.4.1.9B.1 — Verify the winner-only downstream candidate

The selected handoff must be narrower than the audit package.

This check confirms that every candidate artifact contains **Full-History Regime-Aware only**, covers exactly five targets × three horizons, retains complete feature-importance and learning-history evidence, and points to the same shared predictor and forecast-output contracts.

A dedicated manifest then gives 4.6.x one unambiguous entry point for the LightGBM branch.

In [59]:
# ---------------------------------------------------------------------
# 4.4.1.9B.1 — Verify the winner-only candidate and write its manifest
# ---------------------------------------------------------------------

selected_track_contract_df = pd.read_parquet(
    LGBM_SELECTED_TRACK_CONTRACT_PATH
)
selected_registry_df = pd.read_parquet(
    LGBM_SELECTED_MODEL_REGISTRY_PATH
)
selected_support_df = pd.read_parquet(
    LGBM_SELECTED_SUPPORT_PATH
)
selected_importance_df = pd.read_parquet(
    LGBM_SELECTED_IMPORTANCE_PATH
)
selected_learning_history_df = pd.read_parquet(
    LGBM_SELECTED_LEARNING_HISTORY_PATH
)

expected_selected_job_keys = set(
    lgbm_full_validation_jobs_df.loc[
        lgbm_full_validation_jobs_df["modeling_track"].eq(
            LGBM_SELECTED_MODELING_TRACK
        ),
        FINAL_JOB_KEYS,
    ].itertuples(index=False, name=None)
)

selected_registry_job_keys = set(
    selected_registry_df[FINAL_JOB_KEYS]
    .itertuples(index=False, name=None)
)

selected_support_job_keys = set(
    selected_support_df[FINAL_JOB_KEYS]
    .itertuples(index=False, name=None)
)

selected_importance_job_keys = set(
    selected_importance_df[FINAL_JOB_KEYS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

selected_learning_job_keys = set(
    selected_learning_history_df[FINAL_JOB_KEYS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)


# Scan only the three job-key columns from the large selected prediction file.
# This is enough to prove that Post-CP rows cannot enter 4.6.x.
selected_prediction_file = pq.ParquetFile(
    LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH
)

selected_prediction_job_keys = set()
selected_prediction_tracks = set()

for batch in selected_prediction_file.iter_batches(
    columns=FINAL_JOB_KEYS,
    batch_size=250_000,
):
    batch_df = batch.to_pandas()

    selected_prediction_tracks.update(
        batch_df["modeling_track"].unique().tolist()
    )
    selected_prediction_job_keys.update(
        batch_df[FINAL_JOB_KEYS]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

selected_prediction_rows = int(
    selected_prediction_file.metadata.num_rows
)

del selected_prediction_file


def selected_qa_row(check, passed, detail):
    """Create one compact QA result for the winner-only handoff."""
    return {
        "check": check,
        "passed": bool(passed),
        "detail": str(detail),
    }


expected_rounds = int(
    selected_lgbm_params["n_estimators"]
)

lgbm_selected_qa_df = pd.DataFrame([
    selected_qa_row(
        "Selected-track contract names Full-History only",
        (
            len(selected_track_contract_df) == 1
            and selected_track_contract_df["selected_modeling_track"]
            .eq(LGBM_SELECTED_MODELING_TRACK)
            .all()
        ),
        LGBM_SELECTED_MODELING_TRACK_LABEL,
    ),
    selected_qa_row(
        "Selected registry contains exactly 15 jobs",
        (
            len(selected_registry_df) == 15
            and selected_registry_job_keys == expected_selected_job_keys
        ),
        f"{len(selected_registry_job_keys)}/15 Target × Horizon jobs",
    ),
    selected_qa_row(
        "Selected support evidence matches the same 15 jobs",
        selected_support_job_keys == expected_selected_job_keys,
        f"{len(selected_support_job_keys)}/15 jobs",
    ),
    selected_qa_row(
        "Selected feature importance matches the same 15 jobs",
        (
            selected_importance_job_keys == expected_selected_job_keys
            and len(selected_importance_df) == 15 * 85
        ),
        f"{len(selected_importance_df):,} rows",
    ),
    selected_qa_row(
        "Selected learning history contains all 400 trees for all 15 jobs",
        (
            selected_learning_job_keys == expected_selected_job_keys
            and len(selected_learning_history_df) == 15 * expected_rounds
            and not selected_learning_history_df.duplicated(
                FINAL_JOB_KEYS + ["boosting_round"]
            ).any()
        ),
        f"{len(selected_learning_history_df):,} rows",
    ),
    selected_qa_row(
        "Selected row-level predictions contain Full-History only",
        (
            selected_prediction_tracks == {LGBM_SELECTED_MODELING_TRACK}
            and selected_prediction_job_keys == expected_selected_job_keys
            and selected_prediction_rows == expected_selected_prediction_rows
        ),
        f"{selected_prediction_rows:,} rows · 15 jobs",
    ),
    selected_qa_row(
        "Shared predictor registry remains exact",
        predictor_registry_exact,
        "425 rows · 85 ordered inputs for each target",
    ),
    selected_qa_row(
        "Shared output contract remains exact",
        (
            len(lgbm_forecast_output_contract_df) == 5
            and set(lgbm_forecast_output_contract_df["metric"])
            == set(FORECAST_TARGETS)
        ),
        "5/5 targets",
    ),
    selected_qa_row(
        "Final holdout remains excluded",
        prediction_holdout_excluded,
        (
            "Selected predictions are a subset of the validated "
            "ordinary-validation handoff"
        ),
    ),
])

if not lgbm_selected_qa_df["passed"].all():
    failed_checks = lgbm_selected_qa_df.loc[
        ~lgbm_selected_qa_df["passed"],
        ["check", "detail"],
    ]

    raise RuntimeError(
        "Winner-only LightGBM handoff failed QA:\n"
        + failed_checks.to_string(index=False)
    )

atomic_parquet_write(
    lgbm_selected_qa_df,
    LGBM_SELECTED_QA_PATH,
)


# ---------------------------------------------------------------------
# Build one dedicated manifest for the tree candidate used by 4.6.x
# ---------------------------------------------------------------------

def selected_manifest_row(artifact, path, role):
    """Describe one Parquet artifact in the selected LightGBM handoff."""
    parquet_file = pq.ParquetFile(path)

    return {
        "artifact": artifact,
        "path": str(path),
        "rows": int(parquet_file.metadata.num_rows),
        "columns": len(parquet_file.schema_arrow.names),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
        "role": role,
    }


selected_manifest_specs = [
    (
        "Selected track contract",
        LGBM_SELECTED_TRACK_CONTRACT_PATH,
        "Declares the sole LightGBM candidate for 4.6.x",
    ),
    (
        "Selected model registry",
        LGBM_SELECTED_MODEL_REGISTRY_PATH,
        "15 Full-History Target × Horizon definitions and validation metrics",
    ),
    (
        "Selected support summary",
        LGBM_SELECTED_SUPPORT_PATH,
        "Winner-only zero-floor validation evidence",
    ),
    (
        "Selected feature importance",
        LGBM_SELECTED_IMPORTANCE_PATH,
        "Winner-only gain and SHAP evidence",
    ),
    (
        "Selected learning history",
        LGBM_SELECTED_LEARNING_HISTORY_PATH,
        "Winner-only tree-by-tree training and validation loss",
    ),
    (
        "Selected validation predictions",
        LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH,
        "Winner-only row-level evidence for cross-family evaluation",
    ),
    (
        "Shared predictor registry",
        LGBM_FINAL_PREDICTOR_REGISTRY_PATH,
        "85 ordered LightGBM inputs for each target",
    ),
    (
        "Shared forecast-output contract",
        LGBM_FINAL_OUTPUT_CONTRACT_PATH,
        "Five-target final prediction rule",
    ),
    (
        "Selected QA",
        LGBM_SELECTED_QA_PATH,
        "Closeout checks for the winner-only candidate",
    ),
]

lgbm_selected_manifest_df = pd.DataFrame([
    selected_manifest_row(artifact, path, role)
    for artifact, path, role in selected_manifest_specs
])

atomic_parquet_write(
    lgbm_selected_manifest_df,
    LGBM_SELECTED_MANIFEST_PATH,
)

display(lgbm_selected_qa_df)
display(lgbm_selected_manifest_df)

,check,passed,detail
0,Selected-track contract names Full-History only,True,Full-History
1,Selected registry contains exactly 15 jobs,True,15/15 Target × Horizon jobs
2,Selected support evidence matches the same 15 ...,True,15/15 jobs
3,Selected feature importance matches the same 1...,True,"1,275 rows"
4,Selected learning history contains all 400 tre...,True,"6,000 rows"
5,Selected row-level predictions contain Full-Hi...,True,"4,311,531 rows · 15 jobs"
6,Shared predictor registry remains exact,True,425 rows · 85 ordered inputs for each target
7,Shared output contract remains exact,True,5/5 targets
8,Final holdout remains excluded,True,Selected predictions are a subset of the valid...


,artifact,path,rows,columns,size_mb,role
0,Selected track contract,pipeline_data/4.4.1.final_tables/lgbm_selected...,1,8,0.007,Declares the sole LightGBM candidate for 4.6.x
1,Selected model registry,pipeline_data/4.4.1.final_tables/lgbm_selected...,15,51,0.037,15 Full-History Target × Horizon definitions a...
2,Selected support summary,pipeline_data/4.4.1.final_tables/lgbm_selected...,15,14,0.011,Winner-only zero-floor validation evidence
3,Selected feature importance,pipeline_data/4.4.1.final_tables/lgbm_selected...,1275,16,0.067,Winner-only gain and SHAP evidence
4,Selected learning history,pipeline_data/4.4.1.final_tables/lgbm_selected...,6000,14,0.222,Winner-only tree-by-tree training and validati...
5,Selected validation predictions,pipeline_data/4.4.1.final_tables/lgbm_selected...,4311531,47,359.263,Winner-only row-level evidence for cross-famil...
6,Shared predictor registry,pipeline_data/4.4.1.final_tables/lgbm_predicto...,425,6,0.006,85 ordered LightGBM inputs for each target
7,Shared forecast-output contract,pipeline_data/4.4.1.final_tables/lgbm_forecast...,5,6,0.005,Five-target final prediction rule
8,Selected QA,pipeline_data/4.4.1.final_tables/lgbm_selected...,9,3,0.003,Closeout checks for the winner-only candidate


**Findings.** The selected LightGBM handoff passes its closeout checks.

The downstream candidate contains exactly **15 Full-History Regime-Aware jobs**: five forecasting targets × three horizons. No Post-CP job appears in the selected model registry, support evidence, feature importance, learning history, or row-level validation file.

The selected learning-history artifact preserves all **400 trees × 15 jobs = 6,000 rows** of training and validation diagnostics. The shared predictor registry still contains the exact 85-input contract for every target, and the shared output contract continues to apply the same final forecast rule.

The selected manifest is therefore the authoritative LightGBM entry point for 4.6.x. The larger 30-job files remain available for audit, but they are not an additional candidate roster.

#### 4.4.1.9B.2 — Read the Selected LightGBM Errors on a Common Scale

Native-unit MAE remains useful because it tells us how far a forecast misses in the units of the thing being predicted. But Taxi trips, Subway riders, and average speed live on very different numerical scales, so those raw MAEs cannot be compared directly across targets.

**Relative MAE** expresses accumulated absolute forecast error as a percentage of the amount of mobility actually observed:

$$
\text{Relative MAE (\%)} =
100 \times
\frac{\sum |\text{prediction\_final} - \text{actual}|}
{\sum |\text{actual}|}
$$

A Relative MAE of 10%, for example, means that accumulated forecast error is equal to about 10% of the observed mobility amount. Lower values are better.

This is **not MAPE**: we do not calculate a percentage error for every individual row and average those percentages. The aggregate observed-activity denominator is more stable when individual observations are small or zero.

The view below describes the **already-selected Full-History Regime-Aware LightGBM system** across its 15 Target × Horizon jobs. It does not reopen the tree-family, LightGBM-capacity, representation, refit-cadence, or modeling-track decisions. Cross-family comparisons still belong in 4.6.1, where all four finalized families will be rescored on exact common future rows.

In [60]:
# ---------------------------------------------------------------------
# 4.4.1.9B.2 — Relative MAE of the selected LightGBM system
# ---------------------------------------------------------------------
# WHY: native MAE is meaningful within one target, but trips, riders, and
# speed use incompatible numerical scales. Relative MAE divides accumulated
# absolute error by accumulated observed mobility, giving the selected 15
# LightGBM jobs one percentage-based interpretation without changing selection.
#
# The selected prediction handoff contains more than four million rows. Read
# only the six fields needed here in bounded Parquet batches and immediately
# reduce them to Target × Horizon totals rather than loading the full handoff.

relative_partial_summaries = []

selected_prediction_file = pq.ParquetFile(
    LGBM_SELECTED_VALIDATION_PREDICTIONS_PATH
)

relative_columns = [
    "metric",
    "horizon",
    "modeling_track",
    "actual",
    "prediction_final",
    "model_prediction_available",
]

missing_relative_columns = (
    set(relative_columns)
    - set(selected_prediction_file.schema_arrow.names)
)

if missing_relative_columns:
    raise KeyError(
        "Selected LightGBM predictions are missing fields required for "
        f"Relative MAE: {sorted(missing_relative_columns)}"
    )


# ---------------------------------------------------------------------
# 1) Stream and reduce the winner-only row-level handoff
# ---------------------------------------------------------------------
# Availability is kept explicit rather than silently dropping missing values.
# Relative MAE is calculated only where both the observed value and the final
# model forecast are finite.

for batch in selected_prediction_file.iter_batches(
    columns=relative_columns,
    batch_size=250_000,
):
    batch_df = batch.to_pandas()

    actual = pd.to_numeric(
        batch_df["actual"],
        errors="coerce",
    ).to_numpy(dtype=float)

    prediction = pd.to_numeric(
        batch_df["prediction_final"],
        errors="coerce",
    ).to_numpy(dtype=float)

    declared_available = (
        batch_df["model_prediction_available"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    finite_actual = np.isfinite(actual)
    finite_prediction = np.isfinite(prediction)

    # The persisted availability flag should describe whether a final model
    # prediction physically exists. If it does not, stop rather than quietly
    # redefining the selected handoff's support.
    if not np.array_equal(
        declared_available,
        finite_prediction,
    ):
        raise RuntimeError(
            "model_prediction_available does not match finite "
            "prediction_final support in the selected LightGBM handoff."
        )

    batch_df["relative_mae_scored"] = (
        finite_actual & finite_prediction
    )
    batch_df["absolute_actual"] = np.where(
        batch_df["relative_mae_scored"],
        np.abs(actual),
        0.0,
    )
    batch_df["absolute_error"] = np.where(
        batch_df["relative_mae_scored"],
        np.abs(prediction - actual),
        0.0,
    )

    relative_partial_summaries.append(
        batch_df.groupby(
            ["metric", "horizon", "modeling_track"],
            observed=True,
            as_index=False,
        ).agg(
            validation_rows=("prediction_final", "size"),
            scored_rows=("relative_mae_scored", "sum"),
            observed_abs_sum=("absolute_actual", "sum"),
            absolute_error_sum=("absolute_error", "sum"),
        )
    )

del selected_prediction_file


# ---------------------------------------------------------------------
# 2) Combine batch summaries into the 15 selected forecasting jobs
# ---------------------------------------------------------------------

lgbm_relative_mae_df = (
    pd.concat(
        relative_partial_summaries,
        ignore_index=True,
    )
    .groupby(
        ["metric", "horizon", "modeling_track"],
        observed=True,
        as_index=False,
    )
    .agg(
        validation_rows=("validation_rows", "sum"),
        scored_rows=("scored_rows", "sum"),
        observed_abs_sum=("observed_abs_sum", "sum"),
        absolute_error_sum=("absolute_error_sum", "sum"),
    )
)

if len(lgbm_relative_mae_df) != 15:
    raise RuntimeError(
        "Expected exactly 15 selected LightGBM Target × Horizon jobs; "
        f"found {len(lgbm_relative_mae_df)}."
    )

if set(lgbm_relative_mae_df["modeling_track"]) != {
    LGBM_SELECTED_MODELING_TRACK
}:
    raise RuntimeError(
        "Relative-MAE evidence contains a modeling track outside the "
        "selected Full-History LightGBM handoff."
    )

if (lgbm_relative_mae_df["scored_rows"] <= 0).any():
    raise RuntimeError(
        "At least one selected LightGBM job contains no scoreable forecasts."
    )

if (lgbm_relative_mae_df["observed_abs_sum"] <= 0).any():
    raise RuntimeError(
        "At least one selected LightGBM job has no positive observed-activity "
        "denominator, so Relative MAE cannot be interpreted safely."
    )

lgbm_relative_mae_df["coverage_pct"] = (
    100.0
    * lgbm_relative_mae_df["scored_rows"]
    / lgbm_relative_mae_df["validation_rows"]
)

lgbm_relative_mae_df["native_mae"] = (
    lgbm_relative_mae_df["absolute_error_sum"]
    / lgbm_relative_mae_df["scored_rows"]
)

lgbm_relative_mae_df["relative_mae_pct"] = (
    100.0
    * lgbm_relative_mae_df["absolute_error_sum"]
    / lgbm_relative_mae_df["observed_abs_sum"]
)


# ---------------------------------------------------------------------
# 3) Reader-facing table
# ---------------------------------------------------------------------

target_order_lookup = {
    metric: index
    for index, metric in enumerate(FORECAST_TARGETS)
}

lgbm_relative_display_df = (
    lgbm_relative_mae_df
    .assign(
        Target=lambda frame:
            frame["metric"].map(TARGET_SHORT_LABELS),
        _target_order=lambda frame:
            frame["metric"].map(target_order_lookup),
    )
    .sort_values(["_target_order", "horizon"])
    [
        [
            "Target",
            "horizon",
            "validation_rows",
            "scored_rows",
            "coverage_pct",
            "native_mae",
            "relative_mae_pct",
        ]
    ]
    .rename(
        columns={
            "horizon": "Horizon",
            "validation_rows": "Validation rows",
            "scored_rows": "Scored rows",
            "coverage_pct": "Coverage (%)",
            "native_mae": "Native MAE",
            "relative_mae_pct": "Relative MAE (%)",
        }
    )
)

display(
    lgbm_relative_display_df.round(
        {
            "Coverage (%)": 1,
            "Native MAE": 3,
            "Relative MAE (%)": 1,
        }
    )
)


# ---------------------------------------------------------------------
# 4) Reader-facing 5 × 3 heatmap
# ---------------------------------------------------------------------
# All 15 cells share one sequential scale: darker cells indicate larger
# forecast error relative to observed mobility. Native MAE and scored coverage
# remain available in hover so the percentage view does not hide either the
# original units or forecast availability.

relative_heatmap_df = lgbm_relative_mae_df.copy()
relative_heatmap_df["Target"] = (
    relative_heatmap_df["metric"].map(TARGET_SHORT_LABELS)
)

target_display_order = [
    TARGET_SHORT_LABELS[metric]
    for metric in FORECAST_TARGETS
]

relative_matrix_df = (
    relative_heatmap_df
    .pivot(
        index="Target",
        columns="horizon",
        values="relative_mae_pct",
    )
    .reindex(
        index=target_display_order,
        columns=FORECAST_HORIZONS,
    )
)

native_mae_matrix_df = (
    relative_heatmap_df
    .pivot(
        index="Target",
        columns="horizon",
        values="native_mae",
    )
    .reindex(
        index=target_display_order,
        columns=FORECAST_HORIZONS,
    )
)

coverage_matrix_df = (
    relative_heatmap_df
    .pivot(
        index="Target",
        columns="horizon",
        values="coverage_pct",
    )
    .reindex(
        index=target_display_order,
        columns=FORECAST_HORIZONS,
    )
)

if (
    relative_matrix_df.isna().any().any()
    or native_mae_matrix_df.isna().any().any()
    or coverage_matrix_df.isna().any().any()
):
    raise RuntimeError(
        "The selected LightGBM Relative-MAE surface is incomplete."
    )

if RUN_VISUALS:
    fig = go.Figure(
        data=go.Heatmap(
            z=relative_matrix_df.to_numpy(),
            x=[
                f"h={horizon}"
                for horizon in FORECAST_HORIZONS
            ],
            y=relative_matrix_df.index,
            zmin=0,
            zmax=float(relative_matrix_df.max().max()),
            colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
            text=np.vectorize(
                lambda value: f"{value:.1f}%"
            )(relative_matrix_df.to_numpy()),
            texttemplate="%{text}",
            customdata=np.dstack([
                native_mae_matrix_df.to_numpy(),
                coverage_matrix_df.to_numpy(),
            ]),
            colorbar=dict(
                title="Relative<br>MAE (%)"
            ),
            hovertemplate=(
                "<b>%{y} · %{x}</b><br>"
                "Relative MAE: %{z:.3f}%<br>"
                "Native MAE: %{customdata[0]:.3f}<br>"
                "Scored coverage: %{customdata[1]:.1f}%"
                "<extra></extra>"
            ),
        )
    )

    fig = apply_branding(fig)

    fig.update_layout(
        title="Relative forecast error of the selected LightGBM system",
        width=780,
        height=500,
        margin=dict(
            l=120,
            r=100,
            t=80,
            b=65,
        ),
    )

    fig.update_xaxes(title=None)
    fig.update_yaxes(
        title=None,
        autorange="reversed",
    )

    fig.show()

,Target,Horizon,Validation rows,Scored rows,Coverage (%),Native MAE,Relative MAE (%)
12,Taxi trips,1,353847,353847,100.0,14.077,13.5
13,Taxi trips,2,353847,353847,100.0,15.078,14.5
14,Taxi trips,5,353847,353847,100.0,14.151,13.6
9,Taxi speed,1,168305,168305,100.0,1.772,13.1
10,Taxi speed,2,168305,168305,100.0,1.793,13.2
11,Taxi speed,5,168305,168305,100.0,1.745,12.9
3,FHVHV trips,1,353847,353847,100.0,47.632,9.2
4,FHVHV trips,2,353847,353847,100.0,52.439,10.1
5,FHVHV trips,5,353847,353847,100.0,49.978,9.7
0,FHVHV speed,1,352319,352319,100.0,0.938,5.6


**Findings.** The selected LightGBM system provides **complete validation coverage across all 15 Target × Horizon jobs**, so every Relative MAE value reflects the full selected validation surface rather than a subset of successfully predicted rows.

Relative forecast difficulty varies substantially by target. **FHVHV speed is the strongest LightGBM result**, with Relative MAE of only **5.6%–5.8%** across all three horizons. FHVHV trips are also comparatively accurate at **9.2%–10.1%**, while Subway ranges from **8.8% to 10.1%**. The two Taxi targets are more difficult on this relative scale: Taxi speed remains near **13%**, while Taxi trips produce the highest Relative MAE values at **13.5%–14.5%**.

The horizon pattern is **not simply monotonic deterioration**. For four of the five targets, Relative MAE is highest at **h=2** and improves again somewhat at **h=5**: Taxi trips move from **13.5% → 14.5% → 13.6%**, FHVHV trips from **9.2% → 10.1% → 9.7%**, Subway from **8.8% → 10.1% → 9.3%**, and Taxi speed from **13.1% → 13.2% → 12.9%**. FHVHV speed is nearly flat at **5.6% → 5.8% → 5.8%**. This suggests that forecast difficulty depends on the particular Target × Horizon combination rather than increasing mechanically with forecast distance.

These percentages describe the already-selected Full-History LightGBM system only. They make error magnitude comparable across differently scaled mobility targets, but they should not yet be interpreted as evidence that LightGBM is better or worse than Classical, Neural, or Transformer models. That comparison belongs in 4.6.1, where every family will be rescored on the exact same future observations.

### 4.4.1.9C — Show the completed LightGBM selection path

The final figure summarizes how the tree-model branch narrowed from a broad family screen to one downstream candidate.

The sequence captures the decisions already established in the notebook:

- tree-family selection;
- LightGBM capacity;
- target representation;
- refit cadence;
- full validation across both modeling tracks;
- final modeling-track selection after the explainability sanity check.

The figure introduces no new modeling decision. It simply makes the completed selection path explicit.

In [61]:
# ---------------------------------------------------------------------
# 4.4.1.9C — Visual summary of the tree-model selection path
# ---------------------------------------------------------------------

if RUN_VISUALS:
    stage_rows = [
        {
            "x": 0,
            "title": "Tree-family<br>screen",
            "headline": "5 families",
            "detail": "Accuracy<br>+ runtime",
            "color": BRAND_COLORS["seafoam"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 1,
            "title": "Finalist<br>review",
            "headline": "Extra Trees<br>+ LightGBM",
            "detail": "Accuracy vs<br>feasibility",
            "color": BRAND_COLORS["pale_peach"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 2,
            "title": "LightGBM<br>capacity",
            "headline": "More<br>expressive",
            "detail": "18/20<br>screen wins",
            "color": BRAND_COLORS["seafoam"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 3,
            "title": "Target<br>scale",
            "headline": "Raw<br>values",
            "detail": "Raw won all<br>6 count tests",
            "color": BRAND_COLORS["seafoam"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 4,
            "title": "Retraining",
            "headline": "Validation-start<br>only",
            "detail": "Lower MAE in<br>8/10 tests",
            "color": BRAND_COLORS["seafoam"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 5,
            "title": "Full<br>validation",
            "headline": "29/30<br>benchmark wins",
            "detail": "Both tracks<br>evaluated",
            "color": BRAND_COLORS["pale_peach"],
            "text_color": BRAND_COLORS["dark_teal"],
        },
        {
            "x": 6,
            "title": "Track<br>winner",
            "headline": "Full-History",
            "detail": "15 jobs<br>advance",
            "color": BRAND_COLORS["dark_teal"],
            "text_color": "white",
        },
    ]

    selection_path_fig = go.Figure()

    for left_stage, right_stage in zip(stage_rows[:-1], stage_rows[1:]):
        selection_path_fig.add_annotation(
            x=right_stage["x"] - 0.36,
            y=0.40,
            ax=left_stage["x"] + 0.36,
            ay=0.40,
            xref="x",
            yref="y",
            axref="x",
            ayref="y",
            showarrow=True,
            arrowhead=3,
            arrowsize=1.0,
            arrowwidth=2,
            arrowcolor=BRAND_COLORS["dark_teal"],
            text="",
        )

    for stage in stage_rows:
        x = stage["x"]

        selection_path_fig.add_shape(
            type="rect",
            x0=x - 0.36,
            x1=x + 0.36,
            y0=0.26,
            y1=0.74,
            line={
                "color": BRAND_COLORS["dark_teal"],
                "width": 1.5,
            },
            fillcolor=stage["color"],
        )

        selection_path_fig.add_annotation(
            x=x,
            y=0.64,
            text=f"<b>{stage['title']}</b>",
            showarrow=False,
            align="center",
            font={
                "size": 11,
                "color": stage["text_color"],
            },
        )

        selection_path_fig.add_annotation(
            x=x,
            y=0.50,
            text=f"<b>{stage['headline']}</b>",
            showarrow=False,
            align="center",
            font={
                "size": 12,
                "color": stage["text_color"],
            },
        )

        selection_path_fig.add_annotation(
            x=x,
            y=0.32,
            text=stage["detail"],
            showarrow=False,
            align="center",
            font={
                "size": 9,
                "color": stage["text_color"],
            },
        )

    selection_path_fig.update_xaxes(
        range=[-0.50, 6.50],
        visible=False,
        fixedrange=True,
    )
    selection_path_fig.update_yaxes(
        range=[0, 1],
        visible=False,
        fixedrange=True,
    )

    selection_path_fig.update_layout(
        title=(
            "Tree Forecasting Selection — "
            "From Family Screen to Full-History Winner"
        ),
        width=960,
        height=650,
        margin=dict(l=35, r=35, t=95, b=90),
        showlegend=False,
    )

    selection_path_fig.add_annotation(
        x=0,
        y=0.08,
        xref="paper",
        yref="paper",
        xanchor="left",
        showarrow=False,
        align="left",
        text=(
            "<b>Selected tree representative:</b> LightGBM · "
            "Full-History Regime-Aware · More expressive · "
            "Raw targets · Validation-start-only"
        ),
        font={
            "size": 12,
            "color": BRAND_COLORS["dark_teal"],
        },
    )

    apply_branding(selection_path_fig)
    selection_path_fig.show()

**Findings.** The tree-model branch now closes with one explicit downstream representative.

LightGBM advances with the **More expressive** capacity profile, **raw target values**, and **validation-start-only fitting**. Full validation was completed across both ordinary modeling tracks so the history choice could be evaluated rather than assumed.

The final comparison selects **Full-History Regime-Aware**. Its stronger forecasting performance is not contradicted by the explainability evidence, so the longer historical training record is retained.

The LightGBM candidate entering 4.6.x is therefore:

**Full-History Regime-Aware · LightGBM · More expressive · raw targets · validation-start-only · 400 trees.**

Post-CP remains part of the audit record but no longer participates as a downstream tree-model candidate.

### Section Summary

Section 9 closes the LightGBM branch with a clear distinction between **evidence retained for audit** and **the model selected for downstream comparison**.

The 30-job ordinary-validation experiment remains preserved across Full-History and Post-CP, including validation metrics, row-level predictions, support-rule evidence, and explainability evidence. The full two-track learning-history diagnostic remains available in the restartable intermediate checkpoints.

The downstream candidate is narrower. **Full-History Regime-Aware** is the selected ordinary-forecasting LightGBM track, producing **15 Target × Horizon jobs** across the five mobility outcomes and three forecast horizons.

The selected package preserves the frozen More expressive LightGBM definition, raw target representation, validation-start-only fitting, all 400 tree-by-tree training and validation measurements, the 85-input predictor contract, zero-floor output rule, feature importance, and winner-only row-level validation predictions.

**4.6.x should use `lgbm_selected_manifest.parquet` as the authoritative LightGBM handoff.** The larger 30-job files remain audit evidence and should not be treated as additional tree-model candidates.

The final holdout remains untouched. Chapter 5 can reuse the selected LightGBM family, capacity, representation, predictor ordering, and forecast-output contracts when it builds its separate pre-CP counterfactual production surface; the Full-History modeling track selected here remains the ordinary-forecasting representative for 4.6.x.

## Notebook Close

Notebook **4.4.1 Train Tree-Based Panel Forecasting Models** completes the tree-based forecasting branch.

The notebook began with a broad comparison of practical tree-model families and ultimately selected **LightGBM**. Its frozen modeling configuration uses:

- **More expressive LightGBM capacity**
- **400 trees**
- **raw target values**
- **validation-start-only fitting**
- **Taxi Zone preserved as a categorical input**
- **zero-flooring only at final forecast output**

Both Full-History Regime-Aware and Post-CP Primary were carried through complete ordinary validation so the training-history choice could be evaluated rather than assumed. LightGBM beats the seasonal benchmark in **29 of 30 jobs** overall.

The head-to-head track comparison is clearer: **Full-History has higher benchmark-relative skill and lower NMAE in all 15 matched Target × Horizon comparisons**. Its median benchmark improvement is **25.290% versus 16.360%** for Post-CP, and its median NMAE is **0.101 versus 0.114**.

The explainability comparison supports that performance result rather than raising a warning. The two histories retain substantial agreement in feature structure, with a median SHAP-rank correlation of **0.799** and a median **8-of-10 Top-10 feature overlap**. Full-History changes the weighting of familiar forecasting signals rather than introducing an unexpected dependency.

> ✅ **Selected tree-model representative: Full-History Regime-Aware LightGBM.**

The final handoff therefore separates the complete two-track audit evidence from the **15-job Full-History candidate** that advances to 4.6.x. Downstream model-family comparison should begin from `lgbm_selected_manifest.parquet`, not from the 30-job audit registry.

No **cross-family** forecasting winner is declared here; that comparison still belongs downstream. But there is now exactly **one tree-model representative**, not another permutation of LightGBM tracks.

The final holdout remains untouched. Chapter 5 can reuse the selected LightGBM modeling configuration and predictor/output contracts when constructing its separate pre-CP counterfactual workflow.

### 4.4.1 Final Inventory of Files

#### Selected LightGBM handoff for 4.6.x

| File | Description |
|---|---|
| `lgbm_selected_manifest.parquet` | **Authoritative 4.6.x entry point.** Inventory of the selected Full-History LightGBM candidate and its supporting contracts. |
| `lgbm_selected_track_contract.parquet` | One-row contract declaring Full-History Regime-Aware as the sole LightGBM candidate for downstream model-family comparison. |
| `lgbm_selected_model_registry.parquet` | 15 selected Full-History Target × Horizon jobs with the frozen LightGBM definition and validation metrics. |
| `lgbm_selected_support_constraint_summary.parquet` | Zero-floor validation evidence for the selected 15 jobs. |
| `lgbm_selected_feature_importance.parquet` | 1,275 gain and SHAP feature-importance rows for the selected 15 models. |
| `lgbm_selected_learning_history.parquet` | 6,000 tree-by-tree training and validation observations: 15 selected jobs × 400 trees. |
| `lgbm_selected_row_level_validation_predictions.parquet` | 4,311,531 Full-History ordinary-validation prediction rows used for downstream cross-family evaluation. |
| `lgbm_selected_qa.parquet` | Nine closeout checks confirming that the selected handoff contains Full-History only, preserves all 15 jobs, and excludes the final holdout. |

The selected package shares the common `lgbm_predictor_registry.parquet` and `lgbm_forecast_output_contract.parquet` contracts listed below.

#### Complete LightGBM audit evidence

| File | Description |
|---|---|
| `lgbm_final_model_registry.parquet` | 30-row registry preserving both Full-History and Post-CP validation jobs. |
| `lgbm_forecast_output_contract.parquet` | Five-row contract defining the shared zero lower bound, raw-prediction preservation, and final `max(0, raw_prediction)` rule. |
| `lgbm_support_constraint_summary.parquet` | Raw versus zero-floored validation evidence across all 30 jobs. |
| `lgbm_feature_importance.parquet` | 2,550 gain and SHAP feature-importance rows across the 30 validation models. |
| `lgbm_row_level_validation_predictions.parquet` | 8,846,655 ordinary-validation rows containing observed outcomes, raw and final LightGBM forecasts, benchmark forecasts, error fields, forecast identifiers, geographic context, temporal buckets, and policy regime. |
| `lgbm_predictor_registry.parquet` | 425-row ordered predictor contract: 85 LightGBM inputs for each of the five targets, including categorical-treatment flags. |
| `lgbm_final_qa.parquet` | Twenty audit-layer integrity checks covering the complete model roster, configuration, benchmark evidence, support rule, explainability, prediction handoff, predictor contract, and untouched holdout. |
| `lgbm_final_manifest.parquet` | Inventory of the reusable 30-job audit artifacts. |

The complete **12,000-row, two-track learning-history diagnostic** remains in the restartable `4.4.1.intermediate_tables/` checkpoint tree. Its selected 6,000-row Full-History subset is promoted into the final handoff above.

For downstream model-family comparison, use **`lgbm_selected_manifest.parquet`**. The 30-job files remain supporting audit evidence rather than additional LightGBM candidates.

---

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>